# 02 · ETL — Integración del dataset maestro

## Integración temporal y espacial de fuentes urbanas para el análisis de la calidad del aire en Barcelona

Este notebook desarrolla la segunda etapa del proceso ETL del proyecto, centrada en la **integración progresiva de las fuentes previamente depuradas y validadas** en `01_ETL_Preprocesamiento_Fuentes.ipynb`.

La contaminación atmosférica constituye el núcleo del dataset maestro. Sobre esta estructura se incorporan sucesivamente variables meteorológicas y diferentes indicadores de actividad urbana potencialmente relacionados con la calidad del aire:

- meteorología;
- tráfico rodado;
- contaminación acústica;
- transporte aéreo;
- transporte marítimo.

La integración combina relaciones **temporales y espaciales** en función de la naturaleza de cada fuente. Para garantizar la trazabilidad del proceso se utiliza una estrategia secuencial basada en checkpoints, verificando después de cada incorporación la conservación de la estructura y granularidad del dataset.

La unidad fundamental del maestro queda definida mediante la combinación:

`fecha + estacion_fisica + contaminant`

Los valores ausentes derivados de limitaciones reales de cobertura se conservan explícitamente. En esta fase no se realiza imputación, tratamiento de valores extremos ni transformación orientada al aprendizaje automático.

El proceso genera finalmente el **Checkpoint 06**, correspondiente al dataset maestro multifuente que servirá como punto de partida para las posteriores fases de visualización, análisis exploratorio y modelado.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 1 · ESTRATEGIA DE INTEGRACIÓN Y CONSTRUCCIÓN DEL DATASET MAESTRO

La integración se plantea como un proceso **incremental, reproducible y trazable**. En lugar de combinar simultáneamente todas las fuentes, cada bloque de información se incorpora de forma independiente sobre el último dataset validado.

Esta estrategia permite controlar después de cada integración:

- el número de registros y variables;
- la unicidad de la clave principal;
- la posible aparición de duplicados;
- la conservación de la granularidad original;
- la correspondencia temporal y/o espacial entre fuentes;
- la cobertura de las nuevas variables;
- los valores ausentes generados o conservados;
- la integridad del archivo exportado.

La secuencia de construcción del dataset maestro es:

`Contaminación atmosférica`  
→ `+ Meteorología`  
→ `+ Tráfico rodado`  
→ `+ Contaminación acústica`  
→ `+ Transporte aéreo`  
→ `+ Transporte marítimo`

Cada integración validada genera un checkpoint independiente:

- **Checkpoint 01** — contaminación atmosférica;
- **Checkpoint 02** — contaminación + meteorología;
- **Checkpoint 03** — contaminación + meteorología + tráfico rodado;
- **Checkpoint 04** — contaminación + meteorología + tráfico rodado + ruido;
- **Checkpoint 05** — contaminación + meteorología + tráfico rodado + ruido + transporte aéreo;
- **Checkpoint 06** — contaminación + meteorología + tráfico rodado + ruido + transporte aéreo + transporte marítimo.

El Checkpoint 06 constituye el **dataset maestro definitivo de la fase de integración**. Una auditoría global final permite verificar su integridad antes de utilizarlo en las siguientes etapas del proyecto.

# 2 · CONTAMINACIÓN ATMOSFÉRICA — CONSTRUCCIÓN DEL DATASET BASE

La contaminación atmosférica constituye el **bloque de referencia del dataset maestro** y determina la granularidad sobre la que se integrarán posteriormente las restantes fuentes.

Se parte del dataset diario previamente depurado en la fase ETL anterior, correspondiente a los cuatro contaminantes seleccionados para el estudio:

- `NO2`;
- `O3`;
- `PM10`;
- `PM2.5`.

Antes de generar el primer checkpoint se prepara la referencia espacial de las estaciones, se audita el dataset temporal y se armonizan los diferentes identificadores históricos utilizados por la red de monitorización.

Esta distinción permite conservar simultáneamente la trazabilidad temporal mediante `estacion_fisica` y una referencia espacial homogénea mediante `estacion_geo`.

El bloque finaliza con la generación del **Checkpoint 01**, que constituye la base sobre la que se realizarán todas las integraciones posteriores.

> **Objetivo del bloque:** construir y validar la estructura base de contaminación atmosférica, garantizando su coherencia temporal y espacial antes de incorporar variables explicativas externas.

### 2.1. Preparación espacial de las estaciones de contaminación atmosférica

Como primer paso de la fase de preintegración se construye una tabla auxiliar con la localización de las estaciones de calidad del aire utilizadas en el estudio.

Para cada estación se incorporan:

- nombre de la estación física;
- latitud y longitud en coordenadas geográficas (`EPSG:4326`);
- tipología de estación;
- coordenadas proyectadas `X_ETRS89` e `Y_ETRS89`.

Las coordenadas se transforman al sistema **ETRS89 / UTM zona 31N (`EPSG:25831`)**, que se utilizará como sistema de referencia común para las posteriores operaciones espaciales.

Esta transformación permitirá calcular distancias y establecer relaciones espaciales entre las estaciones de contaminación y otras fuentes georreferenciadas del proyecto, especialmente meteorología, contaminación acústica y tráfico viario.

El archivo generado se almacena dentro de la estructura de `PREINTEGRACION`, manteniéndolo separado de los datasets originales y de los archivos limpios producidos durante el ETL.

> **Objetivo de la etapa:** disponer de un catálogo espacial homogéneo de las estaciones de contaminación atmosférica que pueda utilizarse como referencia en la construcción del dataset maestro espacio-temporal.

In [ ]:
# ============================================================
# PREINTEGRACIÓN — CONTAMINACIÓN ATMOSFÉRICA
# CREACIÓN Y EXPORTACIÓN DE ESTACIONES
# ============================================================

import pandas as pd
import geopandas as gpd
from pathlib import Path


# ============================================================
# 1. DATOS DE LAS ESTACIONES
# ============================================================

datos_estaciones = [
    {
        "estacion_fisica": "Eixample",
        "lat": 41.3853,
        "lon": 2.1538,
        "tipo": "Tráfico"
    },
    {
        "estacion_fisica": "Gràcia",
        "lat": 41.3987,
        "lon": 2.1534,
        "tipo": "Tráfico"
    },
    {
        "estacion_fisica": "Ciutadella",
        "lat": 41.3864,
        "lon": 2.1874,
        "tipo": "Fondo"
    },
    {
        "estacion_fisica": "Palau Reial",
        "lat": 41.3875,
        "lon": 2.1153,
        "tipo": "Fondo"
    },
    {
        "estacion_fisica": "Vall d'Hebron",
        "lat": 41.4261,
        "lon": 2.1478,
        "tipo": "Fondo"
    },
    {
        "estacion_fisica": "Poblenou",
        "lat": 41.4039,
        "lon": 2.2045,
        "tipo": "Fondo"
    },
    {
        "estacion_fisica": "Sants",
        "lat": 41.3791,
        "lon": 2.1328,
        "tipo": "Fondo"
    },
    {
        "estacion_fisica": "Observatori Fabra",
        "lat": 41.4184,
        "lon": 2.1239,
        "tipo": "Fondo regional"
    }
]

df_estaciones = pd.DataFrame(datos_estaciones)


# ============================================================
# 2. CREAR GEODATAFRAME EN WGS84
# ============================================================

gdf_estaciones = gpd.GeoDataFrame(
    df_estaciones,
    geometry=gpd.points_from_xy(
        df_estaciones["lon"],
        df_estaciones["lat"]
    ),
    crs="EPSG:4326"
)


# ============================================================
# 3. TRANSFORMAR A ETRS89 / UTM ZONA 31N
# ============================================================

gdf_estaciones = gdf_estaciones.to_crs(
    "EPSG:25831"
)


# ============================================================
# 4. EXTRAER COORDENADAS PROYECTADAS
# ============================================================

gdf_estaciones["X_ETRS89"] = gdf_estaciones.geometry.x
gdf_estaciones["Y_ETRS89"] = gdf_estaciones.geometry.y


# ============================================================
# 5. CREAR DATAFRAME FINAL
# ============================================================

df_estaciones_contaminacion = (
    gdf_estaciones
    .drop(columns=["geometry"])
    .copy()
)


# ============================================================
# 6. DEFINIR RUTA DE PREINTEGRACIÓN
# ============================================================

ruta_carpeta = Path(
    "/content/drive/MyDrive/TFM/"
    "11_Machine_Learning_Temporal/"
    "PREINTEGRACION/"
    "06_Contaminacion_Atmosferica"
)

ruta_carpeta.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 7. DEFINIR ARCHIVO DE SALIDA
# ============================================================

ruta_csv = (
    ruta_carpeta /
    "estaciones_contaminacion_atmosferica.csv"
)


# ============================================================
# 8. EXPORTAR CSV
# ============================================================

df_estaciones_contaminacion.to_csv(
    ruta_csv,
    index=False,
    encoding="utf-8"
)


# ============================================================
# 9. VERIFICACIÓN
# ============================================================

if not ruta_csv.exists():
    raise FileNotFoundError(
        "No se ha creado correctamente "
        "estaciones_contaminacion_atmosferica.csv"
    )


# ============================================================
# 10. RESULTADO FINAL
# ============================================================

print("=" * 80)
print("ESTACIONES DE CONTAMINACIÓN ATMOSFÉRICA")
print("=" * 80)

print(
    f"\n✓ Número de estaciones: "
    f"{len(df_estaciones_contaminacion)}"
)

print(
    "✓ CRS original: "
    "EPSG:4326 — WGS84"
)

print(
    "✓ CRS proyectado: "
    "EPSG:25831 — ETRS89 / UTM zona 31N"
)

print("\n✓ Archivo guardado en:")
print(ruta_csv)

print(
    "\n✓ Archivo existe:",
    ruta_csv.exists()
)


# ============================================================
# 11. VISUALIZACIÓN
# ============================================================

print("\nTabla de estaciones:")

display(df_estaciones_contaminacion)

ESTACIONES DE CONTAMINACIÓN ATMOSFÉRICA

✓ Número de estaciones: 8
✓ CRS original: EPSG:4326 — WGS84
✓ CRS proyectado: EPSG:25831 — ETRS89 / UTM zona 31N

✓ Archivo guardado en:
/content/drive/MyDrive/TFM/11_Machine_Learning_Temporal/PREINTEGRACION/06_Contaminacion_Atmosferica/estaciones_contaminacion_atmosferica.csv

✓ Archivo existe: True

Tabla de estaciones:


,estacion_fisica,lat,lon,tipo,X_ETRS89,Y_ETRS89
0,Eixample,41.3853,2.1538,Tráfico,429249.000364,4.581876e+06
1,Gràcia,41.3987,2.1534,Tráfico,429230.091303,4.583364e+06
2,Ciutadella,41.3864,2.1874,Fondo,432059.474731,4.581971e+06
3,Palau Reial,41.3875,2.1153,Fondo,426032.466989,4.582152e+06
4,Vall d'Hebron,41.4261,2.1478,Fondo,428791.895985,4.586410e+06
5,Poblenou,41.4039,2.2045,Fondo,433507.035285,4.583901e+06
6,Sants,41.3791,2.1328,Fondo,427486.280382,4.581205e+06
7,Observatori Fabra,41.4184,2.1239,Fondo regional,426786.192307,4.585575e+06


#### Resultado — Estaciones de contaminación atmosférica

Se genera correctamente el archivo `estaciones_contaminacion_atmosferica.csv`, que contiene la información espacial correspondiente a las **8 estaciones físicas** consideradas en el estudio.

Para cada estación se conservan las coordenadas originales en `EPSG:4326` y se incorporan las coordenadas proyectadas en **ETRS89 / UTM zona 31N (`EPSG:25831`)**, permitiendo realizar posteriormente cálculos métricos de distancia y proximidad.

La tabla final incluye las variables:

- `estacion_fisica`;
- `lat`;
- `lon`;
- `tipo`;
- `X_ETRS89`;
- `Y_ETRS89`.

El archivo queda almacenado en:

`11_Machine_Learning_Temporal/PREINTEGRACION/06_Contaminacion_Atmosferica/`

y constituye la referencia espacial del bloque de contaminación atmosférica para las siguientes etapas de preintegración.

> **Resultado:** la red de estaciones de calidad del aire queda georreferenciada y preparada para su asociación con las observaciones temporales de contaminación y con las restantes fuentes espaciales del proyecto.

> **Nota metodológica sobre la red de estaciones:**  
> Las estaciones consideradas no presentan una exposición ambiental homogénea. La red incluye estaciones de **tráfico** (Eixample y Gràcia), estaciones de **fondo urbano** y una estación de **fondo regional** (Observatori Fabra). Esta diferenciación se conserva mediante la variable `tipo`, ya que puede explicar parte de la variabilidad espacial observada en las concentraciones de contaminantes y podrá utilizarse posteriormente como variable explicativa o de estratificación en el análisis.
>
> Asimismo, la tabla representa **estaciones físicas**, independientemente de los posibles cambios históricos en los códigos administrativos utilizados por las fuentes originales. La correspondencia entre códigos históricos y estación física se resuelve previamente durante el proceso de limpieza, evitando duplicar artificialmente localizaciones en la fase de integración.

### 2.2. Carga y comprobación del dataset limpio de contaminación atmosférica

Una vez definida la referencia espacial de las estaciones, se carga el dataset diario de contaminación atmosférica previamente sometido al proceso de limpieza y control de calidad.

En esta etapa no se realizan todavía transformaciones ni integraciones espaciales. El objetivo es verificar la estructura del dataset que servirá como base temporal del modelo, comprobando:

- dimensiones del conjunto de datos;
- variables disponibles y tipos de datos;
- periodo temporal cubierto;
- contaminantes incluidos;
- estaciones físicas presentes;
- valores nulos;
- duplicados;
- distribución del número de observaciones por estación y contaminante.

Esta comprobación permite validar que el dataset limpio mantiene la estructura esperada antes de incorporar la información espacial de las estaciones.

> **Objetivo de la etapa:** validar el dataset temporal de contaminación que actuará como tabla base para la posterior preintegración espacio-temporal.

In [ ]:
# ============================================================
# PREINTEGRACIÓN — CONTAMINACIÓN ATMOSFÉRICA
# CARGA Y AUDITORÍA DEL DATASET LIMPIO
# ============================================================

import pandas as pd
from pathlib import Path


# ============================================================
# 1. RUTA DE TRABAJO
# ============================================================

ruta_carpeta = Path(
    "/content/drive/MyDrive/TFM/"
    "11_Machine_Learning_Temporal/"
    "PREINTEGRACION/"
    "06_Contaminacion_Atmosferica"
)

ruta_contaminacion = (
    ruta_carpeta /
    "df_contaminacion_atm_2018_2024_Limpio.csv"
)


# ============================================================
# 2. COMPROBAR EXISTENCIA
# ============================================================

if not ruta_contaminacion.exists():
    raise FileNotFoundError(
        f"No se encuentra el archivo:\n{ruta_contaminacion}"
    )


# ============================================================
# 3. CARGAR DATASET
# ============================================================

df_contaminacion = pd.read_csv(
    ruta_contaminacion
)


# ============================================================
# 4. INFORMACIÓN GENERAL
# ============================================================

print("=" * 80)
print("DATASET LIMPIO — CONTAMINACIÓN ATMOSFÉRICA")
print("=" * 80)

print("\nArchivo cargado:")
print(ruta_contaminacion)

print(
    f"\n✓ Filas: {df_contaminacion.shape[0]:,}"
)

print(
    f"✓ Columnas: {df_contaminacion.shape[1]}"
)

print("\nColumnas disponibles:")

for columna in df_contaminacion.columns:
    print(f"  • {columna}")


# ============================================================
# 5. TIPOS DE DATOS
# ============================================================

print("\n" + "=" * 80)
print("TIPOS DE DATOS")
print("=" * 80)

print(df_contaminacion.dtypes)


# ============================================================
# 6. CONVERTIR FECHA PARA LA AUDITORÍA
# ============================================================

if "fecha" in df_contaminacion.columns:

    df_contaminacion["fecha"] = pd.to_datetime(
        df_contaminacion["fecha"],
        errors="coerce"
    )

    print("\n" + "=" * 80)
    print("COBERTURA TEMPORAL")
    print("=" * 80)

    print(
        "\nFecha inicial:",
        df_contaminacion["fecha"].min()
    )

    print(
        "Fecha final:",
        df_contaminacion["fecha"].max()
    )

    print(
        "Número de fechas diferentes:",
        df_contaminacion["fecha"].nunique()
    )


# ============================================================
# 7. CONTAMINANTES
# ============================================================

if "contaminant" in df_contaminacion.columns:

    print("\n" + "=" * 80)
    print("CONTAMINANTES")
    print("=" * 80)

    print(
        df_contaminacion["contaminant"]
        .value_counts(dropna=False)
    )


# ============================================================
# 8. ESTACIONES FÍSICAS
# ============================================================

if "estacion_fisica" in df_contaminacion.columns:

    print("\n" + "=" * 80)
    print("ESTACIONES FÍSICAS")
    print("=" * 80)

    print(
        f"\nNúmero de estaciones: "
        f"{df_contaminacion['estacion_fisica'].nunique()}"
    )

    print("\nEstaciones encontradas:")

    for estacion in sorted(
        df_contaminacion["estacion_fisica"]
        .dropna()
        .unique()
    ):
        print(f"  • {estacion}")


# ============================================================
# 9. VALORES NULOS
# ============================================================

print("\n" + "=" * 80)
print("VALORES NULOS")
print("=" * 80)

nulos = df_contaminacion.isna().sum()

print(nulos)


# ============================================================
# 10. DUPLICADOS COMPLETOS
# ============================================================

print("\n" + "=" * 80)
print("DUPLICADOS COMPLETOS")
print("=" * 80)

duplicados_completos = (
    df_contaminacion
    .duplicated()
    .sum()
)

print(
    f"\nDuplicados completos: "
    f"{duplicados_completos:,}"
)


# ============================================================
# 11. DUPLICADOS EN LA CLAVE TEMPORAL-ESPACIAL
# ============================================================

columnas_clave = [
    "fecha",
    "estacion_fisica",
    "contaminant"
]

if all(
    columna in df_contaminacion.columns
    for columna in columnas_clave
):

    duplicados_clave = (
        df_contaminacion
        .duplicated(
            subset=columnas_clave
        )
        .sum()
    )

    print(
        "\nDuplicados "
        "fecha + estación + contaminante: "
        f"{duplicados_clave:,}"
    )


# ============================================================
# 12. OBSERVACIONES POR ESTACIÓN Y CONTAMINANTE
# ============================================================

if all(
    columna in df_contaminacion.columns
    for columna in ["estacion_fisica", "contaminant"]
):

    print("\n" + "=" * 80)
    print("OBSERVACIONES POR ESTACIÓN Y CONTAMINANTE")
    print("=" * 80)

    tabla_estacion_contaminante = pd.crosstab(
        df_contaminacion["estacion_fisica"],
        df_contaminacion["contaminant"]
    )

    display(tabla_estacion_contaminante)


# ============================================================
# 13. PRIMERAS FILAS
# ============================================================

print("\n" + "=" * 80)
print("PRIMERAS FILAS DEL DATASET")
print("=" * 80)

display(
    df_contaminacion.head(10)
)


# ============================================================
# 14. CIERRE
# ============================================================

print("\n" + "=" * 80)
print("AUDITORÍA FINALIZADA")
print("=" * 80)

print(
    "\n✓ Dataset cargado correctamente."
)

print(
    "✓ No se ha sobrescrito ni modificado "
    "ningún archivo de origen."
)

print(
    "✓ Preparado para comprobar la correspondencia "
    "con el catálogo espacial de estaciones."
)

DATASET LIMPIO — CONTAMINACIÓN ATMOSFÉRICA

Archivo cargado:
/content/drive/MyDrive/TFM/11_Machine_Learning_Temporal/PREINTEGRACION/06_Contaminacion_Atmosferica/df_contaminacion_atm_2018_2024_Limpio.csv

✓ Filas: 43,642
✓ Columnas: 8

Columnas disponibles:
  • fecha
  • codi_estacio
  • nom_cabina
  • contaminant
  • valor
  • horas_validas
  • codi_estacio_str
  • estacion_fisica

TIPOS DE DATOS
fecha                object
codi_estacio         object
nom_cabina           object
contaminant          object
valor               float64
horas_validas         int64
codi_estacio_str     object
estacion_fisica      object
dtype: object

COBERTURA TEMPORAL

Fecha inicial: 2018-06-12 00:00:00
Fecha final: 2024-12-31 00:00:00
Número de fechas diferentes: 2249

CONTAMINANTES
contaminant
NO2      17188
O3       12766
PM10     11954
PM2.5     1734
Name: count, dtype: int64

ESTACIONES FÍSICAS

Número de estaciones: 10

Estaciones encontradas:
  • Ciutadella
  • Eixample
  • Gràcia
  • Observ Fabra

contaminant,NO2,O3,PM10,PM2.5
estacion_fisica,,,,
Ciutadella,2219,2209,0,0
Eixample,2134,2119,2104,618
Gràcia,2159,2158,1803,0
Observ Fabra,168,168,104,0
Palau Reial,2142,2138,2009,1070
Poblenou,1996,1780,1899,0
Sants_4,2028,0,1980,0
Sants_42,1990,0,0,0
Sants_hist,165,0,0,0



PRIMERAS FILAS DEL DATASET


,fecha,codi_estacio,nom_cabina,contaminant,valor,horas_validas,codi_estacio_str,estacion_fisica
0,2018-06-12,I2,Barcelona - Poblenou,NO2,21.041667,24,I2,Poblenou
1,2018-06-12,I2,Barcelona - Poblenou,PM10,24.695652,23,I2,Poblenou
2,2018-06-12,ID,Barcelona - Sants,NO2,19.708333,24,ID,Sants_hist
3,2018-06-12,IH,Barcelona - Eixample,PM10,29.458333,24,IH,Eixample
4,2018-06-12,IJ,Barcelona - Gràcia,NO2,36.105263,19,IJ,Gràcia
5,2018-06-12,IJ,Barcelona - Gràcia,O3,67.421053,19,IJ,Gràcia
6,2018-06-12,IJ,Barcelona - Gràcia,PM10,23.625000,24,IJ,Gràcia
7,2018-06-12,IL,Barcelona - Ciutadella,NO2,15.521739,23,IL,Ciutadella
8,2018-06-12,IL,Barcelona - Ciutadella,O3,68.565217,23,IL,Ciutadella
9,2018-06-12,IN,Barcelona - Vall Hebron,NO2,16.708333,24,IN,Vall Hebron



AUDITORÍA FINALIZADA

✓ Dataset cargado correctamente.
✓ No se ha sobrescrito ni modificado ningún archivo de origen.
✓ Preparado para comprobar la correspondencia con el catálogo espacial de estaciones.


#### Resultado de la auditoría del dataset de contaminación

La auditoría confirma que el dataset limpio de contaminación atmosférica se encuentra preparado para iniciar la fase de preintegración. El conjunto contiene **43.642 observaciones**, correspondientes al periodo comprendido entre el **12 de junio de 2018 y el 31 de diciembre de 2024**, e incluye los contaminantes `NO2`, `O3`, `PM10` y `PM2.5`.

No se detectan valores nulos, duplicados completos ni duplicados para la combinación `fecha + estacion_fisica + contaminant`, por lo que se mantiene el dataset sin modificaciones adicionales.

##### Homogeneización de la referencia espacial

La auditoría identifica **10 denominaciones de estación** en el dataset temporal frente a las **8 localizaciones físicas** definidas en el catálogo espacial. Esta diferencia se debe principalmente a la conservación de tres identificadores históricos asociados a la estación de Sants:

- `Sants_hist`
- `Sants_4`
- `Sants_42`

Estas denominaciones se conservarán en la variable original `estacion_fisica`, evitando perder la trazabilidad histórica de las observaciones. Para la integración espacial se creará una nueva variable, `estacion_geo`, que asociará las tres series a una única localización física: `Sants`.

Asimismo, se homogeneizarán exclusivamente para el cruce espacial las siguientes diferencias nominales:

- `Vall Hebron` → `Vall d'Hebron`
- `Observ Fabra` → `Observatori Fabra`

De esta forma se mantiene separada la **identidad temporal e histórica de la estación** de su **localización física**, evitando modificar el dataset limpio de origen.

##### Cobertura por contaminante

La disponibilidad de contaminantes no es homogénea entre todas las estaciones. Mientras que `NO2` presenta la cobertura espacial y temporal más amplia, `PM2.5` dispone de un número considerablemente menor de observaciones y se concentra principalmente en determinadas estaciones.

Esta heterogeneidad se conservará y será considerada posteriormente durante el análisis exploratorio y el modelado, evitando asumir una cobertura uniforme de la red de medida.

> **Conclusión:** el dataset supera los controles de integridad necesarios para continuar con la preintegración. El siguiente paso consiste en construir la clave espacial `estacion_geo` y asociar a cada observación las coordenadas y la tipología correspondientes a su estación física, verificando que el cruce espacial alcance una correspondencia completa antes de incorporar nuevas fuentes de información.

### 2.3. Homogeneización espacial e integración con el catálogo de estaciones

El dataset temporal de contaminación conserva las denominaciones históricas utilizadas durante el proceso de limpieza, mientras que el análisis espacial requiere trabajar con una única localización física por estación.

La auditoría previa identificó **10 denominaciones temporales** frente a **8 localizaciones físicas**. Esta diferencia se debe principalmente a la existencia de tres identificadores históricos asociados a la estación de Sants:

- `Sants_hist`
- `Sants_4`
- `Sants_42`

Para preservar la trazabilidad del dataset limpio, la variable original `estacion_fisica` se mantiene sin modificaciones. Paralelamente se crea una nueva clave espacial, `estacion_geo`, que representa la localización física utilizada en las operaciones espaciales.

La correspondencia aplicada es:

- `Sants_hist` → `Sants`
- `Sants_4` → `Sants`
- `Sants_42` → `Sants`
- `Vall Hebron` → `Vall d'Hebron`
- `Observ Fabra` → `Observatori Fabra`

El resto de estaciones mantiene su denominación original.

A continuación, el dataset temporal se integra con el catálogo espacial de estaciones mediante una relación `many_to_one`, incorporando:

- latitud;
- longitud;
- tipología de estación;
- coordenadas proyectadas `X_ETRS89` e `Y_ETRS89`.

Se aplican controles estrictos para verificar que:

- el número de filas no cambia tras la integración;
- cada observación temporal encuentra una única estación espacial;
- no quedan observaciones sin coordenadas;
- no se introducen duplicados en la clave temporal-espacial.

> **Objetivo de la etapa:** asociar a cada observación de contaminación una localización física única y homogénea, manteniendo al mismo tiempo la trazabilidad histórica de los códigos de estación.

In [ ]:
# ============================================================
# 2.3. HOMOGENEIZACIÓN ESPACIAL E INTEGRACIÓN
# CONTAMINACIÓN + CATÁLOGO DE ESTACIONES
# ============================================================

import pandas as pd
from pathlib import Path


# ============================================================
# 1. RUTAS DE TRABAJO
# ============================================================

ruta_carpeta = Path(
    "/content/drive/MyDrive/TFM/"
    "11_Machine_Learning_Temporal/"
    "PREINTEGRACION/"
    "06_Contaminacion_Atmosferica"
)

ruta_contaminacion = (
    ruta_carpeta /
    "df_contaminacion_atm_2018_2024_Limpio.csv"
)

ruta_estaciones = (
    ruta_carpeta /
    "estaciones_contaminacion_atmosferica.csv"
)

ruta_salida = (
    ruta_carpeta /
    "df_contaminacion_atm_preintegrado.csv"
)


# ============================================================
# 2. COMPROBAR EXISTENCIA DE LOS ARCHIVOS
# ============================================================

if not ruta_contaminacion.exists():
    raise FileNotFoundError(
        f"No se encuentra el dataset de contaminación:\n"
        f"{ruta_contaminacion}"
    )

if not ruta_estaciones.exists():
    raise FileNotFoundError(
        f"No se encuentra el catálogo de estaciones:\n"
        f"{ruta_estaciones}"
    )


# ============================================================
# 3. CARGAR DATASETS
# ============================================================

df_contaminacion = pd.read_csv(
    ruta_contaminacion
)

df_estaciones = pd.read_csv(
    ruta_estaciones
)


# ============================================================
# 4. NORMALIZAR FECHA
# ============================================================

df_contaminacion["fecha"] = pd.to_datetime(
    df_contaminacion["fecha"],
    errors="coerce"
)

if df_contaminacion["fecha"].isna().any():
    raise ValueError(
        "Se han detectado fechas no convertibles "
        "en el dataset de contaminación."
    )


# ============================================================
# 5. CREAR CLAVE ESPACIAL HOMOGÉNEA
# ============================================================

mapeo_estacion_geo = {
    "Sants_hist": "Sants",
    "Sants_4": "Sants",
    "Sants_42": "Sants",
    "Vall Hebron": "Vall d'Hebron",
    "Observ Fabra": "Observatori Fabra"
}

df_contaminacion["estacion_geo"] = (
    df_contaminacion["estacion_fisica"]
    .replace(mapeo_estacion_geo)
)


# ============================================================
# 6. CONTROL DE CORRESPONDENCIA ANTES DEL MERGE
# ============================================================

estaciones_temporales = set(
    df_contaminacion["estacion_geo"]
    .dropna()
    .unique()
)

estaciones_catalogo = set(
    df_estaciones["estacion_fisica"]
    .dropna()
    .unique()
)

sin_correspondencia = (
    estaciones_temporales
    - estaciones_catalogo
)

print("=" * 80)
print("CONTROL PREVIO DE CORRESPONDENCIA")
print("=" * 80)

print(
    f"\nEstaciones temporales homogeneizadas: "
    f"{len(estaciones_temporales)}"
)

print(
    f"Estaciones en catálogo espacial: "
    f"{len(estaciones_catalogo)}"
)

print("\nEstaciones temporales sin correspondencia:")

if sin_correspondencia:
    for estacion in sorted(sin_correspondencia):
        print("  ⚠", estacion)

    raise ValueError(
        "Existen estaciones sin correspondencia espacial."
    )

else:
    print("  ✓ Ninguna")


# ============================================================
# 7. PREPARAR CATÁLOGO ESPACIAL PARA EL MERGE
# ============================================================

columnas_estaciones = [
    "estacion_fisica",
    "lat",
    "lon",
    "tipo",
    "X_ETRS89",
    "Y_ETRS89"
]

faltantes_catalogo = [
    columna
    for columna in columnas_estaciones
    if columna not in df_estaciones.columns
]

if faltantes_catalogo:
    raise KeyError(
        "Faltan columnas en el catálogo espacial:\n"
        f"{faltantes_catalogo}"
    )


df_estaciones_merge = (
    df_estaciones[columnas_estaciones]
    .rename(
        columns={
            "estacion_fisica": "estacion_geo"
        }
    )
    .copy()
)


# ============================================================
# 8. COMPROBAR UNICIDAD DEL CATÁLOGO ESPACIAL
# ============================================================

if not df_estaciones_merge["estacion_geo"].is_unique:
    duplicadas = (
        df_estaciones_merge.loc[
            df_estaciones_merge["estacion_geo"]
            .duplicated(keep=False),
            "estacion_geo"
        ]
        .unique()
        .tolist()
    )

    raise ValueError(
        "El catálogo espacial contiene estaciones duplicadas:\n"
        f"{duplicadas}"
    )


# ============================================================
# 9. MERGE ESPACIAL MANY_TO_ONE
# ============================================================

filas_antes = len(df_contaminacion)

df_contaminacion_preintegrado = (
    df_contaminacion
    .merge(
        df_estaciones_merge,
        on="estacion_geo",
        how="left",
        validate="many_to_one"
    )
)

filas_despues = len(
    df_contaminacion_preintegrado
)


# ============================================================
# 10. CONTROL DEL NÚMERO DE FILAS
# ============================================================

if filas_antes != filas_despues:
    raise ValueError(
        "El número de filas ha cambiado tras el merge.\n"
        f"Antes: {filas_antes:,}\n"
        f"Después: {filas_despues:,}"
    )


# ============================================================
# 11. CONTROL DE COBERTURA ESPACIAL
# ============================================================

columnas_coordenadas = [
    "lat",
    "lon",
    "X_ETRS89",
    "Y_ETRS89"
]

nulos_espaciales = (
    df_contaminacion_preintegrado[
        columnas_coordenadas
    ]
    .isna()
    .any(axis=1)
    .sum()
)

print("\n" + "=" * 80)
print("CONTROL DE COBERTURA ESPACIAL")
print("=" * 80)

print(
    f"\nFilas antes del merge: "
    f"{filas_antes:,}"
)

print(
    f"Filas después del merge: "
    f"{filas_despues:,}"
)

print(
    f"Filas sin coordenadas completas: "
    f"{nulos_espaciales:,}"
)

if nulos_espaciales > 0:
    raise ValueError(
        "Existen observaciones sin coordenadas "
        "después de la integración."
    )


# ============================================================
# 12. CONTROL DE DUPLICADOS EN LA CLAVE TEMPORAL-ESPACIAL
# ============================================================

clave_control = [
    "fecha",
    "estacion_fisica",
    "contaminant"
]

duplicados_clave = (
    df_contaminacion_preintegrado
    .duplicated(
        subset=clave_control
    )
    .sum()
)

print(
    f"\nDuplicados "
    f"fecha + estacion_fisica + contaminant: "
    f"{duplicados_clave:,}"
)

if duplicados_clave > 0:
    raise ValueError(
        "Se han introducido duplicados durante "
        "la preintegración."
    )


# ============================================================
# 13. RESUMEN DE ESTACIONES
# ============================================================

print("\n" + "=" * 80)
print("CORRESPONDENCIA DE ESTACIONES")
print("=" * 80)

tabla_correspondencia = (
    df_contaminacion_preintegrado[
        [
            "estacion_fisica",
            "estacion_geo",
            "tipo"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "estacion_geo",
            "estacion_fisica"
        ]
    )
    .reset_index(drop=True)
)

display(tabla_correspondencia)


# ============================================================
# 14. RESUMEN FINAL DEL DATASET PREINTEGRADO
# ============================================================

print("\n" + "=" * 80)
print("RESUMEN DEL DATASET PREINTEGRADO")
print("=" * 80)

print(
    f"\nRegistros: "
    f"{len(df_contaminacion_preintegrado):,}"
)

print(
    f"Estaciones históricas: "
    f"{df_contaminacion_preintegrado['estacion_fisica'].nunique()}"
)

print(
    f"Localizaciones físicas: "
    f"{df_contaminacion_preintegrado['estacion_geo'].nunique()}"
)

print(
    f"Contaminantes: "
    f"{df_contaminacion_preintegrado['contaminant'].nunique()}"
)

print(
    "\nPeriodo:"
)

print(
    df_contaminacion_preintegrado["fecha"].min(),
    "→",
    df_contaminacion_preintegrado["fecha"].max()
)


# ============================================================
# 15. GUARDAR DATASET PREINTEGRADO
# ============================================================

df_contaminacion_preintegrado.to_csv(
    ruta_salida,
    index=False,
    encoding="utf-8"
)


# ============================================================
# 16. VERIFICACIÓN DE EXPORTACIÓN
# ============================================================

if not ruta_salida.exists():
    raise FileNotFoundError(
        "No se ha creado correctamente "
        "el dataset preintegrado."
    )


# ============================================================
# 17. CIERRE
# ============================================================

print("\n" + "=" * 80)
print("PREINTEGRACIÓN DE CONTAMINACIÓN FINALIZADA")
print("=" * 80)

print(
    "\n✓ Número de filas conservado."
)

print(
    "✓ Correspondencia espacial completa."
)

print(
    "✓ Sin duplicados introducidos."
)

print(
    "✓ Trazabilidad histórica conservada "
    "en 'estacion_fisica'."
)

print(
    "✓ Localización física homogénea "
    "disponible en 'estacion_geo'."
)

print("\n✓ Archivo generado:")
print(ruta_salida)

CONTROL PREVIO DE CORRESPONDENCIA

Estaciones temporales homogeneizadas: 8
Estaciones en catálogo espacial: 8

Estaciones temporales sin correspondencia:
  ✓ Ninguna

CONTROL DE COBERTURA ESPACIAL

Filas antes del merge: 43,642
Filas después del merge: 43,642
Filas sin coordenadas completas: 0

Duplicados fecha + estacion_fisica + contaminant: 0

CORRESPONDENCIA DE ESTACIONES


,estacion_fisica,estacion_geo,tipo
0,Ciutadella,Ciutadella,Fondo
1,Eixample,Eixample,Tráfico
2,Gràcia,Gràcia,Tráfico
3,Observ Fabra,Observatori Fabra,Fondo regional
4,Palau Reial,Palau Reial,Fondo
5,Poblenou,Poblenou,Fondo
6,Sants_4,Sants,Fondo
7,Sants_42,Sants,Fondo
8,Sants_hist,Sants,Fondo
9,Vall Hebron,Vall d'Hebron,Fondo



RESUMEN DEL DATASET PREINTEGRADO

Registros: 43,642
Estaciones históricas: 10
Localizaciones físicas: 8
Contaminantes: 4

Periodo:
2018-06-12 00:00:00 → 2024-12-31 00:00:00

PREINTEGRACIÓN DE CONTAMINACIÓN FINALIZADA

✓ Número de filas conservado.
✓ Correspondencia espacial completa.
✓ Sin duplicados introducidos.
✓ Trazabilidad histórica conservada en 'estacion_fisica'.
✓ Localización física homogénea disponible en 'estacion_geo'.

✓ Archivo generado:
/content/drive/MyDrive/TFM/11_Machine_Learning_Temporal/PREINTEGRACION/06_Contaminacion_Atmosferica/df_contaminacion_atm_preintegrado.csv


#### Resultado — Preintegración espacial de contaminación atmosférica

La homogeneización espacial se completa correctamente. Las **10 denominaciones históricas** presentes en el dataset temporal se asocian a **8 localizaciones físicas**, manteniendo la variable original `estacion_fisica` para preservar la trazabilidad y utilizando `estacion_geo` como clave espacial común.

La integración con el catálogo de estaciones se realiza mediante una relación `many_to_one`, sin modificar la granularidad original del dataset.

Los controles posteriores confirman:

- **43.642 registros** antes y después de la integración;
- **8 localizaciones físicas** correctamente identificadas;
- **0 observaciones sin correspondencia espacial**;
- **0 observaciones sin coordenadas**;
- **0 duplicados** introducidos en la clave `fecha + estacion_fisica + contaminant`;
- conservación de las coordenadas geográficas y proyectadas de cada estación;
- conservación de la tipología de estación.

El dataset resultante se guarda como:

`df_contaminacion_atm_preintegrado.csv`

y constituye la base espacio-temporal del bloque de contaminación atmosférica para las siguientes etapas de integración.

> **Conclusión:** la contaminación atmosférica queda completamente preintegrada, con trazabilidad histórica y referencia espacial homogénea, y puede utilizarse como tabla base para la incorporación progresiva de las restantes fuentes.

### 2.4. Creación del primer checkpoint del dataset maestro — Contaminación atmosférica

Una vez finalizada y validada la preintegración espacial del bloque de contaminación atmosférica, se genera la primera versión del **dataset maestro** que servirá como base para la incorporación progresiva del resto de fuentes de información.

El dataset de partida contiene las observaciones diarias de contaminación atmosférica junto con la localización y tipología de las estaciones de medida. La correspondencia espacial ha sido previamente homogeneizada mediante la variable `estacion_geo`, manteniendo `estacion_fisica` para conservar la trazabilidad histórica.

En esta etapa no se realiza ninguna transformación adicional de los datos. El objetivo es generar un **checkpoint independiente y reproducible** antes de comenzar las sucesivas integraciones.

Se verifica que:

- se conserva el número de registros del dataset preintegrado;
- no existen observaciones sin geolocalización;
- no aparecen duplicados en la clave `fecha + estacion_fisica + contaminant`;
- se mantienen las 8 localizaciones físicas;
- se conserva la cobertura temporal y espacial;
- el archivo exportado mantiene exactamente las dimensiones del dataset validado.

La primera versión del maestro se almacena en la carpeta `INTEGRACION` como:

`01_maestro_contaminacion.csv`

Las siguientes fuentes se incorporarán progresivamente sobre versiones sucesivas del maestro, conservando cada checkpoint para garantizar la trazabilidad completa del proceso de integración.

In [ ]:
# ============================================================
# 2.4. CREACIÓN DEL PRIMER CHECKPOINT DEL DATASET MAESTRO
# CONTAMINACIÓN ATMOSFÉRICA
# ============================================================

import pandas as pd
from pathlib import Path


# ============================================================
# 1. DEFINIR RUTAS
# ============================================================

ruta_preintegracion = Path(
    "/content/drive/MyDrive/TFM/"
    "11_Machine_Learning_Temporal/"
    "PREINTEGRACION/"
    "06_Contaminacion_Atmosferica"
)

ruta_integracion = Path(
    "/content/drive/MyDrive/TFM/"
    "11_Machine_Learning_Temporal/"
    "INTEGRACION"
)

ruta_origen = (
    ruta_preintegracion /
    "df_contaminacion_atm_preintegrado.csv"
)

ruta_maestro_01 = (
    ruta_integracion /
    "01_maestro_contaminacion.csv"
)


# ============================================================
# 2. COMPROBAR EXISTENCIA DE LAS RUTAS
# ============================================================

if not ruta_preintegracion.exists():
    raise FileNotFoundError(
        "No existe la carpeta de PREINTEGRACION:\n"
        f"{ruta_preintegracion}"
    )

if not ruta_integracion.exists():
    raise FileNotFoundError(
        "No existe la carpeta INTEGRACION:\n"
        f"{ruta_integracion}\n\n"
        "Se detiene el proceso para evitar crear "
        "una carpeta en una ubicación incorrecta."
    )

if not ruta_origen.exists():
    raise FileNotFoundError(
        "No se encuentra el dataset preintegrado:\n"
        f"{ruta_origen}"
    )


# ============================================================
# 3. CARGAR DATASET PREINTEGRADO
# ============================================================

df_maestro_01 = pd.read_csv(
    ruta_origen
)

df_maestro_01["fecha"] = pd.to_datetime(
    df_maestro_01["fecha"],
    errors="coerce"
)


# ============================================================
# 4. CONTROL DE FECHAS
# ============================================================

fechas_invalidas = (
    df_maestro_01["fecha"]
    .isna()
    .sum()
)

if fechas_invalidas > 0:
    raise ValueError(
        f"Se han detectado {fechas_invalidas:,} "
        "fechas no válidas."
    )


# ============================================================
# 5. CONTROL DE DIMENSIONES
# ============================================================

print("=" * 80)
print("CHECKPOINT 01 — CONTAMINACIÓN ATMOSFÉRICA")
print("=" * 80)

print("\nArchivo de origen:")
print(ruta_origen)

print(
    f"\n✓ Filas: "
    f"{len(df_maestro_01):,}"
)

print(
    f"✓ Columnas: "
    f"{df_maestro_01.shape[1]}"
)


# ============================================================
# 6. CONTROL DE COBERTURA TEMPORAL
# ============================================================

print(
    "\n✓ Periodo:",
    df_maestro_01["fecha"].min(),
    "→",
    df_maestro_01["fecha"].max()
)

print(
    f"✓ Fechas diferentes: "
    f"{df_maestro_01['fecha'].nunique():,}"
)


# ============================================================
# 7. CONTROL DE ESTACIONES Y CONTAMINANTES
# ============================================================

print(
    f"\n✓ Denominaciones históricas: "
    f"{df_maestro_01['estacion_fisica'].nunique()}"
)

print(
    f"✓ Localizaciones físicas: "
    f"{df_maestro_01['estacion_geo'].nunique()}"
)

print(
    f"✓ Contaminantes: "
    f"{df_maestro_01['contaminant'].nunique()}"
)


# ============================================================
# 8. CONTROL DE COBERTURA ESPACIAL
# ============================================================

columnas_espaciales = [
    "estacion_geo",
    "lat",
    "lon",
    "X_ETRS89",
    "Y_ETRS89"
]

columnas_faltantes = [
    columna
    for columna in columnas_espaciales
    if columna not in df_maestro_01.columns
]

if columnas_faltantes:
    raise KeyError(
        "Faltan variables espaciales necesarias:\n"
        f"{columnas_faltantes}"
    )

filas_sin_geolocalizacion = (
    df_maestro_01[
        columnas_espaciales
    ]
    .isna()
    .any(axis=1)
    .sum()
)

print(
    f"\n✓ Filas sin geolocalización completa: "
    f"{filas_sin_geolocalizacion:,}"
)

if filas_sin_geolocalizacion > 0:
    raise ValueError(
        "Existen registros sin geolocalización completa."
    )


# ============================================================
# 9. CONTROL DE DUPLICADOS
# ============================================================

clave_maestro = [
    "fecha",
    "estacion_fisica",
    "contaminant"
]

duplicados_maestro = (
    df_maestro_01
    .duplicated(
        subset=clave_maestro
    )
    .sum()
)

print(
    "✓ Duplicados "
    "fecha + estacion_fisica + contaminant: "
    f"{duplicados_maestro:,}"
)

if duplicados_maestro > 0:
    raise ValueError(
        "El dataset contiene duplicados "
        "en la clave temporal-espacial."
    )


# ============================================================
# 10. CONTROL DE NULOS
# ============================================================

nulos_totales = (
    df_maestro_01
    .isna()
    .sum()
)

print("\n" + "=" * 80)
print("VALORES NULOS")
print("=" * 80)

print(nulos_totales)


# ============================================================
# 11. GUARDAR CHECKPOINT 01
# ============================================================

df_maestro_01.to_csv(
    ruta_maestro_01,
    index=False,
    encoding="utf-8"
)


# ============================================================
# 12. VERIFICAR ARCHIVO EXPORTADO
# ============================================================

if not ruta_maestro_01.exists():
    raise FileNotFoundError(
        "No se ha generado correctamente "
        "el checkpoint 01."
    )

df_control = pd.read_csv(
    ruta_maestro_01
)

if df_control.shape != df_maestro_01.shape:
    raise ValueError(
        "Las dimensiones del archivo exportado "
        "no coinciden con el dataset original.\n"
        f"Original: {df_maestro_01.shape}\n"
        f"Exportado: {df_control.shape}"
    )


# ============================================================
# 13. TAMAÑO DEL ARCHIVO GENERADO
# ============================================================

tamano_mb = (
    ruta_maestro_01.stat().st_size
    / (1024 ** 2)
)


# ============================================================
# 14. CONTROL FINAL
# ============================================================

print("\n" + "=" * 80)
print("CHECKPOINT 01 CREADO CORRECTAMENTE")
print("=" * 80)

print(
    f"\n✓ Registros conservados: "
    f"{len(df_maestro_01):,}"
)

print(
    f"✓ Variables disponibles: "
    f"{df_maestro_01.shape[1]}"
)

print(
    f"✓ Localizaciones físicas: "
    f"{df_maestro_01['estacion_geo'].nunique()}"
)

print(
    f"✓ Contaminantes: "
    f"{df_maestro_01['contaminant'].nunique()}"
)

print(
    f"✓ Tamaño del archivo: "
    f"{tamano_mb:.2f} MB"
)

print(
    "✓ Sin pérdida de información "
    "durante la exportación."
)

print(
    "✓ Dataset preparado para incorporar "
    "la siguiente fuente."
)

print("\nArchivo maestro generado:")
print(ruta_maestro_01)

CHECKPOINT 01 — CONTAMINACIÓN ATMOSFÉRICA

Archivo de origen:
/content/drive/MyDrive/TFM/11_Machine_Learning_Temporal/PREINTEGRACION/06_Contaminacion_Atmosferica/df_contaminacion_atm_preintegrado.csv

✓ Filas: 43,642
✓ Columnas: 14

✓ Periodo: 2018-06-12 00:00:00 → 2024-12-31 00:00:00
✓ Fechas diferentes: 2,249

✓ Denominaciones históricas: 10
✓ Localizaciones físicas: 8
✓ Contaminantes: 4

✓ Filas sin geolocalización completa: 0
✓ Duplicados fecha + estacion_fisica + contaminant: 0

VALORES NULOS
fecha               0
codi_estacio        0
nom_cabina          0
contaminant         0
valor               0
horas_validas       0
codi_estacio_str    0
estacion_fisica     0
estacion_geo        0
lat                 0
lon                 0
tipo                0
X_ETRS89            0
Y_ETRS89            0
dtype: int64

CHECKPOINT 01 CREADO CORRECTAMENTE

✓ Registros conservados: 43,642
✓ Variables disponibles: 14
✓ Localizaciones físicas: 8
✓ Contaminantes: 4
✓ Tamaño del archivo: 5.77 MB
✓ 

#### Resultado — Checkpoint 01

Se genera correctamente el primer checkpoint del dataset maestro:

`01_maestro_contaminacion.csv`

El archivo conserva íntegramente la información validada durante la preintegración:

- **43.642 observaciones**;
- **14 variables**;
- **2.249 fechas** entre 2018-06-12 y 2024-12-31;
- **8 localizaciones físicas**;
- **4 contaminantes**;
- **0 valores nulos**;
- **0 observaciones sin geolocalización**;
- **0 duplicados** en la clave `fecha + estacion_fisica + contaminant`.

Este archivo constituye la **base oficial de la integración progresiva**. Las siguientes fuentes se incorporarán mediante nuevos checkpoints, sin sobrescribir las versiones anteriores.

# 3 · INTEGRACIÓN DE METEOROLOGÍA
El primer bloque explicativo incorporado al dataset maestro corresponde a las **condiciones meteorológicas**, fundamentales para interpretar los procesos de dispersión, acumulación y transformación de los contaminantes atmosféricos.

La integración requiere combinar dos dimensiones: la correspondencia espacial entre las estaciones de calidad del aire y las estaciones meteorológicas, y la correspondencia temporal diaria de las observaciones.

Para ello se prepara y valida la red meteorológica, se establece una asociación espacial con las estaciones de contaminación y posteriormente se incorporan al maestro las variables de temperatura, humedad, presión atmosférica, precipitación, radiación y viento disponibles.

Las diferencias de cobertura existentes entre variables meteorológicas se conservan explícitamente y no se realiza ninguna imputación en esta fase.

El bloque finaliza con la generación del **Checkpoint 02**.

> **Objetivo del bloque:** incorporar al dataset maestro las condiciones meteorológicas asociadas espacial y temporalmente a cada estación de calidad del aire, manteniendo la trazabilidad y la cobertura real de la fuente.


### 3.1. Preparación espacial de las estaciones meteorológicas

Como primer paso del bloque meteorológico se construye el catálogo espacial de las estaciones utilizadas en el estudio.

La red meteorológica considerada está formada por cuatro estaciones:

- `D5` — Observatori Fabra
- `X2` — Zoo
- `X4` — El Raval
- `X8` — Zona Universitària

Para cada estación se conservan su código identificador, nombre y coordenadas geográficas originales en `EPSG:4326` (WGS84).

Posteriormente, las coordenadas se transforman al sistema proyectado **ETRS89 / UTM zona 31N (`EPSG:25831`)**, utilizado como sistema espacial común durante el proceso de preintegración.

Esta referencia espacial permitirá posteriormente establecer la correspondencia entre las estaciones meteorológicas y las estaciones de contaminación atmosférica mediante criterios de proximidad espacial.

El catálogo resultante se almacena como:

`estaciones_meteorologia.csv`

dentro de:

`11_Machine_Learning_Temporal/PREINTEGRACION/08_Meteorologia/`

> **Objetivo:** disponer de una referencia espacial homogénea, trazable y reproducible de las estaciones meteorológicas antes de incorporar las observaciones meteorológicas al dataset maestro.

In [ ]:
# ============================================================
# 3.1. PREPARACIÓN ESPACIAL DE LAS ESTACIONES METEOROLÓGICAS
# ============================================================

import pandas as pd
import geopandas as gpd
from pathlib import Path


# ============================================================
# 1. DEFINIR DATOS DE LAS ESTACIONES METEOROLÓGICAS
# ============================================================

datos_estaciones_meteo = [
    {
        "codigo": "D5",
        "estacion": "Observatori Fabra",
        "lat": 41.41864,
        "lon": 2.12379
    },
    {
        "codigo": "X2",
        "estacion": "Zoo",
        "lat": 41.38740,
        "lon": 2.18720
    },
    {
        "codigo": "X4",
        "estacion": "El Raval",
        "lat": 41.38070,
        "lon": 2.16860
    },
    {
        "codigo": "X8",
        "estacion": "Zona Universitària",
        "lat": 41.38470,
        "lon": 2.11280
    }
]

df_estaciones_meteo = pd.DataFrame(
    datos_estaciones_meteo
)


# ============================================================
# 2. CONTROL INICIAL
# ============================================================

if df_estaciones_meteo["codigo"].duplicated().any():
    raise ValueError(
        "Existen códigos de estación meteorológica duplicados."
    )

if df_estaciones_meteo[
    ["lat", "lon"]
].isna().any().any():
    raise ValueError(
        "Existen estaciones meteorológicas sin coordenadas."
    )


# ============================================================
# 3. CREAR GEODATAFRAME EN WGS84
# ============================================================
# Coordenadas originales:
# EPSG:4326 — WGS84
#
# Sistema proyectado de trabajo:
# EPSG:25831 — ETRS89 / UTM zona 31N
# ============================================================

gdf_estaciones_meteo = gpd.GeoDataFrame(
    df_estaciones_meteo,
    geometry=gpd.points_from_xy(
        df_estaciones_meteo["lon"],
        df_estaciones_meteo["lat"]
    ),
    crs="EPSG:4326"
)


# ============================================================
# 4. TRANSFORMAR A ETRS89 / UTM ZONA 31N
# ============================================================

gdf_estaciones_meteo = (
    gdf_estaciones_meteo
    .to_crs("EPSG:25831")
)


# ============================================================
# 5. EXTRAER COORDENADAS PROYECTADAS
# ============================================================

gdf_estaciones_meteo["X_ETRS89"] = (
    gdf_estaciones_meteo.geometry.x
)

gdf_estaciones_meteo["Y_ETRS89"] = (
    gdf_estaciones_meteo.geometry.y
)


# ============================================================
# 6. CREAR TABLA FINAL
# ============================================================

df_estaciones_meteo_final = (
    gdf_estaciones_meteo[
        [
            "codigo",
            "estacion",
            "lat",
            "lon",
            "X_ETRS89",
            "Y_ETRS89"
        ]
    ]
    .copy()
)


# ============================================================
# 7. DEFINIR RUTA DE PREINTEGRACIÓN
# ============================================================

ruta_meteo = Path(
    "/content/drive/MyDrive/TFM/"
    "11_Machine_Learning_Temporal/"
    "PREINTEGRACION/"
    "08_Meteorologia"
)


# ============================================================
# 8. COMPROBAR QUE LA CARPETA YA EXISTE
# ============================================================
# No se crea automáticamente para evitar escribir
# accidentalmente en una ruta incorrecta.
# ============================================================

if not ruta_meteo.exists():
    raise FileNotFoundError(
        "No existe la carpeta de preintegración meteorológica:\n"
        f"{ruta_meteo}"
    )


# ============================================================
# 9. DEFINIR ARCHIVO DE SALIDA
# ============================================================

ruta_estaciones_meteo = (
    ruta_meteo /
    "estaciones_meteorologia.csv"
)


# ============================================================
# 10. EXPORTAR CSV
# ============================================================

df_estaciones_meteo_final.to_csv(
    ruta_estaciones_meteo,
    index=False,
    encoding="utf-8"
)


# ============================================================
# 11. VERIFICAR EXPORTACIÓN MEDIANTE RELECTURA
# ============================================================

if not ruta_estaciones_meteo.exists():
    raise FileNotFoundError(
        "No se ha creado correctamente "
        "estaciones_meteorologia.csv."
    )

df_control_meteo = pd.read_csv(
    ruta_estaciones_meteo
)

if df_control_meteo.shape != df_estaciones_meteo_final.shape:
    raise ValueError(
        "Las dimensiones del archivo exportado "
        "no coinciden con la tabla original."
    )


# ============================================================
# 12. CONTROL DE COORDENADAS PROYECTADAS
# ============================================================

nulos_utm = (
    df_control_meteo[
        ["X_ETRS89", "Y_ETRS89"]
    ]
    .isna()
    .any(axis=1)
    .sum()
)

if nulos_utm > 0:
    raise ValueError(
        "Existen estaciones sin coordenadas proyectadas."
    )


# ============================================================
# 13. RESULTADO FINAL
# ============================================================

print("=" * 80)
print("ESTACIONES METEOROLÓGICAS — PREINTEGRACIÓN")
print("=" * 80)

print(
    f"\n✓ Número de estaciones: "
    f"{len(df_control_meteo)}"
)

print(
    f"✓ Códigos únicos: "
    f"{df_control_meteo['codigo'].nunique()}"
)

print(
    "✓ CRS original: "
    "EPSG:4326 — WGS84"
)

print(
    "✓ CRS proyectado: "
    "EPSG:25831 — ETRS89 / UTM zona 31N"
)

print(
    f"✓ Estaciones sin coordenadas proyectadas: "
    f"{nulos_utm}"
)

print("\n✓ Archivo guardado en:")
print(ruta_estaciones_meteo)

print(
    "\n✓ Archivo existe:",
    ruta_estaciones_meteo.exists()
)


# ============================================================
# 14. VISUALIZACIÓN
# ============================================================

print("\nTabla de estaciones meteorológicas:")

display(
    df_control_meteo
)


# ============================================================
# 15. CIERRE
# ============================================================

print("\n" + "=" * 80)
print("PREPARACIÓN ESPACIAL 3.1 FINALIZADA")
print("=" * 80)

print(
    "\n✓ Catálogo meteorológico preparado "
    "para la preintegración."
)

ESTACIONES METEOROLÓGICAS — PREINTEGRACIÓN

✓ Número de estaciones: 4
✓ Códigos únicos: 4
✓ CRS original: EPSG:4326 — WGS84
✓ CRS proyectado: EPSG:25831 — ETRS89 / UTM zona 31N
✓ Estaciones sin coordenadas proyectadas: 0

✓ Archivo guardado en:
/content/drive/MyDrive/TFM/11_Machine_Learning_Temporal/PREINTEGRACION/08_Meteorologia/estaciones_meteorologia.csv

✓ Archivo existe: True

Tabla de estaciones meteorológicas:


,codigo,estacion,lat,lon,X_ETRS89,Y_ETRS89
0,D5,Observatori Fabra,41.41864,2.12379,426777.269335,4.585602e+06
1,X2,Zoo,41.38740,2.18720,432043.794132,4.582082e+06
2,X4,El Raval,41.38070,2.16860,430481.544114,4.581353e+06
3,X8,Zona Universitària,41.38470,2.11280,425820.263624,4.581844e+06



PREPARACIÓN ESPACIAL 3.1 FINALIZADA

✓ Catálogo meteorológico preparado para la preintegración.


#### Resultado — Preparación espacial de las estaciones meteorológicas

La preparación espacial del bloque meteorológico se completa correctamente.

Se obtiene un catálogo formado por **4 estaciones meteorológicas y 4 códigos únicos**, conservando para cada una las coordenadas geográficas originales y sus correspondientes coordenadas proyectadas en `EPSG:25831`.

Los controles realizados confirman:

- **4 estaciones meteorológicas**;
- **4 códigos de estación únicos**;
- **0 estaciones sin coordenadas proyectadas**;
- transformación correcta de `EPSG:4326` a `EPSG:25831`;
- exportación correcta del catálogo espacial.

El archivo resultante se guarda como:

`estaciones_meteorologia.csv`

y queda preparado para su utilización durante la preintegración del dataset meteorológico.

> **Conclusión:** el catálogo espacial meteorológico queda validado y preparado para asociar las observaciones meteorológicas con la estructura espacio-temporal del dataset maestro.

### 3.2. Carga y auditoría del dataset meteorológico limpio

Una vez preparado el catálogo espacial de estaciones meteorológicas, se carga y audita el dataset meteorológico limpio correspondiente al periodo 2018–2024.

El dataset presenta una estructura diaria por estación meteorológica e incluye variables de temperatura, humedad relativa, presión atmosférica, precipitación, radiación solar y viento.

En esta etapa se comprueba:

- la estructura y dimensiones del dataset;
- la cobertura temporal global y por estación;
- la correspondencia entre los códigos de estación del dataset y el catálogo espacial;
- la unicidad de la clave `Fecha + Estacion`;
- la presencia de valores nulos;
- la disponibilidad de cada variable meteorológica por estación;
- los rangos y estadísticos descriptivos de las variables numéricas.

La disponibilidad de las variables se analiza por estación antes de realizar cualquier imputación, ya que las diferencias de cobertura pueden responder a la instrumentación y disponibilidad histórica de cada observatorio.

No se realiza todavía ninguna integración con el dataset maestro ni se modifican los datos originales.

> **Objetivo:** validar la estructura, granularidad, cobertura temporal y disponibilidad de las variables meteorológicas antes de definir la estrategia de preintegración espacial con las estaciones de contaminación atmosférica.

In [ ]:
# ============================================================
# 3.2. CARGA Y AUDITORÍA DEL DATASET METEOROLÓGICO LIMPIO
# ============================================================

import pandas as pd
from pathlib import Path


# ============================================================
# 1. DEFINIR RUTAS
# ============================================================

ruta_meteo = Path(
    "/content/drive/MyDrive/TFM/"
    "11_Machine_Learning_Temporal/"
    "PREINTEGRACION/"
    "08_Meteorologia"
)

ruta_dataset_meteo = (
    ruta_meteo /
    "df_meteorologia_2018_2024_Limpio.csv"
)

ruta_estaciones_meteo = (
    ruta_meteo /
    "estaciones_meteorologia.csv"
)


# ============================================================
# 2. COMPROBAR EXISTENCIA DE CARPETA Y ARCHIVOS
# ============================================================

if not ruta_meteo.exists():
    raise FileNotFoundError(
        "No existe la carpeta meteorológica:\n"
        f"{ruta_meteo}"
    )

if not ruta_dataset_meteo.exists():
    raise FileNotFoundError(
        "No se encuentra el dataset meteorológico limpio:\n"
        f"{ruta_dataset_meteo}"
    )

if not ruta_estaciones_meteo.exists():
    raise FileNotFoundError(
        "No se encuentra el catálogo espacial de estaciones:\n"
        f"{ruta_estaciones_meteo}"
    )


# ============================================================
# 3. CARGAR DATASETS
# ============================================================

df_meteo = pd.read_csv(
    ruta_dataset_meteo
)

df_estaciones_meteo = pd.read_csv(
    ruta_estaciones_meteo
)


# ============================================================
# 4. COMPROBAR COLUMNAS FUNDAMENTALES
# ============================================================

columnas_obligatorias = [
    "Fecha",
    "Estacion"
]

faltantes = [
    columna
    for columna in columnas_obligatorias
    if columna not in df_meteo.columns
]

if faltantes:
    raise KeyError(
        "Faltan columnas fundamentales en el dataset:\n"
        f"{faltantes}"
    )

if "codigo" not in df_estaciones_meteo.columns:
    raise KeyError(
        "El catálogo espacial no contiene "
        "la columna 'codigo'."
    )


# ============================================================
# 5. INFORMACIÓN GENERAL
# ============================================================

print("=" * 80)
print("DATASET LIMPIO — METEOROLOGÍA")
print("=" * 80)

print("\nArchivo meteorológico:")
print(ruta_dataset_meteo)

print("\nCatálogo espacial:")
print(ruta_estaciones_meteo)

print(
    f"\n✓ Filas: "
    f"{df_meteo.shape[0]:,}"
)

print(
    f"✓ Columnas: "
    f"{df_meteo.shape[1]}"
)

print("\nColumnas disponibles:")

for columna in df_meteo.columns:
    print(f"  • {columna}")


# ============================================================
# 6. TIPOS DE DATOS
# ============================================================

print("\n" + "=" * 80)
print("TIPOS DE DATOS")
print("=" * 80)

print(
    df_meteo.dtypes
)


# ============================================================
# 7. CONVERTIR FECHA
# ============================================================

df_meteo["Fecha"] = pd.to_datetime(
    df_meteo["Fecha"],
    errors="coerce"
)

fechas_invalidas = (
    df_meteo["Fecha"]
    .isna()
    .sum()
)

if fechas_invalidas > 0:
    raise ValueError(
        f"Se han detectado {fechas_invalidas:,} "
        "fechas no válidas."
    )


# ============================================================
# 8. COBERTURA TEMPORAL GLOBAL
# ============================================================

print("\n" + "=" * 80)
print("COBERTURA TEMPORAL")
print("=" * 80)

print(
    "\nFecha inicial:",
    df_meteo["Fecha"].min()
)

print(
    "Fecha final:",
    df_meteo["Fecha"].max()
)

print(
    "Número de fechas diferentes:",
    df_meteo["Fecha"].nunique()
)

print(
    f"Fechas no válidas: "
    f"{fechas_invalidas:,}"
)


# ============================================================
# 9. ESTACIONES DEL DATASET
# ============================================================

estaciones_dataset = sorted(
    df_meteo["Estacion"]
    .dropna()
    .astype(str)
    .unique()
)

print("\n" + "=" * 80)
print("ESTACIONES METEOROLÓGICAS")
print("=" * 80)

print(
    f"\n✓ Número de códigos detectados: "
    f"{len(estaciones_dataset)}"
)

print("\nCódigos encontrados:")

for estacion in estaciones_dataset:
    print(f"  • {estacion}")


# ============================================================
# 10. CORRESPONDENCIA CON EL CATÁLOGO ESPACIAL
# ============================================================

codigos_dataset = set(
    df_meteo["Estacion"]
    .dropna()
    .astype(str)
)

codigos_catalogo = set(
    df_estaciones_meteo["codigo"]
    .dropna()
    .astype(str)
)

sin_catalogo = (
    codigos_dataset - codigos_catalogo
)

sin_datos = (
    codigos_catalogo - codigos_dataset
)

print("\n" + "=" * 80)
print("CORRESPONDENCIA CON EL CATÁLOGO ESPACIAL")
print("=" * 80)

print(
    "\nCódigos del dataset sin correspondencia:",
    sorted(sin_catalogo)
    if sin_catalogo
    else "✓ Ninguno"
)

print(
    "Códigos del catálogo sin observaciones:",
    sorted(sin_datos)
    if sin_datos
    else "✓ Ninguno"
)

if sin_catalogo:
    raise ValueError(
        "Existen estaciones meteorológicas "
        "sin correspondencia en el catálogo espacial."
    )


# ============================================================
# 11. DUPLICADOS COMPLETOS
# ============================================================

duplicados_completos = (
    df_meteo
    .duplicated()
    .sum()
)


# ============================================================
# 12. DUPLICADOS EN LA CLAVE FECHA + ESTACION
# ============================================================

duplicados_clave = (
    df_meteo
    .duplicated(
        subset=[
            "Fecha",
            "Estacion"
        ]
    )
    .sum()
)

print("\n" + "=" * 80)
print("CONTROL DE DUPLICADOS")
print("=" * 80)

print(
    f"\n✓ Duplicados completos: "
    f"{duplicados_completos:,}"
)

print(
    f"✓ Duplicados Fecha + Estacion: "
    f"{duplicados_clave:,}"
)

if duplicados_clave > 0:
    print(
        "\n⚠ Existen duplicados en la clave "
        "Fecha + Estacion. Deberán revisarse "
        "antes de la preintegración."
    )


# ============================================================
# 13. VALORES NULOS
# ============================================================

nulos = (
    df_meteo
    .isna()
    .sum()
)

porcentaje_nulos = (
    df_meteo
    .isna()
    .mean()
    .mul(100)
    .round(2)
)

tabla_nulos = pd.DataFrame({
    "n_nulos": nulos,
    "porcentaje_nulos": porcentaje_nulos
})

print("\n" + "=" * 80)
print("VALORES NULOS")
print("=" * 80)

display(
    tabla_nulos
)


# ============================================================
# 14. COBERTURA TEMPORAL POR ESTACIÓN
# ============================================================

cobertura_estaciones = (
    df_meteo
    .groupby("Estacion")
    .agg(
        fecha_inicio=(
            "Fecha",
            "min"
        ),
        fecha_fin=(
            "Fecha",
            "max"
        ),
        fechas_disponibles=(
            "Fecha",
            "nunique"
        ),
        observaciones=(
            "Fecha",
            "size"
        )
    )
    .reset_index()
    .sort_values("Estacion")
)

print("\n" + "=" * 80)
print("COBERTURA TEMPORAL POR ESTACIÓN")
print("=" * 80)

display(
    cobertura_estaciones
)


# ============================================================
# 15. VARIABLES METEOROLÓGICAS
# ============================================================

variables_meteo = [
    "TM",
    "TN",
    "TX",
    "HRM",
    "HRN",
    "HRX",
    "PM",
    "PN",
    "PX",
    "PPT",
    "RS24h",
    "DVM10",
    "DVVX10",
    "VVM10",
    "VVX10"
]

variables_disponibles = [
    variable
    for variable in variables_meteo
    if variable in df_meteo.columns
]

variables_ausentes = [
    variable
    for variable in variables_meteo
    if variable not in df_meteo.columns
]

print("\n" + "=" * 80)
print("VARIABLES METEOROLÓGICAS")
print("=" * 80)

print(
    f"\n✓ Variables esperadas: "
    f"{len(variables_meteo)}"
)

print(
    f"✓ Variables disponibles: "
    f"{len(variables_disponibles)}"
)

if variables_ausentes:
    print(
        "⚠ Variables ausentes:",
        variables_ausentes
    )
else:
    print(
        "✓ Todas las variables meteorológicas "
        "esperadas están presentes."
    )


# ============================================================
# 16. OBSERVACIONES VÁLIDAS POR ESTACIÓN Y VARIABLE
# ============================================================

cobertura_variables = (
    df_meteo
    .groupby("Estacion")[
        variables_disponibles
    ]
    .count()
)

print("\n" + "=" * 80)
print("OBSERVACIONES VÁLIDAS POR ESTACIÓN Y VARIABLE")
print("=" * 80)

display(
    cobertura_variables
)


# ============================================================
# 17. DISPONIBILIDAD (%) POR ESTACIÓN Y VARIABLE
# ============================================================

disponibilidad_variables = (
    df_meteo
    .groupby("Estacion")[
        variables_disponibles
    ]
    .agg(
        lambda serie:
        serie.notna().mean() * 100
    )
    .round(2)
)

print("\n" + "=" * 80)
print("DISPONIBILIDAD (%) POR ESTACIÓN Y VARIABLE")
print("=" * 80)

display(
    disponibilidad_variables
)


# ============================================================
# 18. RESUMEN ESTADÍSTICO
# ============================================================

print("\n" + "=" * 80)
print("RESUMEN ESTADÍSTICO DE VARIABLES METEOROLÓGICAS")
print("=" * 80)

resumen_estadistico = (
    df_meteo[
        variables_disponibles
    ]
    .describe()
    .T
)

display(
    resumen_estadistico
)


# ============================================================
# 19. COMPROBACIÓN DE RANGOS BÁSICOS
# ============================================================
# No elimina ni modifica valores.
# Solo identifica posibles valores físicamente sospechosos
# que puedan requerir revisión posterior.
# ============================================================

print("\n" + "=" * 80)
print("CONTROL BÁSICO DE RANGOS")
print("=" * 80)

controles_rango = {}

if "HRM" in df_meteo.columns:
    controles_rango["HRM_fuera_0_100"] = (
        ~df_meteo["HRM"].between(
            0,
            100,
            inclusive="both"
        )
        & df_meteo["HRM"].notna()
    ).sum()

if "HRN" in df_meteo.columns:
    controles_rango["HRN_fuera_0_100"] = (
        ~df_meteo["HRN"].between(
            0,
            100,
            inclusive="both"
        )
        & df_meteo["HRN"].notna()
    ).sum()

if "HRX" in df_meteo.columns:
    controles_rango["HRX_fuera_0_100"] = (
        ~df_meteo["HRX"].between(
            0,
            100,
            inclusive="both"
        )
        & df_meteo["HRX"].notna()
    ).sum()

if "PPT" in df_meteo.columns:
    controles_rango["PPT_negativa"] = (
        (df_meteo["PPT"] < 0)
        & df_meteo["PPT"].notna()
    ).sum()

if "VVM10" in df_meteo.columns:
    controles_rango["VVM10_negativa"] = (
        (df_meteo["VVM10"] < 0)
        & df_meteo["VVM10"].notna()
    ).sum()

if "VVX10" in df_meteo.columns:
    controles_rango["VVX10_negativa"] = (
        (df_meteo["VVX10"] < 0)
        & df_meteo["VVX10"].notna()
    ).sum()

if "DVM10" in df_meteo.columns:
    controles_rango["DVM10_fuera_0_360"] = (
        ~df_meteo["DVM10"].between(
            0,
            360,
            inclusive="both"
        )
        & df_meteo["DVM10"].notna()
    ).sum()

if "DVVX10" in df_meteo.columns:
    controles_rango["DVVX10_fuera_0_360"] = (
        ~df_meteo["DVVX10"].between(
            0,
            360,
            inclusive="both"
        )
        & df_meteo["DVVX10"].notna()
    ).sum()

for control, numero in controles_rango.items():
    print(
        f"✓ {control}: "
        f"{numero:,}"
    )


# ============================================================
# 20. PRIMERAS FILAS
# ============================================================

print("\n" + "=" * 80)
print("PRIMERAS FILAS DEL DATASET")
print("=" * 80)

display(
    df_meteo.head(10)
)


# ============================================================
# 21. RESUMEN FINAL DE AUDITORÍA
# ============================================================

print("\n" + "=" * 80)
print("AUDITORÍA METEOROLÓGICA FINALIZADA")
print("=" * 80)

print(
    f"\n✓ Dataset: "
    f"{df_meteo.shape[0]:,} filas × "
    f"{df_meteo.shape[1]} columnas"
)

print(
    f"✓ Periodo: "
    f"{df_meteo['Fecha'].min()} "
    f"→ "
    f"{df_meteo['Fecha'].max()}"
)

print(
    f"✓ Fechas diferentes: "
    f"{df_meteo['Fecha'].nunique():,}"
)

print(
    f"✓ Estaciones detectadas: "
    f"{df_meteo['Estacion'].nunique()}"
)

print(
    f"✓ Variables meteorológicas: "
    f"{len(variables_disponibles)}"
)

print(
    f"✓ Duplicados completos: "
    f"{duplicados_completos:,}"
)

print(
    f"✓ Duplicados Fecha + Estacion: "
    f"{duplicados_clave:,}"
)

print(
    "✓ Correspondencia con catálogo espacial: "
    + (
        "completa"
        if not sin_catalogo
        else "incompleta"
    )
)

print(
    "\n✓ No se ha sobrescrito ni modificado "
    "ningún archivo de origen."
)

print(
    "✓ Dataset preparado para definir "
    "la estrategia de preintegración meteorológica."
)

DATASET LIMPIO — METEOROLOGÍA

Archivo meteorológico:
/content/drive/MyDrive/TFM/11_Machine_Learning_Temporal/PREINTEGRACION/08_Meteorologia/df_meteorologia_2018_2024_Limpio.csv

Catálogo espacial:
/content/drive/MyDrive/TFM/11_Machine_Learning_Temporal/PREINTEGRACION/08_Meteorologia/estaciones_meteorologia.csv

✓ Filas: 10,136
✓ Columnas: 17

Columnas disponibles:
  • Fecha
  • Estacion
  • TM
  • TN
  • TX
  • HRM
  • HRN
  • HRX
  • PM
  • PN
  • PX
  • PPT
  • RS24h
  • DVM10
  • DVVX10
  • VVM10
  • VVX10

TIPOS DE DATOS
Fecha        object
Estacion     object
TM          float64
TN          float64
TX          float64
HRM         float64
HRN         float64
HRX         float64
PM          float64
PN          float64
PX          float64
PPT         float64
RS24h       float64
DVM10       float64
DVVX10      float64
VVM10       float64
VVX10       float64
dtype: object

COBERTURA TEMPORAL

Fecha inicial: 2018-01-01 00:00:00
Fecha final: 2024-12-31 00:00:00
Número de fechas diferent

,n_nulos,porcentaje_nulos
Fecha,0,0.00
Estacion,0,0.00
TM,0,0.00
TN,0,0.00
TX,0,0.00
HRM,0,0.00
HRN,2920,28.81
HRX,2920,28.81
PM,4655,45.93
PN,4655,45.93



COBERTURA TEMPORAL POR ESTACIÓN


,Estacion,fecha_inicio,fecha_fin,fechas_disponibles,observaciones
0,D5,2018-01-01,2024-12-31,2557,2557
1,X2,2018-01-01,2024-09-30,2465,2465
2,X4,2018-01-01,2024-12-31,2557,2557
3,X8,2018-01-01,2024-12-31,2557,2557



VARIABLES METEOROLÓGICAS

✓ Variables esperadas: 15
✓ Variables disponibles: 15
✓ Todas las variables meteorológicas esperadas están presentes.

OBSERVACIONES VÁLIDAS POR ESTACIÓN Y VARIABLE


...


RESUMEN ESTADÍSTICO DE VARIABLES METEOROLÓGICAS


,count,mean,std,min,25%,50%,75%,max
TM,10136.0,17.495424,6.032170,-0.7,12.6,16.9,22.7,33.4
TN,10136.0,14.123056,6.025070,-2.4,9.3,13.6,19.3,29.5
TX,10136.0,21.785093,6.247192,1.2,16.6,21.2,27.1,39.8
HRM,10136.0,68.131102,12.212608,22.0,60.0,69.0,77.0,100.0
HRN,7216.0,47.587390,13.273874,7.0,38.0,47.0,57.0,100.0
HRX,7216.0,87.380469,11.506744,36.0,80.0,90.0,97.0,100.0
PM,5481.0,996.746816,20.587990,942.5,972.9,1006.1,1012.5,1033.7
PN,5481.0,994.569513,20.712482,936.6,971.0,1003.9,1010.6,1032.8
PX,5481.0,999.074621,20.537422,946.8,975.0,1008.3,1014.7,1035.7
PPT,7671.0,1.548638,6.664773,0.0,0.0,0.0,0.0,139.9



CONTROL BÁSICO DE RANGOS
✓ HRM_fuera_0_100: 0
✓ HRN_fuera_0_100: 0
✓ HRX_fuera_0_100: 0
✓ PPT_negativa: 0
✓ VVM10_negativa: 0
✓ VVX10_negativa: 0
✓ DVM10_fuera_0_360: 0
✓ DVVX10_fuera_0_360: 0

PRIMERAS FILAS DEL DATASET


,Fecha,Estacion,TM,TN,TX,HRM,HRN,HRX,PM,PN,PX,PPT,RS24h,DVM10,DVVX10,VVM10,VVX10
0,2018-01-01,D5,9.1,6.0,13.0,56.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,273.0,NaN,5.0,18.2
1,2018-01-01,X2,13.0,9.8,15.5,44.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2018-01-01,X4,13.1,10.2,15.8,44.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,304.0,NaN,5.1,17.2
3,2018-01-01,X8,12.4,9.2,15.8,46.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,299.0,NaN,5.3,17.3
4,2018-01-02,D5,10.9,7.8,15.8,68.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,303.0,NaN,4.1,17.2
5,2018-01-02,X2,13.9,8.7,17.6,57.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2018-01-02,X4,14.6,11.5,18.2,55.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,296.0,NaN,2.6,11.6
7,2018-01-02,X8,13.9,10.3,18.1,58.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,297.0,NaN,3.5,14.2
8,2018-01-03,D5,14.4,11.2,18.7,68.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,305.0,NaN,6.3,17.9
9,2018-01-03,X2,17.6,11.7,21.8,57.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



AUDITORÍA METEOROLÓGICA FINALIZADA

✓ Dataset: 10,136 filas × 17 columnas
✓ Periodo: 2018-01-01 00:00:00 → 2024-12-31 00:00:00
✓ Fechas diferentes: 2,557
✓ Estaciones detectadas: 4
✓ Variables meteorológicas: 15
✓ Duplicados completos: 0
✓ Duplicados Fecha + Estacion: 0
✓ Correspondencia con catálogo espacial: completa

✓ No se ha sobrescrito ni modificado ningún archivo de origen.
✓ Dataset preparado para definir la estrategia de preintegración meteorológica.


#### Resultado — Auditoría del dataset meteorológico

La auditoría confirma una estructura meteorológica diaria consistente y adecuada para la fase de preintegración.

El dataset contiene **10.136 registros**, correspondientes a **4 estaciones meteorológicas** y **15 variables meteorológicas**, con una cobertura global comprendida entre el **1 de enero de 2018 y el 31 de diciembre de 2024**.

Los controles realizados confirman:

- ausencia de fechas inválidas;
- ausencia de duplicados completos;
- unicidad de la clave `Fecha + Estacion`;
- correspondencia completa entre los códigos del dataset y el catálogo espacial de estaciones;
- presencia de las 15 variables meteorológicas previstas;
- rangos físicamente coherentes en los controles básicos realizados.

La cobertura temporal es completa hasta el 31/12/2024 para `D5`, `X4` y `X8`, mientras que `X2` dispone de observaciones hasta el 30/09/2024.

La disponibilidad de las variables no es homogénea entre estaciones. Las variables `TM`, `TN`, `TX` y `HRM` presentan cobertura completa, mientras que otras variables muestran disponibilidad parcial. En particular, la estación `X2` no dispone de registros de presión atmosférica, precipitación, radiación solar ni viento.

Estas ausencias se consideran **estructurales respecto a la disponibilidad de las estaciones** y no se imputan en esta fase. Su tratamiento se definirá posteriormente en función de la estrategia espacial de integración y de las variables finalmente utilizadas en los modelos.

> **Conclusión:** el dataset meteorológico presenta una estructura temporal y espacial consistente y queda validado para definir la estrategia de asociación entre las estaciones meteorológicas y las estaciones de contaminación atmosférica.

### 3.3. Asociación espacial entre estaciones de contaminación y estaciones meteorológicas

Una vez validadas las estaciones meteorológicas y las estaciones de contaminación atmosférica, se construye una relación espacial entre ambas redes de medida.

Dado que existen **8 localizaciones físicas de contaminación** y **4 estaciones meteorológicas**, la integración no puede realizarse mediante una correspondencia directa por nombre. En su lugar, se utiliza la proximidad espacial como criterio inicial de asociación.

Para cada estación de contaminación se calculan las distancias euclídeas, en metros, respecto a las cuatro estaciones meteorológicas utilizando las coordenadas proyectadas en **ETRS89 / UTM zona 31N (`EPSG:25831`)**.

A partir de estas distancias se identifican:

- la estación meteorológica más próxima a cada estación de contaminación;
- la distancia mínima entre ambas;
- la segunda estación meteorológica más cercana;
- la diferencia de distancia entre la primera y la segunda opción.

Este análisis permite evaluar la robustez espacial de la asignación antes de incorporar las variables meteorológicas al dataset maestro.

> **Objetivo:** construir una correspondencia espacial reproducible entre ambas redes de medida y validar que la estación meteorológica asignada a cada estación de contaminación sea razonable desde el punto de vista geográfico.

In [ ]:
# ============================================================
# 3.3. ASOCIACIÓN ESPACIAL ENTRE ESTACIONES
# CONTAMINACIÓN ATMOSFÉRICA + METEOROLOGÍA
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. DEFINIR RUTAS
# ============================================================

ruta_contaminacion = Path(
    "/content/drive/MyDrive/TFM/"
    "11_Machine_Learning_Temporal/"
    "PREINTEGRACION/"
    "06_Contaminacion_Atmosferica/"
    "estaciones_contaminacion_atmosferica.csv"
)

ruta_meteo = Path(
    "/content/drive/MyDrive/TFM/"
    "11_Machine_Learning_Temporal/"
    "PREINTEGRACION/"
    "08_Meteorologia/"
    "estaciones_meteorologia.csv"
)

ruta_salida = Path(
    "/content/drive/MyDrive/TFM/"
    "11_Machine_Learning_Temporal/"
    "PREINTEGRACION/"
    "08_Meteorologia/"
    "correspondencia_contaminacion_meteorologia.csv"
)


# ============================================================
# 2. COMPROBAR EXISTENCIA DE LOS ARCHIVOS
# ============================================================

if not ruta_contaminacion.exists():
    raise FileNotFoundError(
        "No se encuentra el catálogo de estaciones "
        "de contaminación:\n"
        f"{ruta_contaminacion}"
    )

if not ruta_meteo.exists():
    raise FileNotFoundError(
        "No se encuentra el catálogo de estaciones "
        "meteorológicas:\n"
        f"{ruta_meteo}"
    )


# ============================================================
# 3. CARGAR CATÁLOGOS
# ============================================================

df_cont_est = pd.read_csv(
    ruta_contaminacion
)

df_meteo_est = pd.read_csv(
    ruta_meteo
)


# ============================================================
# 4. COMPROBAR COLUMNAS NECESARIAS
# ============================================================

columnas_cont = [
    "estacion_fisica",
    "X_ETRS89",
    "Y_ETRS89"
]

columnas_meteo = [
    "codigo",
    "estacion",
    "X_ETRS89",
    "Y_ETRS89"
]

faltantes_cont = [
    c for c in columnas_cont
    if c not in df_cont_est.columns
]

faltantes_meteo = [
    c for c in columnas_meteo
    if c not in df_meteo_est.columns
]

if faltantes_cont:
    raise KeyError(
        "Faltan columnas en estaciones de contaminación:\n"
        f"{faltantes_cont}"
    )

if faltantes_meteo:
    raise KeyError(
        "Faltan columnas en estaciones meteorológicas:\n"
        f"{faltantes_meteo}"
    )


# ============================================================
# 5. CONTROL DE UNICIDAD
# ============================================================

if not df_cont_est["estacion_fisica"].is_unique:
    raise ValueError(
        "El catálogo de contaminación contiene "
        "estaciones físicas duplicadas."
    )

if not df_meteo_est["codigo"].is_unique:
    raise ValueError(
        "El catálogo meteorológico contiene "
        "códigos duplicados."
    )


# ============================================================
# 6. CALCULAR MATRIZ DE DISTANCIAS
# ============================================================

resultados = []

for _, fila_cont in df_cont_est.iterrows():

    x_cont = fila_cont["X_ETRS89"]
    y_cont = fila_cont["Y_ETRS89"]

    distancias = []

    for _, fila_meteo in df_meteo_est.iterrows():

        x_meteo = fila_meteo["X_ETRS89"]
        y_meteo = fila_meteo["Y_ETRS89"]

        distancia_m = np.sqrt(
            (x_cont - x_meteo) ** 2 +
            (y_cont - y_meteo) ** 2
        )

        distancias.append({
            "codigo_meteo": fila_meteo["codigo"],
            "estacion_meteo": fila_meteo["estacion"],
            "distancia_m": distancia_m
        })

    # Ordenar por distancia
    distancias = sorted(
        distancias,
        key=lambda x: x["distancia_m"]
    )

    primera = distancias[0]
    segunda = distancias[1]

    resultados.append({
        "estacion_contaminacion": fila_cont["estacion_fisica"],
        "codigo_meteo_asignado": primera["codigo_meteo"],
        "estacion_meteo_asignada": primera["estacion_meteo"],
        "distancia_m": primera["distancia_m"],
        "codigo_meteo_2": segunda["codigo_meteo"],
        "estacion_meteo_2": segunda["estacion_meteo"],
        "distancia_2_m": segunda["distancia_m"],
        "diferencia_1_2_m": (
            segunda["distancia_m"]
            - primera["distancia_m"]
        )
    })


# ============================================================
# 7. CREAR DATAFRAME DE CORRESPONDENCIA
# ============================================================

df_correspondencia_meteo = pd.DataFrame(
    resultados
)


# ============================================================
# 8. ORDENAR POR ESTACIÓN DE CONTAMINACIÓN
# ============================================================

df_correspondencia_meteo = (
    df_correspondencia_meteo
    .sort_values(
        "estacion_contaminacion"
    )
    .reset_index(drop=True)
)


# ============================================================
# 9. AÑADIR DISTANCIAS EN KM
# ============================================================

df_correspondencia_meteo["distancia_km"] = (
    df_correspondencia_meteo["distancia_m"]
    / 1000
)

df_correspondencia_meteo["distancia_2_km"] = (
    df_correspondencia_meteo["distancia_2_m"]
    / 1000
)

df_correspondencia_meteo["diferencia_1_2_km"] = (
    df_correspondencia_meteo["diferencia_1_2_m"]
    / 1000
)


# ============================================================
# 10. CONTROL GENERAL
# ============================================================

print("=" * 80)
print("ASOCIACIÓN ESPACIAL — CONTAMINACIÓN / METEOROLOGÍA")
print("=" * 80)

print(
    f"\n✓ Estaciones de contaminación: "
    f"{len(df_cont_est)}"
)

print(
    f"✓ Estaciones meteorológicas: "
    f"{len(df_meteo_est)}"
)

print(
    f"✓ Correspondencias generadas: "
    f"{len(df_correspondencia_meteo)}"
)


# ============================================================
# 11. RESUMEN DE ASIGNACIONES
# ============================================================

print("\n" + "=" * 80)
print("ASIGNACIÓN METEOROLÓGICA POR ESTACIÓN")
print("=" * 80)

display(
    df_correspondencia_meteo[
        [
            "estacion_contaminacion",
            "codigo_meteo_asignado",
            "estacion_meteo_asignada",
            "distancia_km",
            "codigo_meteo_2",
            "estacion_meteo_2",
            "distancia_2_km",
            "diferencia_1_2_km"
        ]
    ]
)


# ============================================================
# 12. DISTRIBUCIÓN DE ESTACIONES METEOROLÓGICAS ASIGNADAS
# ============================================================

print("\n" + "=" * 80)
print("DISTRIBUCIÓN DE ASIGNACIONES")
print("=" * 80)

tabla_asignaciones = (
    df_correspondencia_meteo
    ["codigo_meteo_asignado"]
    .value_counts()
    .rename("n_estaciones_contaminacion")
    .to_frame()
)

display(
    tabla_asignaciones
)


# ============================================================
# 13. CONTROL DE DISTANCIAS
# ============================================================

print("\n" + "=" * 80)
print("CONTROL DE DISTANCIAS")
print("=" * 80)

print(
    f"\nDistancia mínima: "
    f"{df_correspondencia_meteo['distancia_km'].min():.3f} km"
)

print(
    f"Distancia máxima: "
    f"{df_correspondencia_meteo['distancia_km'].max():.3f} km"
)

print(
    f"Distancia media: "
    f"{df_correspondencia_meteo['distancia_km'].mean():.3f} km"
)

print(
    f"Distancia mediana: "
    f"{df_correspondencia_meteo['distancia_km'].median():.3f} km"
)


# ============================================================
# 14. IDENTIFICAR ASIGNACIONES POTENCIALMENTE AMBIGUAS
# ============================================================
# Se muestran casos donde la primera y segunda estación
# meteorológica están separadas por menos de 1 km de diferencia.
# Esto NO invalida la asignación; sirve como control.
# ============================================================

umbral_ambiguedad_km = 1.0

ambiguas = (
    df_correspondencia_meteo[
        df_correspondencia_meteo[
            "diferencia_1_2_km"
        ] < umbral_ambiguedad_km
    ]
    .copy()
)

print("\n" + "=" * 80)
print("POSIBLES ASIGNACIONES ESPACIALMENTE PRÓXIMAS")
print("=" * 80)

print(
    f"\nCasos con diferencia entre primera y segunda "
    f"estación < {umbral_ambiguedad_km} km: "
    f"{len(ambiguas)}"
)

if len(ambiguas) > 0:

    display(
        ambiguas[
            [
                "estacion_contaminacion",
                "codigo_meteo_asignado",
                "distancia_km",
                "codigo_meteo_2",
                "distancia_2_km",
                "diferencia_1_2_km"
            ]
        ]
    )

else:

    print(
        "✓ No se detectan asignaciones "
        "especialmente ambiguas."
    )


# ============================================================
# 15. GUARDAR TABLA DE CORRESPONDENCIA
# ============================================================

df_correspondencia_meteo.to_csv(
    ruta_salida,
    index=False,
    encoding="utf-8"
)


# ============================================================
# 16. VERIFICAR EXPORTACIÓN
# ============================================================

if not ruta_salida.exists():
    raise FileNotFoundError(
        "No se ha generado correctamente "
        "la tabla de correspondencia."
    )


# ============================================================
# 17. CIERRE
# ============================================================

print("\n" + "=" * 80)
print("ASOCIACIÓN ESPACIAL 3.3 FINALIZADA")
print("=" * 80)

print(
    "\n✓ Correspondencia espacial calculada "
    "para todas las estaciones de contaminación."
)

print(
    "✓ Primera y segunda estación meteorológica "
    "más cercana identificadas."
)

print(
    "✓ Distancias calculadas en EPSG:25831."
)

print("\n✓ Archivo generado:")
print(ruta_salida)

ASOCIACIÓN ESPACIAL — CONTAMINACIÓN / METEOROLOGÍA

✓ Estaciones de contaminación: 8
✓ Estaciones meteorológicas: 4
✓ Correspondencias generadas: 8

ASIGNACIÓN METEOROLÓGICA POR ESTACIÓN


,estacion_contaminacion,codigo_meteo_asignado,estacion_meteo_asignada,distancia_km,codigo_meteo_2,estacion_meteo_2,distancia_2_km,diferencia_1_2_km
0,Ciutadella,X2,Zoo,0.112276,X4,El Raval,1.694597,1.582321
1,Eixample,X4,El Raval,1.338783,X2,Zoo,2.802405,1.463622
2,Gràcia,X4,El Raval,2.368291,X2,Zoo,3.091818,0.723528
3,Observatori Fabra,D5,Observatori Fabra,0.028187,X8,Zona Universitària,3.854875,3.826687
4,Palau Reial,X8,Zona Universitària,0.374614,D5,Observatori Fabra,3.529399,3.154786
5,Poblenou,X2,Zoo,2.334007,X4,El Raval,3.955137,1.621130
6,Sants,X8,Zona Universitària,1.784226,X4,El Raval,2.998932,1.214707
7,Vall d'Hebron,D5,Observatori Fabra,2.170686,X4,El Raval,5.331948,3.161262



DISTRIBUCIÓN DE ASIGNACIONES


,n_estaciones_contaminacion
codigo_meteo_asignado,
X2,2
X4,2
D5,2
X8,2



CONTROL DE DISTANCIAS

Distancia mínima: 0.028 km
Distancia máxima: 2.368 km
Distancia media: 1.314 km
Distancia mediana: 1.562 km

POSIBLES ASIGNACIONES ESPACIALMENTE PRÓXIMAS

Casos con diferencia entre primera y segunda estación < 1.0 km: 1


,estacion_contaminacion,codigo_meteo_asignado,distancia_km,codigo_meteo_2,distancia_2_km,diferencia_1_2_km
2,Gràcia,X4,2.368291,X2,3.091818,0.723528



ASOCIACIÓN ESPACIAL 3.3 FINALIZADA

✓ Correspondencia espacial calculada para todas las estaciones de contaminación.
✓ Primera y segunda estación meteorológica más cercana identificadas.
✓ Distancias calculadas en EPSG:25831.

✓ Archivo generado:
/content/drive/MyDrive/TFM/11_Machine_Learning_Temporal/PREINTEGRACION/08_Meteorologia/correspondencia_contaminacion_meteorologia.csv


#### Resultado — Asociación espacial contaminación–meteorología

La asociación espacial entre las estaciones de contaminación atmosférica y las estaciones meteorológicas se completa correctamente mediante distancia euclídea en `EPSG:25831`.

Las **8 estaciones físicas de contaminación** quedan asociadas a las **4 estaciones meteorológicas**, asignando a cada una la estación meteorológica más próxima.

La distribución resultante es equilibrada, con dos estaciones de contaminación asociadas a cada estación meteorológica.

Las distancias entre ambas redes son reducidas:

- distancia mínima: **0,028 km**;
- distancia máxima: **2,368 km**;
- distancia media: **1,314 km**;
- distancia mediana: **1,562 km**.

La única asociación espacialmente menos diferenciada corresponde a `Gràcia`, cuya estación meteorológica más próxima es `X4 — El Raval`, aunque `X2 — Zoo` constituye una segunda alternativa relativamente cercana.

La correspondencia obtenida se almacena como:

`correspondencia_contaminacion_meteorologia.csv`

y se utilizará como tabla de referencia para incorporar las observaciones meteorológicas al dataset maestro.

> **Conclusión:** la proximidad espacial entre ambas redes permite establecer una correspondencia reproducible y suficientemente próxima para la integración meteorológica a escala urbana.

### 3.4. Preintegración temporal y espacial de la información meteorológica

Una vez establecida la correspondencia espacial entre las estaciones de contaminación atmosférica y las estaciones meteorológicas, se prepara el bloque meteorológico para su posterior incorporación al dataset maestro.

La tabla de correspondencia obtenida en la etapa anterior permite asignar a cada una de las **8 estaciones físicas de contaminación** la estación meteorológica más próxima. Esta relación se utiliza para trasladar las observaciones meteorológicas diarias desde la red meteorológica hacia la estructura espacial de la red de calidad del aire.

La preintegración se realiza manteniendo la granularidad diaria de los datos meteorológicos y conservando las **15 variables meteorológicas disponibles**.

Durante esta etapa:

- se verifica nuevamente la unicidad de la clave `Fecha + Estacion` del dataset meteorológico;
- se incorpora el código y nombre de la estación meteorológica asignada;
- se replica la información meteorológica diaria para las estaciones de contaminación asociadas a cada observatorio;
- se conserva la distancia entre ambas estaciones como variable de trazabilidad espacial;
- se mantienen los valores ausentes estructurales detectados durante la auditoría, sin realizar imputaciones;
- se comprueba la unicidad de la nueva clave `fecha + estacion_geo`;
- se genera un archivo meteorológico preintegrado independiente antes de realizar el `merge` con el dataset maestro.

El resultado se almacena como:

`df_meteorologia_preintegrado.csv`

dentro de:

`11_Machine_Learning_Temporal/PREINTEGRACION/08_Meteorologia/`

> **Objetivo:** generar una tabla meteorológica diaria adaptada a las localizaciones físicas de las estaciones de contaminación, manteniendo la trazabilidad de la estación meteorológica de origen y sin alterar todavía el dataset maestro.

In [ ]:
# ============================================================
# 3.4. PREINTEGRACIÓN TEMPORAL Y ESPACIAL
# DE LA INFORMACIÓN METEOROLÓGICA
# ============================================================

import pandas as pd
from pathlib import Path


# ============================================================
# 1. DEFINIR RUTAS
# ============================================================

ruta_base_meteo = Path(
    "/content/drive/MyDrive/TFM/"
    "11_Machine_Learning_Temporal/"
    "PREINTEGRACION/"
    "08_Meteorologia"
)

ruta_dataset_meteo = (
    ruta_base_meteo /
    "df_meteorologia_2018_2024_Limpio.csv"
)

ruta_correspondencia = (
    ruta_base_meteo /
    "correspondencia_contaminacion_meteorologia.csv"
)

ruta_salida = (
    ruta_base_meteo /
    "df_meteorologia_preintegrado.csv"
)


# ============================================================
# 2. COMPROBAR EXISTENCIA DE ARCHIVOS
# ============================================================

if not ruta_dataset_meteo.exists():
    raise FileNotFoundError(
        "No se encuentra el dataset meteorológico:\n"
        f"{ruta_dataset_meteo}"
    )

if not ruta_correspondencia.exists():
    raise FileNotFoundError(
        "No se encuentra la tabla de correspondencia "
        "contaminación-meteorología:\n"
        f"{ruta_correspondencia}"
    )


# ============================================================
# 3. CARGAR DATOS
# ============================================================

df_meteo = pd.read_csv(
    ruta_dataset_meteo
)

df_correspondencia = pd.read_csv(
    ruta_correspondencia
)


# ============================================================
# 4. NORMALIZAR FECHA
# ============================================================

df_meteo["Fecha"] = pd.to_datetime(
    df_meteo["Fecha"],
    errors="coerce"
)

fechas_invalidas = (
    df_meteo["Fecha"]
    .isna()
    .sum()
)

if fechas_invalidas > 0:
    raise ValueError(
        f"Existen {fechas_invalidas:,} "
        "fechas meteorológicas no válidas."
    )


# ============================================================
# 5. COMPROBAR COLUMNAS NECESARIAS
# ============================================================

columnas_correspondencia = [
    "estacion_contaminacion",
    "codigo_meteo_asignado",
    "estacion_meteo_asignada",
    "distancia_m",
    "distancia_km"
]

faltantes = [
    columna
    for columna in columnas_correspondencia
    if columna not in df_correspondencia.columns
]

if faltantes:
    raise KeyError(
        "Faltan columnas en la tabla de correspondencia:\n"
        f"{faltantes}"
    )

columnas_meteo_necesarias = [
    "Fecha",
    "Estacion"
]

faltantes_meteo = [
    columna
    for columna in columnas_meteo_necesarias
    if columna not in df_meteo.columns
]

if faltantes_meteo:
    raise KeyError(
        "Faltan columnas fundamentales "
        "en el dataset meteorológico:\n"
        f"{faltantes_meteo}"
    )


# ============================================================
# 6. COMPROBAR UNICIDAD FECHA + ESTACION
# ============================================================

duplicados_meteo = (
    df_meteo
    .duplicated(
        subset=[
            "Fecha",
            "Estacion"
        ]
    )
    .sum()
)

if duplicados_meteo > 0:
    raise ValueError(
        "Existen duplicados Fecha + Estacion "
        "en el dataset meteorológico."
    )


# ============================================================
# 7. COMPROBAR UNICIDAD DE LA ASIGNACIÓN
# ============================================================
# Cada estación de contaminación debe tener exactamente
# una estación meteorológica asignada.
# ============================================================

duplicados_correspondencia = (
    df_correspondencia
    .duplicated(
        subset=[
            "estacion_contaminacion"
        ]
    )
    .sum()
)

if duplicados_correspondencia > 0:
    raise ValueError(
        "Existen estaciones de contaminación "
        "con más de una asignación meteorológica."
    )


# ============================================================
# 8. PREPARAR TABLA DE ASIGNACIÓN
# ============================================================

df_asignacion = (
    df_correspondencia[
        [
            "estacion_contaminacion",
            "codigo_meteo_asignado",
            "estacion_meteo_asignada",
            "distancia_m",
            "distancia_km"
        ]
    ]
    .rename(
        columns={
            "estacion_contaminacion": "estacion_geo"
        }
    )
    .copy()
)


# ============================================================
# 9. EXPANDIR INFORMACIÓN METEOROLÓGICA
# ============================================================
# Cada estación meteorológica puede estar asociada a varias
# estaciones de contaminación.
#
# Además, cada estación meteorológica tiene múltiples fechas.
#
# Por ello, el cruce por código meteorológico es
# intencionadamente MANY-TO-MANY.
#
# La unicidad correcta se comprobará después sobre:
# fecha + estacion_geo
# ============================================================

df_meteo_pre = pd.merge(
    df_asignacion,
    df_meteo,
    left_on="codigo_meteo_asignado",
    right_on="Estacion",
    how="left",
    validate="many_to_many"
)


# ============================================================
# 10. RENOMBRAR VARIABLES DE TRAZABILIDAD
# ============================================================

df_meteo_pre = (
    df_meteo_pre
    .rename(
        columns={
            "Fecha": "fecha",
            "Estacion": "codigo_meteo_origen",
            "distancia_m": "distancia_meteo_m",
            "distancia_km": "distancia_meteo_km"
        }
    )
)


# ============================================================
# 11. COMPROBAR CONSISTENCIA DE CÓDIGOS
# ============================================================

inconsistencias_codigo = (
    df_meteo_pre[
        "codigo_meteo_asignado"
    ].astype(str)
    !=
    df_meteo_pre[
        "codigo_meteo_origen"
    ].astype(str)
)

n_inconsistencias = (
    inconsistencias_codigo
    .sum()
)

if n_inconsistencias > 0:
    raise ValueError(
        "Se han detectado inconsistencias entre "
        "la estación meteorológica asignada "
        "y la estación de origen."
    )


# ============================================================
# 12. ELIMINAR COLUMNA REDUNDANTE
# ============================================================

df_meteo_pre = (
    df_meteo_pre
    .drop(
        columns=[
            "codigo_meteo_origen"
        ]
    )
    .rename(
        columns={
            "codigo_meteo_asignado": "codigo_meteo"
        }
    )
)


# ============================================================
# 13. DEFINIR VARIABLES METEOROLÓGICAS
# ============================================================

variables_meteo = [
    "TM",
    "TN",
    "TX",
    "HRM",
    "HRN",
    "HRX",
    "PM",
    "PN",
    "PX",
    "PPT",
    "RS24h",
    "DVM10",
    "DVVX10",
    "VVM10",
    "VVX10"
]


# ============================================================
# 14. COMPROBAR VARIABLES DISPONIBLES
# ============================================================

variables_faltantes = [
    variable
    for variable in variables_meteo
    if variable not in df_meteo_pre.columns
]

if variables_faltantes:
    raise KeyError(
        "Faltan variables meteorológicas:\n"
        f"{variables_faltantes}"
    )


# ============================================================
# 15. ORDENAR COLUMNAS
# ============================================================

columnas_finales = [
    "fecha",
    "estacion_geo",
    "codigo_meteo",
    "estacion_meteo_asignada",
    "distancia_meteo_m",
    "distancia_meteo_km"
] + variables_meteo

df_meteo_pre = (
    df_meteo_pre[
        columnas_finales
    ]
    .sort_values(
        [
            "fecha",
            "estacion_geo"
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# 16. CONTROL DE FILAS SIN FECHA
# ============================================================

filas_sin_fecha = (
    df_meteo_pre["fecha"]
    .isna()
    .sum()
)

if filas_sin_fecha > 0:
    raise ValueError(
        "Existen asociaciones sin observaciones "
        "meteorológicas."
    )


# ============================================================
# 17. CONTROL DE UNICIDAD FECHA + ESTACION_GEO
# ============================================================

duplicados_preintegracion = (
    df_meteo_pre
    .duplicated(
        subset=[
            "fecha",
            "estacion_geo"
        ]
    )
    .sum()
)

if duplicados_preintegracion > 0:
    raise ValueError(
        "La preintegración ha generado duplicados "
        "en fecha + estacion_geo."
    )


# ============================================================
# 18. CONTROL DE FILAS ESPERADAS
# ============================================================
# Como cada estación meteorológica está asignada a dos
# estaciones de contaminación:
#
# D5: 2557 x 2
# X2: 2465 x 2
# X4: 2557 x 2
# X8: 2557 x 2
#
# Total esperado = 20.272 filas
# ============================================================

filas_esperadas = (
    df_asignacion
    .merge(
        df_meteo[
            ["Fecha", "Estacion"]
        ],
        left_on="codigo_meteo_asignado",
        right_on="Estacion",
        how="left"
    )
    .shape[0]
)

print("=" * 80)
print("PREINTEGRACIÓN METEOROLÓGICA")
print("=" * 80)

print(
    f"\n✓ Filas esperadas: "
    f"{filas_esperadas:,}"
)

print(
    f"✓ Filas generadas: "
    f"{len(df_meteo_pre):,}"
)

if len(df_meteo_pre) != filas_esperadas:
    raise ValueError(
        "El número de filas generado no coincide "
        "con el número esperado."
    )


# ============================================================
# 19. RESUMEN GENERAL
# ============================================================

print(
    f"\n✓ Columnas: "
    f"{df_meteo_pre.shape[1]}"
)

print(
    f"✓ Estaciones de contaminación: "
    f"{df_meteo_pre['estacion_geo'].nunique()}"
)

print(
    f"✓ Estaciones meteorológicas de origen: "
    f"{df_meteo_pre['codigo_meteo'].nunique()}"
)

print(
    f"✓ Fechas diferentes: "
    f"{df_meteo_pre['fecha'].nunique():,}"
)

print(
    "\n✓ Periodo:",
    df_meteo_pre["fecha"].min(),
    "→",
    df_meteo_pre["fecha"].max()
)

print(
    f"\n✓ Duplicados fecha + estacion_geo: "
    f"{duplicados_preintegracion:,}"
)


# ============================================================
# 20. COBERTURA TEMPORAL POR ESTACIÓN DE CONTAMINACIÓN
# ============================================================

cobertura_preintegrada = (
    df_meteo_pre
    .groupby(
        [
            "estacion_geo",
            "codigo_meteo"
        ]
    )
    .agg(
        fecha_inicio=(
            "fecha",
            "min"
        ),
        fecha_fin=(
            "fecha",
            "max"
        ),
        fechas_disponibles=(
            "fecha",
            "nunique"
        ),
        distancia_km=(
            "distancia_meteo_km",
            "first"
        )
    )
    .reset_index()
    .sort_values(
        "estacion_geo"
    )
)

print("\n" + "=" * 80)
print("COBERTURA POR ESTACIÓN DE CONTAMINACIÓN")
print("=" * 80)

display(
    cobertura_preintegrada
)


# ============================================================
# 21. CONTROL DE NULOS
# ============================================================
# Los nulos estructurales se mantienen deliberadamente.
# No se realiza imputación en esta fase.
# ============================================================

nulos_variables = (
    df_meteo_pre[
        variables_meteo
    ]
    .isna()
    .sum()
)

porcentaje_nulos = (
    df_meteo_pre[
        variables_meteo
    ]
    .isna()
    .mean()
    .mul(100)
    .round(2)
)

tabla_nulos_pre = pd.DataFrame({
    "n_nulos": nulos_variables,
    "porcentaje_nulos": porcentaje_nulos
})

print("\n" + "=" * 80)
print("VALORES NULOS — VARIABLES METEOROLÓGICAS")
print("=" * 80)

display(
    tabla_nulos_pre
)


# ============================================================
# 22. DISPONIBILIDAD POR ESTACIÓN DE CONTAMINACIÓN
# ============================================================

disponibilidad_pre = (
    df_meteo_pre
    .groupby("estacion_geo")[
        variables_meteo
    ]
    .agg(
        lambda serie:
        serie.notna().mean() * 100
    )
    .round(2)
)

print("\n" + "=" * 80)
print("DISPONIBILIDAD (%) POR ESTACIÓN DE CONTAMINACIÓN")
print("=" * 80)

display(
    disponibilidad_pre
)


# ============================================================
# 23. GUARDAR DATASET PREINTEGRADO
# ============================================================

df_meteo_pre.to_csv(
    ruta_salida,
    index=False,
    encoding="utf-8"
)


# ============================================================
# 24. VERIFICAR EXPORTACIÓN
# ============================================================

if not ruta_salida.exists():
    raise FileNotFoundError(
        "No se ha generado correctamente "
        "df_meteorologia_preintegrado.csv."
    )

df_control = pd.read_csv(
    ruta_salida
)

if df_control.shape != df_meteo_pre.shape:
    raise ValueError(
        "Las dimensiones del archivo exportado "
        "no coinciden con el dataset preintegrado.\n"
        f"Original: {df_meteo_pre.shape}\n"
        f"Exportado: {df_control.shape}"
    )


# ============================================================
# 25. PRIMERAS FILAS
# ============================================================

print("\n" + "=" * 80)
print("PRIMERAS FILAS DEL DATASET PREINTEGRADO")
print("=" * 80)

display(
    df_meteo_pre.head(10)
)


# ============================================================
# 26. CIERRE
# ============================================================

print("\n" + "=" * 80)
print("PREINTEGRACIÓN METEOROLÓGICA 3.4 FINALIZADA")
print("=" * 80)

print(
    f"\n✓ Registros: "
    f"{len(df_meteo_pre):,}"
)

print(
    f"✓ Localizaciones de contaminación: "
    f"{df_meteo_pre['estacion_geo'].nunique()}"
)

print(
    f"✓ Estaciones meteorológicas de origen: "
    f"{df_meteo_pre['codigo_meteo'].nunique()}"
)

print(
    "✓ Sin duplicados en fecha + estacion_geo."
)

print(
    "✓ Nulos meteorológicos estructurales conservados."
)

print(
    "✓ No se ha modificado ningún archivo de origen."
)

print("\n✓ Archivo generado:")
print(ruta_salida)

print(
    "\n✓ Dataset preparado para su integración "
    "con el checkpoint 01 del dataset maestro."
)

PREINTEGRACIÓN METEOROLÓGICA

✓ Filas esperadas: 20,272
✓ Filas generadas: 20,272

✓ Columnas: 21
✓ Estaciones de contaminación: 8
✓ Estaciones meteorológicas de origen: 4
✓ Fechas diferentes: 2,557

✓ Periodo: 2018-01-01 00:00:00 → 2024-12-31 00:00:00

✓ Duplicados fecha + estacion_geo: 0

COBERTURA POR ESTACIÓN DE CONTAMINACIÓN


,estacion_geo,codigo_meteo,fecha_inicio,fecha_fin,fechas_disponibles,distancia_km
0,Ciutadella,X2,2018-01-01,2024-09-30,2465,0.112276
1,Eixample,X4,2018-01-01,2024-12-31,2557,1.338783
2,Gràcia,X4,2018-01-01,2024-12-31,2557,2.368291
3,Observatori Fabra,D5,2018-01-01,2024-12-31,2557,0.028187
4,Palau Reial,X8,2018-01-01,2024-12-31,2557,0.374614
5,Poblenou,X2,2018-01-01,2024-09-30,2465,2.334007
6,Sants,X8,2018-01-01,2024-12-31,2557,1.784226
7,Vall d'Hebron,D5,2018-01-01,2024-12-31,2557,2.170686



VALORES NULOS — VARIABLES METEOROLÓGICAS


,n_nulos,porcentaje_nulos
TM,0,0.00
TN,0,0.00
TX,0,0.00
HRM,0,0.00
HRN,5840,28.81
HRX,5840,28.81
PM,9310,45.93
PN,9310,45.93
PX,9310,45.93
PPT,4930,24.32



DISPONIBILIDAD (%) POR ESTACIÓN DE CONTAMINACIÓN


,TM,TN,TX,HRM,HRN,HRX,PM,PN,PX,PPT,RS24h,DVM10,DVVX10,VVM10,VVX10
estacion_geo,,,,,,,,,,,,,,,
Ciutadella,100.0,100.0,100.0,100.0,70.39,70.39,0.00,0.00,0.00,0.0,0.00,0.0,0.00,0.0,0.0
Eixample,100.0,100.0,100.0,100.0,71.45,71.45,71.45,71.45,71.45,100.0,71.45,100.0,71.45,100.0,100.0
Gràcia,100.0,100.0,100.0,100.0,71.45,71.45,71.45,71.45,71.45,100.0,71.45,100.0,71.45,100.0,100.0
Observatori Fabra,100.0,100.0,100.0,100.0,71.45,71.45,71.45,71.45,71.45,100.0,71.45,100.0,71.45,100.0,100.0
Palau Reial,100.0,100.0,100.0,100.0,71.45,71.45,71.45,71.45,71.45,100.0,71.45,100.0,71.45,100.0,100.0
Poblenou,100.0,100.0,100.0,100.0,70.39,70.39,0.00,0.00,0.00,0.0,0.00,0.0,0.00,0.0,0.0
Sants,100.0,100.0,100.0,100.0,71.45,71.45,71.45,71.45,71.45,100.0,71.45,100.0,71.45,100.0,100.0
Vall d'Hebron,100.0,100.0,100.0,100.0,71.45,71.45,71.45,71.45,71.45,100.0,71.45,100.0,71.45,100.0,100.0



PRIMERAS FILAS DEL DATASET PREINTEGRADO


,fecha,estacion_geo,codigo_meteo,estacion_meteo_asignada,distancia_meteo_m,distancia_meteo_km,TM,TN,TX,HRM,...,HRX,PM,PN,PX,PPT,RS24h,DVM10,DVVX10,VVM10,VVX10
0,2018-01-01,Ciutadella,X2,Zoo,112.275603,0.112276,13.0,9.8,15.5,44.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2018-01-01,Eixample,X4,El Raval,1338.783004,1.338783,13.1,10.2,15.8,44.0,...,NaN,NaN,NaN,NaN,0.0,NaN,304.0,NaN,5.1,17.2
2,2018-01-01,Gràcia,X4,El Raval,2368.290681,2.368291,13.1,10.2,15.8,44.0,...,NaN,NaN,NaN,NaN,0.0,NaN,304.0,NaN,5.1,17.2
3,2018-01-01,Observatori Fabra,D5,Observatori Fabra,28.187224,0.028187,9.1,6.0,13.0,56.0,...,NaN,NaN,NaN,NaN,0.0,NaN,273.0,NaN,5.0,18.2
4,2018-01-01,Palau Reial,X8,Zona Universitària,374.613600,0.374614,12.4,9.2,15.8,46.0,...,NaN,NaN,NaN,NaN,0.0,NaN,299.0,NaN,5.3,17.3
5,2018-01-01,Poblenou,X2,Zoo,2334.007369,2.334007,13.0,9.8,15.5,44.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2018-01-01,Sants,X8,Zona Universitària,1784.225612,1.784226,12.4,9.2,15.8,46.0,...,NaN,NaN,NaN,NaN,0.0,NaN,299.0,NaN,5.3,17.3
7,2018-01-01,Vall d'Hebron,D5,Observatori Fabra,2170.685620,2.170686,9.1,6.0,13.0,56.0,...,NaN,NaN,NaN,NaN,0.0,NaN,273.0,NaN,5.0,18.2
8,2018-01-02,Ciutadella,X2,Zoo,112.275603,0.112276,13.9,8.7,17.6,57.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2018-01-02,Eixample,X4,El Raval,1338.783004,1.338783,14.6,11.5,18.2,55.0,...,NaN,NaN,NaN,NaN,0.0,NaN,296.0,NaN,2.6,11.6



PREINTEGRACIÓN METEOROLÓGICA 3.4 FINALIZADA

✓ Registros: 20,272
✓ Localizaciones de contaminación: 8
✓ Estaciones meteorológicas de origen: 4
✓ Sin duplicados en fecha + estacion_geo.
✓ Nulos meteorológicos estructurales conservados.
✓ No se ha modificado ningún archivo de origen.

✓ Archivo generado:
/content/drive/MyDrive/TFM/11_Machine_Learning_Temporal/PREINTEGRACION/08_Meteorologia/df_meteorologia_preintegrado.csv

✓ Dataset preparado para su integración con el checkpoint 01 del dataset maestro.


#### Resultado — Preintegración meteorológica

La preintegración temporal y espacial del bloque meteorológico se completa correctamente.

A partir de la correspondencia espacial definida previamente, las observaciones meteorológicas diarias se trasladan a las **8 localizaciones físicas de contaminación atmosférica**, conservando la estación meteorológica de origen y la distancia entre ambas redes.

El dataset resultante contiene:

- **20.272 registros**;
- **21 variables**;
- **8 estaciones de contaminación**;
- **4 estaciones meteorológicas de origen**;
- **2.557 fechas diferentes**;
- **0 duplicados** en la clave `fecha + estacion_geo`.

La cobertura temporal depende de la estación meteorológica asignada. Las estaciones asociadas a `X2` (`Ciutadella` y `Poblenou`) presentan datos hasta el **30/09/2024**, mientras que el resto mantiene cobertura hasta el **31/12/2024**.

Los valores ausentes detectados durante la auditoría se conservan sin imputación, al considerarse ausencias estructurales asociadas a la disponibilidad de determinadas variables y estaciones.

El archivo generado se almacena como:

`df_meteorologia_preintegrado.csv`

dentro de:

`11_Machine_Learning_Temporal/PREINTEGRACION/08_Meteorologia/`

> **Conclusión:** el bloque meteorológico queda preparado para su integración con el primer checkpoint del dataset maestro, manteniendo la trazabilidad espacial y temporal de los datos originales.

### 3.5. Integración de meteorología con el dataset maestro — Checkpoint 02

Una vez completada la preintegración meteorológica, se incorporan las variables meteorológicas al dataset maestro de contaminación atmosférica.

La integración se realiza mediante la clave compuesta:

`fecha + estacion_geo`

donde `estacion_geo` representa la localización física homogeneizada de cada estación de contaminación.

Se utiliza un `left join` tomando como referencia el dataset maestro de contaminación, de forma que se conserva la totalidad de sus registros aunque para alguna combinación fecha–estación no exista información meteorológica disponible.

Durante la integración se comprueba:

- la unicidad de `fecha + estacion_geo` en el bloque meteorológico;
- la conservación exacta del número de registros del dataset maestro;
- la ausencia de multiplicación de filas;
- la cobertura meteorológica obtenida tras el cruce;
- las combinaciones fecha–estación sin correspondencia meteorológica;
- la disponibilidad de cada variable meteorológica en el dataset integrado;
- la conservación de la trazabilidad de la estación meteorológica asignada.

El resultado constituye el **Checkpoint 02** del proceso incremental de integración.

> **Objetivo:** incorporar de forma controlada la dimensión meteorológica al dataset maestro sin alterar la estructura original de las observaciones de contaminación atmosférica.

In [ ]:
# ============================================================
# 3.5. INTEGRACIÓN METEOROLÓGICA CON EL DATASET MAESTRO
# CHECKPOINT 02
# ============================================================

import pandas as pd
from pathlib import Path


# ============================================================
# 1. DEFINIR RUTAS
# ============================================================

ruta_integracion = Path(
    "/content/drive/MyDrive/TFM/"
    "11_Machine_Learning_Temporal/"
    "INTEGRACION"
)

ruta_maestro_01 = (
    ruta_integracion /
    "01_maestro_contaminacion.csv"
)

ruta_meteo_pre = Path(
    "/content/drive/MyDrive/TFM/"
    "11_Machine_Learning_Temporal/"
    "PREINTEGRACION/"
    "08_Meteorologia/"
    "df_meteorologia_preintegrado.csv"
)

ruta_maestro_02 = (
    ruta_integracion /
    "02_maestro_contaminacion_meteorologia.csv"
)


# ============================================================
# 2. COMPROBAR EXISTENCIA DE ARCHIVOS
# ============================================================

if not ruta_integracion.exists():
    raise FileNotFoundError(
        "No existe la carpeta INTEGRACION:\n"
        f"{ruta_integracion}"
    )

if not ruta_maestro_01.exists():
    raise FileNotFoundError(
        "No se encuentra el Checkpoint 01:\n"
        f"{ruta_maestro_01}"
    )

if not ruta_meteo_pre.exists():
    raise FileNotFoundError(
        "No se encuentra el dataset meteorológico "
        "preintegrado:\n"
        f"{ruta_meteo_pre}"
    )


# ============================================================
# 3. CARGAR DATASETS
# ============================================================

df_maestro_01 = pd.read_csv(
    ruta_maestro_01
)

df_meteo_pre = pd.read_csv(
    ruta_meteo_pre
)


# ============================================================
# 4. INFORMACIÓN INICIAL
# ============================================================

print("=" * 80)
print("CHECKPOINT 02 — INTEGRACIÓN METEOROLÓGICA")
print("=" * 80)

print("\nDataset maestro de origen:")
print(ruta_maestro_01)

print("\nDataset meteorológico:")
print(ruta_meteo_pre)

print(
    f"\n✓ Filas maestro 01: "
    f"{len(df_maestro_01):,}"
)

print(
    f"✓ Columnas maestro 01: "
    f"{df_maestro_01.shape[1]}"
)

print(
    f"\n✓ Filas meteorología: "
    f"{len(df_meteo_pre):,}"
)

print(
    f"✓ Columnas meteorología: "
    f"{df_meteo_pre.shape[1]}"
)


# ============================================================
# 5. COMPROBAR COLUMNAS DE INTEGRACIÓN
# ============================================================

clave_integracion = [
    "fecha",
    "estacion_geo"
]

faltantes_maestro = [
    columna
    for columna in clave_integracion
    if columna not in df_maestro_01.columns
]

faltantes_meteo = [
    columna
    for columna in clave_integracion
    if columna not in df_meteo_pre.columns
]

if faltantes_maestro:
    raise KeyError(
        "Faltan columnas de integración "
        "en el maestro 01:\n"
        f"{faltantes_maestro}"
    )

if faltantes_meteo:
    raise KeyError(
        "Faltan columnas de integración "
        "en meteorología:\n"
        f"{faltantes_meteo}"
    )


# ============================================================
# 6. NORMALIZAR FECHAS
# ============================================================

df_maestro_01["fecha"] = pd.to_datetime(
    df_maestro_01["fecha"],
    errors="coerce"
)

df_meteo_pre["fecha"] = pd.to_datetime(
    df_meteo_pre["fecha"],
    errors="coerce"
)

fechas_invalidas_maestro = (
    df_maestro_01["fecha"]
    .isna()
    .sum()
)

fechas_invalidas_meteo = (
    df_meteo_pre["fecha"]
    .isna()
    .sum()
)

if fechas_invalidas_maestro > 0:
    raise ValueError(
        f"El maestro contiene "
        f"{fechas_invalidas_maestro:,} fechas inválidas."
    )

if fechas_invalidas_meteo > 0:
    raise ValueError(
        f"Meteorología contiene "
        f"{fechas_invalidas_meteo:,} fechas inválidas."
    )


# ============================================================
# 7. COMPROBAR UNICIDAD DEL BLOQUE METEOROLÓGICO
# ============================================================

duplicados_meteo = (
    df_meteo_pre
    .duplicated(
        subset=clave_integracion
    )
    .sum()
)

print("\n" + "=" * 80)
print("CONTROL PREVIO DE LA CLAVE DE INTEGRACIÓN")
print("=" * 80)

print(
    f"\n✓ Duplicados meteorología "
    f"fecha + estacion_geo: "
    f"{duplicados_meteo:,}"
)

if duplicados_meteo > 0:
    raise ValueError(
        "El bloque meteorológico no es único "
        "en fecha + estacion_geo. "
        "Se detiene la integración."
    )


# ============================================================
# 8. VARIABLES METEOROLÓGICAS
# ============================================================

variables_meteo = [
    "TM",
    "TN",
    "TX",
    "HRM",
    "HRN",
    "HRX",
    "PM",
    "PN",
    "PX",
    "PPT",
    "RS24h",
    "DVM10",
    "DVVX10",
    "VVM10",
    "VVX10"
]

columnas_trazabilidad_meteo = [
    "codigo_meteo",
    "estacion_meteo_asignada",
    "distancia_meteo_m",
    "distancia_meteo_km"
]

columnas_meteo_integrar = (
    clave_integracion
    + columnas_trazabilidad_meteo
    + variables_meteo
)

faltantes_variables = [
    columna
    for columna in columnas_meteo_integrar
    if columna not in df_meteo_pre.columns
]

if faltantes_variables:
    raise KeyError(
        "Faltan variables necesarias "
        "en el bloque meteorológico:\n"
        f"{faltantes_variables}"
    )


# ============================================================
# 9. PREPARAR BLOQUE METEOROLÓGICO PARA EL MERGE
# ============================================================

df_meteo_merge = (
    df_meteo_pre[
        columnas_meteo_integrar
    ]
    .copy()
)


# ============================================================
# 10. GUARDAR DIMENSIONES ANTES DEL MERGE
# ============================================================

filas_antes = len(
    df_maestro_01
)

columnas_antes = (
    df_maestro_01.shape[1]
)


# ============================================================
# 11. INTEGRAR METEOROLOGÍA
# ============================================================
# El maestro contiene múltiples contaminantes para una misma
# combinación fecha + estacion_geo.
#
# Meteorología contiene como máximo una fila por:
# fecha + estacion_geo
#
# Por tanto, la relación correcta es MANY-TO-ONE.
# ============================================================

df_maestro_02 = pd.merge(
    df_maestro_01,
    df_meteo_merge,
    on=[
        "fecha",
        "estacion_geo"
    ],
    how="left",
    validate="many_to_one",
    indicator=True
)


# ============================================================
# 12. CONTROL DE FILAS DESPUÉS DEL MERGE
# ============================================================

filas_despues = len(
    df_maestro_02
)

columnas_despues = (
    df_maestro_02.shape[1]
)

print("\n" + "=" * 80)
print("CONTROL DE DIMENSIONES")
print("=" * 80)

print(
    f"\nFilas antes del merge: "
    f"{filas_antes:,}"
)

print(
    f"Filas después del merge: "
    f"{filas_despues:,}"
)

print(
    f"\nColumnas antes del merge: "
    f"{columnas_antes}"
)

print(
    f"Columnas después del merge: "
    f"{columnas_despues}"
)

if filas_antes != filas_despues:
    raise ValueError(
        "ERROR: el número de filas del maestro "
        "ha cambiado durante la integración.\n"
        f"Antes: {filas_antes:,}\n"
        f"Después: {filas_despues:,}"
    )

print(
    "\n✓ Número de registros conservado."
)


# ============================================================
# 13. CONTROL DEL RESULTADO DEL MERGE
# ============================================================

resultado_merge = (
    df_maestro_02["_merge"]
    .value_counts()
)

print("\n" + "=" * 80)
print("RESULTADO DEL CRUCE")
print("=" * 80)

print(
    resultado_merge
)


# ============================================================
# 14. FILAS CON / SIN CORRESPONDENCIA METEOROLÓGICA
# ============================================================

filas_con_meteo = (
    df_maestro_02["_merge"]
    .eq("both")
    .sum()
)

filas_sin_meteo = (
    df_maestro_02["_merge"]
    .eq("left_only")
    .sum()
)

porcentaje_con_meteo = (
    filas_con_meteo
    / filas_despues
    * 100
)

porcentaje_sin_meteo = (
    filas_sin_meteo
    / filas_despues
    * 100
)

print("\n" + "=" * 80)
print("COBERTURA DEL CRUCE METEOROLÓGICO")
print("=" * 80)

print(
    f"\n✓ Filas con correspondencia meteorológica: "
    f"{filas_con_meteo:,} "
    f"({porcentaje_con_meteo:.2f} %)"
)

print(
    f"✓ Filas sin correspondencia meteorológica: "
    f"{filas_sin_meteo:,} "
    f"({porcentaje_sin_meteo:.2f} %)"
)


# ============================================================
# 15. ANALIZAR COMBINACIONES SIN METEOROLOGÍA
# ============================================================

if filas_sin_meteo > 0:

    sin_meteo = (
        df_maestro_02.loc[
            df_maestro_02["_merge"] == "left_only",
            [
                "fecha",
                "estacion_geo"
            ]
        ]
        .drop_duplicates()
        .sort_values(
            [
                "fecha",
                "estacion_geo"
            ]
        )
    )

    print("\n" + "=" * 80)
    print("COMBINACIONES FECHA-ESTACIÓN SIN METEOROLOGÍA")
    print("=" * 80)

    print(
        f"\nCombinaciones únicas sin meteorología: "
        f"{len(sin_meteo):,}"
    )

    print("\nPrimeros casos:")

    display(
        sin_meteo.head(20)
    )

else:

    print(
        "\n✓ Todas las filas del maestro "
        "encuentran correspondencia meteorológica."
    )


# ============================================================
# 16. COBERTURA POR ESTACIÓN DE CONTAMINACIÓN
# ============================================================

cobertura_estacion = (
    df_maestro_02
    .assign(
        tiene_meteo=(
            df_maestro_02["_merge"]
            == "both"
        )
    )
    .groupby(
        "estacion_geo"
    )
    .agg(
        registros=(
            "fecha",
            "size"
        ),
        registros_con_meteo=(
            "tiene_meteo",
            "sum"
        )
    )
)

cobertura_estacion[
    "porcentaje_con_meteo"
] = (
    cobertura_estacion[
        "registros_con_meteo"
    ]
    /
    cobertura_estacion[
        "registros"
    ]
    * 100
).round(2)

cobertura_estacion = (
    cobertura_estacion
    .reset_index()
)

print("\n" + "=" * 80)
print("COBERTURA METEOROLÓGICA POR ESTACIÓN")
print("=" * 80)

display(
    cobertura_estacion
)


# ============================================================
# 17. DISPONIBILIDAD DE VARIABLES METEOROLÓGICAS
# EN EL DATASET MAESTRO
# ============================================================

nulos_meteo_maestro = (
    df_maestro_02[
        variables_meteo
    ]
    .isna()
    .sum()
)

porcentaje_nulos_meteo_maestro = (
    df_maestro_02[
        variables_meteo
    ]
    .isna()
    .mean()
    .mul(100)
    .round(2)
)

tabla_nulos_meteo_maestro = pd.DataFrame({
    "n_nulos": nulos_meteo_maestro,
    "porcentaje_nulos": porcentaje_nulos_meteo_maestro
})

print("\n" + "=" * 80)
print("NULOS METEOROLÓGICOS EN EL DATASET MAESTRO")
print("=" * 80)

display(
    tabla_nulos_meteo_maestro
)


# ============================================================
# 18. CONTROL DE TRAZABILIDAD METEOROLÓGICA
# ============================================================

print("\n" + "=" * 80)
print("TRAZABILIDAD DE LA ASIGNACIÓN METEOROLÓGICA")
print("=" * 80)

tabla_trazabilidad = (
    df_maestro_02[
        [
            "estacion_geo",
            "codigo_meteo",
            "estacion_meteo_asignada",
            "distancia_meteo_km"
        ]
    ]
    .dropna(
        subset=[
            "codigo_meteo"
        ]
    )
    .drop_duplicates()
    .sort_values(
        "estacion_geo"
    )
)

display(
    tabla_trazabilidad
)


# ============================================================
# 19. ELIMINAR INDICADOR AUXILIAR DEL MERGE
# ============================================================

df_maestro_02 = (
    df_maestro_02
    .drop(
        columns=[
            "_merge"
        ]
    )
)


# ============================================================
# 20. CONTROL DE DUPLICADOS DEL MAESTRO
# ============================================================
# Conservamos la clave original del bloque de contaminación.
# Debe seguir siendo única después de añadir meteorología.
# ============================================================

clave_maestro = [
    "fecha",
    "estacion_fisica",
    "contaminant"
]

faltantes_clave_maestro = [
    columna
    for columna in clave_maestro
    if columna not in df_maestro_02.columns
]

if faltantes_clave_maestro:
    raise KeyError(
        "Faltan columnas de la clave original "
        "del maestro:\n"
        f"{faltantes_clave_maestro}"
    )

duplicados_maestro = (
    df_maestro_02
    .duplicated(
        subset=clave_maestro
    )
    .sum()
)

print("\n" + "=" * 80)
print("CONTROL DE UNICIDAD DEL MAESTRO")
print("=" * 80)

print(
    f"\n✓ Duplicados "
    f"fecha + estacion_fisica + contaminant: "
    f"{duplicados_maestro:,}"
)

if duplicados_maestro > 0:
    raise ValueError(
        "La integración ha introducido duplicados "
        "en la clave original del maestro."
    )


# ============================================================
# 21. GUARDAR CHECKPOINT 02
# ============================================================

df_maestro_02.to_csv(
    ruta_maestro_02,
    index=False,
    encoding="utf-8"
)


# ============================================================
# 22. VERIFICAR ARCHIVO EXPORTADO
# ============================================================

if not ruta_maestro_02.exists():
    raise FileNotFoundError(
        "No se ha generado correctamente "
        "el Checkpoint 02."
    )

df_control_02 = pd.read_csv(
    ruta_maestro_02
)

if df_control_02.shape != df_maestro_02.shape:
    raise ValueError(
        "Las dimensiones del archivo exportado "
        "no coinciden con el dataset integrado.\n"
        f"En memoria: {df_maestro_02.shape}\n"
        f"Exportado: {df_control_02.shape}"
    )


# ============================================================
# 23. RESUMEN FINAL
# ============================================================

print("\n" + "=" * 80)
print("CHECKPOINT 02 CREADO CORRECTAMENTE")
print("=" * 80)

print(
    f"\n✓ Registros conservados: "
    f"{len(df_maestro_02):,}"
)

print(
    f"✓ Variables disponibles: "
    f"{df_maestro_02.shape[1]}"
)

print(
    f"✓ Localizaciones físicas: "
    f"{df_maestro_02['estacion_geo'].nunique()}"
)

print(
    f"✓ Contaminantes: "
    f"{df_maestro_02['contaminant'].nunique()}"
)

print(
    f"✓ Filas con correspondencia meteorológica: "
    f"{filas_con_meteo:,} "
    f"({porcentaje_con_meteo:.2f} %)"
)

print(
    f"✓ Filas sin correspondencia meteorológica: "
    f"{filas_sin_meteo:,} "
    f"({porcentaje_sin_meteo:.2f} %)"
)

print(
    "✓ Sin multiplicación ni pérdida "
    "de registros del maestro."
)

print(
    "✓ Sin duplicados introducidos "
    "en la clave original."
)

print(
    "✓ Nulos meteorológicos conservados "
    "sin imputación."
)

print(
    "✓ Trazabilidad de la estación "
    "meteorológica conservada."
)

print("\nArchivo maestro generado:")
print(ruta_maestro_02)

CHECKPOINT 02 — INTEGRACIÓN METEOROLÓGICA

Dataset maestro de origen:
/content/drive/MyDrive/TFM/11_Machine_Learning_Temporal/INTEGRACION/01_maestro_contaminacion.csv

Dataset meteorológico:
/content/drive/MyDrive/TFM/11_Machine_Learning_Temporal/PREINTEGRACION/08_Meteorologia/df_meteorologia_preintegrado.csv

✓ Filas maestro 01: 43,642
✓ Columnas maestro 01: 14

✓ Filas meteorología: 20,272
✓ Columnas meteorología: 21

CONTROL PREVIO DE LA CLAVE DE INTEGRACIÓN

✓ Duplicados meteorología fecha + estacion_geo: 0

CONTROL DE DIMENSIONES

Filas antes del merge: 43,642
Filas después del merge: 43,642

Columnas antes del merge: 14
Columnas después del merge: 34

✓ Número de registros conservado.

RESULTADO DEL CRUCE
_merge
both          43215
left_only       427
right_only        0
Name: count, dtype: int64

COBERTURA DEL CRUCE METEOROLÓGICO

✓ Filas con correspondencia meteorológica: 43,215 (99.02 %)
✓ Filas sin correspondencia meteorológica: 427 (0.98 %)

COMBINACIONES FECHA-ESTACIÓN SIN 

,fecha,estacion_geo
41742,2024-10-01,Ciutadella
41751,2024-10-01,Poblenou
41763,2024-10-02,Ciutadella
41772,2024-10-02,Poblenou
41784,2024-10-03,Ciutadella
41793,2024-10-03,Poblenou
41805,2024-10-04,Ciutadella
41814,2024-10-04,Poblenou
41826,2024-10-05,Ciutadella
41835,2024-10-05,Poblenou



COBERTURA METEOROLÓGICA POR ESTACIÓN


,estacion_geo,registros,registros_con_meteo,porcentaje_con_meteo
0,Ciutadella,4428,4244,95.84
1,Eixample,6975,6975,100.00
2,Gràcia,6120,6120,100.00
3,Observatori Fabra,440,440,100.00
4,Palau Reial,7359,7359,100.00
5,Poblenou,5675,5432,95.72
6,Sants,6163,6163,100.00
7,Vall d'Hebron,6482,6482,100.00



NULOS METEOROLÓGICOS EN EL DATASET MAESTRO


,n_nulos,porcentaje_nulos
TM,427,0.98
TN,427,0.98
TX,427,0.98
HRM,427,0.98
HRN,7568,17.34
HRX,7568,17.34
PM,15904,36.44
PN,15904,36.44
PX,15904,36.44
PPT,10103,23.15



TRAZABILIDAD DE LA ASIGNACIÓN METEOROLÓGICA


,estacion_geo,codigo_meteo,estacion_meteo_asignada,distancia_meteo_km
7,Ciutadella,X2,Zoo,0.112276
3,Eixample,X4,El Raval,1.338783
4,Gràcia,X4,El Raval,2.368291
396,Observatori Fabra,D5,Observatori Fabra,0.028187
12,Palau Reial,X8,Zona Universitària,0.374614
0,Poblenou,X2,Zoo,2.334007
2,Sants,X8,Zona Universitària,1.784226
9,Vall d'Hebron,D5,Observatori Fabra,2.170686



CONTROL DE UNICIDAD DEL MAESTRO

✓ Duplicados fecha + estacion_fisica + contaminant: 0

CHECKPOINT 02 CREADO CORRECTAMENTE

✓ Registros conservados: 43,642
✓ Variables disponibles: 33
✓ Localizaciones físicas: 8
✓ Contaminantes: 4
✓ Filas con correspondencia meteorológica: 43,215 (99.02 %)
✓ Filas sin correspondencia meteorológica: 427 (0.98 %)
✓ Sin multiplicación ni pérdida de registros del maestro.
✓ Sin duplicados introducidos en la clave original.
✓ Nulos meteorológicos conservados sin imputación.
✓ Trazabilidad de la estación meteorológica conservada.

Archivo maestro generado:
/content/drive/MyDrive/TFM/11_Machine_Learning_Temporal/INTEGRACION/02_maestro_contaminacion_meteorologia.csv


/tmp/ipykernel_949/1737455644.py:683: DtypeWarning: Columns (1,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_control_02 = pd.read_csv(


#### Resultado — Checkpoint 02: contaminación atmosférica + meteorología

La integración meteorológica con el dataset maestro se completa correctamente mediante la clave `fecha + estacion_geo`.

El proceso conserva íntegramente la granularidad original del bloque de contaminación atmosférica:

- **43.642 registros** antes y después de la integración;
- **33 variables** en el nuevo dataset maestro;
- **8 localizaciones físicas**;
- **4 contaminantes**;
- **0 duplicados** introducidos en la clave `fecha + estacion_fisica + contaminant`.

La correspondencia meteorológica alcanza el **99,02 % de los registros** del maestro. Las **427 observaciones sin correspondencia (0,98 %)** se concentran en `Ciutadella` y `Poblenou` durante el último trimestre de 2024, debido a que ambas estaciones se encuentran asociadas al observatorio meteorológico `X2`, cuya serie finaliza el 30/09/2024.

Los valores ausentes propios de determinadas variables meteorológicas se conservan sin imputación durante esta fase, manteniendo la estructura real de disponibilidad de las fuentes.

El segundo checkpoint se almacena como:

`02_maestro_contaminacion_meteorologia.csv`

dentro de la carpeta:

`11_Machine_Learning_Temporal/INTEGRACION/`

> **Conclusión:** la dimensión meteorológica queda incorporada al dataset maestro sin pérdida ni multiplicación de registros, conservando además la trazabilidad de la estación meteorológica utilizada para cada localización de contaminación.

### 3.6. Exportación del dataset maestro tras la integración meteorológica

Una vez validada la incorporación de las variables meteorológicas al dataset maestro, se guarda una copia del resultado dentro del bloque de preintegración meteorológica.

Esta exportación permite conservar un producto final específico del bloque, independiente del sistema incremental de checkpoints almacenado en la carpeta `INTEGRACION`.

El archivo conserva los **43.642 registros** del dataset maestro y las **33 variables** disponibles tras incorporar la información meteorológica, sin pérdida ni multiplicación de observaciones.

> **Objetivo:** conservar una copia trazable y reproducible del dataset resultante al cierre del bloque meteorológico.

In [ ]:
# ============================================================
# 3.6. EXPORTACIÓN DEL DATASET MAESTRO
# TRAS LA INTEGRACIÓN METEOROLÓGICA
# ============================================================

import pandas as pd
from pathlib import Path


# ============================================================
# 1. DEFINIR RUTAS
# ============================================================

# Checkpoint oficial generado en 3.5
ruta_checkpoint_02 = Path(
    "/content/drive/MyDrive/TFM/"
    "11_Machine_Learning_Temporal/"
    "INTEGRACION/"
    "02_maestro_contaminacion_meteorologia.csv"
)

# Carpeta correspondiente al bloque meteorológico
ruta_meteo = Path(
    "/content/drive/MyDrive/TFM/"
    "11_Machine_Learning_Temporal/"
    "PREINTEGRACION/"
    "08_Meteorologia"
)

# Copia de cierre del bloque meteorológico
ruta_salida_meteo = (
    ruta_meteo /
    "df_maestro_contaminacion_meteorologia.csv"
)


# ============================================================
# 2. COMPROBAR EXISTENCIA DE RUTAS
# ============================================================

if not ruta_checkpoint_02.exists():
    raise FileNotFoundError(
        "No se encuentra el Checkpoint 02:\n"
        f"{ruta_checkpoint_02}"
    )

if not ruta_meteo.exists():
    raise FileNotFoundError(
        "No existe la carpeta de meteorología:\n"
        f"{ruta_meteo}"
    )


# ============================================================
# 3. CARGAR CHECKPOINT 02
# ============================================================
# low_memory=False evita el DtypeWarning observado durante
# la lectura de algunas columnas con tipos mixtos.
# No modifica los datos del archivo.
# ============================================================

df_maestro_meteo = pd.read_csv(
    ruta_checkpoint_02,
    low_memory=False
)


# ============================================================
# 4. CONTROL PREVIO
# ============================================================

print("=" * 80)
print("3.6 — EXPORTACIÓN DEL MAESTRO TRAS METEOROLOGÍA")
print("=" * 80)

print("\nArchivo de origen:")
print(ruta_checkpoint_02)

print(
    f"\n✓ Filas: "
    f"{df_maestro_meteo.shape[0]:,}"
)

print(
    f"✓ Columnas: "
    f"{df_maestro_meteo.shape[1]}"
)

if "estacion_geo" in df_maestro_meteo.columns:
    print(
        f"✓ Localizaciones físicas: "
        f"{df_maestro_meteo['estacion_geo'].nunique()}"
    )

if "contaminant" in df_maestro_meteo.columns:
    print(
        f"✓ Contaminantes: "
        f"{df_maestro_meteo['contaminant'].nunique()}"
    )


# ============================================================
# 5. CONTROL DE DUPLICADOS EN LA CLAVE DEL MAESTRO
# ============================================================

clave_maestro = [
    "fecha",
    "estacion_fisica",
    "contaminant"
]

faltantes_clave = [
    columna
    for columna in clave_maestro
    if columna not in df_maestro_meteo.columns
]

if faltantes_clave:
    raise KeyError(
        "Faltan columnas de la clave del maestro:\n"
        f"{faltantes_clave}"
    )

duplicados_maestro = (
    df_maestro_meteo
    .duplicated(
        subset=clave_maestro
    )
    .sum()
)

print(
    f"\n✓ Duplicados "
    f"fecha + estacion_fisica + contaminant: "
    f"{duplicados_maestro:,}"
)

if duplicados_maestro > 0:
    raise ValueError(
        "El Checkpoint 02 contiene duplicados "
        "en la clave original del maestro. "
        "Se detiene la exportación."
    )


# ============================================================
# 6. COMPROBAR PRESENCIA DEL BLOQUE METEOROLÓGICO
# ============================================================

variables_meteo = [
    "TM",
    "TN",
    "TX",
    "HRM",
    "HRN",
    "HRX",
    "PM",
    "PN",
    "PX",
    "PPT",
    "RS24h",
    "DVM10",
    "DVVX10",
    "VVM10",
    "VVX10"
]

variables_meteo_presentes = [
    variable
    for variable in variables_meteo
    if variable in df_maestro_meteo.columns
]

variables_meteo_faltantes = [
    variable
    for variable in variables_meteo
    if variable not in df_maestro_meteo.columns
]

print(
    f"\n✓ Variables meteorológicas presentes: "
    f"{len(variables_meteo_presentes)}/"
    f"{len(variables_meteo)}"
)

if variables_meteo_faltantes:
    raise KeyError(
        "Faltan variables meteorológicas "
        "en el Checkpoint 02:\n"
        f"{variables_meteo_faltantes}"
    )


# ============================================================
# 7. GUARDAR COPIA DE CIERRE EN METEOROLOGÍA
# ============================================================

df_maestro_meteo.to_csv(
    ruta_salida_meteo,
    index=False,
    encoding="utf-8"
)


# ============================================================
# 8. COMPROBAR QUE EL ARCHIVO SE HA GENERADO
# ============================================================

if not ruta_salida_meteo.exists():
    raise FileNotFoundError(
        "No se ha generado correctamente "
        "la copia del maestro en meteorología."
    )


# ============================================================
# 9. VOLVER A CARGAR PARA VALIDAR LA EXPORTACIÓN
# ============================================================

df_control_meteo = pd.read_csv(
    ruta_salida_meteo,
    low_memory=False
)


# ============================================================
# 10. COMPROBAR DIMENSIONES
# ============================================================

if df_control_meteo.shape != df_maestro_meteo.shape:
    raise ValueError(
        "Las dimensiones del archivo exportado "
        "no coinciden con el Checkpoint 02.\n"
        f"Origen: {df_maestro_meteo.shape}\n"
        f"Exportado: {df_control_meteo.shape}"
    )


# ============================================================
# 11. COMPROBAR DUPLICADOS TRAS LA EXPORTACIÓN
# ============================================================

duplicados_exportado = (
    df_control_meteo
    .duplicated(
        subset=clave_maestro
    )
    .sum()
)

if duplicados_exportado != duplicados_maestro:
    raise ValueError(
        "El control de duplicados no coincide "
        "tras la exportación."
    )


# ============================================================
# 12. TAMAÑO DEL ARCHIVO
# ============================================================

tamano_mb = (
    ruta_salida_meteo.stat().st_size
    / (1024 ** 2)
)


# ============================================================
# 13. CIERRE
# ============================================================

print("\n" + "=" * 80)
print("BLOQUE METEOROLÓGICO CERRADO CORRECTAMENTE")
print("=" * 80)

print(
    f"\n✓ Registros conservados: "
    f"{df_control_meteo.shape[0]:,}"
)

print(
    f"✓ Variables disponibles: "
    f"{df_control_meteo.shape[1]}"
)

print(
    f"✓ Variables meteorológicas: "
    f"{len(variables_meteo_presentes)}"
)

print(
    f"✓ Duplicados en la clave del maestro: "
    f"{duplicados_exportado:,}"
)

print(
    f"✓ Tamaño del archivo: "
    f"{tamano_mb:.2f} MB"
)

print(
    "✓ Sin pérdida ni multiplicación de registros."
)

print(
    "✓ Integridad del Checkpoint 02 verificada "
    "tras la exportación."
)

print(
    "✓ El archivo oficial de INTEGRACION "
    "no ha sido modificado."
)

print("\n✓ Copia de cierre guardada en:")
print(ruta_salida_meteo)

3.6 — EXPORTACIÓN DEL MAESTRO TRAS METEOROLOGÍA

Archivo de origen:
/content/drive/MyDrive/TFM/11_Machine_Learning_Temporal/INTEGRACION/02_maestro_contaminacion_meteorologia.csv

✓ Filas: 43,642
✓ Columnas: 33
✓ Localizaciones físicas: 8
✓ Contaminantes: 4

✓ Duplicados fecha + estacion_fisica + contaminant: 0

✓ Variables meteorológicas presentes: 15/15

BLOQUE METEOROLÓGICO CERRADO CORRECTAMENTE

✓ Registros conservados: 43,642
✓ Variables disponibles: 33
✓ Variables meteorológicas: 15
✓ Duplicados en la clave del maestro: 0
✓ Tamaño del archivo: 10.74 MB
✓ Sin pérdida ni multiplicación de registros.
✓ Integridad del Checkpoint 02 verificada tras la exportación.
✓ El archivo oficial de INTEGRACION no ha sido modificado.

✓ Copia de cierre guardada en:
/content/drive/MyDrive/TFM/11_Machine_Learning_Temporal/PREINTEGRACION/08_Meteorologia/df_maestro_contaminacion_meteorologia.csv


#### Cierre del bloque 3 — Meteorología

El bloque meteorológico queda completamente preparado, validado e integrado con el dataset maestro.

El proceso ha incluido:

- preparación espacial de las 4 estaciones meteorológicas;
- auditoría del dataset meteorológico diario 2018–2024;
- cálculo de la correspondencia espacial entre las estaciones meteorológicas y las 8 estaciones físicas de contaminación;
- generación del dataset meteorológico preintegrado;
- integración mediante `fecha + estacion_geo`;
- creación del **Checkpoint 02**;
- exportación de una copia específica de cierre del bloque meteorológico.

El dataset maestro resultante contiene:

- **43.642 registros**;
- **33 variables**;
- **15 variables meteorológicas** incorporadas;
- **8 localizaciones físicas**;
- **4 contaminantes**;
- **0 duplicados** en la clave `fecha + estacion_fisica + contaminant`.

La integración meteorológica mantiene la totalidad de las observaciones originales de contaminación y conserva los valores ausentes estructurales sin imputación.

Los archivos de referencia resultantes son:

- `INTEGRACION/02_maestro_contaminacion_meteorologia.csv`
- `PREINTEGRACION/08_Meteorologia/df_maestro_contaminacion_meteorologia.csv`

> **Conclusión:** el bloque meteorológico queda cerrado y el Checkpoint 02 se establece como nuevo dataset maestro de partida para la incorporación de la siguiente fuente de información.

# 4 · INTEGRACIÓN DEL TRÁFICO RODADO Y AFOROS

El segundo bloque explicativo incorpora información relativa al **tráfico rodado**, uno de los principales componentes de la movilidad urbana y una fuente relevante de emisiones atmosféricas.

A diferencia de las fuentes estrictamente temporales, los datos de tráfico requieren una caracterización espacial de la relación entre los puntos de aforo y las estaciones de calidad del aire.

El bloque comprende la construcción y validación del catálogo histórico de aforos, el análisis de su distribución espacial respecto a las estaciones de contaminación y la generación de indicadores diarios de presión de tráfico para diferentes entornos espaciales.

Además de las variables de intensidad, se conservan indicadores de cobertura, número de aforos y distancia, permitiendo evaluar posteriormente la representatividad espacial de las estimaciones obtenidas.

Tras validar su disponibilidad temporal y espacial, los indicadores se incorporan al dataset maestro.

El bloque finaliza con la generación del **Checkpoint 03**.

> **Objetivo del bloque:** caracterizar la presión del tráfico rodado en el entorno de las estaciones de calidad del aire mediante indicadores diarios espacialmente contextualizados.

## 4.1. Construcción del catálogo espacial histórico de aforos

La incorporación del tráfico rodado al dataset maestro requiere disponer de una referencia espacial fiable para los puntos de aforo utilizados durante el periodo **2018–2024**.

Los archivos de intensidad de tráfico contienen el identificador `Id_aforament`, que permite mantener la trazabilidad temporal de las observaciones, pero no incorporan directamente coordenadas ni geometrías que permitan localizar espacialmente cada punto de medida.

Además, la red de aforos no permanece constante durante todo el periodo analizado. La auditoría inicial muestra un incremento progresivo del número de identificadores disponibles, desde **613 puntos de aforo en 2018 hasta 838 en 2024**. Por este motivo, no se adopta un catálogo correspondiente a un único año como representación de toda la serie histórica.

Para reconstruir la componente espacial se utilizan dos grupos de archivos complementarios:

- los archivos anuales `aforament_detall_valor`, que contienen las observaciones de intensidad de tráfico y permiten identificar los `Id_aforament` utilizados en cada año;
- los archivos anuales `aforament_descripcio`, almacenados en la subcarpeta `GEOLOCALIZACION_AFOROS`, destinados a recuperar la información descriptiva y espacial asociada a dichos identificadores.

La construcción del catálogo espacial histórico se organiza en cuatro etapas:

### 4.1.1. Auditoría de los archivos anuales de intensidad de tráfico

Se analiza la estructura de los archivos `aforament_detall_valor` correspondientes a 2018–2024, comprobando su homogeneidad, los identificadores disponibles en cada año y la existencia de posibles variables espaciales.

### 4.1.2. Auditoría de los archivos de descripción y geolocalización

Se analizan los archivos `aforament_descripcio` para identificar las variables de localización, sistema de referencia espacial, información descriptiva disponible y correspondencia con `Id_aforament`.

### 4.1.3. Consolidación del catálogo espacial histórico 2018–2024

A partir de los archivos descriptivos anuales se construirá un catálogo espacial consolidado, comprobando la estabilidad de la localización de cada `Id_aforament` y preservando la trazabilidad temporal cuando existan cambios entre años.

### 4.1.4. Validación de cobertura respecto al dataset diario limpio

Finalmente, se contrastará el catálogo espacial obtenido con los identificadores presentes en `df_aforos_2018_2024_Diario_Limpio.csv`, cuantificando la proporción de observaciones y puntos de aforo que pueden ser correctamente geolocalizados.

En esta fase no se realiza todavía ninguna asignación entre los puntos de aforo y las estaciones de contaminación. La estrategia espacial —proximidad, radios de influencia o métricas ponderadas por distancia— se definirá posteriormente a partir de la distribución real de la red.

> **Objetivo:** construir un catálogo espacial histórico, único, trazable y validado de los puntos de aforo utilizados entre 2018 y 2024, que permita caracterizar posteriormente la presión de tráfico rodado en el entorno de las estaciones de calidad del aire.

In [ ]:
# ============================================================
# 4.1. CONSTRUCCIÓN DEL CATÁLOGO ESPACIAL HISTÓRICO DE AFOROS
# AUDITORÍA INICIAL DE LAS FUENTES DISPONIBLES
# ============================================================

import pandas as pd
from pathlib import Path


# ============================================================
# 1. DEFINIR RUTAS
# ============================================================

ruta_trafico = Path(
    "/content/drive/MyDrive/TFM/"
    "11_Machine_Learning_Temporal/"
    "PREINTEGRACION/"
    "02_Trafico_Rodado_y_Aforos"
)

ruta_geo = (
    ruta_trafico /
    "GEOLOCALIZACION_AFOROS"
)


# ============================================================
# 2. DEFINIR ARCHIVOS
# ============================================================

archivos_detall = {
    anio: ruta_trafico / f"{anio}_aforament_detall_valor.csv"
    for anio in range(2018, 2025)
}

archivos_geo = {
    anio: ruta_geo / f"{anio}_aforament_descripcio.csv"
    for anio in range(2018, 2025)
}

ruta_limpio = (
    ruta_trafico /
    "df_aforos_2018_2024_Diario_Limpio.csv"
)


# ============================================================
# 3. COMPROBAR EXISTENCIA DE CARPETAS
# ============================================================

if not ruta_trafico.exists():
    raise FileNotFoundError(
        "No existe la carpeta de tráfico:\n"
        f"{ruta_trafico}"
    )

if not ruta_geo.exists():
    raise FileNotFoundError(
        "No existe la carpeta de geolocalización:\n"
        f"{ruta_geo}"
    )


# ============================================================
# 4. COMPROBAR EXISTENCIA DE TODOS LOS ARCHIVOS
# ============================================================

archivos_esperados = (
    list(archivos_detall.values())
    + list(archivos_geo.values())
    + [ruta_limpio]
)

archivos_faltantes = [
    ruta
    for ruta in archivos_esperados
    if not ruta.exists()
]

if archivos_faltantes:
    raise FileNotFoundError(
        "Faltan los siguientes archivos:\n\n"
        + "\n".join(
            str(ruta)
            for ruta in archivos_faltantes
        )
    )


# ============================================================
# 5. CARGAR ARCHIVOS ANUALES
# ============================================================

datasets_detall = {}
datasets_geo = {}

for anio in range(2018, 2025):

    datasets_detall[anio] = pd.read_csv(
        archivos_detall[anio],
        low_memory=False
    )

    datasets_geo[anio] = pd.read_csv(
        archivos_geo[anio],
        low_memory=False
    )


# ============================================================
# 6. CARGAR DATASET DIARIO LIMPIO
# ============================================================

df_aforos_limpio = pd.read_csv(
    ruta_limpio,
    low_memory=False
)


# ============================================================
# 7. INFORMACIÓN GENERAL
# ============================================================

print("=" * 80)
print("4.1 — CATÁLOGO ESPACIAL HISTÓRICO DE AFOROS")
print("=" * 80)

print("\nCarpeta de tráfico:")
print(ruta_trafico)

print("\nCarpeta de geolocalización:")
print(ruta_geo)

print(
    f"\n✓ Archivos de intensidad localizados: "
    f"{len(datasets_detall)}"
)

print(
    f"✓ Archivos de geolocalización localizados: "
    f"{len(datasets_geo)}"
)

print("✓ Dataset diario limpio localizado")


# ============================================================
# 8. DIMENSIONES DE LOS ARCHIVOS ANUALES
# ============================================================

resumen = []

for anio in range(2018, 2025):

    df_detall = datasets_detall[anio]
    df_geo = datasets_geo[anio]

    resumen.append({
        "anio": anio,
        "filas_detall": df_detall.shape[0],
        "columnas_detall": df_detall.shape[1],
        "filas_geo": df_geo.shape[0],
        "columnas_geo": df_geo.shape[1]
    })

df_resumen_fuentes = pd.DataFrame(resumen)

print("\n" + "=" * 80)
print("DIMENSIONES DE LAS FUENTES ANUALES")
print("=" * 80)

display(df_resumen_fuentes)


# ============================================================
# 9. ESTRUCTURA DE LOS ARCHIVOS DE INTENSIDAD
# ============================================================

print("\n" + "=" * 80)
print("COLUMNAS — ARCHIVOS DE INTENSIDAD")
print("=" * 80)

for anio, df in datasets_detall.items():

    print(f"\n{anio}:")

    for columna in df.columns:
        print(f"  • {columna}")


# ============================================================
# 10. ESTRUCTURA DE LOS ARCHIVOS DE GEOLOCALIZACIÓN
# ============================================================

print("\n" + "=" * 80)
print("COLUMNAS — ARCHIVOS DE GEOLOCALIZACIÓN")
print("=" * 80)

for anio, df in datasets_geo.items():

    print(f"\n{anio}:")

    for columna in df.columns:
        print(f"  • {columna}")


# ============================================================
# 11. COMPARAR ESQUEMA DE GEOLOCALIZACIÓN ENTRE AÑOS
# ============================================================

columnas_geo_por_anio = {
    anio: set(df.columns)
    for anio, df in datasets_geo.items()
}

columnas_geo_comunes = set.intersection(
    *columnas_geo_por_anio.values()
)

columnas_geo_totales = set.union(
    *columnas_geo_por_anio.values()
)

print("\n" + "=" * 80)
print("ESTABILIDAD DEL ESQUEMA DE GEOLOCALIZACIÓN")
print("=" * 80)

print(
    f"\n✓ Columnas presentes en todos los años: "
    f"{len(columnas_geo_comunes)}"
)

for columna in sorted(columnas_geo_comunes):
    print(f"  • {columna}")

print(
    f"\n✓ Total de columnas diferentes "
    f"detectadas 2018–2024: "
    f"{len(columnas_geo_totales)}"
)


# ============================================================
# 12. BUSCAR VARIABLES ESPACIALES
# ============================================================

terminos_espaciales = [
    "lat",
    "lon",
    "long",
    "coord",
    "utm",
    "etrs",
    "x",
    "y",
    "geometry",
    "geom",
    "wkt",
    "adre",
    "carrer",
    "via",
    "tram",
    "ubic"
]

columnas_espaciales = []

for columna in columnas_geo_totales:

    nombre = str(columna).lower().strip()

    if any(
        termino in nombre
        for termino in terminos_espaciales
    ):
        columnas_espaciales.append(columna)

columnas_espaciales = sorted(
    set(columnas_espaciales)
)

print("\n" + "=" * 80)
print("POSIBLES VARIABLES ESPACIALES")
print("=" * 80)

if columnas_espaciales:

    for columna in columnas_espaciales:
        print(f"  • {columna}")

else:
    print(
        "\n⚠ No se han identificado automáticamente "
        "variables espaciales por su nombre."
    )


# ============================================================
# 13. ANALIZAR Id_aforament POR AÑO
# ============================================================

print("\n" + "=" * 80)
print("IDENTIFICADORES DE AFORO POR AÑO")
print("=" * 80)

resumen_ids = []

for anio in range(2018, 2025):

    df_detall = datasets_detall[anio]
    df_geo = datasets_geo[anio]

    ids_detall = (
        df_detall["Id_aforament"]
        .dropna()
        .astype(str)
        .str.strip()
        if "Id_aforament" in df_detall.columns
        else pd.Series(dtype=str)
    )

    ids_geo = (
        df_geo["Id_aforament"]
        .dropna()
        .astype(str)
        .str.strip()
        if "Id_aforament" in df_geo.columns
        else pd.Series(dtype=str)
    )

    set_detall = set(ids_detall)
    set_geo = set(ids_geo)

    comunes = set_detall & set_geo
    sin_geo = set_detall - set_geo

    cobertura = (
        len(comunes) /
        len(set_detall) * 100
        if len(set_detall) > 0
        else 0
    )

    resumen_ids.append({
        "anio": anio,
        "ids_intensidad": len(set_detall),
        "ids_geolocalizacion": len(set_geo),
        "ids_con_correspondencia": len(comunes),
        "ids_sin_geolocalizacion": len(sin_geo),
        "cobertura_pct": round(cobertura, 2)
    })

df_resumen_ids = pd.DataFrame(resumen_ids)

display(df_resumen_ids)


# ============================================================
# 14. COMPROBAR DUPLICADOS DE Id_aforament
# EN LOS CATÁLOGOS GEO
# ============================================================

print("\n" + "=" * 80)
print("DUPLICADOS DE Id_aforament — GEOLOCALIZACIÓN")
print("=" * 80)

resumen_duplicados = []

for anio, df in datasets_geo.items():

    if "Id_aforament" in df.columns:

        duplicados = (
            df.duplicated(
                subset=["Id_aforament"]
            ).sum()
        )

        resumen_duplicados.append({
            "anio": anio,
            "filas": len(df),
            "ids_unicos":
                df["Id_aforament"].nunique(
                    dropna=True
                ),
            "duplicados_id":
                duplicados
        })

df_duplicados_geo = pd.DataFrame(
    resumen_duplicados
)

display(df_duplicados_geo)


# ============================================================
# 15. MUESTRA DEL CATÁLOGO MÁS RECIENTE
# ============================================================

print("\n" + "=" * 80)
print("MUESTRA DEL CATÁLOGO DE GEOLOCALIZACIÓN — 2024")
print("=" * 80)

display(
    datasets_geo[2024].head(10)
)


# ============================================================
# 16. TIPOS DE DATOS — GEOLOCALIZACIÓN 2024
# ============================================================

print("\n" + "=" * 80)
print("TIPOS DE DATOS — GEOLOCALIZACIÓN 2024")
print("=" * 80)

print(
    datasets_geo[2024].dtypes
)


# ============================================================
# 17. MUESTRA DE VARIABLES ESPACIALES
# ============================================================

columnas_muestra = []

if "Id_aforament" in datasets_geo[2024].columns:
    columnas_muestra.append(
        "Id_aforament"
    )

columnas_muestra += [
    columna
    for columna in columnas_espaciales
    if (
        columna in datasets_geo[2024].columns
        and columna not in columnas_muestra
    )
]

if columnas_muestra:

    print("\n" + "=" * 80)
    print("MUESTRA DE INFORMACIÓN ESPACIAL — 2024")
    print("=" * 80)

    display(
        datasets_geo[2024][
            columnas_muestra
        ].head(20)
    )


# ============================================================
# 18. DATASET DIARIO LIMPIO
# ============================================================

print("\n" + "=" * 80)
print("DATASET DIARIO LIMPIO")
print("=" * 80)

print(
    f"\n✓ Filas: "
    f"{df_aforos_limpio.shape[0]:,}"
)

print(
    f"✓ Columnas: "
    f"{df_aforos_limpio.shape[1]}"
)

if "Id_aforament" in df_aforos_limpio.columns:

    print(
        f"✓ Id_aforament únicos: "
        f"{df_aforos_limpio['Id_aforament'].nunique():,}"
    )

print("\nColumnas:")

for columna in df_aforos_limpio.columns:
    print(f"  • {columna}")


# ============================================================
# 19. CIERRE
# ============================================================

print("\n" + "=" * 80)
print("AUDITORÍA INICIAL 4.1 FINALIZADA")
print("=" * 80)

print(
    "\n✓ Los 7 archivos de intensidad "
    "han sido localizados."
)

print(
    "✓ Los 7 archivos de geolocalización "
    "han sido localizados."
)

print(
    "✓ Se ha comprobado la estructura "
    "de ambas fuentes."
)

print(
    "✓ Se ha analizado la correspondencia "
    "de Id_aforament por año."
)

print(
    "✓ Se han buscado las variables "
    "espaciales disponibles."
)

print(
    "✓ No se ha modificado ni sobrescrito "
    "ningún archivo de origen."
)

print(
    "\n✓ Preparado para consolidar "
    "el catálogo espacial histórico 2018–2024."
)

4.1 — CATÁLOGO ESPACIAL HISTÓRICO DE AFOROS

Carpeta de tráfico:
/content/drive/MyDrive/TFM/11_Machine_Learning_Temporal/PREINTEGRACION/02_Trafico_Rodado_y_Aforos

Carpeta de geolocalización:
/content/drive/MyDrive/TFM/11_Machine_Learning_Temporal/PREINTEGRACION/02_Trafico_Rodado_y_Aforos/GEOLOCALIZACION_AFOROS

✓ Archivos de intensidad localizados: 7
✓ Archivos de geolocalización localizados: 7
✓ Dataset diario limpio localizado

DIMENSIONES DE LAS FUENTES ANUALES


,anio,filas_detall,columnas_detall,filas_geo,columnas_geo
0,2018,36780,6,691,13
1,2019,41100,6,685,13
2,2020,42840,6,714,13
3,2021,43440,6,724,13
4,2022,49435,6,824,13
5,2023,49920,6,832,13
6,2024,50280,6,838,13



COLUMNAS — ARCHIVOS DE INTENSIDAD

2018:
  • Any
  • Id_aforament
  • Mes
  • Codi_tipus_dia
  • Desc_tipus_dia
  • Valor_IMD

2019:
  • Any
  • Id_aforament
  • Mes
  • Codi_tipus_dia
  • Desc_tipus_dia
  • Valor_IMD

2020:
  • Any
  • Id_aforament
  • Mes
  • Codi_tipus_dia
  • Desc_tipus_dia
  • Valor_IMD

2021:
  • Any
  • Id_aforament
  • Mes
  • Codi_tipus_dia
  • Desc_tipus_dia
  • Valor_IMD

2022:
  • Any
  • Id_aforament
  • Mes
  • Codi_tipus_dia
  • Desc_tipus_dia
  • Valor_IMD

2023:
  • Any
  • Id_aforament
  • Mes
  • Codi_tipus_dia
  • Desc_tipus_dia
  • Valor_IMD

2024:
  • Any
  • Id_aforament
  • Mes
  • Codi_tipus_dia
  • Desc_tipus_dia
  • Valor_IMD

COLUMNAS — ARCHIVOS DE GEOLOCALIZACIÓN

2018:
  • Id_aforament
  • Desc_aforament
  • Codi_tipus_aforament
  • Desc_tipus_aforament
  • Num_carrils
  • Codi_districte
  • Codi_barri
  • Codi_tipus_equip_mesura
  • Desc_tipus_equip_mesura
  • Longitud
  • Latitud
  • X_ETRS89
  • Y_ETRS89

2019:
  • Id_aforament
  • Des

,anio,ids_intensidad,ids_geolocalizacion,ids_con_correspondencia,ids_sin_geolocalizacion,cobertura_pct
0,2018,613,691,613,0,100.0
1,2019,685,685,685,0,100.0
2,2020,714,714,714,0,100.0
3,2021,724,724,724,0,100.0
4,2022,824,824,824,0,100.0
5,2023,832,832,832,0,100.0
6,2024,838,838,838,0,100.0



DUPLICADOS DE Id_aforament — GEOLOCALIZACIÓN


...


MUESTRA DEL CATÁLOGO DE GEOLOCALIZACIÓN — 2024


,Id_aforament,Desc_aforament,Codi_tipus_aforament,Desc_tipus_aforament,Num_carrils,Codi_districte,Codi_Barri,Codi_tipus_equip_mesura,Desc_tipus_equip_mesura,Longitud,Latitud,X_ETRS89,Y_ETRS89
0,10001,ARAGÓ - NAVAS (Llobregat),1,Trànsit,2,10,65,1,Espira,2.190735,41.411680,432364.559,4584775.026
1,10002,BAC DE RODA - BOLIVIA (Pujada),1,Trànsit,3,10,71,1,Espira,2.198983,41.411337,433053.504,4584730.520
2,10005,ARAGÓ - BILBAO (Besòs),1,Trànsit,2,10,65,1,Espira,2.191886,41.412286,432461.330,4584841.413
3,10007,CANTÀBRIA - GUIPÚSCOA (Baixada),1,Trànsit,2,10,72,1,Espira,2.201899,41.419286,433305.332,4585610.784
4,10008,CANTÀBRIA - GUIPÚSCOA (Pujada),1,Trànsit,2,10,72,1,Espira,2.201063,41.420043,433236.312,4585695.525
5,10009,GUIPÚSCOA - MARESME (Llobregat),1,Trànsit,2,10,73,1,Espira,2.203021,41.420926,433400.838,4585792.000
6,1001,AV. DIAGONAL - TORRE MELINA (Entrada),1,Trànsit,3,4,21,1,Espira,2.109525,41.383736,425545.349,4581739.280
7,10010,GUIPÚSCOA - PUIGCERDÀ (Besós),1,Trànsit,2,10,73,1,Espira,2.203240,41.420792,433419.001,4585776.911
8,10011,RBLA. PRIM - CONCILI DE TRENTO (Baixada),1,Trànsit,2,10,73,1,Espira,2.206291,41.420903,433674.023,4585786.904
9,10012,RBLA. PRIM - CONCILI DE TRENTO (Pujada),1,Trànsit,2,10,73,1,Espira,2.205820,41.421791,433635.622,4585885.890



TIPOS DE DATOS — GEOLOCALIZACIÓN 2024
Id_aforament                object
Desc_aforament              object
Codi_tipus_aforament         int64
Desc_tipus_aforament        object
Num_carrils                  int64
Codi_districte               int64
Codi_Barri                   int64
Codi_tipus_equip_mesura      int64
Desc_tipus_equip_mesura     object
Longitud                   float64
Latitud                    float64
X_ETRS89                   float64
Y_ETRS89                   float64
dtype: object

MUESTRA DE INFORMACIÓN ESPACIAL — 2024


,Id_aforament,Latitud,Longitud,X_ETRS89,Y_ETRS89
0,10001,41.411680,2.190735,432364.559,4584775.026
1,10002,41.411337,2.198983,433053.504,4584730.520
2,10005,41.412286,2.191886,432461.330,4584841.413
3,10007,41.419286,2.201899,433305.332,4585610.784
4,10008,41.420043,2.201063,433236.312,4585695.525
5,10009,41.420926,2.203021,433400.838,4585792.000
6,1001,41.383736,2.109525,425545.349,4581739.280
7,10010,41.420792,2.203240,433419.001,4585776.911
8,10011,41.420903,2.206291,433674.023,4585786.904
9,10012,41.421791,2.205820,433635.622,4585885.890



DATASET DIARIO LIMPIO

✓ Filas: 1,910,502
✓ Columnas: 9
✓ Id_aforament únicos: 922

Columnas:
  • Fecha
  • Any
  • Mes
  • Id_aforament
  • Codi_tipus_dia
  • Desc_tipus_dia
  • Valor_IMD_original
  • Valor_IMD
  • IMD_imputado

AUDITORÍA INICIAL 4.1 FINALIZADA

✓ Los 7 archivos de intensidad han sido localizados.
✓ Los 7 archivos de geolocalización han sido localizados.
✓ Se ha comprobado la estructura de ambas fuentes.
✓ Se ha analizado la correspondencia de Id_aforament por año.
✓ Se han buscado las variables espaciales disponibles.
✓ No se ha modificado ni sobrescrito ningún archivo de origen.

✓ Preparado para consolidar el catálogo espacial histórico 2018–2024.


#### Resultado — Construcción de la base espacial de aforos

La auditoría conjunta de las fuentes temporales y espaciales confirma la disponibilidad de información suficiente para reconstruir la red histórica de aforos correspondiente al periodo 2018–2024.

Los archivos `aforament_descripcio` contienen, además de `Id_aforament`, información descriptiva del punto de medida, número de carriles, distrito, barrio, tipología de aforo y equipo de medida, así como coordenadas geográficas (`Longitud`, `Latitud`) y proyectadas (`X_ETRS89`, `Y_ETRS89`).

La correspondencia entre los identificadores presentes en los archivos anuales de intensidad y sus respectivos catálogos espaciales alcanza el **100 % en todos los años analizados**, sin detectarse duplicados de `Id_aforament` dentro de los catálogos anuales.

La red presenta una evolución temporal significativa, mientras que el dataset diario consolidado contiene **922 identificadores únicos** a lo largo del periodo completo. Este comportamiento confirma la conveniencia de construir un catálogo histórico a partir de los siete años disponibles en lugar de adoptar la configuración espacial de un único año.

Se detecta únicamente una diferencia menor de nomenclatura entre años en la variable de barrio (`Codi_barri` / `Codi_Barri`), que será normalizada durante la consolidación.

> **Conclusión:** la componente espacial del tráfico presenta cobertura completa respecto a las observaciones de intensidad. Las fuentes quedan preparadas para construir el catálogo histórico consolidado de aforos y analizar posteriormente su distribución respecto a las estaciones de contaminación.

## 4.2. Consolidación y validación del catálogo espacial histórico de aforos

Una vez confirmada la correspondencia completa entre los registros de intensidad de tráfico y los catálogos anuales de geolocalización, se procede a construir una referencia espacial histórica única para la red de aforos del periodo **2018–2024**.

La consolidación se realiza inicialmente mediante la clave `anio_catalogo + Id_aforament`, preservando la configuración de la red correspondiente a cada año y evitando asumir que todos los puntos de medida mantienen necesariamente una localización constante durante todo el periodo.

Durante la auditoría preliminar se detectaron pequeñas diferencias de precisión en algunas coordenadas y determinados valores incompatibles con el sistema proyectado utilizado en el estudio. Por este motivo, antes de generar el catálogo definitivo se incorpora una fase específica de **validación y control de coherencia espacial**.

El procedimiento comprende:

- homogeneización del esquema de los siete catálogos anuales;
- normalización de variables cuya denominación cambia entre años;
- validación de las coordenadas geográficas (`Longitud`, `Latitud`);
- validación de las coordenadas proyectadas (`X_ETRS89`, `Y_ETRS89`);
- detección de coordenadas incompatibles con el ámbito espacial de Barcelona;
- reconstrucción de coordenadas proyectadas a partir de WGS84 cuando las coordenadas ETRS89 originales sean inválidas y las geográficas sean plausibles;
- comparación entre las coordenadas proyectadas originales y las reconstruidas;
- análisis de la estabilidad espacial de cada `Id_aforament` mediante una tolerancia métrica;
- identificación de posibles cambios reales de localización;
- comprobación de la cobertura respecto al dataset diario limpio;
- validación estricta mediante la clave `año + Id_aforament`.

Las coordenadas originales se conservan durante todo el proceso para garantizar la trazabilidad. Las coordenadas validadas se almacenan en variables independientes y serán las utilizadas posteriormente en los cálculos de distancia y presión de tráfico.

No se eliminan registros por diferencias espaciales entre años. Cuando un identificador presente cambios de localización superiores a la tolerancia establecida, se mantiene su posición correspondiente a cada año.

> **Objetivo:** obtener un catálogo espacial histórico validado, trazable y temporalmente consistente de la red de aforos 2018–2024, preparado para calcular posteriormente la relación espacial entre tráfico rodado y estaciones de calidad del aire.

In [ ]:
# ============================================================
# 4.2. CONSOLIDACIÓN Y VALIDACIÓN DEL CATÁLOGO
# ESPACIAL HISTÓRICO DE AFOROS 2018–2024
# ============================================================

import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path


# ============================================================
# 1. RUTAS
# ============================================================

ruta_trafico = Path(
    "/content/drive/MyDrive/TFM/"
    "11_Machine_Learning_Temporal/"
    "PREINTEGRACION/"
    "02_Trafico_Rodado_y_Aforos"
)

ruta_geo = (
    ruta_trafico /
    "GEOLOCALIZACION_AFOROS"
)

ruta_limpio = (
    ruta_trafico /
    "df_aforos_2018_2024_Diario_Limpio.csv"
)

ruta_salida = (
    ruta_geo /
    "catalogo_espacial_aforos_2018_2024.csv"
)


# ============================================================
# 2. ARCHIVOS ANUALES
# ============================================================

archivos_geo = {
    anio: (
        ruta_geo /
        f"{anio}_aforament_descripcio.csv"
    )
    for anio in range(2018, 2025)
}


# ============================================================
# 3. COMPROBAR EXISTENCIA
# ============================================================

archivos_necesarios = (
    list(archivos_geo.values())
    + [ruta_limpio]
)

faltantes = [
    ruta
    for ruta in archivos_necesarios
    if not ruta.exists()
]

if faltantes:
    raise FileNotFoundError(
        "Faltan los siguientes archivos:\n\n"
        + "\n".join(
            str(ruta)
            for ruta in faltantes
        )
    )


# ============================================================
# 4. CARGAR Y HOMOGENEIZAR CATÁLOGOS
# ============================================================

catalogos = []

for anio, ruta_archivo in archivos_geo.items():

    df = pd.read_csv(
        ruta_archivo,
        dtype={"Id_aforament": str},
        low_memory=False
    )

    # --------------------------------------------------------
    # Homogeneizar nomenclatura
    # --------------------------------------------------------

    if "Codi_Barri" in df.columns:
        df = df.rename(
            columns={
                "Codi_Barri": "Codi_barri"
            }
        )

    # --------------------------------------------------------
    # Normalizar identificador
    # --------------------------------------------------------

    df["Id_aforament"] = (
        df["Id_aforament"]
        .astype("string")
        .str.strip()
    )

    # --------------------------------------------------------
    # Incorporar año del catálogo
    # --------------------------------------------------------

    df.insert(
        0,
        "anio_catalogo",
        anio
    )

    catalogos.append(df)


# ============================================================
# 5. CONSOLIDAR LOS SIETE AÑOS
# ============================================================

df_catalogo = pd.concat(
    catalogos,
    ignore_index=True,
    sort=False
)

print("=" * 80)
print("4.2 — CONSOLIDACIÓN Y VALIDACIÓN ESPACIAL DE AFOROS")
print("=" * 80)

print(
    f"\n✓ Registros históricos: "
    f"{len(df_catalogo):,}"
)

print(
    f"✓ Id_aforament únicos: "
    f"{df_catalogo['Id_aforament'].nunique():,}"
)

print(
    f"✓ Periodo: "
    f"{df_catalogo['anio_catalogo'].min()} "
    f"→ "
    f"{df_catalogo['anio_catalogo'].max()}"
)


# ============================================================
# 6. VALIDAR CLAVE HISTÓRICA
# ============================================================

duplicados_clave = (
    df_catalogo
    .duplicated(
        subset=[
            "anio_catalogo",
            "Id_aforament"
        ]
    )
    .sum()
)

print(
    f"\n✓ Duplicados "
    f"año + Id_aforament: "
    f"{duplicados_clave:,}"
)

if duplicados_clave > 0:
    raise ValueError(
        "Existen duplicados en la clave "
        "anio_catalogo + Id_aforament."
    )


# ============================================================
# 7. COMPROBAR VARIABLES ESPACIALES
# ============================================================

columnas_espaciales = [
    "Longitud",
    "Latitud",
    "X_ETRS89",
    "Y_ETRS89"
]

faltantes_espaciales = [
    columna
    for columna in columnas_espaciales
    if columna not in df_catalogo.columns
]

if faltantes_espaciales:
    raise KeyError(
        "Faltan variables espaciales:\n"
        f"{faltantes_espaciales}"
    )


# ============================================================
# 8. CONVERTIR COORDENADAS A NUMÉRICO
# ============================================================

for columna in columnas_espaciales:

    df_catalogo[columna] = pd.to_numeric(
        df_catalogo[columna],
        errors="coerce"
    )


# ============================================================
# 9. CONSERVAR COORDENADAS ORIGINALES
# ============================================================

df_catalogo["Longitud_original"] = (
    df_catalogo["Longitud"]
)

df_catalogo["Latitud_original"] = (
    df_catalogo["Latitud"]
)

df_catalogo["X_ETRS89_original"] = (
    df_catalogo["X_ETRS89"]
)

df_catalogo["Y_ETRS89_original"] = (
    df_catalogo["Y_ETRS89"]
)


# ============================================================
# 10. VALIDAR COORDENADAS GEOGRÁFICAS
# ============================================================
# Se utilizan límites amplios alrededor del municipio
# únicamente como control de plausibilidad.
# No constituyen un recorte administrativo.
# ============================================================

lon_min, lon_max = 1.90, 2.30
lat_min, lat_max = 41.25, 41.55

df_catalogo["wgs84_valida"] = (
    df_catalogo["Longitud"].between(
        lon_min,
        lon_max
    )
    &
    df_catalogo["Latitud"].between(
        lat_min,
        lat_max
    )
)


# ============================================================
# 11. VALIDAR COORDENADAS ETRS89 / UTM 31N
# ============================================================
# Rangos amplios compatibles con Barcelona y su entorno.
# ============================================================

x_min, x_max = 400000, 450000
y_min, y_max = 4560000, 4610000

df_catalogo["etrs89_original_valida"] = (
    df_catalogo["X_ETRS89"].between(
        x_min,
        x_max
    )
    &
    df_catalogo["Y_ETRS89"].between(
        y_min,
        y_max
    )
)


# ============================================================
# 12. RESUMEN DE VALIDEZ ORIGINAL
# ============================================================

print("\n" + "=" * 80)
print("VALIDACIÓN DE COORDENADAS ORIGINALES")
print("=" * 80)

print(
    f"\n✓ Registros con WGS84 plausible: "
    f"{df_catalogo['wgs84_valida'].sum():,}"
    f"/{len(df_catalogo):,}"
)

print(
    f"✓ Registros con ETRS89 original plausible: "
    f"{df_catalogo['etrs89_original_valida'].sum():,}"
    f"/{len(df_catalogo):,}"
)

print(
    f"✓ ETRS89 originales anómalas: "
    f"{(~df_catalogo['etrs89_original_valida']).sum():,}"
)


# ============================================================
# 13. REPROYECTAR WGS84 → ETRS89 / UTM 31N
# ============================================================
# Reconstruimos coordenadas proyectadas para todos los
# registros con WGS84 válida.
# ============================================================

df_wgs_valido = (
    df_catalogo[
        df_catalogo["wgs84_valida"]
    ]
    .copy()
)

gdf_wgs = gpd.GeoDataFrame(
    df_wgs_valido,
    geometry=gpd.points_from_xy(
        df_wgs_valido["Longitud"],
        df_wgs_valido["Latitud"]
    ),
    crs="EPSG:4326"
)

gdf_etrs = gdf_wgs.to_crs(
    "EPSG:25831"
)

df_catalogo[
    "X_ETRS89_recalculada"
] = np.nan

df_catalogo[
    "Y_ETRS89_recalculada"
] = np.nan

df_catalogo.loc[
    gdf_etrs.index,
    "X_ETRS89_recalculada"
] = gdf_etrs.geometry.x.values

df_catalogo.loc[
    gdf_etrs.index,
    "Y_ETRS89_recalculada"
] = gdf_etrs.geometry.y.values


# ============================================================
# 14. COMPARAR ETRS89 ORIGINAL Y RECALCULADA
# ============================================================

df_catalogo["diferencia_etrs_m"] = np.sqrt(
    (
        df_catalogo["X_ETRS89_original"]
        - df_catalogo["X_ETRS89_recalculada"]
    ) ** 2
    +
    (
        df_catalogo["Y_ETRS89_original"]
        - df_catalogo["Y_ETRS89_recalculada"]
    ) ** 2
)


print("\n" + "=" * 80)
print("COHERENCIA WGS84 ↔ ETRS89")
print("=" * 80)

diferencias_validas = (
    df_catalogo.loc[
        df_catalogo[
            "etrs89_original_valida"
        ]
        &
        df_catalogo[
            "wgs84_valida"
        ],
        "diferencia_etrs_m"
    ]
    .dropna()
)

if len(diferencias_validas) > 0:

    print(
        f"\nMediana diferencia: "
        f"{diferencias_validas.median():.3f} m"
    )

    print(
        f"Percentil 95: "
        f"{diferencias_validas.quantile(0.95):.3f} m"
    )

    print(
        f"Máxima diferencia: "
        f"{diferencias_validas.max():.3f} m"
    )


# ============================================================
# 15. CONSTRUIR COORDENADAS ETRS89 VALIDADAS
# ============================================================
# Criterio:
#
# - Si ETRS89 original es plausible, se conserva.
# - Si ETRS89 original es inválida pero WGS84 es válida,
#   se sustituye por la coordenada reproyectada.
# - Si ambas fuentes son inválidas, el registro queda marcado
#   para revisión y NO se inventa ninguna coordenada.
# ============================================================

df_catalogo["X_ETRS89_validada"] = np.where(
    df_catalogo["etrs89_original_valida"],
    df_catalogo["X_ETRS89_original"],
    np.where(
        df_catalogo["wgs84_valida"],
        df_catalogo["X_ETRS89_recalculada"],
        np.nan
    )
)

df_catalogo["Y_ETRS89_validada"] = np.where(
    df_catalogo["etrs89_original_valida"],
    df_catalogo["Y_ETRS89_original"],
    np.where(
        df_catalogo["wgs84_valida"],
        df_catalogo["Y_ETRS89_recalculada"],
        np.nan
    )
)


# ============================================================
# 16. TRAZABILIDAD DE LA COORDENADA UTILIZADA
# ============================================================

df_catalogo["fuente_coordenada"] = np.select(
    [
        df_catalogo["etrs89_original_valida"],
        (
            ~df_catalogo["etrs89_original_valida"]
            &
            df_catalogo["wgs84_valida"]
        )
    ],
    [
        "ETRS89_original",
        "WGS84_reproyectada"
    ],
    default="sin_coordenada_valida"
)


# ============================================================
# 17. CONTROL DE COORDENADAS FINALES
# ============================================================

sin_coordenada_final = (
    df_catalogo[
        [
            "X_ETRS89_validada",
            "Y_ETRS89_validada"
        ]
    ]
    .isna()
    .any(axis=1)
)

print("\n" + "=" * 80)
print("COORDENADAS VALIDADAS")
print("=" * 80)

print(
    f"\n✓ ETRS89 originales conservadas: "
    f"{(df_catalogo['fuente_coordenada'] == 'ETRS89_original').sum():,}"
)

print(
    f"✓ ETRS89 reconstruidas desde WGS84: "
    f"{(df_catalogo['fuente_coordenada'] == 'WGS84_reproyectada').sum():,}"
)

print(
    f"✓ Registros sin coordenada válida: "
    f"{sin_coordenada_final.sum():,}"
)


# ============================================================
# 18. ESTABILIDAD ESPACIAL HISTÓRICA
# ============================================================
# Se utiliza una tolerancia de 5 metros.
#
# Diferencias inferiores o iguales a 5 m se consideran
# equivalentes a efectos de estabilidad espacial.
# ============================================================

TOLERANCIA_ESTABILIDAD_M = 5.0

df_estabilidad = (
    df_catalogo[
        [
            "Id_aforament",
            "anio_catalogo",
            "X_ETRS89_validada",
            "Y_ETRS89_validada"
        ]
    ]
    .dropna(
        subset=[
            "X_ETRS89_validada",
            "Y_ETRS89_validada"
        ]
    )
    .copy()
)

resumen_estabilidad = (
    df_estabilidad
    .groupby("Id_aforament")
    .agg(
        anios_presentes=(
            "anio_catalogo",
            "nunique"
        ),
        X_min=(
            "X_ETRS89_validada",
            "min"
        ),
        X_max=(
            "X_ETRS89_validada",
            "max"
        ),
        Y_min=(
            "Y_ETRS89_validada",
            "min"
        ),
        Y_max=(
            "Y_ETRS89_validada",
            "max"
        )
    )
    .reset_index()
)

resumen_estabilidad["rango_X_m"] = (
    resumen_estabilidad["X_max"]
    - resumen_estabilidad["X_min"]
)

resumen_estabilidad["rango_Y_m"] = (
    resumen_estabilidad["Y_max"]
    - resumen_estabilidad["Y_min"]
)

resumen_estabilidad["desplazamiento_max_m"] = np.sqrt(
    resumen_estabilidad["rango_X_m"] ** 2
    +
    resumen_estabilidad["rango_Y_m"] ** 2
)

resumen_estabilidad["estable_5m"] = (
    resumen_estabilidad[
        "desplazamiento_max_m"
    ]
    <= TOLERANCIA_ESTABILIDAD_M
)


# ============================================================
# 19. RESUMEN DE ESTABILIDAD
# ============================================================

n_ids_analizados = len(
    resumen_estabilidad
)

n_ids_estables = int(
    resumen_estabilidad[
        "estable_5m"
    ].sum()
)

n_ids_cambio = (
    n_ids_analizados
    - n_ids_estables
)

pct_estables = (
    n_ids_estables
    / n_ids_analizados
    * 100
    if n_ids_analizados > 0
    else np.nan
)

print("\n" + "=" * 80)
print("ESTABILIDAD ESPACIAL HISTÓRICA")
print("=" * 80)

print(
    f"\n✓ Tolerancia utilizada: "
    f"{TOLERANCIA_ESTABILIDAD_M:.1f} m"
)

print(
    f"✓ IDs analizados: "
    f"{n_ids_analizados:,}"
)

print(
    f"✓ IDs espacialmente estables: "
    f"{n_ids_estables:,} "
    f"({pct_estables:.2f} %)"
)

print(
    f"✓ IDs con posible cambio real: "
    f"{n_ids_cambio:,}"
)


# ============================================================
# 20. MOSTRAR POSIBLES CAMBIOS REALES
# ============================================================

ids_cambio_real = (
    resumen_estabilidad[
        ~resumen_estabilidad[
            "estable_5m"
        ]
    ]
    .sort_values(
        "desplazamiento_max_m",
        ascending=False
    )
)

if len(ids_cambio_real) > 0:

    print("\n" + "=" * 80)
    print("IDs CON POSIBLE CAMBIO REAL DE LOCALIZACIÓN")
    print("=" * 80)

    display(
        ids_cambio_real
    )

else:

    print(
        "\n✓ No se detectan cambios de localización "
        "superiores a la tolerancia."
    )


# ============================================================
# 21. COBERTURA RESPECTO AL DATASET DIARIO
# ============================================================

df_claves_limpio = pd.read_csv(
    ruta_limpio,
    usecols=[
        "Any",
        "Id_aforament"
    ],
    dtype={
        "Id_aforament": str
    },
    low_memory=False
)

df_claves_limpio[
    "Id_aforament"
] = (
    df_claves_limpio[
        "Id_aforament"
    ]
    .astype("string")
    .str.strip()
)

claves_limpio = (
    df_claves_limpio[
        [
            "Any",
            "Id_aforament"
        ]
    ]
    .drop_duplicates()
    .rename(
        columns={
            "Any": "anio_catalogo"
        }
    )
)

claves_catalogo = (
    df_catalogo[
        [
            "anio_catalogo",
            "Id_aforament",
            "X_ETRS89_validada",
            "Y_ETRS89_validada"
        ]
    ]
    .copy()
)

claves_catalogo[
    "geolocalizacion_valida"
] = (
    claves_catalogo[
        [
            "X_ETRS89_validada",
            "Y_ETRS89_validada"
        ]
    ]
    .notna()
    .all(axis=1)
)

control_cobertura = pd.merge(
    claves_limpio,
    claves_catalogo,
    on=[
        "anio_catalogo",
        "Id_aforament"
    ],
    how="left",
    validate="one_to_one"
)

control_cobertura[
    "geolocalizacion_valida"
] = (
    control_cobertura[
        "geolocalizacion_valida"
    ]
    .fillna(False)
    .astype(bool)
)


# ============================================================
# 22. COBERTURA ESTRICTA POR AÑO
# ============================================================

resumen_cobertura = (
    control_cobertura
    .groupby("anio_catalogo")
    .agg(
        ids_trafico=(
            "Id_aforament",
            "size"
        ),
        ids_geolocalizados=(
            "geolocalizacion_valida",
            "sum"
        )
    )
    .reset_index()
)

resumen_cobertura[
    "ids_sin_geo"
] = (
    resumen_cobertura[
        "ids_trafico"
    ]
    - resumen_cobertura[
        "ids_geolocalizados"
    ]
)

resumen_cobertura[
    "cobertura_pct"
] = (
    resumen_cobertura[
        "ids_geolocalizados"
    ]
    / resumen_cobertura[
        "ids_trafico"
    ]
    * 100
).round(2)

print("\n" + "=" * 80)
print("COBERTURA ESTRICTA AÑO + Id_aforament")
print("=" * 80)

display(
    resumen_cobertura
)


# ============================================================
# 23. COBERTURA GLOBAL DE IDs
# ============================================================

ids_limpio = set(
    df_claves_limpio[
        "Id_aforament"
    ]
    .dropna()
    .astype(str)
    .str.strip()
)

ids_catalogo_validos = set(
    df_catalogo.loc[
        df_catalogo[
            [
                "X_ETRS89_validada",
                "Y_ETRS89_validada"
            ]
        ]
        .notna()
        .all(axis=1),
        "Id_aforament"
    ]
    .dropna()
    .astype(str)
    .str.strip()
)

ids_con_geo = (
    ids_limpio
    & ids_catalogo_validos
)

ids_sin_geo = (
    ids_limpio
    - ids_catalogo_validos
)

cobertura_global = (
    len(ids_con_geo)
    / len(ids_limpio)
    * 100
    if len(ids_limpio) > 0
    else 0
)

print("\n" + "=" * 80)
print("COBERTURA GLOBAL")
print("=" * 80)

print(
    f"\n✓ IDs del dataset diario: "
    f"{len(ids_limpio):,}"
)

print(
    f"✓ IDs con geolocalización válida: "
    f"{len(ids_con_geo):,}"
)

print(
    f"✓ IDs sin geolocalización válida: "
    f"{len(ids_sin_geo):,}"
)

print(
    f"✓ Cobertura global: "
    f"{cobertura_global:.2f} %"
)


# ============================================================
# 24. CONTROL FINAL ANTES DE EXPORTAR
# ============================================================

if duplicados_clave != 0:
    raise ValueError(
        "No se puede exportar: "
        "existen duplicados históricos."
    )

if ids_sin_geo:
    print(
        "\n⚠ Existen IDs del dataset temporal "
        "sin geolocalización válida."
    )

    print(
        "Se conservarán identificados para revisión."
    )


# ============================================================
# 25. ORDENAR EL CATÁLOGO
# ============================================================

df_catalogo = (
    df_catalogo
    .sort_values(
        [
            "anio_catalogo",
            "Id_aforament"
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# 26. EXPORTAR CATÁLOGO VALIDADO
# ============================================================

df_catalogo.to_csv(
    ruta_salida,
    index=False,
    encoding="utf-8"
)


# ============================================================
# 27. VERIFICAR EXPORTACIÓN
# ============================================================

if not ruta_salida.exists():
    raise FileNotFoundError(
        "No se ha generado el catálogo espacial."
    )

df_control = pd.read_csv(
    ruta_salida,
    dtype={"Id_aforament": str},
    low_memory=False
)

if df_control.shape != df_catalogo.shape:
    raise ValueError(
        "Las dimensiones del archivo exportado "
        "no coinciden con el catálogo validado."
    )


# ============================================================
# 28. CIERRE
# ============================================================

print("\n" + "=" * 80)
print("CATÁLOGO ESPACIAL HISTÓRICO VALIDADO")
print("=" * 80)

print(
    f"\n✓ Registros históricos: "
    f"{len(df_catalogo):,}"
)

print(
    f"✓ Id_aforament únicos: "
    f"{df_catalogo['Id_aforament'].nunique():,}"
)

print(
    f"✓ Duplicados año + Id_aforament: "
    f"{duplicados_clave:,}"
)

print(
    f"✓ Coordenadas ETRS89 reconstruidas: "
    f"{(df_catalogo['fuente_coordenada'] == 'WGS84_reproyectada').sum():,}"
)

print(
    f"✓ Registros sin coordenada válida: "
    f"{sin_coordenada_final.sum():,}"
)

print(
    f"✓ IDs espacialmente estables "
    f"(≤ {TOLERANCIA_ESTABILIDAD_M:.0f} m): "
    f"{n_ids_estables:,}/{n_ids_analizados:,}"
)

print(
    f"✓ Cobertura global del dataset diario: "
    f"{cobertura_global:.2f} %"
)

print(
    "\n✓ Catálogo validado guardado en:"
)

print(
    ruta_salida
)

print(
    "\n✓ Coordenadas originales conservadas "
    "para garantizar la trazabilidad."
)

print(
    "✓ Preparado para el análisis espacial "
    "aforos ↔ estaciones de contaminación."
)

4.2 — CONSOLIDACIÓN Y VALIDACIÓN ESPACIAL DE AFOROS

✓ Registros históricos: 5,308
✓ Id_aforament únicos: 931
✓ Periodo: 2018 → 2024

✓ Duplicados año + Id_aforament: 0

VALIDACIÓN DE COORDENADAS ORIGINALES

✓ Registros con WGS84 plausible: 5,308/5,308
✓ Registros con ETRS89 original plausible: 5,290/5,308
✓ ETRS89 originales anómalas: 18

COHERENCIA WGS84 ↔ ETRS89

Mediana diferencia: 0.005 m
Percentil 95: 208.796 m
Máxima diferencia: 2507.177 m

COORDENADAS VALIDADAS

✓ ETRS89 originales conservadas: 5,290
✓ ETRS89 reconstruidas desde WGS84: 18
✓ Registros sin coordenada válida: 0

ESTABILIDAD ESPACIAL HISTÓRICA

✓ Tolerancia utilizada: 5.0 m
✓ IDs analizados: 931
✓ IDs espacialmente estables: 918 (98.60 %)
✓ IDs con posible cambio real: 13

IDs CON POSIBLE CAMBIO REAL DE LOCALIZACIÓN


,Id_aforament,anios_presentes,X_min,X_max,Y_min,Y_max,rango_X_m,rango_Y_m,desplazamiento_max_m,estable_5m
221,20093,7,431453.230000,432143.211,4.582603e+06,4584255.250,689.981000,1651.99000,1790.291803,False
596,3027,7,429780.939000,430212.698,4.580723e+06,4580723.220,431.759000,0.00000,431.759000,False
597,3028,6,429780.939000,430212.690,4.580730e+06,4580730.361,431.751000,0.00000,431.751000,False
479,20338,3,430210.616358,430299.550,4.583212e+06,4583415.830,88.933642,203.50143,222.085625,False
356,20219,5,430250.790000,430356.570,4.584531e+06,4584643.990,105.780000,113.47000,155.128493,False
274,20141,7,431905.320000,431995.780,4.582754e+06,4582845.911,90.460000,91.71100,128.817387,False
273,20140,7,432016.190000,432051.100,4.582698e+06,4582789.528,34.910000,91.41800,97.856828,False
395,20255,5,429904.526000,429949.186,4.581790e+06,4581794.015,44.660000,3.64100,44.808174,False
194,20066,7,431527.880000,431538.644,4.582601e+06,4582610.203,10.764000,9.33300,14.246704,False
229,20100,7,429936.960000,429946.490,4.583931e+06,4583939.971,9.530000,9.22100,13.260759,False



COBERTURA ESTRICTA AÑO + Id_aforament


,anio_catalogo,ids_trafico,ids_geolocalizados,ids_sin_geo,cobertura_pct
0,2018,613,613,0,100.0
1,2019,685,685,0,100.0
2,2020,714,714,0,100.0
3,2021,724,724,0,100.0
4,2022,824,824,0,100.0
5,2023,832,832,0,100.0
6,2024,838,838,0,100.0



COBERTURA GLOBAL

✓ IDs del dataset diario: 922
✓ IDs con geolocalización válida: 922
✓ IDs sin geolocalización válida: 0
✓ Cobertura global: 100.00 %

CATÁLOGO ESPACIAL HISTÓRICO VALIDADO

✓ Registros históricos: 5,308
✓ Id_aforament únicos: 931
✓ Duplicados año + Id_aforament: 0
✓ Coordenadas ETRS89 reconstruidas: 18
✓ Registros sin coordenada válida: 0
✓ IDs espacialmente estables (≤ 5 m): 918/931
✓ Cobertura global del dataset diario: 100.00 %

✓ Catálogo validado guardado en:
/content/drive/MyDrive/TFM/11_Machine_Learning_Temporal/PREINTEGRACION/02_Trafico_Rodado_y_Aforos/GEOLOCALIZACION_AFOROS/catalogo_espacial_aforos_2018_2024.csv

✓ Coordenadas originales conservadas para garantizar la trazabilidad.
✓ Preparado para el análisis espacial aforos ↔ estaciones de contaminación.


#### Resultado — Consolidación y validación espacial del catálogo histórico

La consolidación de los catálogos anuales genera una base histórica formada por **5.308 registros y 931 `Id_aforament` únicos**, sin duplicados en la clave `anio_catalogo + Id_aforament`.

La validación espacial detecta **18 registros con coordenadas ETRS89 originales no plausibles**. En todos estos casos las coordenadas geográficas WGS84 son válidas, por lo que las coordenadas proyectadas pueden reconstruirse mediante transformación a **ETRS89 / UTM zona 31N (EPSG:25831)**. Como resultado, el catálogo final no presenta registros sin geolocalización válida.

El análisis de estabilidad temporal, utilizando una tolerancia de **5 m**, muestra que **918 de los 931 identificadores (98,60 %)** mantienen una localización espacial estable. Los **13 identificadores restantes** presentan variaciones superiores al umbral establecido y se conservan mediante su posición específica para cada año, evitando imponer artificialmente una localización única.

La cobertura respecto al dataset diario de tráfico es completa: los **922 `Id_aforament` utilizados en las observaciones temporales disponen de geolocalización válida**, alcanzándose además una cobertura del **100 % para la clave `año + Id_aforament` en todos los años del periodo 2018–2024**.

Las coordenadas originales se mantienen junto con las coordenadas recalculadas y validadas, preservando la trazabilidad completa del proceso.

> **Conclusión:** se obtiene un catálogo espacial histórico validado, temporalmente consistente y con cobertura completa de los datos de tráfico utilizados en el estudio. La red queda preparada para analizar su relación espacial con las estaciones de calidad del aire.

## 4.3. Caracterización espacial de la red de aforos respecto a las estaciones de contaminación

Una vez construido y validado el catálogo espacial histórico de aforos, se analiza la relación geométrica entre la red de tráfico rodado y las estaciones de calidad del aire.

El objetivo de esta etapa no es todavía incorporar una variable de tráfico al dataset maestro, sino determinar qué estrategia espacial representa de forma más adecuada la presión del tráfico en el entorno de cada estación.

La elevada densidad de puntos de aforo permite superar una asignación basada exclusivamente en el punto más próximo. Por este motivo se analiza, para cada estación y año:

- la distancia al aforo más cercano;
- las distancias a los primeros vecinos espaciales;
- el número de aforos disponibles dentro de radios de **250, 500, 750 y 1.000 m**;
- la distribución de las distancias de los aforos contenidos en cada radio;
- la estabilidad temporal de la cobertura espacial entre 2018 y 2024.

Todas las distancias se calculan en **ETRS89 / UTM zona 31N (EPSG:25831)**, utilizando las coordenadas validadas del catálogo histórico de aforos.

Los radios analizados en esta fase tienen carácter diagnóstico y no constituyen todavía una selección definitiva. La elección de la escala espacial utilizada para construir los indicadores de tráfico se realizará a partir de los resultados obtenidos, buscando un equilibrio entre proximidad a la estación, número de observaciones disponibles y estabilidad temporal.

> **Objetivo:** caracterizar empíricamente la estructura espacial de la red de aforos alrededor de las estaciones de contaminación y seleccionar posteriormente una estrategia de agregación espacial adecuada para representar la presión de tráfico rodado.

In [ ]:
# ============================================================
# 4.3. CARACTERIZACIÓN ESPACIAL DE LA RED DE AFOROS
# RESPECTO A LAS ESTACIONES DE CONTAMINACIÓN
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. RUTAS
# ============================================================

ruta_base = Path(
    "/content/drive/MyDrive/TFM/"
    "11_Machine_Learning_Temporal"
)

ruta_trafico = (
    ruta_base /
    "PREINTEGRACION" /
    "02_Trafico_Rodado_y_Aforos"
)

ruta_geo_aforos = (
    ruta_trafico /
    "GEOLOCALIZACION_AFOROS"
)

ruta_catalogo_aforos = (
    ruta_geo_aforos /
    "catalogo_espacial_aforos_2018_2024.csv"
)

ruta_contaminacion = (
    ruta_base /
    "PREINTEGRACION" /
    "06_Contaminacion_Atmosferica" /
    "df_contaminacion_atm_preintegrado.csv"
)


# ============================================================
# 2. COMPROBAR ARCHIVOS
# ============================================================

for ruta in [
    ruta_catalogo_aforos,
    ruta_contaminacion
]:

    if not ruta.exists():
        raise FileNotFoundError(
            f"No se encuentra el archivo:\n{ruta}"
        )


# ============================================================
# 3. CARGAR CATÁLOGO HISTÓRICO DE AFOROS
# ============================================================

df_aforos_geo = pd.read_csv(
    ruta_catalogo_aforos,
    dtype={"Id_aforament": str},
    low_memory=False
)

df_aforos_geo["Id_aforament"] = (
    df_aforos_geo["Id_aforament"]
    .astype("string")
    .str.strip()
)


# ============================================================
# 4. CARGAR ESTACIONES DE CONTAMINACIÓN
# ============================================================
# El dataset de contaminación contiene varias observaciones
# temporales por estación. Para el análisis espacial solo
# necesitamos las localizaciones físicas únicas.
# ============================================================

df_contaminacion = pd.read_csv(
    ruta_contaminacion,
    low_memory=False
)


# ============================================================
# 5. COMPROBAR VARIABLES NECESARIAS
# ============================================================

columnas_aforos_necesarias = [
    "anio_catalogo",
    "Id_aforament",
    "X_ETRS89_validada",
    "Y_ETRS89_validada"
]

faltantes_aforos = [
    columna
    for columna in columnas_aforos_necesarias
    if columna not in df_aforos_geo.columns
]

if faltantes_aforos:
    raise KeyError(
        "Faltan variables necesarias en el "
        "catálogo de aforos:\n"
        f"{faltantes_aforos}"
    )


columnas_contaminacion_necesarias = [
    "estacion_geo",
    "X_ETRS89",
    "Y_ETRS89"
]

faltantes_contaminacion = [
    columna
    for columna in columnas_contaminacion_necesarias
    if columna not in df_contaminacion.columns
]

if faltantes_contaminacion:
    raise KeyError(
        "Faltan variables necesarias en contaminación:\n"
        f"{faltantes_contaminacion}"
    )


# ============================================================
# 6. CONSTRUIR CATÁLOGO DE ESTACIONES DE CONTAMINACIÓN
# ============================================================

df_estaciones = (
    df_contaminacion[
        [
            "estacion_geo",
            "X_ETRS89",
            "Y_ETRS89"
        ]
    ]
    .drop_duplicates()
    .copy()
)


# ============================================================
# 7. VALIDAR UNA POSICIÓN POR estacion_geo
# ============================================================

control_estaciones = (
    df_estaciones
    .groupby("estacion_geo")
    .agg(
        posiciones=(
            "X_ETRS89",
            "size"
        )
    )
    .reset_index()
)

if (
    control_estaciones["posiciones"] > 1
).any():

    raise ValueError(
        "Alguna estacion_geo presenta más de "
        "una pareja de coordenadas."
    )


df_estaciones = (
    df_estaciones
    .drop_duplicates(
        subset=["estacion_geo"]
    )
    .reset_index(drop=True)
)


# ============================================================
# 8. CONVERTIR COORDENADAS A NUMÉRICO
# ============================================================

for columna in [
    "X_ETRS89_validada",
    "Y_ETRS89_validada"
]:

    df_aforos_geo[columna] = pd.to_numeric(
        df_aforos_geo[columna],
        errors="coerce"
    )


for columna in [
    "X_ETRS89",
    "Y_ETRS89"
]:

    df_estaciones[columna] = pd.to_numeric(
        df_estaciones[columna],
        errors="coerce"
    )


# ============================================================
# 9. CONTROL PREVIO
# ============================================================

if (
    df_aforos_geo[
        [
            "X_ETRS89_validada",
            "Y_ETRS89_validada"
        ]
    ]
    .isna()
    .any(axis=None)
):

    raise ValueError(
        "Existen coordenadas validadas nulas "
        "en el catálogo de aforos."
    )


if (
    df_estaciones[
        [
            "X_ETRS89",
            "Y_ETRS89"
        ]
    ]
    .isna()
    .any(axis=None)
):

    raise ValueError(
        "Existen coordenadas nulas en las "
        "estaciones de contaminación."
    )


print("=" * 80)
print("4.3 — CARACTERIZACIÓN ESPACIAL AFOROS ↔ CONTAMINACIÓN")
print("=" * 80)

print(
    f"\n✓ Estaciones de contaminación: "
    f"{df_estaciones['estacion_geo'].nunique()}"
)

print(
    f"✓ Registros históricos de aforos: "
    f"{len(df_aforos_geo):,}"
)

print(
    f"✓ Id_aforament históricos únicos: "
    f"{df_aforos_geo['Id_aforament'].nunique():,}"
)

print(
    f"✓ Periodo: "
    f"{df_aforos_geo['anio_catalogo'].min()} "
    f"→ "
    f"{df_aforos_geo['anio_catalogo'].max()}"
)


# ============================================================
# 10. MOSTRAR ESTACIONES UTILIZADAS
# ============================================================

print("\n" + "=" * 80)
print("ESTACIONES DE CONTAMINACIÓN")
print("=" * 80)

display(
    df_estaciones
    .sort_values("estacion_geo")
)


# ============================================================
# 11. CALCULAR TODAS LAS DISTANCIAS
# ============================================================
# Se genera una matriz estación × aforo para cada año.
#
# 8 estaciones × ~600–840 aforos/año es un volumen pequeño
# y permite mantener una tabla completamente trazable.
# ============================================================

resultados_distancias = []

for anio in sorted(
    df_aforos_geo[
        "anio_catalogo"
    ].unique()
):

    df_aforos_anio = (
        df_aforos_geo[
            df_aforos_geo[
                "anio_catalogo"
            ] == anio
        ]
        .copy()
    )

    for _, estacion in df_estaciones.iterrows():

        dx = (
            df_aforos_anio[
                "X_ETRS89_validada"
            ].to_numpy()
            - estacion["X_ETRS89"]
        )

        dy = (
            df_aforos_anio[
                "Y_ETRS89_validada"
            ].to_numpy()
            - estacion["Y_ETRS89"]
        )

        distancia = np.sqrt(
            dx ** 2
            + dy ** 2
        )

        df_temp = pd.DataFrame({
            "anio_catalogo": anio,
            "estacion_geo":
                estacion["estacion_geo"],
            "Id_aforament":
                df_aforos_anio[
                    "Id_aforament"
                ].to_numpy(),
            "distancia_m":
                distancia
        })

        resultados_distancias.append(
            df_temp
        )


df_distancias = pd.concat(
    resultados_distancias,
    ignore_index=True
)


# ============================================================
# 12. CONTROL DE LA MATRIZ DE DISTANCIAS
# ============================================================

print("\n" + "=" * 80)
print("MATRIZ DE DISTANCIAS")
print("=" * 80)

print(
    f"\n✓ Relaciones estación–aforo calculadas: "
    f"{len(df_distancias):,}"
)

print(
    f"✓ Distancia mínima observada: "
    f"{df_distancias['distancia_m'].min():.2f} m"
)

print(
    f"✓ Distancia máxima observada: "
    f"{df_distancias['distancia_m'].max():.2f} m"
)


# ============================================================
# 13. DISTANCIA AL AFORO MÁS CERCANO
# ============================================================

idx_min = (
    df_distancias
    .groupby(
        [
            "anio_catalogo",
            "estacion_geo"
        ]
    )["distancia_m"]
    .idxmin()
)

df_aforo_mas_cercano = (
    df_distancias
    .loc[
        idx_min,
        [
            "anio_catalogo",
            "estacion_geo",
            "Id_aforament",
            "distancia_m"
        ]
    ]
    .sort_values(
        [
            "estacion_geo",
            "anio_catalogo"
        ]
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 80)
print("AFORO MÁS CERCANO POR ESTACIÓN Y AÑO")
print("=" * 80)

display(
    df_aforo_mas_cercano
)


# ============================================================
# 14. RESUMEN DEL AFORO MÁS CERCANO POR ESTACIÓN
# ============================================================

resumen_cercano = (
    df_aforo_mas_cercano
    .groupby("estacion_geo")
    .agg(
        distancia_min_m=(
            "distancia_m",
            "min"
        ),
        distancia_mediana_m=(
            "distancia_m",
            "median"
        ),
        distancia_max_m=(
            "distancia_m",
            "max"
        ),
        aforos_cercanos_distintos=(
            "Id_aforament",
            "nunique"
        )
    )
    .reset_index()
)

print("\n" + "=" * 80)
print("RESUMEN DE PROXIMIDAD")
print("=" * 80)

display(
    resumen_cercano
)


# ============================================================
# 15. PRIMEROS 5 VECINOS POR ESTACIÓN Y AÑO
# ============================================================

df_vecinos = (
    df_distancias
    .sort_values(
        [
            "anio_catalogo",
            "estacion_geo",
            "distancia_m"
        ]
    )
    .groupby(
        [
            "anio_catalogo",
            "estacion_geo"
        ],
        group_keys=False
    )
    .head(5)
    .copy()
)

df_vecinos["orden_vecino"] = (
    df_vecinos
    .groupby(
        [
            "anio_catalogo",
            "estacion_geo"
        ]
    )
    .cumcount()
    + 1
)

print("\n" + "=" * 80)
print("CINCO AFOROS MÁS PRÓXIMOS")
print("=" * 80)

display(
    df_vecinos[
        [
            "anio_catalogo",
            "estacion_geo",
            "orden_vecino",
            "Id_aforament",
            "distancia_m"
        ]
    ]
)


# ============================================================
# 16. RADIOS DIAGNÓSTICOS
# ============================================================

radios_m = [
    250,
    500,
    750,
    1000
]


# ============================================================
# 17. NÚMERO DE AFOROS DENTRO DE CADA RADIO
# ============================================================

resultados_radios = []

grupos = (
    df_distancias
    .groupby(
        [
            "anio_catalogo",
            "estacion_geo"
        ]
    )
)

for (
    anio,
    estacion
), grupo in grupos:

    fila = {
        "anio_catalogo": anio,
        "estacion_geo": estacion
    }

    for radio in radios_m:

        dentro = (
            grupo[
                grupo["distancia_m"]
                <= radio
            ]
        )

        fila[
            f"n_aforos_{radio}m"
        ] = len(dentro)

        fila[
            f"dist_mediana_{radio}m"
        ] = (
            dentro[
                "distancia_m"
            ].median()
            if len(dentro) > 0
            else np.nan
        )

    resultados_radios.append(
        fila
    )


df_radios = pd.DataFrame(
    resultados_radios
)


# ============================================================
# 18. TABLA COMPLETA DE COBERTURA POR RADIO
# ============================================================

print("\n" + "=" * 80)
print("COBERTURA ESPACIAL POR RADIO — ESTACIÓN Y AÑO")
print("=" * 80)

display(
    df_radios
    .sort_values(
        [
            "estacion_geo",
            "anio_catalogo"
        ]
    )
)


# ============================================================
# 19. RESUMEN POR ESTACIÓN Y RADIO
# ============================================================

resumen_radios = []

for estacion in sorted(
    df_radios[
        "estacion_geo"
    ].unique()
):

    df_est = (
        df_radios[
            df_radios[
                "estacion_geo"
            ] == estacion
        ]
    )

    for radio in radios_m:

        columna_n = (
            f"n_aforos_{radio}m"
        )

        resumen_radios.append({
            "estacion_geo": estacion,
            "radio_m": radio,

            "aforos_min":
                int(
                    df_est[
                        columna_n
                    ].min()
                ),

            "aforos_mediana":
                float(
                    df_est[
                        columna_n
                    ].median()
                ),

            "aforos_max":
                int(
                    df_est[
                        columna_n
                    ].max()
                ),

            "anios_sin_aforos":
                int(
                    (
                        df_est[
                            columna_n
                        ] == 0
                    ).sum()
                )
        })


df_resumen_radios = pd.DataFrame(
    resumen_radios
)

print("\n" + "=" * 80)
print("RESUMEN DE COBERTURA POR ESTACIÓN Y RADIO")
print("=" * 80)

display(
    df_resumen_radios
)


# ============================================================
# 20. RESUMEN GLOBAL DE LOS RADIOS
# ============================================================
# Queremos saber si algún radio deja estaciones/años
# sin representación de tráfico.
# ============================================================

resumen_global_radios = []

for radio in radios_m:

    columna_n = (
        f"n_aforos_{radio}m"
    )

    resumen_global_radios.append({

        "radio_m":
            radio,

        "min_aforos_estacion_anio":
            int(
                df_radios[
                    columna_n
                ].min()
            ),

        "mediana_aforos_estacion_anio":
            float(
                df_radios[
                    columna_n
                ].median()
            ),

        "max_aforos_estacion_anio":
            int(
                df_radios[
                    columna_n
                ].max()
            ),

        "casos_sin_aforos":
            int(
                (
                    df_radios[
                        columna_n
                    ] == 0
                ).sum()
            ),

        "pct_estacion_anio_con_cobertura":
            round(
                (
                    df_radios[
                        columna_n
                    ] > 0
                ).mean()
                * 100,
                2
            )
    })


df_resumen_global_radios = pd.DataFrame(
    resumen_global_radios
)

print("\n" + "=" * 80)
print("COMPARACIÓN GLOBAL DE RADIOS")
print("=" * 80)

display(
    df_resumen_global_radios
)


# ============================================================
# 21. DISTRIBUCIÓN DE DISTANCIAS A LOS 5 PRIMEROS VECINOS
# ============================================================

resumen_vecinos = (
    df_vecinos
    .groupby(
        "orden_vecino"
    )["distancia_m"]
    .agg(
        minimo_m="min",
        mediana_m="median",
        media_m="mean",
        p75_m=lambda x:
            x.quantile(0.75),
        p95_m=lambda x:
            x.quantile(0.95),
        maximo_m="max"
    )
    .reset_index()
)

print("\n" + "=" * 80)
print("DISTANCIAS A LOS PRIMEROS VECINOS")
print("=" * 80)

display(
    resumen_vecinos
)


# ============================================================
# 22. CONTROL DE ESTABILIDAD TEMPORAL POR RADIO
# ============================================================
# Coeficiente de variación del número de aforos alrededor
# de cada estación durante 2018–2024.
# ============================================================

estabilidad_radios = []

for estacion in sorted(
    df_radios[
        "estacion_geo"
    ].unique()
):

    df_est = (
        df_radios[
            df_radios[
                "estacion_geo"
            ] == estacion
        ]
    )

    for radio in radios_m:

        valores = (
            df_est[
                f"n_aforos_{radio}m"
            ]
        )

        media = valores.mean()
        std = valores.std()

        cv = (
            std / media
            if media > 0
            else np.nan
        )

        estabilidad_radios.append({
            "estacion_geo":
                estacion,
            "radio_m":
                radio,
            "media_aforos":
                round(
                    media,
                    2
                ),
            "std_aforos":
                round(
                    std,
                    2
                ),
            "cv":
                round(
                    cv,
                    3
                )
                if not np.isnan(cv)
                else np.nan
        })


df_estabilidad_radios = pd.DataFrame(
    estabilidad_radios
)

print("\n" + "=" * 80)
print("ESTABILIDAD TEMPORAL DE LA COBERTURA")
print("=" * 80)

display(
    df_estabilidad_radios
)


# ============================================================
# 23. CONTROL FINAL
# ============================================================

n_combinaciones_esperadas = (
    df_estaciones[
        "estacion_geo"
    ].nunique()
    *
    df_aforos_geo[
        "anio_catalogo"
    ].nunique()
)

n_combinaciones_obtenidas = (
    df_radios[
        [
            "anio_catalogo",
            "estacion_geo"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

print("\n" + "=" * 80)
print("CONTROL FINAL 4.3")
print("=" * 80)

print(
    f"\n✓ Combinaciones estación-año esperadas: "
    f"{n_combinaciones_esperadas}"
)

print(
    f"✓ Combinaciones estación-año obtenidas: "
    f"{n_combinaciones_obtenidas}"
)

print(
    f"✓ Radios evaluados: "
    f"{radios_m}"
)

print(
    f"✓ Relaciones espaciales calculadas: "
    f"{len(df_distancias):,}"
)

print(
    "✓ No se ha realizado todavía ninguna "
    "agregación de intensidad de tráfico."
)

print(
    "✓ No se ha modificado el dataset maestro."
)

print(
    "\n✓ Preparado para seleccionar la escala "
    "espacial de los indicadores de tráfico."
)

4.3 — CARACTERIZACIÓN ESPACIAL AFOROS ↔ CONTAMINACIÓN

✓ Estaciones de contaminación: 8
✓ Registros históricos de aforos: 5,308
✓ Id_aforament históricos únicos: 931
✓ Periodo: 2018 → 2024

ESTACIONES DE CONTAMINACIÓN


,estacion_geo,X_ETRS89,Y_ETRS89
4,Ciutadella,432059.474731,4.581971e+06
2,Eixample,429249.000364,4.581876e+06
3,Gràcia,429230.091303,4.583364e+06
7,Observatori Fabra,426786.192307,4.585575e+06
6,Palau Reial,426032.466989,4.582152e+06
0,Poblenou,433507.035285,4.583901e+06
1,Sants,427486.280382,4.581205e+06
5,Vall d'Hebron,428791.895985,4.586410e+06



MATRIZ DE DISTANCIAS

✓ Relaciones estación–aforo calculadas: 42,464
✓ Distancia mínima observada: 111.35 m
✓ Distancia máxima observada: 9922.06 m

AFORO MÁS CERCANO POR ESTACIÓN Y AÑO


,anio_catalogo,estacion_geo,Id_aforament,distancia_m
0,2018,Ciutadella,5016,269.680868
1,2019,Ciutadella,5016,269.680868
2,2020,Ciutadella,5016,269.673767
3,2021,Ciutadella,5016,269.673767
4,2022,Ciutadella,5016,269.673767
5,2023,Ciutadella,5016,269.673767
6,2024,Ciutadella,5008,383.762003
7,2018,Eixample,4071,151.143897
8,2019,Eixample,4071,151.143897
9,2020,Eixample,4071,151.150196



RESUMEN DE PROXIMIDAD


...


DISTANCIAS A LOS PRIMEROS VECINOS


,orden_vecino,minimo_m,mediana_m,media_m,p75_m,p95_m,maximo_m
0,1,111.348261,226.751043,310.127840,401.222427,896.469531,896.472751
1,2,135.268502,239.353418,340.542917,441.386591,896.469531,896.472751
2,3,144.564431,302.241896,412.108788,455.078963,1074.962570,1074.964444
3,4,159.365498,317.228897,443.085509,493.025635,1080.062671,1080.062671
4,5,159.916798,405.205147,504.682273,527.129862,1206.545957,1206.545957



ESTABILIDAD TEMPORAL DE LA COBERTURA


,estacion_geo,radio_m,media_aforos,std_aforos,cv
0,Ciutadella,250,0.00,0.00,NaN
1,Ciutadella,500,13.14,1.46,0.111
2,Ciutadella,750,37.00,1.53,0.041
3,Ciutadella,1000,47.57,1.40,0.029
4,Eixample,250,3.29,2.36,0.718
5,Eixample,500,19.57,6.43,0.328
6,Eixample,750,50.14,11.07,0.221
7,Eixample,1000,87.71,11.60,0.132
8,Gràcia,250,4.00,0.00,0.000
9,Gràcia,500,12.00,1.41,0.118



CONTROL FINAL 4.3

✓ Combinaciones estación-año esperadas: 56
✓ Combinaciones estación-año obtenidas: 56
✓ Radios evaluados: [250, 500, 750, 1000]
✓ Relaciones espaciales calculadas: 42,464
✓ No se ha realizado todavía ninguna agregación de intensidad de tráfico.
✓ No se ha modificado el dataset maestro.

✓ Preparado para seleccionar la escala espacial de los indicadores de tráfico.


#### Resultado — Caracterización espacial de la red de aforos

El análisis espacial confirma una distribución heterogénea de la red de aforos alrededor de las estaciones de calidad del aire. La distancia al punto de medida de tráfico más próximo oscila aproximadamente entre **111 m y 896 m**, reflejando diferencias importantes entre estaciones situadas en ámbitos urbanos densos y estaciones periféricas o de fondo.

Los radios de **250, 500 y 750 m** no proporcionan cobertura completa para todas las estaciones y años. En particular, la estación Observatori Fabra no dispone de aforos dentro de los primeros 750 m. El radio de **1.000 m** constituye la primera escala analizada que garantiza representación de tráfico para el **100 % de las 56 combinaciones estación-año**, con un mínimo de dos aforos disponibles.

Sin embargo, el número de puntos incluidos dentro de 1 km presenta una elevada heterogeneidad espacial, desde únicamente 2 aforos en Observatori Fabra hasta más de 100 en determinados años de Eixample. Por este motivo, una agregación basada exclusivamente en la media aritmética podría asignar el mismo peso a puntos con grados de proximidad muy diferentes.

Se adopta, por tanto, **1.000 m como radio espacial máximo de referencia**, manteniendo la distancia individual estación-aforo para construir indicadores ponderados por proximidad. Paralelamente se conservarán indicadores correspondientes a escalas inferiores como variables candidatas cuando exista cobertura suficiente.

> **Conclusión:** la caracterización empírica de la red justifica utilizar una estrategia multiescala y ponderada por distancia, evitando seleccionar arbitrariamente un único aforo o tratar de forma equivalente todos los puntos situados dentro del área de influencia.

## 4.4. Construcción de indicadores diarios de presión de tráfico

A partir de la relación espacial caracterizada entre las estaciones de calidad del aire y la red histórica de aforos, se construyen indicadores diarios destinados a representar la intensidad del tráfico rodado en el entorno de cada estación.

La elevada heterogeneidad en la densidad de puntos de medida desaconseja utilizar exclusivamente el aforo más próximo o una media simple sobre un radio arbitrario. Por este motivo se adopta una estrategia **multiescala y ponderada por distancia**.

Se calculan indicadores para radios de **500, 750 y 1.000 m**, manteniendo 1.000 m como escala espacial de referencia por ser la primera de las analizadas que proporciona cobertura para todas las estaciones y años.

Para cada combinación `fecha + estacion_geo` se obtienen:

- intensidad media de tráfico de los aforos disponibles;
- intensidad total observada dentro del radio;
- intensidad ponderada mediante distancia inversa (IDW);
- número de aforos con observación disponible;
- distancia mínima al aforo con información de tráfico ese día;
- cobertura relativa respecto al número de aforos espacialmente disponibles.

La ponderación IDW asigna mayor influencia a los puntos de medida próximos a la estación, evitando que los aforos situados en el límite del área de influencia tengan el mismo peso que aquellos localizados en sus inmediaciones.

La asignación espacial se realiza utilizando la posición histórica correspondiente a cada `Id_aforament` y año, conservando así los cambios documentados en la configuración de la red.

Los indicadores se calculan inicialmente como una tabla independiente con granularidad `fecha + estacion_geo`. Su integración con el dataset maestro se realizará únicamente después de comprobar su cobertura temporal, unicidad y coherencia.

> **Objetivo:** transformar las observaciones diarias de la red de aforos en variables cuantitativas de exposición al tráfico, espacialmente comparables y directamente utilizables en el posterior análisis estadístico y predictivo.

In [ ]:
# ============================================================
# 4.4. CONSTRUCCIÓN DE INDICADORES DIARIOS
# DE PRESIÓN DE TRÁFICO
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. RUTAS
# ============================================================

ruta_base = Path(
    "/content/drive/MyDrive/TFM/"
    "11_Machine_Learning_Temporal"
)

ruta_trafico = (
    ruta_base /
    "PREINTEGRACION" /
    "02_Trafico_Rodado_y_Aforos"
)

ruta_geo = (
    ruta_trafico /
    "GEOLOCALIZACION_AFOROS"
)

ruta_aforos_diarios = (
    ruta_trafico /
    "df_aforos_2018_2024_Diario_Limpio.csv"
)

ruta_catalogo_geo = (
    ruta_geo /
    "catalogo_espacial_aforos_2018_2024.csv"
)

ruta_contaminacion = (
    ruta_base /
    "PREINTEGRACION" /
    "06_Contaminacion_Atmosferica" /
    "df_contaminacion_atm_preintegrado.csv"
)


# ============================================================
# 2. COMPROBAR ARCHIVOS
# ============================================================

for ruta in [
    ruta_aforos_diarios,
    ruta_catalogo_geo,
    ruta_contaminacion
]:

    if not ruta.exists():

        raise FileNotFoundError(
            f"No se encuentra el archivo:\n{ruta}"
        )


# ============================================================
# 3. CARGAR AFOROS DIARIOS
# ============================================================

df_trafico = pd.read_csv(
    ruta_aforos_diarios,
    dtype={"Id_aforament": str},
    low_memory=False
)

df_trafico["Id_aforament"] = (
    df_trafico["Id_aforament"]
    .astype("string")
    .str.strip()
)


# ============================================================
# 4. AUDITAR COLUMNAS DEL DATASET DE TRÁFICO
# ============================================================

print("=" * 80)
print("4.4 — CONSTRUCCIÓN DE INDICADORES DIARIOS DE TRÁFICO")
print("=" * 80)

print(
    f"\n✓ Registros diarios de tráfico: "
    f"{len(df_trafico):,}"
)

print(
    f"✓ Columnas disponibles: "
    f"{df_trafico.shape[1]}"
)

print("\nColumnas del dataset diario:")

for columna in df_trafico.columns:
    print(f"  • {columna}")


# ============================================================
# 5. IDENTIFICAR VARIABLES NECESARIAS
# ============================================================
# El dataset limpio construido previamente utiliza:
#
# Any
# Id_aforament
# Fecha
# Valor_IMD
#
# Se comprueba explícitamente para evitar continuar
# silenciosamente con un esquema diferente.
# ============================================================

columnas_necesarias = [
    "Any",
    "Id_aforament",
    "Fecha",
    "Valor_IMD"
]

faltantes = [
    columna
    for columna in columnas_necesarias
    if columna not in df_trafico.columns
]

if faltantes:

    raise KeyError(
        "Faltan variables necesarias en "
        "el dataset diario de tráfico:\n"
        f"{faltantes}"
    )


# ============================================================
# 6. NORMALIZAR FECHA, AÑO E INTENSIDAD
# ============================================================

df_trafico["fecha"] = pd.to_datetime(
    df_trafico["Fecha"],
    errors="coerce"
)

df_trafico["anio"] = pd.to_numeric(
    df_trafico["Any"],
    errors="coerce"
).astype("Int64")

df_trafico["Valor_IMD"] = pd.to_numeric(
    df_trafico["Valor_IMD"],
    errors="coerce"
)


# ============================================================
# 7. CONTROL TEMPORAL
# ============================================================

fechas_invalidas = (
    df_trafico["fecha"]
    .isna()
    .sum()
)

anios_invalidos = (
    df_trafico["anio"]
    .isna()
    .sum()
)

print("\n" + "=" * 80)
print("CONTROL TEMPORAL DEL TRÁFICO")
print("=" * 80)

print(
    f"\n✓ Fechas inválidas: "
    f"{fechas_invalidas:,}"
)

print(
    f"✓ Años inválidos: "
    f"{anios_invalidos:,}"
)

print(
    f"✓ Periodo: "
    f"{df_trafico['fecha'].min()} "
    f"→ "
    f"{df_trafico['fecha'].max()}"
)

if fechas_invalidas > 0:
    raise ValueError(
        "Existen fechas inválidas en tráfico."
    )


# ============================================================
# 8. CONTROL DE LA VARIABLE DE INTENSIDAD
# ============================================================

nulos_imd = (
    df_trafico["Valor_IMD"]
    .isna()
    .sum()
)

negativos_imd = (
    df_trafico["Valor_IMD"] < 0
).sum()

print("\n" + "=" * 80)
print("CONTROL DE Valor_IMD")
print("=" * 80)

print(
    f"\n✓ Valores nulos: "
    f"{nulos_imd:,}"
)

print(
    f"✓ Valores negativos: "
    f"{negativos_imd:,}"
)

print(
    f"✓ Valor mínimo: "
    f"{df_trafico['Valor_IMD'].min():,.2f}"
)

print(
    f"✓ Mediana: "
    f"{df_trafico['Valor_IMD'].median():,.2f}"
)

print(
    f"✓ Valor máximo: "
    f"{df_trafico['Valor_IMD'].max():,.2f}"
)


# ============================================================
# 9. COMPROBAR CLAVE DIARIA
# ============================================================

duplicados_trafico = (
    df_trafico
    .duplicated(
        subset=[
            "fecha",
            "Id_aforament"
        ]
    )
    .sum()
)

print(
    f"\n✓ Duplicados fecha + Id_aforament: "
    f"{duplicados_trafico:,}"
)

# No forzamos error aquí porque, si existen,
# deben diagnosticarse antes de decidir su tratamiento.


# ============================================================
# 10. CARGAR CATÁLOGO ESPACIAL HISTÓRICO
# ============================================================

df_geo = pd.read_csv(
    ruta_catalogo_geo,
    dtype={"Id_aforament": str},
    low_memory=False
)

df_geo["Id_aforament"] = (
    df_geo["Id_aforament"]
    .astype("string")
    .str.strip()
)


# ============================================================
# 11. PREPARAR CATÁLOGO GEO PARA EL MERGE
# ============================================================

columnas_geo = [
    "anio_catalogo",
    "Id_aforament",
    "X_ETRS89_validada",
    "Y_ETRS89_validada"
]

faltantes_geo = [
    columna
    for columna in columnas_geo
    if columna not in df_geo.columns
]

if faltantes_geo:

    raise KeyError(
        "Faltan variables en el catálogo espacial:\n"
        f"{faltantes_geo}"
    )


df_geo_merge = (
    df_geo[
        columnas_geo
    ]
    .copy()
)

df_geo_merge[
    "anio_catalogo"
] = pd.to_numeric(
    df_geo_merge["anio_catalogo"],
    errors="coerce"
).astype("Int64")


# ============================================================
# 12. VALIDAR CLAVE DEL CATÁLOGO
# ============================================================

duplicados_geo = (
    df_geo_merge
    .duplicated(
        subset=[
            "anio_catalogo",
            "Id_aforament"
        ]
    )
    .sum()
)

if duplicados_geo > 0:

    raise ValueError(
        "El catálogo espacial contiene duplicados "
        "en año + Id_aforament."
    )


# ============================================================
# 13. INCORPORAR POSICIÓN HISTÓRICA A CADA REGISTRO
# ============================================================

filas_antes_geo = len(
    df_trafico
)

df_trafico_geo = pd.merge(
    df_trafico,
    df_geo_merge,
    left_on=[
        "anio",
        "Id_aforament"
    ],
    right_on=[
        "anio_catalogo",
        "Id_aforament"
    ],
    how="left",
    validate="many_to_one"
)

filas_despues_geo = len(
    df_trafico_geo
)

sin_geo = (
    df_trafico_geo[
        [
            "X_ETRS89_validada",
            "Y_ETRS89_validada"
        ]
    ]
    .isna()
    .any(axis=1)
    .sum()
)

print("\n" + "=" * 80)
print("ASIGNACIÓN ESPACIAL DEL TRÁFICO")
print("=" * 80)

print(
    f"\n✓ Filas antes del merge: "
    f"{filas_antes_geo:,}"
)

print(
    f"✓ Filas después del merge: "
    f"{filas_despues_geo:,}"
)

print(
    f"✓ Registros sin geolocalización: "
    f"{sin_geo:,}"
)

if filas_antes_geo != filas_despues_geo:

    raise ValueError(
        "El merge espacial ha modificado "
        "el número de registros."
    )

if sin_geo > 0:

    raise ValueError(
        "Existen registros de tráfico "
        "sin geolocalización histórica."
    )


# ============================================================
# 14. CARGAR ESTACIONES DE CONTAMINACIÓN
# ============================================================

df_cont = pd.read_csv(
    ruta_contaminacion,
    low_memory=False
)

columnas_estaciones = [
    "estacion_geo",
    "X_ETRS89",
    "Y_ETRS89"
]

df_estaciones = (
    df_cont[
        columnas_estaciones
    ]
    .drop_duplicates()
    .copy()
)

control_posiciones = (
    df_estaciones
    .groupby("estacion_geo")
    .size()
)

if (
    control_posiciones > 1
).any():

    raise ValueError(
        "Alguna estacion_geo tiene "
        "más de una localización."
    )

df_estaciones = (
    df_estaciones
    .drop_duplicates(
        subset=["estacion_geo"]
    )
    .reset_index(drop=True)
)


# ============================================================
# 15. CONSTRUIR RELACIÓN ESPACIAL ESTACIÓN ↔ AFORO
# ============================================================
# La relación se construye sobre el catálogo histórico,
# no sobre los 1.9 M registros diarios.
# Esto evita un producto cartesiano innecesariamente grande.
# ============================================================

relaciones = []

for anio in sorted(
    df_geo_merge[
        "anio_catalogo"
    ].dropna().unique()
):

    geo_anio = (
        df_geo_merge[
            df_geo_merge[
                "anio_catalogo"
            ] == anio
        ]
        .copy()
    )

    for _, estacion in df_estaciones.iterrows():

        dx = (
            geo_anio[
                "X_ETRS89_validada"
            ].to_numpy()
            - estacion["X_ETRS89"]
        )

        dy = (
            geo_anio[
                "Y_ETRS89_validada"
            ].to_numpy()
            - estacion["Y_ETRS89"]
        )

        distancia = np.sqrt(
            dx ** 2
            + dy ** 2
        )

        temp = pd.DataFrame({
            "anio": anio,
            "estacion_geo":
                estacion["estacion_geo"],
            "Id_aforament":
                geo_anio[
                    "Id_aforament"
                ].to_numpy(),
            "distancia_m":
                distancia
        })

        relaciones.append(
            temp
        )


df_relaciones = pd.concat(
    relaciones,
    ignore_index=True
)


# ============================================================
# 16. LIMITAR AL RADIO MÁXIMO DE REFERENCIA
# ============================================================

RADIO_MAX_M = 1000

df_relaciones_1000 = (
    df_relaciones[
        df_relaciones[
            "distancia_m"
        ] <= RADIO_MAX_M
    ]
    .copy()
)

print("\n" + "=" * 80)
print("RELACIONES ESPACIALES ≤ 1.000 m")
print("=" * 80)

print(
    f"\n✓ Relaciones históricas dentro de 1 km: "
    f"{len(df_relaciones_1000):,}"
)


# ============================================================
# 17. INCORPORAR RELACIONES ESPACIALES AL TRÁFICO DIARIO
# ============================================================
# Cada registro diario de un aforo puede contribuir a varias
# estaciones si está dentro de 1 km de ellas.
# ============================================================

df_trafico_rel = pd.merge(
    df_trafico_geo,
    df_relaciones_1000,
    on=[
        "anio",
        "Id_aforament"
    ],
    how="inner",
    validate="many_to_many"
)

print(
    f"✓ Contribuciones diarias estación-aforo: "
    f"{len(df_trafico_rel):,}"
)


# ============================================================
# 18. DEFINIR PESO IDW
# ============================================================
# Peso inversamente proporcional a la distancia.
#
# w = 1 / d
#
# Todas las distancias observadas son > 0, pero se utiliza
# un mínimo de 1 m como protección numérica.
# ============================================================

df_trafico_rel["peso_idw"] = (
    1.0 /
    df_trafico_rel[
        "distancia_m"
    ].clip(lower=1.0)
)

df_trafico_rel[
    "imd_x_peso"
] = (
    df_trafico_rel["Valor_IMD"]
    *
    df_trafico_rel["peso_idw"]
)


# ============================================================
# 19. FUNCIÓN DE AGREGACIÓN POR RADIO
# ============================================================

def construir_indicadores_radio(
    df,
    radio
):

    temp = (
        df[
            df["distancia_m"]
            <= radio
        ]
        .copy()
    )

    if temp.empty:
        return pd.DataFrame()

    # --------------------------------------------------------
    # Agregación básica
    # --------------------------------------------------------

    agregado = (
        temp
        .groupby(
            [
                "fecha",
                "estacion_geo"
            ]
        )
        .agg(
            **{
                f"trafico_media_{radio}m":
                    (
                        "Valor_IMD",
                        "mean"
                    ),

                f"trafico_suma_{radio}m":
                    (
                        "Valor_IMD",
                        "sum"
                    ),

                f"n_aforos_obs_{radio}m":
                    (
                        "Id_aforament",
                        "nunique"
                    ),

                f"dist_aforo_min_{radio}m":
                    (
                        "distancia_m",
                        "min"
                    ),

                f"suma_pesos_{radio}m":
                    (
                        "peso_idw",
                        "sum"
                    ),

                f"suma_imd_pesos_{radio}m":
                    (
                        "imd_x_peso",
                        "sum"
                    )
            }
        )
        .reset_index()
    )

    # --------------------------------------------------------
    # IDW normalizado
    # --------------------------------------------------------

    agregado[
        f"trafico_idw_{radio}m"
    ] = (
        agregado[
            f"suma_imd_pesos_{radio}m"
        ]
        /
        agregado[
            f"suma_pesos_{radio}m"
        ]
    )

    agregado = agregado.drop(
        columns=[
            f"suma_pesos_{radio}m",
            f"suma_imd_pesos_{radio}m"
        ]
    )

    return agregado


# ============================================================
# 20. CALCULAR INDICADORES MULTIESCALA
# ============================================================

radios = [
    500,
    750,
    1000
]

indicadores_por_radio = {}

for radio in radios:

    indicadores_por_radio[
        radio
    ] = construir_indicadores_radio(
        df_trafico_rel,
        radio
    )

    print(
        f"\n✓ Radio {radio} m: "
        f"{len(indicadores_por_radio[radio]):,} "
        f"combinaciones fecha-estación"
    )


# ============================================================
# 21. UNIFICAR INDICADORES MULTIESCALA
# ============================================================

df_indicadores_trafico = None

for radio in radios:

    df_radio = (
        indicadores_por_radio[
            radio
        ]
    )

    if df_indicadores_trafico is None:

        df_indicadores_trafico = (
            df_radio.copy()
        )

    else:

        df_indicadores_trafico = pd.merge(
            df_indicadores_trafico,
            df_radio,
            on=[
                "fecha",
                "estacion_geo"
            ],
            how="outer",
            validate="one_to_one"
        )


# ============================================================
# 22. NÚMERO TOTAL DE AFOROS ESPACIALMENTE DISPONIBLES
# ============================================================
# Esta cantidad depende de estación + año, no del día.
# Permitirá medir la cobertura temporal efectiva.
# ============================================================

cobertura_espacial = []

for radio in radios:

    temp = (
        df_relaciones[
            df_relaciones[
                "distancia_m"
            ] <= radio
        ]
        .groupby(
            [
                "anio",
                "estacion_geo"
            ]
        )["Id_aforament"]
        .nunique()
        .reset_index(
            name=f"n_aforos_disponibles_{radio}m"
        )
    )

    cobertura_espacial.append(
        temp
    )


df_cobertura_espacial = (
    cobertura_espacial[0]
)

for temp in cobertura_espacial[1:]:

    df_cobertura_espacial = pd.merge(
        df_cobertura_espacial,
        temp,
        on=[
            "anio",
            "estacion_geo"
        ],
        how="outer",
        validate="one_to_one"
    )


# ============================================================
# 23. AÑADIR AÑO A LOS INDICADORES
# ============================================================

df_indicadores_trafico[
    "anio"
] = (
    df_indicadores_trafico[
        "fecha"
    ]
    .dt.year
    .astype("Int64")
)


# ============================================================
# 24. INCORPORAR COBERTURA ESPACIAL TEÓRICA
# ============================================================

df_indicadores_trafico = pd.merge(
    df_indicadores_trafico,
    df_cobertura_espacial,
    on=[
        "anio",
        "estacion_geo"
    ],
    how="left",
    validate="many_to_one"
)


# ============================================================
# 25. CALCULAR COBERTURA DIARIA
# ============================================================

for radio in radios:

    obs = (
        f"n_aforos_obs_{radio}m"
    )

    disponibles = (
        f"n_aforos_disponibles_{radio}m"
    )

    cobertura = (
        f"cobertura_aforos_{radio}m"
    )

    df_indicadores_trafico[
        cobertura
    ] = (
        df_indicadores_trafico[
            obs
        ]
        /
        df_indicadores_trafico[
            disponibles
        ]
    )


# ============================================================
# 26. CREAR VARIABLE PRINCIPAL DE DISTANCIA MÍNIMA
# ============================================================

df_indicadores_trafico[
    "dist_aforo_min_m"
] = (
    df_indicadores_trafico[
        "dist_aforo_min_1000m"
    ]
)


# ============================================================
# 27. ORDENAR
# ============================================================

df_indicadores_trafico = (
    df_indicadores_trafico
    .sort_values(
        [
            "fecha",
            "estacion_geo"
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# 28. COMPROBAR UNICIDAD DE LA CLAVE
# ============================================================

duplicados_indicadores = (
    df_indicadores_trafico
    .duplicated(
        subset=[
            "fecha",
            "estacion_geo"
        ]
    )
    .sum()
)

print("\n" + "=" * 80)
print("CONTROL DEL DATASET DE INDICADORES")
print("=" * 80)

print(
    f"\n✓ Filas: "
    f"{len(df_indicadores_trafico):,}"
)

print(
    f"✓ Columnas: "
    f"{df_indicadores_trafico.shape[1]}"
)

print(
    f"✓ Duplicados fecha + estacion_geo: "
    f"{duplicados_indicadores:,}"
)

if duplicados_indicadores > 0:

    raise ValueError(
        "Los indicadores no presentan "
        "una clave fecha + estación única."
    )


# ============================================================
# 29. COBERTURA TEMPORAL POR ESTACIÓN
# ============================================================

resumen_temporal = (
    df_indicadores_trafico
    .groupby("estacion_geo")
    .agg(
        fecha_inicio=(
            "fecha",
            "min"
        ),
        fecha_fin=(
            "fecha",
            "max"
        ),
        dias_con_trafico=(
            "fecha",
            "nunique"
        ),
        trafico_idw_1000m_mediana=(
            "trafico_idw_1000m",
            "median"
        ),
        cobertura_1000m_mediana=(
            "cobertura_aforos_1000m",
            "median"
        )
    )
    .reset_index()
)

print("\n" + "=" * 80)
print("COBERTURA TEMPORAL POR ESTACIÓN")
print("=" * 80)

display(
    resumen_temporal
)


# ============================================================
# 30. COBERTURA DE LOS INDICADORES MULTIESCALA
# ============================================================

resumen_variables = []

for radio in radios:

    columna = (
        f"trafico_idw_{radio}m"
    )

    n_validos = (
        df_indicadores_trafico[
            columna
        ]
        .notna()
        .sum()
    )

    resumen_variables.append({
        "radio_m": radio,
        "registros_validos": n_validos,
        "registros_totales":
            len(df_indicadores_trafico),
        "cobertura_pct":
            round(
                n_validos
                /
                len(df_indicadores_trafico)
                * 100,
                2
            )
    })


df_resumen_variables = pd.DataFrame(
    resumen_variables
)

print("\n" + "=" * 80)
print("COBERTURA DE INDICADORES IDW")
print("=" * 80)

display(
    df_resumen_variables
)


# ============================================================
# 31. ESTADÍSTICOS DE LOS INDICADORES PRINCIPALES
# ============================================================

columnas_principales = [
    "trafico_idw_500m",
    "trafico_idw_750m",
    "trafico_idw_1000m",
    "trafico_media_1000m",
    "trafico_suma_1000m",
    "n_aforos_obs_1000m",
    "cobertura_aforos_1000m",
    "dist_aforo_min_m"
]

columnas_principales = [
    columna
    for columna in columnas_principales
    if columna in df_indicadores_trafico.columns
]

print("\n" + "=" * 80)
print("ESTADÍSTICOS DE LOS INDICADORES PRINCIPALES")
print("=" * 80)

display(
    df_indicadores_trafico[
        columnas_principales
    ]
    .describe()
    .T
)


# ============================================================
# 32. CONTROL DE VALORES INFINITOS
# ============================================================

columnas_numericas = (
    df_indicadores_trafico
    .select_dtypes(
        include=[np.number]
    )
    .columns
)

n_inf = (
    np.isinf(
        df_indicadores_trafico[
            columnas_numericas
        ]
    )
    .sum()
    .sum()
)

print(
    f"\n✓ Valores infinitos detectados: "
    f"{n_inf:,}"
)


# ============================================================
# 33. CONTROL FINAL
# ============================================================

print("\n" + "=" * 80)
print("CONTROL FINAL 4.4")
print("=" * 80)

print(
    f"\n✓ Granularidad final: "
    f"fecha + estacion_geo"
)

print(
    f"✓ Radios calculados: "
    f"{radios}"
)

print(
    f"✓ Variable principal candidata: "
    f"trafico_idw_1000m"
)

print(
    f"✓ Duplicados de la clave: "
    f"{duplicados_indicadores:,}"
)

print(
    "✓ Se conserva información multiescala."
)

print(
    "✓ Se conserva el número de aforos observados."
)

print(
    "✓ Se incorpora cobertura diaria de la red."
)

print(
    "✓ No se ha modificado todavía "
    "el dataset maestro."
)

print(
    "\n✓ Preparado para auditar los indicadores "
    "antes de su integración."
)

4.4 — CONSTRUCCIÓN DE INDICADORES DIARIOS DE TRÁFICO

✓ Registros diarios de tráfico: 1,910,502
✓ Columnas disponibles: 9

Columnas del dataset diario:
  • Fecha
  • Any
  • Mes
  • Id_aforament
  • Codi_tipus_dia
  • Desc_tipus_dia
  • Valor_IMD_original
  • Valor_IMD
  • IMD_imputado

CONTROL TEMPORAL DEL TRÁFICO

✓ Fechas inválidas: 0
✓ Años inválidos: 0
✓ Periodo: 2018-01-01 00:00:00 → 2024-12-31 00:00:00

CONTROL DE Valor_IMD

✓ Valores nulos: 170,810
✓ Valores negativos: 0
✓ Valor mínimo: 0.00
✓ Mediana: 5,125.00
✓ Valor máximo: 503,838.00

✓ Duplicados fecha + Id_aforament: 0

ASIGNACIÓN ESPACIAL DEL TRÁFICO

✓ Filas antes del merge: 1,910,502
✓ Filas después del merge: 1,910,502
✓ Registros sin geolocalización: 0

RELACIONES ESPACIALES ≤ 1.000 m

✓ Relaciones históricas dentro de 1 km: 2,159
✓ Contribuciones diarias estación-aforo: 779,533

✓ Radio 500 m: 17,899 combinaciones fecha-estación

✓ Radio 750 m: 17,899 combinaciones fecha-estación

✓ Radio 1000 m: 20,456 combinacione

,estacion_geo,fecha_inicio,fecha_fin,dias_con_trafico,trafico_idw_1000m_mediana,cobertura_1000m_mediana
0,Ciutadella,2018-01-01,2024-12-31,2557,9230.444795,1.0
1,Eixample,2018-01-01,2024-12-31,2557,10434.416367,1.0
2,Gràcia,2018-01-01,2024-12-31,2557,10007.002253,1.0
3,Observatori Fabra,2018-01-01,2024-12-31,2557,63061.000000,1.0
4,Palau Reial,2018-01-01,2024-12-31,2557,7986.508254,1.0
5,Poblenou,2018-01-01,2024-12-31,2557,3359.044973,1.0
6,Sants,2018-01-01,2024-12-31,2557,8210.889585,1.0
7,Vall d'Hebron,2018-01-01,2024-12-31,2557,37913.246851,1.0



COBERTURA DE INDICADORES IDW


,radio_m,registros_validos,registros_totales,cobertura_pct
0,500,17899,20456,87.5
1,750,17899,20456,87.5
2,1000,20456,20456,100.0



ESTADÍSTICOS DE LOS INDICADORES PRINCIPALES


,count,mean,std,min,25%,50%,75%,max
trafico_idw_500m,17899.0,16071.301414,20907.153615,49.367917,5134.057299,9100.266456,13504.918003,8.059800e+04
trafico_idw_750m,17899.0,16118.085285,19543.434886,200.186678,6855.705130,9749.736971,12413.222813,8.230890e+04
trafico_idw_1000m,20456.0,17754.353584,19216.056887,0.000000,6855.813615,9575.066183,16571.194823,7.511300e+04
trafico_media_1000m,20275.0,18648.484726,18516.963112,527.437500,7913.425926,10430.004167,18123.253521,7.511300e+04
trafico_suma_1000m,20456.0,379564.417029,273697.557793,0.000000,172489.000000,311215.000000,463087.500000,1.321002e+06
n_aforos_obs_1000m,20456.0,38.107792,26.246046,2.000000,16.750000,42.000000,49.000000,1.040000e+02
cobertura_aforos_1000m,20456.0,0.989983,0.031368,0.846154,1.000000,1.000000,1.000000,1.000000e+00
dist_aforo_min_m,20456.0,310.131772,243.864672,111.348261,146.384323,226.751043,401.222427,8.964728e+02



✓ Valores infinitos detectados: 0

CONTROL FINAL 4.4

✓ Granularidad final: fecha + estacion_geo
✓ Radios calculados: [500, 750, 1000]
✓ Variable principal candidata: trafico_idw_1000m
✓ Duplicados de la clave: 0
✓ Se conserva información multiescala.
✓ Se conserva el número de aforos observados.
✓ Se incorpora cobertura diaria de la red.
✓ No se ha modificado todavía el dataset maestro.

✓ Preparado para auditar los indicadores antes de su integración.


#### Resultado — Construcción de indicadores diarios de presión de tráfico

La construcción de los indicadores diarios de tráfico parte de **1.910.502 observaciones**, correspondientes al periodo completo **2018–2024**, sin duplicados en la clave `fecha + Id_aforament` y con correspondencia espacial completa con el catálogo histórico validado.

La combinación de las observaciones temporales con las relaciones espaciales estación-aforo permite generar una tabla final con granularidad **`fecha + estacion_geo`**, formada por **20.456 combinaciones estación-día y sin duplicados**.

Se construyen indicadores de tráfico para radios de **500, 750 y 1.000 m**, incluyendo:

- intensidad media de tráfico;
- intensidad acumulada;
- intensidad ponderada mediante distancia inversa (IDW);
- número de aforos observados;
- cobertura diaria de la red;
- distancia al aforo disponible más próximo.

Los radios de 500 y 750 m proporcionan una cobertura del **87,5 %**, mientras que el radio de **1.000 m alcanza el 100 % de las combinaciones estación-día**, confirmando su idoneidad como escala espacial de referencia.

La variable `trafico_idw_1000m` se mantiene como **indicador principal candidato de presión de tráfico**, ya que combina cobertura espacial completa con una ponderación que otorga mayor influencia a los aforos situados más próximos a cada estación. Los indicadores correspondientes a escalas inferiores se conservan como variables candidatas para el posterior análisis multivariante y predictivo.

La cobertura efectiva de la red dentro de 1.000 m es elevada, con una cobertura diaria mediana igual a **1,00**, lo que indica que habitualmente se dispone de información para la totalidad de los aforos espacialmente asociados a cada estación.

La auditoría identifica, no obstante, valores ausentes en `Valor_IMD`. Por este motivo, antes de exportar los indicadores se realizará una validación específica destinada a diferenciar correctamente entre **ausencia de observación** y **tráfico realmente igual a cero**, evitando la introducción de falsos ceros durante la agregación ponderada.

> **Conclusión:** se obtiene una estructura diaria multiescala de indicadores de presión de tráfico, espacialmente consistente y con cobertura completa a 1.000 m. Antes de su incorporación al dataset maestro se verificará específicamente el tratamiento de los valores ausentes de intensidad de tráfico.

## 4.5. Auditoría de disponibilidad temporal y validación de los indicadores de tráfico

Antes de exportar e integrar los indicadores de tráfico se realiza una auditoría específica de su disponibilidad temporal y del tratamiento de los valores ausentes.

Aunque el radio de 1.000 m proporciona cobertura espacial completa, la variable `Valor_IMD` contiene observaciones ausentes. Esta situación debe diferenciarse de una intensidad de tráfico realmente igual a cero, ya que ambos casos tienen significados distintos desde el punto de vista analítico.

Se comprueba la relación entre `Valor_IMD_original`, `Valor_IMD` e `IMD_imputado`, la distribución temporal de los valores ausentes y la disponibilidad efectiva de observaciones para cada combinación `fecha + estacion_geo`.

Asimismo, se revisa el cálculo de los indicadores ponderados mediante distancia inversa (IDW). Cuando ningún aforo asociado a una estación dispone de un valor IMD válido en una fecha determinada, el indicador debe mantenerse como `NaN` y no convertirse artificialmente en cero.

No se realiza en esta etapa ninguna imputación adicional.

> **Objetivo:** garantizar que los indicadores de tráfico distingan correctamente entre ausencia de información y tráfico observado igual a cero, preservando la trazabilidad del dato antes de su integración con el dataset maestro.

In [ ]:
# ============================================================
# 4.5. AUDITORÍA DE DISPONIBILIDAD TEMPORAL Y
# VALIDACIÓN DE LOS INDICADORES DE TRÁFICO
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 1. CONTROL DE VARIABLES NECESARIAS
# ============================================================

variables_necesarias = [
    "Valor_IMD",
    "Valor_IMD_original",
    "IMD_imputado"
]

faltantes = [
    col for col in variables_necesarias
    if col not in df_trafico.columns
]

if faltantes:
    raise KeyError(
        "Faltan variables necesarias para la auditoría:\n"
        f"{faltantes}"
    )


print("=" * 80)
print("4.5 — AUDITORÍA Y VALIDACIÓN DE INDICADORES DE TRÁFICO")
print("=" * 80)


# ============================================================
# 2. NORMALIZAR VARIABLES NUMÉRICAS
# ============================================================

df_trafico["Valor_IMD"] = pd.to_numeric(
    df_trafico["Valor_IMD"],
    errors="coerce"
)

df_trafico["Valor_IMD_original"] = pd.to_numeric(
    df_trafico["Valor_IMD_original"],
    errors="coerce"
)


# ============================================================
# 3. DISPONIBILIDAD GENERAL DE IMD
# ============================================================

n_total = len(df_trafico)

n_imd_valido = (
    df_trafico["Valor_IMD"]
    .notna()
    .sum()
)

n_imd_nulo = (
    df_trafico["Valor_IMD"]
    .isna()
    .sum()
)

pct_imd_nulo = (
    n_imd_nulo /
    n_total *
    100
)

n_original_nulo = (
    df_trafico["Valor_IMD_original"]
    .isna()
    .sum()
)


print("\n" + "=" * 80)
print("DISPONIBILIDAD GENERAL DE IMD")
print("=" * 80)

print(f"\n✓ Registros totales: {n_total:,}")
print(f"✓ Valor_IMD válido: {n_imd_valido:,}")

print(
    f"✓ Valor_IMD nulo: "
    f"{n_imd_nulo:,} "
    f"({pct_imd_nulo:.2f} %)"
)

print(
    f"✓ Valor_IMD_original nulo: "
    f"{n_original_nulo:,}"
)


# ============================================================
# 4. AUDITAR LA VARIABLE IMD_imputado
# ============================================================

print("\n" + "=" * 80)
print("VARIABLE IMD_imputado")
print("=" * 80)

print(
    df_trafico["IMD_imputado"]
    .value_counts(dropna=False)
)


# ============================================================
# 5. RELACIÓN ORIGINAL / FINAL / IMPUTACIÓN
# ============================================================

df_control_imd = pd.DataFrame({
    "original_nulo":
        df_trafico["Valor_IMD_original"].isna(),

    "final_nulo":
        df_trafico["Valor_IMD"].isna(),

    "IMD_imputado":
        df_trafico["IMD_imputado"]
})

tabla_control_imd = (
    df_control_imd
    .groupby(
        [
            "original_nulo",
            "final_nulo",
            "IMD_imputado"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="n_registros")
)


print("\n" + "=" * 80)
print("RELACIÓN ORIGINAL / FINAL / IMPUTACIÓN")
print("=" * 80)

display(tabla_control_imd)


# ============================================================
# 6. DISPONIBILIDAD POR AÑO
# ============================================================

resumen_nulos_anio = (
    df_trafico
    .groupby("anio")
    .agg(
        registros=("Valor_IMD", "size"),
        imd_validos=("Valor_IMD", "count")
    )
    .reset_index()
)

resumen_nulos_anio["imd_nulos"] = (
    resumen_nulos_anio["registros"]
    - resumen_nulos_anio["imd_validos"]
)

resumen_nulos_anio["pct_nulos"] = (
    resumen_nulos_anio["imd_nulos"]
    /
    resumen_nulos_anio["registros"]
    * 100
).round(2)


print("\n" + "=" * 80)
print("DISPONIBILIDAD DE IMD POR AÑO")
print("=" * 80)

display(resumen_nulos_anio)


# ============================================================
# 7. CONTROL DE RELACIONES ESTACIÓN-AFORO
# ============================================================

if "df_trafico_rel" not in globals():
    raise NameError(
        "No existe df_trafico_rel. "
        "Debe ejecutarse previamente la celda 4.4."
    )

df_trafico_rel = df_trafico_rel.copy()

df_trafico_rel["imd_valido"] = (
    df_trafico_rel["Valor_IMD"]
    .notna()
)


# ============================================================
# 8. DISPONIBILIDAD DIARIA DENTRO DE 1.000 m
# ============================================================

disponibilidad_1000 = (
    df_trafico_rel
    .groupby(
        [
            "fecha",
            "estacion_geo"
        ]
    )
    .agg(
        n_aforos_total_1000m=(
            "Id_aforament",
            "nunique"
        ),

        n_imd_validos_1000m=(
            "imd_valido",
            "sum"
        )
    )
    .reset_index()
)

disponibilidad_1000["sin_imd_valido_1000m"] = (
    disponibilidad_1000[
        "n_imd_validos_1000m"
    ] == 0
)

n_sin_imd_1000 = (
    disponibilidad_1000[
        "sin_imd_valido_1000m"
    ]
    .sum()
)


print("\n" + "=" * 80)
print("DISPONIBILIDAD DIARIA DENTRO DE 1.000 m")
print("=" * 80)

print(
    f"\n✓ Combinaciones estación-día: "
    f"{len(disponibilidad_1000):,}"
)

print(
    f"✓ Casos sin ningún IMD válido: "
    f"{n_sin_imd_1000:,}"
)

print(
    f"✓ Casos con ≥1 IMD válido: "
    f"{len(disponibilidad_1000) - n_sin_imd_1000:,}"
)


# ============================================================
# 9. DISTRIBUCIÓN DE AFOROS CON IMD VÁLIDO
# ============================================================

print("\n" + "=" * 80)
print("AFOROS CON IMD VÁLIDO POR ESTACIÓN-DÍA")
print("=" * 80)

display(
    disponibilidad_1000[
        "n_imd_validos_1000m"
    ]
    .describe()
    .to_frame(
        name="n_imd_validos_1000m"
    )
)


# ============================================================
# 10. DETECTAR FALSOS CEROS DEL IDW ACTUAL
# ============================================================

control_ceros = pd.merge(
    df_indicadores_trafico[
        [
            "fecha",
            "estacion_geo",
            "trafico_idw_1000m"
        ]
    ],

    disponibilidad_1000[
        [
            "fecha",
            "estacion_geo",
            "n_imd_validos_1000m"
        ]
    ],

    on=[
        "fecha",
        "estacion_geo"
    ],

    how="left",
    validate="one_to_one"
)


falsos_ceros = (
    (
        control_ceros["trafico_idw_1000m"] == 0
    )
    &
    (
        control_ceros["n_imd_validos_1000m"] == 0
    )
)

n_falsos_ceros = falsos_ceros.sum()


print("\n" + "=" * 80)
print("CONTROL DE POSIBLES FALSOS CEROS")
print("=" * 80)

print(
    f"\n✓ IDW 1000 m igual a 0: "
    f"{(control_ceros['trafico_idw_1000m'] == 0).sum():,}"
)

print(
    f"✓ Ceros sin ningún IMD válido: "
    f"{n_falsos_ceros:,}"
)


if n_falsos_ceros > 0:

    print(
        "\nPrimeros casos detectados:"
    )

    display(
        control_ceros.loc[
            falsos_ceros
        ]
        .head(20)
    )


# ============================================================
# 11. FUNCIÓN DE AGREGACIÓN IDW VALIDADA
# ============================================================
# Solo intervienen en el cálculo los aforos que disponen
# realmente de Valor_IMD.
#
# Si una estación-día no dispone de ningún IMD válido,
# la intensidad permanece como NaN.
# ============================================================

def construir_indicadores_radio_validado(
    df,
    radio
):

    temp = (
        df[
            df["distancia_m"] <= radio
        ]
        .copy()
    )

    if temp.empty:
        return pd.DataFrame()

    # --------------------------------------------------------
    # Total de aforos espacialmente presentes ese día
    # --------------------------------------------------------

    total_aforos = (
        temp
        .groupby(
            [
                "fecha",
                "estacion_geo"
            ]
        )["Id_aforament"]
        .nunique()
        .reset_index(
            name=f"n_aforos_total_{radio}m"
        )
    )

    # --------------------------------------------------------
    # Solo observaciones con IMD válido
    # --------------------------------------------------------

    validos = (
        temp[
            temp["Valor_IMD"].notna()
        ]
        .copy()
    )

    validos["peso_idw_validado"] = (
        1.0 /
        validos["distancia_m"]
        .clip(lower=1.0)
    )

    validos["imd_x_peso_validado"] = (
        validos["Valor_IMD"]
        *
        validos["peso_idw_validado"]
    )

    # --------------------------------------------------------
    # Agregación
    # --------------------------------------------------------

    agregado = (
        validos
        .groupby(
            [
                "fecha",
                "estacion_geo"
            ]
        )
        .agg(
            **{
                f"trafico_media_{radio}m":
                    ("Valor_IMD", "mean"),

                f"trafico_suma_{radio}m":
                    ("Valor_IMD", "sum"),

                f"n_aforos_obs_{radio}m":
                    ("Id_aforament", "nunique"),

                f"dist_aforo_min_{radio}m":
                    ("distancia_m", "min"),

                "_suma_pesos":
                    ("peso_idw_validado", "sum"),

                "_suma_imd_pesos":
                    ("imd_x_peso_validado", "sum")
            }
        )
        .reset_index()
    )

    agregado[
        f"trafico_idw_{radio}m"
    ] = (
        agregado["_suma_imd_pesos"]
        /
        agregado["_suma_pesos"]
    )

    agregado = agregado.drop(
        columns=[
            "_suma_pesos",
            "_suma_imd_pesos"
        ]
    )

    # --------------------------------------------------------
    # Recuperar estación-día aunque no tenga IMD válido
    # --------------------------------------------------------

    resultado = pd.merge(
        total_aforos,
        agregado,

        on=[
            "fecha",
            "estacion_geo"
        ],

        how="left",
        validate="one_to_one"
    )

    # n_aforos_obs sí puede expresarse como 0.
    # Los indicadores de intensidad permanecen NaN.

    columna_obs = (
        f"n_aforos_obs_{radio}m"
    )

    resultado[columna_obs] = (
        resultado[columna_obs]
        .fillna(0)
        .astype(int)
    )

    return resultado


# ============================================================
# 12. RECALCULAR INDICADORES MULTIESCALA
# ============================================================

radios = [
    500,
    750,
    1000
]

indicadores_validados = {}

for radio in radios:

    indicadores_validados[radio] = (
        construir_indicadores_radio_validado(
            df_trafico_rel,
            radio
        )
    )

    print(
        f"\n✓ Radio {radio} m recalculado: "
        f"{len(indicadores_validados[radio]):,} "
        f"combinaciones estación-día"
    )


# ============================================================
# 13. UNIFICAR INDICADORES VALIDADOS
# ============================================================

df_indicadores_trafico_validado = None

for radio in radios:

    temp = indicadores_validados[radio]

    if df_indicadores_trafico_validado is None:

        df_indicadores_trafico_validado = (
            temp.copy()
        )

    else:

        df_indicadores_trafico_validado = pd.merge(
            df_indicadores_trafico_validado,
            temp,

            on=[
                "fecha",
                "estacion_geo"
            ],

            how="outer",
            validate="one_to_one"
        )


# ============================================================
# 14. AÑADIR AÑO
# ============================================================

df_indicadores_trafico_validado["anio"] = (
    df_indicadores_trafico_validado["fecha"]
    .dt.year
    .astype("Int64")
)


# ============================================================
# 15. INCORPORAR COBERTURA ESPACIAL TEÓRICA
# ============================================================

df_indicadores_trafico_validado = pd.merge(
    df_indicadores_trafico_validado,
    df_cobertura_espacial,

    on=[
        "anio",
        "estacion_geo"
    ],

    how="left",
    validate="many_to_one"
)


# ============================================================
# 16. COBERTURA EFECTIVA
# ============================================================

for radio in radios:

    obs = (
        f"n_aforos_obs_{radio}m"
    )

    disponibles = (
        f"n_aforos_disponibles_{radio}m"
    )

    cobertura = (
        f"cobertura_aforos_{radio}m"
    )

    df_indicadores_trafico_validado[cobertura] = (
        df_indicadores_trafico_validado[obs]
        /
        df_indicadores_trafico_validado[disponibles]
    )


# ============================================================
# 17. DISTANCIA MÍNIMA DE REFERENCIA
# ============================================================

df_indicadores_trafico_validado[
    "dist_aforo_min_m"
] = (
    df_indicadores_trafico_validado[
        "dist_aforo_min_1000m"
    ]
)


# ============================================================
# 18. ORDENAR DATASET
# ============================================================

df_indicadores_trafico_validado = (
    df_indicadores_trafico_validado
    .sort_values(
        [
            "fecha",
            "estacion_geo"
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# 19. CONTROL DE FALSOS CEROS TRAS LA CORRECCIÓN
# ============================================================

control_validado = pd.merge(
    df_indicadores_trafico_validado[
        [
            "fecha",
            "estacion_geo",
            "trafico_idw_1000m",
            "n_aforos_obs_1000m"
        ]
    ],

    disponibilidad_1000[
        [
            "fecha",
            "estacion_geo",
            "n_imd_validos_1000m"
        ]
    ],

    on=[
        "fecha",
        "estacion_geo"
    ],

    how="left",
    validate="one_to_one"
)


falsos_ceros_final = (
    (
        control_validado[
            "trafico_idw_1000m"
        ] == 0
    )
    &
    (
        control_validado[
            "n_imd_validos_1000m"
        ] == 0
    )
)


print("\n" + "=" * 80)
print("CONTROL DEL IDW CORREGIDO")
print("=" * 80)

print(
    f"\n✓ Falsos ceros antes: "
    f"{n_falsos_ceros:,}"
)

print(
    f"✓ Falsos ceros después: "
    f"{falsos_ceros_final.sum():,}"
)

print(
    f"✓ NaN en trafico_idw_1000m: "
    f"{df_indicadores_trafico_validado['trafico_idw_1000m'].isna().sum():,}"
)


# ============================================================
# 20. COBERTURA FINAL POR RADIO
# ============================================================

resumen_cobertura_final = []

for radio in radios:

    columna = (
        f"trafico_idw_{radio}m"
    )

    validos = (
        df_indicadores_trafico_validado[
            columna
        ]
        .notna()
        .sum()
    )

    total = len(
        df_indicadores_trafico_validado
    )

    resumen_cobertura_final.append({

        "radio_m":
            radio,

        "registros_validos":
            validos,

        "registros_totales":
            total,

        "registros_sin_dato":
            total - validos,

        "cobertura_pct":
            round(
                validos /
                total *
                100,
                2
            )
    })


df_cobertura_final = pd.DataFrame(
    resumen_cobertura_final
)

print("\n" + "=" * 80)
print("COBERTURA FINAL DE LOS INDICADORES")
print("=" * 80)

display(
    df_cobertura_final
)


# ============================================================
# 21. COBERTURA FINAL POR ESTACIÓN — 1.000 m
# ============================================================

resumen_estacion_final = (
    df_indicadores_trafico_validado
    .groupby("estacion_geo")
    .agg(
        dias_totales=(
            "fecha",
            "size"
        ),

        dias_con_idw_1000m=(
            "trafico_idw_1000m",
            "count"
        ),

        trafico_idw_mediana=(
            "trafico_idw_1000m",
            "median"
        ),

        cobertura_aforos_mediana=(
            "cobertura_aforos_1000m",
            "median"
        )
    )
    .reset_index()
)

resumen_estacion_final[
    "dias_sin_idw_1000m"
] = (
    resumen_estacion_final["dias_totales"]
    -
    resumen_estacion_final["dias_con_idw_1000m"]
)

resumen_estacion_final[
    "cobertura_temporal_pct"
] = (
    resumen_estacion_final["dias_con_idw_1000m"]
    /
    resumen_estacion_final["dias_totales"]
    * 100
).round(2)


print("\n" + "=" * 80)
print("COBERTURA FINAL POR ESTACIÓN — 1.000 m")
print("=" * 80)

display(
    resumen_estacion_final
)


# ============================================================
# 22. UNICIDAD Y VALORES INFINITOS
# ============================================================

duplicados_finales = (
    df_indicadores_trafico_validado
    .duplicated(
        subset=[
            "fecha",
            "estacion_geo"
        ]
    )
    .sum()
)


columnas_numericas = (
    df_indicadores_trafico_validado
    .select_dtypes(
        include=[np.number]
    )
    .columns
)

n_inf = (
    np.isinf(
        df_indicadores_trafico_validado[
            columnas_numericas
        ]
    )
    .sum()
    .sum()
)


# ============================================================
# 23. CONTROL FINAL
# ============================================================

print("\n" + "=" * 80)
print("CONTROL FINAL 4.5")
print("=" * 80)

print(
    f"\n✓ Filas del dataset validado: "
    f"{len(df_indicadores_trafico_validado):,}"
)

print(
    f"✓ Duplicados fecha + estacion_geo: "
    f"{duplicados_finales:,}"
)

print(
    f"✓ Valores infinitos: "
    f"{n_inf:,}"
)

print(
    f"✓ Falsos ceros IDW eliminados: "
    f"{n_falsos_ceros:,}"
)

print(
    "✓ Ausencia de IMD y tráfico cero "
    "quedan diferenciados."
)

print(
    "✓ No se ha realizado imputación adicional."
)

print(
    "✓ No se ha modificado el dataset maestro."
)

print(
    "\n✓ Indicadores preparados para su "
    "exportación e integración."
)

4.5 — AUDITORÍA Y VALIDACIÓN DE INDICADORES DE TRÁFICO

DISPONIBILIDAD GENERAL DE IMD

✓ Registros totales: 1,910,502
✓ Valor_IMD válido: 1,739,692
✓ Valor_IMD nulo: 170,810 (8.94 %)
✓ Valor_IMD_original nulo: 228,436

VARIABLE IMD_imputado
IMD_imputado
0    1852876
1      57626
Name: count, dtype: int64

RELACIÓN ORIGINAL / FINAL / IMPUTACIÓN


,original_nulo,final_nulo,IMD_imputado,n_registros
0,False,False,0,1682066
1,True,False,1,57626
2,True,True,0,170810



DISPONIBILIDAD DE IMD POR AÑO


,anio,registros,imd_validos,imd_nulos,pct_nulos
0,2018,223745,184227,39518,17.66
1,2019,250025,221351,28674,11.47
2,2020,261324,248820,12504,4.78
3,2021,264260,248786,15474,5.86
4,2022,300760,270907,29853,9.93
5,2023,303680,283747,19933,6.56
6,2024,306708,281854,24854,8.10



DISPONIBILIDAD DIARIA DENTRO DE 1.000 m

✓ Combinaciones estación-día: 20,456
✓ Casos sin ningún IMD válido: 181
✓ Casos con ≥1 IMD válido: 20,275

AFOROS CON IMD VÁLIDO POR ESTACIÓN-DÍA


...


✓ Radio 500 m recalculado: 17,899 combinaciones estación-día

✓ Radio 750 m recalculado: 17,899 combinaciones estación-día

✓ Radio 1000 m recalculado: 20,456 combinaciones estación-día

CONTROL DEL IDW CORREGIDO

✓ Falsos ceros antes: 181
✓ Falsos ceros después: 0
✓ NaN en trafico_idw_1000m: 181

COBERTURA FINAL DE LOS INDICADORES


,radio_m,registros_validos,registros_totales,registros_sin_dato,cobertura_pct
0,500,17899,20456,2557,87.50
1,750,17899,20456,2557,87.50
2,1000,20275,20456,181,99.12



COBERTURA FINAL POR ESTACIÓN — 1.000 m


,estacion_geo,dias_totales,dias_con_idw_1000m,trafico_idw_mediana,cobertura_aforos_mediana,dias_sin_idw_1000m,cobertura_temporal_pct
0,Ciutadella,2557,2557,10048.152902,0.936170,0,100.00
1,Eixample,2557,2557,11798.640191,0.940476,0,100.00
2,Gràcia,2557,2557,10884.958950,0.934426,0,100.00
3,Observatori Fabra,2557,2376,64179.000000,1.000000,181,92.92
4,Palau Reial,2557,2557,9543.017406,0.904762,0,100.00
5,Poblenou,2557,2557,3436.377106,0.976744,0,100.00
6,Sants,2557,2557,9263.826519,0.894737,0,100.00
7,Vall d'Hebron,2557,2557,42243.749372,0.900000,0,100.00



CONTROL FINAL 4.5

✓ Filas del dataset validado: 20,456
✓ Duplicados fecha + estacion_geo: 0
✓ Valores infinitos: 0
✓ Falsos ceros IDW eliminados: 181
✓ Ausencia de IMD y tráfico cero quedan diferenciados.
✓ No se ha realizado imputación adicional.
✓ No se ha modificado el dataset maestro.

✓ Indicadores preparados para su exportación e integración.


#### Resultado — Auditoría y validación de los indicadores de tráfico

La auditoría confirma que el dataset diario de tráfico contiene **1.910.502 registros**, de los cuales **1.739.692 (91,06 %)** disponen de un valor IMD válido. Los **170.810 registros restantes (8,94 %)** conservan su ausencia de información después de la fase previa de limpieza e imputación.

La disponibilidad temporal no es homogénea entre años, por lo que se descarta realizar en esta etapa una imputación adicional indiscriminada. Se preservan los valores ausentes para mantener la trazabilidad del dato y permitir su tratamiento posterior dentro del pipeline de modelado.

El análisis a escala estación-día identifica **181 combinaciones sin ningún IMD válido dentro del radio de 1.000 m**. La formulación inicial del indicador IDW interpretaba estos casos como tráfico igual a cero debido al comportamiento de la operación de agregación.

La reformulación del cálculo IDW permite distinguir explícitamente entre ambas situaciones: cuando no existe ningún aforo con IMD válido, el indicador permanece como `NaN`. De este modo se eliminan los **181 falsos ceros detectados** sin introducir ninguna imputación adicional.

Tras la corrección, `trafico_idw_1000m` dispone de información válida en **20.275 de las 20.456 combinaciones estación-día**, alcanzando una cobertura temporal del **99,12 %**. Las 181 ausencias restantes se concentran en Observatori Fabra, mientras que las otras siete localizaciones presentan cobertura temporal completa.

El dataset validado conserva una clave única `fecha + estacion_geo`, sin duplicados ni valores infinitos.

> **Conclusión:** los indicadores de tráfico quedan validados para su exportación. La ausencia de información y el tráfico observado igual a cero se encuentran correctamente diferenciados, manteniendo los valores ausentes sin imputación adicional y preservando la trazabilidad necesaria para las etapas posteriores de integración y modelado.

## 4.6. Exportación del dataset preintegrado de tráfico rodado

Una vez construidos y validados los indicadores diarios de presión de tráfico, se genera el archivo definitivo correspondiente al bloque de tráfico rodado.

El dataset exportado mantiene una granularidad única `fecha + estacion_geo` e incorpora los indicadores multiescala calculados para radios de 500, 750 y 1.000 m.

Se conserva `trafico_idw_1000m` como principal variable candidata de exposición al tráfico, junto con los indicadores complementarios de intensidad media, intensidad acumulada, número de aforos observados, cobertura de la red y distancia al aforo más próximo.

Los valores ausentes detectados durante la auditoría se mantienen como `NaN`, evitando introducir imputaciones adicionales antes de la integración y preservando la diferencia entre ausencia de información y tráfico realmente igual a cero.

Antes de cerrar el bloque se verifica nuevamente:

- conservación del número de registros;
- unicidad de la clave `fecha + estacion_geo`;
- ausencia de valores infinitos;
- conservación de los valores ausentes validados;
- integridad del archivo después de su escritura y recarga.

El archivo resultante constituye la salida oficial de la fase de preintegración del tráfico rodado y será utilizado posteriormente para su incorporación al dataset maestro.

> **Objetivo:** generar un dataset de indicadores diarios de tráfico validado, trazable y reproducible, preparado para su integración con contaminación atmosférica y meteorología.

In [ ]:
# ============================================================
# 4.6. EXPORTACIÓN DEL DATASET PREINTEGRADO
# DE TRÁFICO RODADO
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. COMPROBAR DATASET VALIDADO EN MEMORIA
# ============================================================

if "df_indicadores_trafico_validado" not in globals():
    raise NameError(
        "No existe df_indicadores_trafico_validado. "
        "Debe ejecutarse previamente la celda 4.5."
    )


print("=" * 80)
print("4.6 — EXPORTACIÓN DEL DATASET PREINTEGRADO DE TRÁFICO")
print("=" * 80)


# ============================================================
# 2. DEFINIR RUTA DE SALIDA
# ============================================================

ruta_base = Path(
    "/content/drive/MyDrive/TFM/"
    "11_Machine_Learning_Temporal"
)

ruta_trafico = (
    ruta_base /
    "PREINTEGRACION" /
    "02_Trafico_Rodado_y_Aforos"
)

ruta_salida = (
    ruta_trafico /
    "df_trafico_rodado_preintegrado.csv"
)


# ============================================================
# 3. COPIA DE TRABAJO
# ============================================================

df_exportar = (
    df_indicadores_trafico_validado
    .copy()
)


# ============================================================
# 4. ORDENAR DATASET
# ============================================================

df_exportar = (
    df_exportar
    .sort_values(
        [
            "fecha",
            "estacion_geo"
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# 5. CONTROL PREVIO — DIMENSIONES
# ============================================================

filas_antes = len(
    df_exportar
)

columnas_antes = (
    df_exportar.shape[1]
)


print("\n" + "=" * 80)
print("CONTROL PREVIO A LA EXPORTACIÓN")
print("=" * 80)

print(
    f"\n✓ Filas: "
    f"{filas_antes:,}"
)

print(
    f"✓ Columnas: "
    f"{columnas_antes}"
)

print(
    f"✓ Estaciones: "
    f"{df_exportar['estacion_geo'].nunique()}"
)

print(
    f"✓ Fechas diferentes: "
    f"{df_exportar['fecha'].nunique():,}"
)

print(
    f"✓ Periodo: "
    f"{df_exportar['fecha'].min()} "
    f"→ "
    f"{df_exportar['fecha'].max()}"
)


# ============================================================
# 6. CONTROL DE LA CLAVE
# ============================================================

duplicados_antes = (
    df_exportar
    .duplicated(
        subset=[
            "fecha",
            "estacion_geo"
        ]
    )
    .sum()
)


print(
    f"\n✓ Duplicados fecha + estacion_geo: "
    f"{duplicados_antes:,}"
)

if duplicados_antes > 0:
    raise ValueError(
        "El dataset presenta duplicados "
        "antes de la exportación."
    )


# ============================================================
# 7. CONTROL DE INFINITOS
# ============================================================

columnas_numericas = (
    df_exportar
    .select_dtypes(
        include=[np.number]
    )
    .columns
)

n_inf_antes = (
    np.isinf(
        df_exportar[
            columnas_numericas
        ]
    )
    .sum()
    .sum()
)


print(
    f"✓ Valores infinitos: "
    f"{n_inf_antes:,}"
)

if n_inf_antes > 0:
    raise ValueError(
        "Existen valores infinitos "
        "antes de la exportación."
    )


# ============================================================
# 8. CONTROL DE VARIABLES PRINCIPALES
# ============================================================

variables_principales = [
    "trafico_idw_500m",
    "trafico_idw_750m",
    "trafico_idw_1000m",
    "trafico_media_1000m",
    "trafico_suma_1000m",
    "n_aforos_obs_1000m",
    "n_aforos_disponibles_1000m",
    "cobertura_aforos_1000m",
    "dist_aforo_min_m"
]

variables_presentes = [
    columna
    for columna in variables_principales
    if columna in df_exportar.columns
]

variables_faltantes = [
    columna
    for columna in variables_principales
    if columna not in df_exportar.columns
]


print("\n" + "=" * 80)
print("VARIABLES PRINCIPALES")
print("=" * 80)

print(
    f"\n✓ Variables esperadas presentes: "
    f"{len(variables_presentes)}/"
    f"{len(variables_principales)}"
)

if variables_faltantes:

    print(
        "\nVariables faltantes:"
    )

    for columna in variables_faltantes:
        print(f"  • {columna}")

    raise KeyError(
        "Faltan variables principales "
        "antes de la exportación."
    )


# ============================================================
# 9. CONTROL ESPECÍFICO DEL IDW DE REFERENCIA
# ============================================================

nulos_idw_1000_antes = (
    df_exportar[
        "trafico_idw_1000m"
    ]
    .isna()
    .sum()
)

ceros_idw_1000_antes = (
    df_exportar[
        "trafico_idw_1000m"
    ]
    .eq(0)
    .sum()
)


print("\n" + "=" * 80)
print("CONTROL DEL INDICADOR PRINCIPAL")
print("=" * 80)

print(
    f"\n✓ NaN en trafico_idw_1000m: "
    f"{nulos_idw_1000_antes:,}"
)

print(
    f"✓ Valores iguales a 0: "
    f"{ceros_idw_1000_antes:,}"
)

print(
    f"✓ Cobertura válida: "
    f"{df_exportar['trafico_idw_1000m'].notna().mean() * 100:.2f} %"
)


# ============================================================
# 10. EXPORTAR
# ============================================================

ruta_trafico.mkdir(
    parents=True,
    exist_ok=True
)

df_exportar.to_csv(
    ruta_salida,
    index=False,
    encoding="utf-8"
)


# ============================================================
# 11. COMPROBAR EXISTENCIA DEL ARCHIVO
# ============================================================

if not ruta_salida.exists():
    raise FileNotFoundError(
        "El archivo no se ha generado correctamente."
    )


# ============================================================
# 12. RECARGAR ARCHIVO EXPORTADO
# ============================================================
# La recarga permite comprobar el archivo real que utilizará
# la siguiente fase, no únicamente el DataFrame en memoria.
# ============================================================

df_control_exportacion = pd.read_csv(
    ruta_salida,
    parse_dates=["fecha"],
    low_memory=False
)


# ============================================================
# 13. CONTROL POSTERIOR — DIMENSIONES
# ============================================================

filas_despues = len(
    df_control_exportacion
)

columnas_despues = (
    df_control_exportacion.shape[1]
)


print("\n" + "=" * 80)
print("CONTROL POSTERIOR A LA EXPORTACIÓN")
print("=" * 80)

print(
    f"\n✓ Filas antes: "
    f"{filas_antes:,}"
)

print(
    f"✓ Filas después: "
    f"{filas_despues:,}"
)

print(
    f"✓ Columnas antes: "
    f"{columnas_antes}"
)

print(
    f"✓ Columnas después: "
    f"{columnas_despues}"
)


if filas_antes != filas_despues:
    raise ValueError(
        "El número de filas ha cambiado "
        "durante la exportación."
    )

if columnas_antes != columnas_despues:
    raise ValueError(
        "El número de columnas ha cambiado "
        "durante la exportación."
    )


# ============================================================
# 14. CONTROL POSTERIOR — CLAVE
# ============================================================

duplicados_despues = (
    df_control_exportacion
    .duplicated(
        subset=[
            "fecha",
            "estacion_geo"
        ]
    )
    .sum()
)


print(
    f"\n✓ Duplicados después de recargar: "
    f"{duplicados_despues:,}"
)

if duplicados_despues > 0:
    raise ValueError(
        "Se detectaron duplicados "
        "después de la exportación."
    )


# ============================================================
# 15. CONTROL POSTERIOR — NaN DEL IDW
# ============================================================

nulos_idw_1000_despues = (
    df_control_exportacion[
        "trafico_idw_1000m"
    ]
    .isna()
    .sum()
)


print(
    f"✓ NaN IDW 1000 m antes: "
    f"{nulos_idw_1000_antes:,}"
)

print(
    f"✓ NaN IDW 1000 m después: "
    f"{nulos_idw_1000_despues:,}"
)


if (
    nulos_idw_1000_antes
    !=
    nulos_idw_1000_despues
):

    raise ValueError(
        "Ha cambiado el número de NaN "
        "durante la exportación."
    )


# ============================================================
# 16. CONTROL POSTERIOR — INFINITOS
# ============================================================

columnas_numericas_control = (
    df_control_exportacion
    .select_dtypes(
        include=[np.number]
    )
    .columns
)

n_inf_despues = (
    np.isinf(
        df_control_exportacion[
            columnas_numericas_control
        ]
    )
    .sum()
    .sum()
)


print(
    f"✓ Valores infinitos después: "
    f"{n_inf_despues:,}"
)


# ============================================================
# 17. TAMAÑO DEL ARCHIVO
# ============================================================

tamano_mb = (
    ruta_salida.stat().st_size
    /
    (1024 ** 2)
)


# ============================================================
# 18. CONTROL FINAL
# ============================================================

print("\n" + "=" * 80)
print("BLOQUE DE TRÁFICO RODADO CERRADO CORRECTAMENTE")
print("=" * 80)

print(
    f"\n✓ Registros conservados: "
    f"{filas_despues:,}"
)

print(
    f"✓ Variables disponibles: "
    f"{columnas_despues}"
)

print(
    f"✓ Estaciones: "
    f"{df_control_exportacion['estacion_geo'].nunique()}"
)

print(
    f"✓ Fechas diferentes: "
    f"{df_control_exportacion['fecha'].nunique():,}"
)

print(
    f"✓ Duplicados de la clave: "
    f"{duplicados_despues:,}"
)

print(
    f"✓ NaN conservados en trafico_idw_1000m: "
    f"{nulos_idw_1000_despues:,}"
)

print(
    f"✓ Valores infinitos: "
    f"{n_inf_despues:,}"
)

print(
    f"✓ Tamaño del archivo: "
    f"{tamano_mb:.2f} MB"
)

print(
    "✓ Ausencia de información y tráfico cero "
    "permanecen diferenciados."
)

print(
    "✓ No se ha realizado ninguna "
    "imputación adicional."
)

print(
    "✓ Integridad verificada tras "
    "la recarga del archivo."
)

print(
    "\n✓ Archivo generado:"
)

print(
    ruta_salida
)

4.6 — EXPORTACIÓN DEL DATASET PREINTEGRADO DE TRÁFICO

CONTROL PREVIO A LA EXPORTACIÓN

✓ Filas: 20,456
✓ Columnas: 28
✓ Estaciones: 8
✓ Fechas diferentes: 2,557
✓ Periodo: 2018-01-01 00:00:00 → 2024-12-31 00:00:00

✓ Duplicados fecha + estacion_geo: 0
✓ Valores infinitos: 0

VARIABLES PRINCIPALES

✓ Variables esperadas presentes: 9/9

CONTROL DEL INDICADOR PRINCIPAL

✓ NaN en trafico_idw_1000m: 181
✓ Valores iguales a 0: 0
✓ Cobertura válida: 99.12 %

CONTROL POSTERIOR A LA EXPORTACIÓN

✓ Filas antes: 20,456
✓ Filas después: 20,456
✓ Columnas antes: 28
✓ Columnas después: 28

✓ Duplicados después de recargar: 0
✓ NaN IDW 1000 m antes: 181
✓ NaN IDW 1000 m después: 181
✓ Valores infinitos después: 0

BLOQUE DE TRÁFICO RODADO CERRADO CORRECTAMENTE

✓ Registros conservados: 20,456
✓ Variables disponibles: 28
✓ Estaciones: 8
✓ Fechas diferentes: 2,557
✓ Duplicados de la clave: 0
✓ NaN conservados en trafico_idw_1000m: 181
✓ Valores infinitos: 0
✓ Tamaño del archivo: 5.44 MB
✓ Ausencia de 

#### Resultado — Exportación del dataset preintegrado de tráfico rodado

El bloque de tráfico rodado queda consolidado en un dataset preintegrado con granularidad única `fecha + estacion_geo`, compuesto por **20.456 registros y 28 variables**, correspondientes a las **8 localizaciones físicas** de calidad del aire y al periodo comprendido entre **2018 y 2024**.

La exportación conserva íntegramente la estructura validada previamente:

- **20.456 registros** antes y después de la escritura;
- **28 variables**;
- **2.557 fechas diferentes**;
- **0 duplicados** en la clave `fecha + estacion_geo`;
- **0 valores infinitos**;
- presencia de las **9 variables principales de tráfico previstas**.

El indicador de referencia `trafico_idw_1000m` presenta una cobertura temporal del **99,12 %**, manteniendo **181 valores ausentes** correspondientes a combinaciones estación-día sin ninguna observación IMD válida. Estos registros permanecen correctamente representados como `NaN`, mientras que no se detecta ningún valor artificialmente igual a cero.

La recarga del archivo exportado reproduce exactamente las dimensiones, la unicidad de la clave y el número de valores ausentes del dataset en memoria, confirmando que no se ha producido pérdida ni alteración de información durante la escritura.

No se realiza ninguna imputación adicional en esta etapa, preservando la trazabilidad de los datos originales y dejando el tratamiento de las ausencias residuales para las etapas posteriores del pipeline analítico y predictivo.

> **Conclusión:** el bloque de tráfico rodado queda cerrado con un dataset espacial y temporalmente validado, sin duplicados ni valores infinitos y con diferenciación explícita entre ausencia de información y tráfico observado igual a cero. El archivo queda preparado para su incorporación al dataset maestro de contaminación atmosférica y meteorología.

## 4.7. Integración del tráfico rodado y generación del Checkpoint 03

Una vez finalizada la preintegración y validación del bloque de tráfico rodado, los indicadores diarios se incorporan al dataset maestro formado previamente por contaminación atmosférica y meteorología.

La integración se realiza mediante la clave común `fecha + estacion_geo`. El dataset de tráfico presenta granularidad única a nivel estación-día, mientras que el maestro contiene diferentes contaminantes para una misma estación y fecha. Por este motivo, la relación esperada es de tipo **many-to-one**, de forma que los indicadores diarios de tráfico se replican únicamente entre los contaminantes correspondientes a la misma estación y fecha, sin modificar el número de observaciones del maestro.

Se utilizan como archivos de entrada:

- `02_maestro_contaminacion_meteorologia.csv`, correspondiente al Checkpoint 02;
- `df_trafico_rodado_preintegrado.csv`, correspondiente al bloque de tráfico validado.

Durante la integración se comprueba:

- unicidad de `fecha + estacion_geo` en el dataset de tráfico;
- conservación exacta del número de registros del maestro;
- ausencia de nuevos duplicados en la clave `fecha + estacion_fisica + contaminant`;
- incorporación correcta de las variables de tráfico;
- cobertura temporal de `trafico_idw_1000m`;
- conservación de los valores ausentes previamente validados;
- integridad del archivo después de su exportación y recarga.

El resultado se almacena como un nuevo checkpoint dentro de la carpeta `INTEGRACION`, manteniendo inalterados los archivos correspondientes a las etapas anteriores.

> **Objetivo:** incorporar los indicadores diarios de presión de tráfico al dataset maestro sin alterar su estructura temporal, espacial ni la trazabilidad de las fuentes previamente integradas.

In [ ]:
# ============================================================
# 4.7. INTEGRACIÓN DEL TRÁFICO RODADO
# CHECKPOINT 03
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. RUTAS
# ============================================================

ruta_base = Path(
    "/content/drive/MyDrive/TFM/"
    "11_Machine_Learning_Temporal"
)

ruta_integracion = (
    ruta_base /
    "INTEGRACION"
)

ruta_maestro_02 = (
    ruta_integracion /
    "02_maestro_contaminacion_meteorologia.csv"
)

ruta_trafico = (
    ruta_base /
    "PREINTEGRACION" /
    "02_Trafico_Rodado_y_Aforos" /
    "df_trafico_rodado_preintegrado.csv"
)

ruta_maestro_03 = (
    ruta_integracion /
    "03_maestro_contaminacion_meteorologia_trafico.csv"
)


# ============================================================
# 2. COMPROBAR EXISTENCIA DE ARCHIVOS
# ============================================================

for ruta in [
    ruta_maestro_02,
    ruta_trafico
]:

    if not ruta.exists():

        raise FileNotFoundError(
            f"No se encuentra el archivo:\n{ruta}"
        )


# ============================================================
# 3. CARGAR DATASET MAESTRO — CHECKPOINT 02
# ============================================================

df_maestro_02 = pd.read_csv(
    ruta_maestro_02,
    low_memory=False
)

df_maestro_02["fecha"] = pd.to_datetime(
    df_maestro_02["fecha"],
    errors="coerce"
)


# ============================================================
# 4. CARGAR DATASET PREINTEGRADO DE TRÁFICO
# ============================================================

df_trafico_pre = pd.read_csv(
    ruta_trafico,
    low_memory=False
)

df_trafico_pre["fecha"] = pd.to_datetime(
    df_trafico_pre["fecha"],
    errors="coerce"
)


# ============================================================
# 5. INFORMACIÓN GENERAL
# ============================================================

print("=" * 80)
print("4.7 — INTEGRACIÓN DEL TRÁFICO RODADO — CHECKPOINT 03")
print("=" * 80)

print("\nCHECKPOINT 02:")

print(
    f"✓ Filas: "
    f"{len(df_maestro_02):,}"
)

print(
    f"✓ Columnas: "
    f"{df_maestro_02.shape[1]}"
)

print("\nDATASET PREINTEGRADO DE TRÁFICO:")

print(
    f"✓ Filas: "
    f"{len(df_trafico_pre):,}"
)

print(
    f"✓ Columnas: "
    f"{df_trafico_pre.shape[1]}"
)


# ============================================================
# 6. CONTROL DE LAS CLAVES DE INTEGRACIÓN
# ============================================================

claves_merge = [
    "fecha",
    "estacion_geo"
]

for nombre, df in [
    ("maestro", df_maestro_02),
    ("tráfico", df_trafico_pre)
]:

    faltantes = [
        col
        for col in claves_merge
        if col not in df.columns
    ]

    if faltantes:

        raise KeyError(
            f"Faltan claves en {nombre}: "
            f"{faltantes}"
        )


# ============================================================
# 7. CONTROL DE FECHAS INVÁLIDAS
# ============================================================

fechas_invalidas_maestro = (
    df_maestro_02["fecha"]
    .isna()
    .sum()
)

fechas_invalidas_trafico = (
    df_trafico_pre["fecha"]
    .isna()
    .sum()
)


print("\n" + "=" * 80)
print("CONTROL DE FECHAS")
print("=" * 80)

print(
    f"\n✓ Fechas inválidas en maestro: "
    f"{fechas_invalidas_maestro:,}"
)

print(
    f"✓ Fechas inválidas en tráfico: "
    f"{fechas_invalidas_trafico:,}"
)


if (
    fechas_invalidas_maestro > 0
    or
    fechas_invalidas_trafico > 0
):

    raise ValueError(
        "Existen fechas inválidas antes del merge."
    )


# ============================================================
# 8. UNICIDAD DEL DATASET DE TRÁFICO
# ============================================================

duplicados_trafico = (
    df_trafico_pre
    .duplicated(
        subset=claves_merge
    )
    .sum()
)


print("\n" + "=" * 80)
print("CARDINALIDAD DEL DATASET DE TRÁFICO")
print("=" * 80)

print(
    f"\n✓ Duplicados fecha + estacion_geo: "
    f"{duplicados_trafico:,}"
)


if duplicados_trafico > 0:

    raise ValueError(
        "El dataset de tráfico no presenta "
        "una clave fecha + estacion_geo única."
    )


# ============================================================
# 9. CONTROL DEL MAESTRO ANTES DEL MERGE
# ============================================================

clave_maestro = [
    "fecha",
    "estacion_fisica",
    "contaminant"
]

faltantes_clave_maestro = [
    col
    for col in clave_maestro
    if col not in df_maestro_02.columns
]

if faltantes_clave_maestro:

    raise KeyError(
        "Faltan columnas de la clave principal "
        "del maestro:\n"
        f"{faltantes_clave_maestro}"
    )


duplicados_maestro_antes = (
    df_maestro_02
    .duplicated(
        subset=clave_maestro
    )
    .sum()
)

filas_antes = len(
    df_maestro_02
)


print("\n" + "=" * 80)
print("CONTROL DEL MAESTRO ANTES DE INTEGRAR")
print("=" * 80)

print(
    f"\n✓ Filas: "
    f"{filas_antes:,}"
)

print(
    f"✓ Duplicados clave principal: "
    f"{duplicados_maestro_antes:,}"
)


if duplicados_maestro_antes > 0:

    raise ValueError(
        "El Checkpoint 02 presenta duplicados "
        "antes de integrar tráfico."
    )


# ============================================================
# 10. COMPROBAR SOLAPAMIENTO DE COLUMNAS
# ============================================================
# Solo deben compartirse las claves de integración.
# Si existen otras columnas comunes, se detiene el proceso
# para evitar sufijos _x / _y silenciosos.
# ============================================================

columnas_comunes = (
    set(df_maestro_02.columns)
    &
    set(df_trafico_pre.columns)
)

columnas_comunes_no_clave = sorted(
    columnas_comunes
    -
    set(claves_merge)
)


print("\n" + "=" * 80)
print("CONTROL DE COLUMNAS COMUNES")
print("=" * 80)

print(
    f"\n✓ Columnas comunes totales: "
    f"{len(columnas_comunes)}"
)

print(
    f"✓ Columnas comunes fuera de la clave: "
    f"{len(columnas_comunes_no_clave)}"
)


if columnas_comunes_no_clave:

    print(
        "\nColumnas comunes no esperadas:"
    )

    for col in columnas_comunes_no_clave:
        print(f"  • {col}")

    raise ValueError(
        "Existen columnas comunes fuera de "
        "fecha + estacion_geo. "
        "Revisar antes de realizar el merge."
    )


# ============================================================
# 11. COBERTURA DE LA CLAVE ANTES DEL MERGE
# ============================================================
# Comprobamos qué combinaciones fecha-estación del maestro
# tienen correspondencia en el dataset de tráfico.
# ============================================================

claves_maestro = (
    df_maestro_02[
        claves_merge
    ]
    .drop_duplicates()
)

claves_trafico = (
    df_trafico_pre[
        claves_merge
    ]
    .drop_duplicates()
)

control_cobertura = pd.merge(
    claves_maestro,
    claves_trafico.assign(
        disponible_trafico=True
    ),
    on=claves_merge,
    how="left",
    validate="one_to_one"
)

control_cobertura[
    "disponible_trafico"
] = (
    control_cobertura[
        "disponible_trafico"
    ]
    .fillna(False)
    .astype(bool)
)

n_claves_maestro = len(
    control_cobertura
)

n_claves_con_trafico = (
    control_cobertura[
        "disponible_trafico"
    ]
    .sum()
)

n_claves_sin_trafico = (
    n_claves_maestro
    -
    n_claves_con_trafico
)


print("\n" + "=" * 80)
print("COBERTURA DE LA CLAVE DE INTEGRACIÓN")
print("=" * 80)

print(
    f"\n✓ Combinaciones fecha-estación del maestro: "
    f"{n_claves_maestro:,}"
)

print(
    f"✓ Con correspondencia en tráfico: "
    f"{n_claves_con_trafico:,}"
)

print(
    f"✓ Sin correspondencia en tráfico: "
    f"{n_claves_sin_trafico:,}"
)

print(
    f"✓ Cobertura de la clave: "
    f"{n_claves_con_trafico / n_claves_maestro * 100:.2f} %"
)


# ============================================================
# 12. VARIABLES DE TRÁFICO QUE SE INCORPORARÁN
# ============================================================

variables_trafico = [
    col
    for col in df_trafico_pre.columns
    if col not in claves_merge
]


print("\n" + "=" * 80)
print("VARIABLES DE TRÁFICO A INCORPORAR")
print("=" * 80)

print(
    f"\n✓ Variables nuevas: "
    f"{len(variables_trafico)}"
)

for col in variables_trafico:
    print(f"  • {col}")


# ============================================================
# 13. INTEGRACIÓN MANY-TO-ONE
# ============================================================

df_maestro_03 = pd.merge(
    df_maestro_02,
    df_trafico_pre,

    on=[
        "fecha",
        "estacion_geo"
    ],

    how="left",

    validate="many_to_one"
)


# ============================================================
# 14. CONTROL DEL NÚMERO DE FILAS
# ============================================================

filas_despues = len(
    df_maestro_03
)

columnas_despues = (
    df_maestro_03.shape[1]
)

columnas_esperadas = (
    df_maestro_02.shape[1]
    +
    len(variables_trafico)
)


print("\n" + "=" * 80)
print("CONTROL DE DIMENSIONES TRAS EL MERGE")
print("=" * 80)

print(
    f"\n✓ Filas antes: "
    f"{filas_antes:,}"
)

print(
    f"✓ Filas después: "
    f"{filas_despues:,}"
)

print(
    f"✓ Columnas antes: "
    f"{df_maestro_02.shape[1]}"
)

print(
    f"✓ Variables de tráfico añadidas: "
    f"{len(variables_trafico)}"
)

print(
    f"✓ Columnas esperadas: "
    f"{columnas_esperadas}"
)

print(
    f"✓ Columnas obtenidas: "
    f"{columnas_despues}"
)


if filas_antes != filas_despues:

    raise ValueError(
        "La integración ha modificado "
        "el número de registros del maestro."
    )


if columnas_despues != columnas_esperadas:

    raise ValueError(
        "El número de columnas después del merge "
        "no coincide con el esperado."
    )


# ============================================================
# 15. DUPLICADOS TRAS EL MERGE
# ============================================================

duplicados_maestro_despues = (
    df_maestro_03
    .duplicated(
        subset=clave_maestro
    )
    .sum()
)


print("\n" + "=" * 80)
print("CONTROL DE DUPLICADOS TRAS EL MERGE")
print("=" * 80)

print(
    f"\n✓ Duplicados antes: "
    f"{duplicados_maestro_antes:,}"
)

print(
    f"✓ Duplicados después: "
    f"{duplicados_maestro_despues:,}"
)


if duplicados_maestro_despues != duplicados_maestro_antes:

    raise ValueError(
        "La integración ha introducido "
        "duplicados en el maestro."
    )


# ============================================================
# 16. CONTROL DEL INDICADOR PRINCIPAL DE TRÁFICO
# ============================================================

if "trafico_idw_1000m" not in df_maestro_03.columns:

    raise KeyError(
        "No se ha incorporado trafico_idw_1000m."
    )


nulos_idw_maestro = (
    df_maestro_03[
        "trafico_idw_1000m"
    ]
    .isna()
    .sum()
)

validos_idw_maestro = (
    df_maestro_03[
        "trafico_idw_1000m"
    ]
    .notna()
    .sum()
)

ceros_idw_maestro = (
    df_maestro_03[
        "trafico_idw_1000m"
    ]
    .eq(0)
    .sum()
)


print("\n" + "=" * 80)
print("COBERTURA DE TRÁFICO EN EL MAESTRO")
print("=" * 80)

print(
    f"\n✓ Registros con trafico_idw_1000m: "
    f"{validos_idw_maestro:,}"
)

print(
    f"✓ Registros sin trafico_idw_1000m: "
    f"{nulos_idw_maestro:,}"
)

print(
    f"✓ Cobertura: "
    f"{validos_idw_maestro / filas_despues * 100:.2f} %"
)

print(
    f"✓ Valores IDW iguales a cero: "
    f"{ceros_idw_maestro:,}"
)


# ============================================================
# 17. COBERTURA POR ESTACIÓN
# ============================================================

cobertura_estacion = (
    df_maestro_03
    .groupby("estacion_geo")
    .agg(
        registros=(
            "fecha",
            "size"
        ),

        registros_con_trafico=(
            "trafico_idw_1000m",
            "count"
        )
    )
    .reset_index()
)

cobertura_estacion[
    "registros_sin_trafico"
] = (
    cobertura_estacion["registros"]
    -
    cobertura_estacion["registros_con_trafico"]
)

cobertura_estacion[
    "cobertura_pct"
] = (
    cobertura_estacion["registros_con_trafico"]
    /
    cobertura_estacion["registros"]
    * 100
).round(2)


print("\n" + "=" * 80)
print("COBERTURA DE TRÁFICO POR ESTACIÓN")
print("=" * 80)

display(
    cobertura_estacion
)


# ============================================================
# 18. CONTROL DE INFINITOS
# ============================================================

columnas_numericas = (
    df_maestro_03
    .select_dtypes(
        include=[np.number]
    )
    .columns
)

n_inf = (
    np.isinf(
        df_maestro_03[
            columnas_numericas
        ]
    )
    .sum()
    .sum()
)


print(
    f"\n✓ Valores infinitos en el maestro: "
    f"{n_inf:,}"
)

if n_inf > 0:

    raise ValueError(
        "Se han detectado valores infinitos "
        "en el Checkpoint 03."
    )


# ============================================================
# 19. ORDENAR EL MAESTRO
# ============================================================

columnas_orden = [
    col
    for col in [
        "fecha",
        "estacion_geo",
        "estacion_fisica",
        "contaminant"
    ]
    if col in df_maestro_03.columns
]

df_maestro_03 = (
    df_maestro_03
    .sort_values(
        columnas_orden
    )
    .reset_index(drop=True)
)


# ============================================================
# 20. EXPORTAR CHECKPOINT 03
# ============================================================

ruta_integracion.mkdir(
    parents=True,
    exist_ok=True
)

df_maestro_03.to_csv(
    ruta_maestro_03,
    index=False,
    encoding="utf-8"
)


# ============================================================
# 21. RECARGAR CHECKPOINT 03
# ============================================================

df_control_03 = pd.read_csv(
    ruta_maestro_03,
    parse_dates=["fecha"],
    low_memory=False
)


# ============================================================
# 22. VERIFICAR INTEGRIDAD DE LA EXPORTACIÓN
# ============================================================

filas_recarga = len(
    df_control_03
)

columnas_recarga = (
    df_control_03.shape[1]
)

duplicados_recarga = (
    df_control_03
    .duplicated(
        subset=clave_maestro
    )
    .sum()
)

nulos_idw_recarga = (
    df_control_03[
        "trafico_idw_1000m"
    ]
    .isna()
    .sum()
)


print("\n" + "=" * 80)
print("VERIFICACIÓN DEL CHECKPOINT 03")
print("=" * 80)

print(
    f"\n✓ Filas exportadas: "
    f"{filas_despues:,}"
)

print(
    f"✓ Filas recargadas: "
    f"{filas_recarga:,}"
)

print(
    f"✓ Columnas exportadas: "
    f"{columnas_despues}"
)

print(
    f"✓ Columnas recargadas: "
    f"{columnas_recarga}"
)

print(
    f"✓ Duplicados tras recarga: "
    f"{duplicados_recarga:,}"
)

print(
    f"✓ NaN IDW antes de exportar: "
    f"{nulos_idw_maestro:,}"
)

print(
    f"✓ NaN IDW tras recarga: "
    f"{nulos_idw_recarga:,}"
)


if filas_recarga != filas_despues:

    raise ValueError(
        "El número de filas ha cambiado "
        "durante la exportación."
    )


if columnas_recarga != columnas_despues:

    raise ValueError(
        "El número de columnas ha cambiado "
        "durante la exportación."
    )


if duplicados_recarga != duplicados_maestro_despues:

    raise ValueError(
        "La recarga ha alterado "
        "la unicidad del maestro."
    )


if nulos_idw_recarga != nulos_idw_maestro:

    raise ValueError(
        "La exportación ha alterado "
        "los NaN de tráfico."
    )


# ============================================================
# 23. TAMAÑO DEL ARCHIVO
# ============================================================

tamano_mb = (
    ruta_maestro_03.stat().st_size
    /
    (1024 ** 2)
)


# ============================================================
# 24. RESUMEN FINAL
# ============================================================

print("\n" + "=" * 80)
print("CHECKPOINT 03 — TRÁFICO RODADO")
print("=" * 80)

print(
    f"\n✓ Registros conservados: "
    f"{filas_recarga:,}"
)

print(
    f"✓ Variables disponibles: "
    f"{columnas_recarga}"
)

print(
    f"✓ Variables de tráfico añadidas: "
    f"{len(variables_trafico)}"
)

print(
    f"✓ Localizaciones físicas: "
    f"{df_control_03['estacion_geo'].nunique()}"
)

print(
    f"✓ Contaminantes: "
    f"{df_control_03['contaminant'].nunique()}"
)

print(
    f"✓ Duplicados de la clave principal: "
    f"{duplicados_recarga:,}"
)

print(
    f"✓ Cobertura trafico_idw_1000m: "
    f"{df_control_03['trafico_idw_1000m'].notna().mean() * 100:.2f} %"
)

print(
    f"✓ NaN conservados en trafico_idw_1000m: "
    f"{nulos_idw_recarga:,}"
)

print(
    f"✓ Valores infinitos: "
    f"{n_inf:,}"
)

print(
    f"✓ Tamaño del archivo: "
    f"{tamano_mb:.2f} MB"
)

print(
    "✓ Sin pérdida ni multiplicación de registros."
)

print(
    "✓ Checkpoint 02 conservado sin modificaciones."
)

print(
    "✓ Integridad del Checkpoint 03 "
    "verificada tras la exportación."
)

print(
    "\n✓ Archivo maestro generado:"
)

print(
    ruta_maestro_03
)

4.7 — INTEGRACIÓN DEL TRÁFICO RODADO — CHECKPOINT 03

CHECKPOINT 02:
✓ Filas: 43,642
✓ Columnas: 33

DATASET PREINTEGRADO DE TRÁFICO:
✓ Filas: 20,456
✓ Columnas: 28

CONTROL DE FECHAS

✓ Fechas inválidas en maestro: 0
✓ Fechas inválidas en tráfico: 0

CARDINALIDAD DEL DATASET DE TRÁFICO

✓ Duplicados fecha + estacion_geo: 0

CONTROL DEL MAESTRO ANTES DE INTEGRAR

✓ Filas: 43,642
✓ Duplicados clave principal: 0

CONTROL DE COLUMNAS COMUNES

✓ Columnas comunes totales: 2
✓ Columnas comunes fuera de la clave: 0

COBERTURA DE LA CLAVE DE INTEGRACIÓN

✓ Combinaciones fecha-estación del maestro: 15,417
✓ Con correspondencia en tráfico: 15,417
✓ Sin correspondencia en tráfico: 0
✓ Cobertura de la clave: 100.00 %

VARIABLES DE TRÁFICO A INCORPORAR

✓ Variables nuevas: 26
  • n_aforos_total_500m
  • trafico_media_500m
  • trafico_suma_500m
  • n_aforos_obs_500m
  • dist_aforo_min_500m
  • trafico_idw_500m
  • n_aforos_total_750m
  • trafico_media_750m
  • trafico_suma_750m
  • n_aforos_obs_750m

,estacion_geo,registros,registros_con_trafico,registros_sin_trafico,cobertura_pct
0,Ciutadella,4428,4428,0,100.00
1,Eixample,6975,6975,0,100.00
2,Gràcia,6120,6120,0,100.00
3,Observatori Fabra,440,355,85,80.68
4,Palau Reial,7359,7359,0,100.00
5,Poblenou,5675,5675,0,100.00
6,Sants,6163,6163,0,100.00
7,Vall d'Hebron,6482,6482,0,100.00



✓ Valores infinitos en el maestro: 0

VERIFICACIÓN DEL CHECKPOINT 03

✓ Filas exportadas: 43,642
✓ Filas recargadas: 43,642
✓ Columnas exportadas: 59
✓ Columnas recargadas: 59
✓ Duplicados tras recarga: 0
✓ NaN IDW antes de exportar: 85
✓ NaN IDW tras recarga: 85

CHECKPOINT 03 — TRÁFICO RODADO

✓ Registros conservados: 43,642
✓ Variables disponibles: 59
✓ Variables de tráfico añadidas: 26
✓ Localizaciones físicas: 8
✓ Contaminantes: 4
✓ Duplicados de la clave principal: 0
✓ Cobertura trafico_idw_1000m: 99.81 %
✓ NaN conservados en trafico_idw_1000m: 85
✓ Valores infinitos: 0
✓ Tamaño del archivo: 22.27 MB
✓ Sin pérdida ni multiplicación de registros.
✓ Checkpoint 02 conservado sin modificaciones.
✓ Integridad del Checkpoint 03 verificada tras la exportación.

✓ Archivo maestro generado:
/content/drive/MyDrive/TFM/11_Machine_Learning_Temporal/INTEGRACION/03_maestro_contaminacion_meteorologia_trafico.csv


#### Resultado — Integración del tráfico rodado y Checkpoint 03

La integración de los indicadores de tráfico rodado con el dataset maestro se realiza mediante la clave `fecha + estacion_geo`, utilizando una relación `many-to-one` coherente con la diferente granularidad de ambas fuentes.

Las **15.417 combinaciones fecha-estación** presentes en el maestro encuentran correspondencia en el dataset preintegrado de tráfico, alcanzándose una **cobertura del 100 % de la clave de integración**.

El proceso conserva exactamente los **43.642 registros** del Checkpoint 02 y amplía el dataset de **33 a 59 variables**, mediante la incorporación de **26 variables derivadas del bloque de tráfico rodado**. No se generan nuevos duplicados en la clave principal `fecha + estacion_fisica + contaminant` ni se detectan valores infinitos.

El indicador principal `trafico_idw_1000m` dispone de información válida en **43.557 de los 43.642 registros**, lo que supone una cobertura del **99,81 %**. Los **85 valores ausentes restantes** se concentran en Observatori Fabra y corresponden a situaciones previamente identificadas en las que no existe ninguna observación IMD válida dentro del radio de referencia.

La cobertura de la clave espacial-temporal y la disponibilidad efectiva del indicador se mantienen conceptualmente diferenciadas: todos los registros del maestro encuentran una combinación fecha-estación correspondiente en el dataset de tráfico, pero las ausencias reales de intensidad permanecen representadas como `NaN`.

No se detectan valores artificiales de tráfico iguales a cero y no se realiza ninguna imputación adicional durante la integración.

La exportación y posterior recarga del Checkpoint 03 reproducen exactamente las **43.642 filas y 59 variables**, manteniendo la unicidad de la clave principal y los 85 valores ausentes del indicador de referencia.

> **Conclusión:** el tráfico rodado queda incorporado al dataset maestro sin pérdida ni multiplicación de registros, con cobertura espacial-temporal completa de la clave y una disponibilidad efectiva del 99,81 % para el indicador principal. El Checkpoint 03 queda validado y preparado para la incorporación de la siguiente fuente de información.

# 5 · PREINTEGRACIÓN E INTEGRACIÓN DE LA CONTAMINACIÓN ACÚSTICA

El tercer bloque explicativo incorpora información procedente de la **red de monitorización del ruido ambiental de Barcelona**.

La contaminación acústica se utiliza como indicador complementario de actividad urbana. Aunque representa un fenómeno ambiental diferente de la contaminación atmosférica, puede reflejar patrones comunes asociados a movilidad, intensidad viaria y actividad antrópica.

La fuente acústica presenta particularidades espaciales y temporales que requieren una fase específica de preintegración. Se audita la cobertura de la red, se valida la geolocalización de los sensores y se establece su relación espacial con las estaciones de calidad del aire.

A partir de esta correspondencia se construyen indicadores acústicos diarios y se selecciona una representación adecuada para su incorporación al dataset maestro, manteniendo explícitamente las limitaciones de cobertura detectadas.

El bloque finaliza con la generación del **Checkpoint 04**.

> **Objetivo del bloque:** incorporar al maestro una caracterización diaria del entorno acústico de las estaciones de calidad del aire mediante la relación espacial con la red de sensores de ruido.

## 5.1. Carga y auditoría inicial de los datos de contaminación acústica

Se cargan los dos datasets preparados para el bloque de contaminación acústica: el dataset diario limpio de mediciones de ruido y el catálogo de equipos de monitorización de la red acústica.

En esta primera etapa se realiza exclusivamente una auditoría estructural de ambas fuentes, sin modificar ni sobrescribir los archivos originales.

Se comprueban las dimensiones, nombres y tipos de variables, cobertura temporal, identificadores de los equipos de medida, valores ausentes y posibles duplicados.

El análisis conjunto de ambas estructuras permitirá determinar posteriormente la estrategia adecuada para geolocalizar las observaciones acústicas y establecer su correspondencia espacial con las estaciones físicas del dataset maestro.

> **Objetivo:** verificar la estructura, calidad y trazabilidad de las dos fuentes acústicas antes de realizar cualquier transformación o integración espacial.

In [ ]:
# ============================================================
# 5.1. CARGA Y AUDITORÍA INICIAL
# CONTAMINACIÓN ACÚSTICA
# ============================================================

import pandas as pd
from pathlib import Path


# ============================================================
# 1. RUTAS
# ============================================================

ruta_base = Path(
    "/content/drive/MyDrive/TFM/"
    "11_Machine_Learning_Temporal"
)

ruta_ruido = (
    ruta_base /
    "PREINTEGRACION" /
    "07_Contaminacion_Acustica"
)

ruta_ruido_diario = (
    ruta_ruido /
    "df_ruido_diario_limpio.csv"
)

ruta_estaciones_ruido = (
    ruta_ruido /
    "XarxaSoroll_EquipsMonitor_Instal.csv"
)


# ============================================================
# 2. COMPROBAR EXISTENCIA DE LOS ARCHIVOS
# ============================================================

archivos = {
    "Dataset diario de ruido":
        ruta_ruido_diario,

    "Catálogo de equipos":
        ruta_estaciones_ruido
}

print("=" * 80)
print("5.1 — CARGA Y AUDITORÍA INICIAL DE CONTAMINACIÓN ACÚSTICA")
print("=" * 80)

for nombre, ruta in archivos.items():

    print(f"\n{nombre}:")
    print(ruta)

    if not ruta.exists():

        raise FileNotFoundError(
            f"No se encuentra el archivo:\n{ruta}"
        )

    print("✓ Archivo encontrado")


# ============================================================
# 3. CARGAR DATASETS
# ============================================================

df_ruido = pd.read_csv(
    ruta_ruido_diario,
    low_memory=False
)

df_estaciones_ruido = pd.read_csv(
    ruta_estaciones_ruido,
    low_memory=False
)


# ============================================================
# 4. DIMENSIONES
# ============================================================

print("\n" + "=" * 80)
print("DIMENSIONES")
print("=" * 80)

print("\nDATASET DIARIO DE RUIDO")

print(
    f"✓ Filas: "
    f"{len(df_ruido):,}"
)

print(
    f"✓ Columnas: "
    f"{df_ruido.shape[1]}"
)


print("\nCATÁLOGO DE EQUIPOS")

print(
    f"✓ Filas: "
    f"{len(df_estaciones_ruido):,}"
)

print(
    f"✓ Columnas: "
    f"{df_estaciones_ruido.shape[1]}"
)


# ============================================================
# 5. COLUMNAS DISPONIBLES
# ============================================================

print("\n" + "=" * 80)
print("COLUMNAS — DATASET DIARIO DE RUIDO")
print("=" * 80)

for columna in df_ruido.columns:
    print(f"  • {columna}")


print("\n" + "=" * 80)
print("COLUMNAS — CATÁLOGO DE EQUIPOS")
print("=" * 80)

for columna in df_estaciones_ruido.columns:
    print(f"  • {columna}")


# ============================================================
# 6. TIPOS DE DATOS
# ============================================================

print("\n" + "=" * 80)
print("TIPOS DE DATOS — RUIDO")
print("=" * 80)

print(
    df_ruido.dtypes
)


print("\n" + "=" * 80)
print("TIPOS DE DATOS — CATÁLOGO")
print("=" * 80)

print(
    df_estaciones_ruido.dtypes
)


# ============================================================
# 7. DETECTAR POSIBLES COLUMNAS DE FECHA
# ============================================================

posibles_fechas = [
    columna
    for columna in df_ruido.columns
    if any(
        termino in columna.lower()
        for termino in [
            "fecha",
            "date",
            "dia",
            "data"
        ]
    )
]


print("\n" + "=" * 80)
print("POSIBLES VARIABLES TEMPORALES")
print("=" * 80)

if posibles_fechas:

    for columna in posibles_fechas:
        print(f"  • {columna}")

else:

    print(
        "No se ha detectado automáticamente "
        "ninguna columna temporal."
    )


# ============================================================
# 8. AUDITORÍA TEMPORAL
# ============================================================

if "fecha" in df_ruido.columns:

    df_ruido["fecha"] = pd.to_datetime(
        df_ruido["fecha"],
        errors="coerce"
    )

    print("\n" + "=" * 80)
    print("COBERTURA TEMPORAL")
    print("=" * 80)

    print(
        "\n✓ Fecha inicial:",
        df_ruido["fecha"].min()
    )

    print(
        "✓ Fecha final:",
        df_ruido["fecha"].max()
    )

    print(
        "✓ Fechas diferentes:",
        f"{df_ruido['fecha'].nunique():,}"
    )

    print(
        "✓ Fechas inválidas:",
        f"{df_ruido['fecha'].isna().sum():,}"
    )


# ============================================================
# 9. DETECTAR POSIBLES IDENTIFICADORES
# ============================================================

terminos_id = [
    "id",
    "codi",
    "codigo",
    "equip",
    "instal",
    "sensor",
    "monitor"
]

posibles_id_ruido = [
    columna
    for columna in df_ruido.columns
    if any(
        termino in columna.lower()
        for termino in terminos_id
    )
]

posibles_id_catalogo = [
    columna
    for columna in df_estaciones_ruido.columns
    if any(
        termino in columna.lower()
        for termino in terminos_id
    )
]


print("\n" + "=" * 80)
print("POSIBLES IDENTIFICADORES")
print("=" * 80)

print("\nDataset diario:")

for columna in posibles_id_ruido:
    print(f"  • {columna}")


print("\nCatálogo de equipos:")

for columna in posibles_id_catalogo:
    print(f"  • {columna}")


# ============================================================
# 10. VALORES NULOS — RUIDO
# ============================================================

print("\n" + "=" * 80)
print("VALORES NULOS — DATASET DIARIO")
print("=" * 80)

nulos_ruido = (
    df_ruido
    .isna()
    .sum()
)

display(
    nulos_ruido
    .to_frame("n_nulos")
)


# ============================================================
# 11. VALORES NULOS — CATÁLOGO
# ============================================================

print("\n" + "=" * 80)
print("VALORES NULOS — CATÁLOGO DE EQUIPOS")
print("=" * 80)

nulos_catalogo = (
    df_estaciones_ruido
    .isna()
    .sum()
)

display(
    nulos_catalogo
    .to_frame("n_nulos")
)


# ============================================================
# 12. DUPLICADOS COMPLETOS
# ============================================================

duplicados_ruido = (
    df_ruido
    .duplicated()
    .sum()
)

duplicados_catalogo = (
    df_estaciones_ruido
    .duplicated()
    .sum()
)


print("\n" + "=" * 80)
print("DUPLICADOS COMPLETOS")
print("=" * 80)

print(
    f"\n✓ Dataset diario de ruido: "
    f"{duplicados_ruido:,}"
)

print(
    f"✓ Catálogo de equipos: "
    f"{duplicados_catalogo:,}"
)


# ============================================================
# 13. VALORES ÚNICOS DE POSIBLES IDENTIFICADORES
# ============================================================

print("\n" + "=" * 80)
print("CARDINALIDAD DE POSIBLES IDENTIFICADORES")
print("=" * 80)

print("\nDATASET DIARIO:")

for columna in posibles_id_ruido:

    print(
        f"  • {columna}: "
        f"{df_ruido[columna].nunique(dropna=True):,} "
        "valores únicos"
    )


print("\nCATÁLOGO:")

for columna in posibles_id_catalogo:

    print(
        f"  • {columna}: "
        f"{df_estaciones_ruido[columna].nunique(dropna=True):,} "
        "valores únicos"
    )


# ============================================================
# 14. PRIMERAS FILAS — RUIDO
# ============================================================

print("\n" + "=" * 80)
print("PRIMERAS FILAS — DATASET DIARIO")
print("=" * 80)

display(
    df_ruido.head(10)
)


# ============================================================
# 15. PRIMERAS FILAS — CATÁLOGO
# ============================================================

print("\n" + "=" * 80)
print("PRIMERAS FILAS — CATÁLOGO DE EQUIPOS")
print("=" * 80)

display(
    df_estaciones_ruido.head(10)
)


# ============================================================
# 16. CIERRE
# ============================================================

print("\n" + "=" * 80)
print("AUDITORÍA 5.1 FINALIZADA")
print("=" * 80)

print(
    "\n✓ Dataset diario cargado correctamente."
)

print(
    "✓ Catálogo de equipos cargado correctamente."
)

print(
    "✓ No se ha modificado ni sobrescrito "
    "ningún archivo."
)

print(
    "✓ Estructura preparada para definir "
    "la estrategia de preintegración acústica."
)

5.1 — CARGA Y AUDITORÍA INICIAL DE CONTAMINACIÓN ACÚSTICA

Dataset diario de ruido:
/content/drive/MyDrive/TFM/11_Machine_Learning_Temporal/PREINTEGRACION/07_Contaminacion_Acustica/df_ruido_diario_limpio.csv
✓ Archivo encontrado

Catálogo de equipos:
/content/drive/MyDrive/TFM/11_Machine_Learning_Temporal/PREINTEGRACION/07_Contaminacion_Acustica/XarxaSoroll_EquipsMonitor_Instal.csv
✓ Archivo encontrado

DIMENSIONES

DATASET DIARIO DE RUIDO
✓ Filas: 316,951
✓ Columnas: 10

CATÁLOGO DE EQUIPOS
✓ Filas: 989
✓ Columnas: 14

COLUMNAS — DATASET DIARIO DE RUIDO
  • Fecha
  • Any
  • Id_Instal
  • LAeq_dia
  • N_validos
  • Observaciones_teoricas
  • Cobertura_corregida_%
  • Dia_valido
  • Resolucion_origen
  • Periodo_fuente

COLUMNAS — CATÁLOGO DE EQUIPOS
  • Id_Instal
  • Codi_Carrer
  • Tipus_Via
  • Nom_Carrer
  • Num_Carrer
  • Codi_Barri
  • Nom_Barri
  • Codi_Districte
  • Nom_Districte
  • Latitud
  • Longitud
  • Data_Instalacio
  • Data_DesInstalacio
  • Font

TIPOS DE DATOS — RUID

,n_nulos
Fecha,0
Any,0
Id_Instal,0
LAeq_dia,0
N_validos,0
Observaciones_teoricas,0
Cobertura_corregida_%,0
Dia_valido,0
Resolucion_origen,0
Periodo_fuente,0



VALORES NULOS — CATÁLOGO DE EQUIPOS


,n_nulos
Id_Instal,0
Codi_Carrer,0
Tipus_Via,0
Nom_Carrer,0
Num_Carrer,8
Codi_Barri,0
Nom_Barri,0
Codi_Districte,0
Nom_Districte,0
Latitud,0



DUPLICADOS COMPLETOS

✓ Dataset diario de ruido: 0
✓ Catálogo de equipos: 0

CARDINALIDAD DE POSIBLES IDENTIFICADORES

DATASET DIARIO:
  • Id_Instal: 628 valores únicos
  • N_validos: 896 valores únicos
  • Cobertura_corregida_%: 909 valores únicos
  • Dia_valido: 2 valores únicos

CATÁLOGO:
  • Id_Instal: 989 valores únicos
  • Codi_Carrer: 311 valores únicos
  • Codi_Barri: 63 valores únicos
  • Codi_Districte: 10 valores únicos
  • Data_Instalacio: 493 valores únicos
  • Data_DesInstalacio: 468 valores únicos

PRIMERAS FILAS — DATASET DIARIO


,Fecha,Any,Id_Instal,LAeq_dia,N_validos,Observaciones_teoricas,Cobertura_corregida_%,Dia_valido,Resolucion_origen,Periodo_fuente
0,2018-01-01,2018,495,72.05,24,24,100.0,True,1 hora,2018-2023
1,2018-01-01,2018,496,71.03,24,24,100.0,True,1 hora,2018-2023
2,2018-01-01,2018,497,71.32,24,24,100.0,True,1 hora,2018-2023
3,2018-01-01,2018,651,67.81,24,24,100.0,True,1 hora,2018-2023
4,2018-01-01,2018,653,67.43,24,24,100.0,True,1 hora,2018-2023
5,2018-01-01,2018,659,61.83,24,24,100.0,True,1 hora,2018-2023
6,2018-01-01,2018,666,61.75,24,24,100.0,True,1 hora,2018-2023
7,2018-01-01,2018,667,61.69,24,24,100.0,True,1 hora,2018-2023
8,2018-01-01,2018,686,75.53,24,24,100.0,True,1 hora,2018-2023
9,2018-01-01,2018,726,62.81,24,24,100.0,True,1 hora,2018-2023



PRIMERAS FILAS — CATÁLOGO DE EQUIPOS


,Id_Instal,Codi_Carrer,Tipus_Via,Nom_Carrer,Num_Carrer,Codi_Barri,Nom_Barri,Codi_Districte,Nom_Districte,Latitud,Longitud,Data_Instalacio,Data_DesInstalacio,Font
0,67,294874,Pg,Salvat Papasseit,32.0,3,la Barceloneta,1,Ciutat Vella,41.380962,2.191743,2013-04-08,2013-06-05,ACTIVITATS/INFRAESTRUCTURES ESPORTIVES
1,86,120807,Via,Favència,377.0,53,la Trinitat Nova,8,Nou Barris,41.447024,2.186277,2013-04-10,2013-05-30,TRÀNSIT
2,87,218606,Pg,Montjuïc,50.0,11,el Poble-sec,3,Sants-Montjuïc,41.371728,2.169036,2013-04-08,2013-10-01,NETEJA
3,106,37408,C,de Beethoven,9.0,26,Sant Gervasi - Galvany,5,Sarrià-Sant Gervasi,41.393038,2.140707,2013-05-23,2013-12-09,OCI
4,107,348408,C,Tuset,17.0,26,Sant Gervasi - Galvany,5,Sarrià-Sant Gervasi,41.396156,2.151563,2013-05-23,2013-12-09,OCI
5,146,16907,C,Anglí,46.0,23,Sarrià,5,Sarrià-Sant Gervasi,41.401879,2.124334,2013-10-01,2013-11-11,INFRAESTRUCTURES FERROVIÀRIES
6,147,44709,Pg,Bonanova,78.0,25,Sant Gervasi - la Bonanova,5,Sarrià-Sant Gervasi,41.402274,2.125322,2013-09-30,2013-11-11,INFRAESTRUCTURES FERROVIÀRIES
7,166,44709,Pg,Bonanova,78.0,25,Sant Gervasi - la Bonanova,5,Sarrià-Sant Gervasi,41.402397,2.124890,2013-11-11,2014-01-30,INFRAESTRUCTURES FERROVIÀRIES
8,167,44709,Pg,Bonanova,78.0,25,Sant Gervasi - la Bonanova,5,Sarrià-Sant Gervasi,41.402397,2.124890,2013-11-11,2013-12-09,INFRAESTRUCTURES FERROVIÀRIES
9,186,348408,C,Tuset,8.0,26,Sant Gervasi - Galvany,5,Sarrià-Sant Gervasi,41.395450,2.152752,2014-01-29,2015-01-16,OCI



AUDITORÍA 5.1 FINALIZADA

✓ Dataset diario cargado correctamente.
✓ Catálogo de equipos cargado correctamente.
✓ No se ha modificado ni sobrescrito ningún archivo.
✓ Estructura preparada para definir la estrategia de preintegración acústica.


### Resultado — 5.1. Auditoría inicial de los datos de contaminación acústica

La auditoría inicial confirma la correcta carga de las dos fuentes correspondientes al bloque de contaminación acústica.

El dataset diario de ruido contiene **316.951 registros y 10 variables**, correspondientes a **628 instalaciones de medida (`Id_Instal`) diferentes**. Entre las variables disponibles se encuentran el indicador acústico diario `LAeq_dia`, el número de observaciones válidas y teóricas, el porcentaje de cobertura, la validez diaria y la información relativa a la resolución y periodo de procedencia de los datos.

El dataset diario presenta una estructura especialmente limpia:

- **0 valores nulos** en las variables disponibles;
- **0 duplicados completos**;
- `LAeq_dia` disponible para la totalidad de los registros;
- identificador `Id_Instal` disponible para todas las observaciones.

El catálogo espacial contiene **989 instalaciones y 14 variables**, con un identificador `Id_Instal` único para cada registro. Incluye las coordenadas geográficas `Latitud` y `Longitud`, además de información de calle, barrio, distrito y fechas de instalación y desinstalación.

Las **989 instalaciones disponen de coordenadas geográficas**, por lo que el catálogo proporciona la información necesaria para abordar posteriormente la correspondencia espacial entre los sensores acústicos y las estaciones físicas de contaminación atmosférica.

La presencia de `Id_Instal` en ambas fuentes establece una clave directa para vincular las mediciones acústicas con su localización espacial.

La cobertura temporal exacta no se determina todavía en esta etapa, ya que la variable temporal del dataset se denomina `Fecha`. Su periodo efectivo, distribución anual, número de sensores activos y unicidad de la clave `Fecha + Id_Instal` se analizarán específicamente en la siguiente fase.

> **Conclusión:** las fuentes acústicas presentan una estructura consistente y adecuada para continuar la preintegración. No se detectan problemas de integridad en la auditoría inicial y existe una clave común que permite geolocalizar las observaciones acústicas antes de realizar su correspondencia con las estaciones del dataset maestro.

## 5.2. Auditoría temporal y cobertura de la red acústica

Una vez verificada la estructura de las fuentes de contaminación acústica, se analiza la cobertura temporal efectiva del dataset diario.

Esta etapa permite determinar el periodo real disponible, la evolución anual del número de observaciones y de instalaciones activas, así como la unicidad de la combinación `Fecha + Id_Instal`.

También se analiza la calidad de las observaciones mediante las variables de cobertura y validez disponibles en el dataset, junto con la distribución del indicador acústico `LAeq_dia`.

No se realiza ninguna imputación ni transformación espacial en esta fase.

> **Objetivo:** caracterizar la disponibilidad temporal y la calidad de las mediciones acústicas antes de construir los indicadores espaciales que se incorporarán posteriormente al dataset maestro.

In [ ]:
# ============================================================
# 5.2. AUDITORÍA TEMPORAL Y COBERTURA
# DE LA RED ACÚSTICA
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 1. COPIA DE CONTROL
# ============================================================

df_ruido_temp = df_ruido.copy()


# ============================================================
# 2. COMPROBAR VARIABLES NECESARIAS
# ============================================================

variables_necesarias = [
    "Fecha",
    "Id_Instal",
    "LAeq_dia",
    "Cobertura_corregida_%",
    "Dia_valido",
    "Resolucion_origen",
    "Periodo_fuente"
]

variables_faltantes = [
    columna
    for columna in variables_necesarias
    if columna not in df_ruido_temp.columns
]

if variables_faltantes:
    raise KeyError(
        "Faltan variables necesarias para la auditoría:\n"
        f"{variables_faltantes}"
    )


print("=" * 80)
print("5.2 — AUDITORÍA TEMPORAL Y COBERTURA DE LA RED ACÚSTICA")
print("=" * 80)


# ============================================================
# 3. CONVERTIR FECHA
# ============================================================

df_ruido_temp["Fecha"] = pd.to_datetime(
    df_ruido_temp["Fecha"],
    errors="coerce"
)

fechas_invalidas = (
    df_ruido_temp["Fecha"]
    .isna()
    .sum()
)


# ============================================================
# 4. COBERTURA TEMPORAL GENERAL
# ============================================================

fecha_inicio = (
    df_ruido_temp["Fecha"]
    .min()
)

fecha_fin = (
    df_ruido_temp["Fecha"]
    .max()
)

n_fechas = (
    df_ruido_temp["Fecha"]
    .nunique()
)


print("\n" + "=" * 80)
print("COBERTURA TEMPORAL GENERAL")
print("=" * 80)

print(
    f"\n✓ Fecha inicial: "
    f"{fecha_inicio}"
)

print(
    f"✓ Fecha final: "
    f"{fecha_fin}"
)

print(
    f"✓ Fechas diferentes: "
    f"{n_fechas:,}"
)

print(
    f"✓ Fechas inválidas: "
    f"{fechas_invalidas:,}"
)

if fechas_invalidas > 0:
    raise ValueError(
        "Existen fechas inválidas en el dataset de ruido."
    )


# ============================================================
# 5. CREAR VARIABLE AÑO
# ============================================================

df_ruido_temp["anio"] = (
    df_ruido_temp["Fecha"]
    .dt.year
    .astype("Int64")
)


# ============================================================
# 6. REGISTROS, SENSORES Y DÍAS POR AÑO
# ============================================================

resumen_anual = (
    df_ruido_temp
    .groupby("anio")
    .agg(
        registros=(
            "Id_Instal",
            "size"
        ),
        sensores=(
            "Id_Instal",
            "nunique"
        ),
        dias=(
            "Fecha",
            "nunique"
        ),
        laeq_media=(
            "LAeq_dia",
            "mean"
        ),
        laeq_mediana=(
            "LAeq_dia",
            "median"
        )
    )
    .reset_index()
)

print("\n" + "=" * 80)
print("COBERTURA POR AÑO")
print("=" * 80)

display(
    resumen_anual
)


# ============================================================
# 7. CONTROL DEL PERIODO OBJETIVO 2018–2024
# ============================================================

anios_objetivo = list(
    range(2018, 2025)
)

anios_disponibles = sorted(
    df_ruido_temp["anio"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)

anios_ausentes = [
    anio
    for anio in anios_objetivo
    if anio not in anios_disponibles
]

print("\n" + "=" * 80)
print("CONTROL DEL PERIODO OBJETIVO 2018–2024")
print("=" * 80)

print(
    "\n✓ Años disponibles:",
    anios_disponibles
)

if anios_ausentes:

    print(
        "⚠ Años sin observaciones:",
        anios_ausentes
    )

else:

    print(
        "✓ Existen observaciones "
        "para todos los años 2018–2024."
    )


# ============================================================
# 8. UNICIDAD FECHA + INSTALACIÓN
# ============================================================

duplicados_clave = (
    df_ruido_temp
    .duplicated(
        subset=[
            "Fecha",
            "Id_Instal"
        ]
    )
    .sum()
)

print("\n" + "=" * 80)
print("UNICIDAD TEMPORAL POR SENSOR")
print("=" * 80)

print(
    f"\n✓ Duplicados Fecha + Id_Instal: "
    f"{duplicados_clave:,}"
)

if duplicados_clave > 0:

    print(
        "\n⚠ Se han detectado duplicados "
        "Fecha + Id_Instal."
    )

    display(
        df_ruido_temp[
            df_ruido_temp.duplicated(
                subset=[
                    "Fecha",
                    "Id_Instal"
                ],
                keep=False
            )
        ]
        .sort_values(
            [
                "Id_Instal",
                "Fecha"
            ]
        )
        .head(20)
    )


# ============================================================
# 9. OBSERVACIONES POR SENSOR
# ============================================================

resumen_sensor = (
    df_ruido_temp
    .groupby("Id_Instal")
    .agg(
        fecha_inicio=(
            "Fecha",
            "min"
        ),
        fecha_fin=(
            "Fecha",
            "max"
        ),
        n_dias=(
            "Fecha",
            "nunique"
        ),
        laeq_mediana=(
            "LAeq_dia",
            "median"
        ),
        cobertura_mediana=(
            "Cobertura_corregida_%",
            "median"
        )
    )
    .reset_index()
)

print("\n" + "=" * 80)
print("DISTRIBUCIÓN DE DÍAS DISPONIBLES POR SENSOR")
print("=" * 80)

display(
    resumen_sensor["n_dias"]
    .describe()
    .to_frame("n_dias")
)


# ============================================================
# 10. CONTINUIDAD TEMPORAL DE LOS SENSORES
# ============================================================

print("\n" + "=" * 80)
print("CONTINUIDAD TEMPORAL DE LOS SENSORES")
print("=" * 80)

print(
    f"\n✓ Sensores totales: "
    f"{len(resumen_sensor):,}"
)

print(
    f"✓ Sensores con ≥ 30 días: "
    f"{(resumen_sensor['n_dias'] >= 30).sum():,}"
)

print(
    f"✓ Sensores con ≥ 90 días: "
    f"{(resumen_sensor['n_dias'] >= 90).sum():,}"
)

print(
    f"✓ Sensores con ≥ 180 días: "
    f"{(resumen_sensor['n_dias'] >= 180).sum():,}"
)

print(
    f"✓ Sensores con ≥ 365 días: "
    f"{(resumen_sensor['n_dias'] >= 365).sum():,}"
)


# ============================================================
# 11. COBERTURA DE LAS MEDICIONES
# ============================================================

print("\n" + "=" * 80)
print("COBERTURA DE LAS MEDICIONES")
print("=" * 80)

display(
    df_ruido_temp[
        "Cobertura_corregida_%"
    ]
    .describe()
    .to_frame(
        "Cobertura_corregida_%"
    )
)


# ============================================================
# 12. VALIDEZ DIARIA
# ============================================================

tabla_validez = (
    df_ruido_temp[
        "Dia_valido"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis("Dia_valido")
    .reset_index(
        name="registros"
    )
)

tabla_validez[
    "porcentaje"
] = (
    tabla_validez["registros"]
    /
    len(df_ruido_temp)
    * 100
).round(2)

print("\n" + "=" * 80)
print("VALIDEZ DIARIA")
print("=" * 80)

display(
    tabla_validez
)


# ============================================================
# 13. RESOLUCIÓN TEMPORAL DE ORIGEN
# ============================================================

tabla_resolucion = (
    df_ruido_temp[
        "Resolucion_origen"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "Resolucion_origen"
    )
    .reset_index(
        name="registros"
    )
)

tabla_resolucion[
    "porcentaje"
] = (
    tabla_resolucion["registros"]
    /
    len(df_ruido_temp)
    * 100
).round(2)

print("\n" + "=" * 80)
print("RESOLUCIÓN TEMPORAL DE ORIGEN")
print("=" * 80)

display(
    tabla_resolucion
)


# ============================================================
# 14. PERIODOS DE FUENTE
# ============================================================

tabla_periodos = (
    df_ruido_temp[
        "Periodo_fuente"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "Periodo_fuente"
    )
    .reset_index(
        name="registros"
    )
)

tabla_periodos[
    "porcentaje"
] = (
    tabla_periodos["registros"]
    /
    len(df_ruido_temp)
    * 100
).round(2)

print("\n" + "=" * 80)
print("PERIODOS DE FUENTE")
print("=" * 80)

display(
    tabla_periodos
)


# ============================================================
# 15. DISTRIBUCIÓN DE LAeq_dia
# ============================================================

print("\n" + "=" * 80)
print("DISTRIBUCIÓN DE LAeq_dia")
print("=" * 80)

display(
    df_ruido_temp[
        "LAeq_dia"
    ]
    .describe()
    .to_frame(
        "LAeq_dia"
    )
)


# ============================================================
# 16. RANGO DE LAeq_dia
# ============================================================

laeq_min = (
    df_ruido_temp[
        "LAeq_dia"
    ].min()
)

laeq_max = (
    df_ruido_temp[
        "LAeq_dia"
    ].max()
)

print("\n" + "=" * 80)
print("RANGO DE LAeq_dia")
print("=" * 80)

print(
    f"\n✓ Mínimo: "
    f"{laeq_min:.2f}"
)

print(
    f"✓ Máximo: "
    f"{laeq_max:.2f}"
)


# ============================================================
# 17. COBERTURA DE LAS MEDICIONES POR AÑO
# ============================================================

cobertura_anual = (
    df_ruido_temp
    .groupby("anio")
    .agg(
        cobertura_media=(
            "Cobertura_corregida_%",
            "mean"
        ),
        cobertura_mediana=(
            "Cobertura_corregida_%",
            "median"
        ),
        cobertura_min=(
            "Cobertura_corregida_%",
            "min"
        ),
        cobertura_max=(
            "Cobertura_corregida_%",
            "max"
        )
    )
    .reset_index()
)

print("\n" + "=" * 80)
print("COBERTURA DE LAS MEDICIONES POR AÑO")
print("=" * 80)

display(
    cobertura_anual
)


# ============================================================
# 18. SENSORES POR AÑO
# ============================================================

sensores_anio = (
    df_ruido_temp
    .groupby("anio")[
        "Id_Instal"
    ]
    .nunique()
    .reset_index(
        name="sensores"
    )
)

print("\n" + "=" * 80)
print("NÚMERO DE SENSORES POR AÑO")
print("=" * 80)

display(
    sensores_anio
)


# ============================================================
# 19. COBERTURA DEL CALENDARIO POR AÑO
# ============================================================

dias_anio = (
    df_ruido_temp
    .groupby("anio")[
        "Fecha"
    ]
    .nunique()
    .reset_index(
        name="dias_con_datos"
    )
)

dias_anio[
    "dias_teoricos"
] = (
    dias_anio["anio"]
    .apply(
        lambda x:
        366
        if pd.Timestamp(
            year=int(x),
            month=12,
            day=31
        ).is_leap_year
        else 365
    )
)

dias_anio[
    "cobertura_calendario_pct"
] = (
    dias_anio[
        "dias_con_datos"
    ]
    /
    dias_anio[
        "dias_teoricos"
    ]
    * 100
).round(2)

print("\n" + "=" * 80)
print("COBERTURA DEL CALENDARIO POR AÑO")
print("=" * 80)

display(
    dias_anio
)


# ============================================================
# 20. CONTROL DE NULOS
# ============================================================

variables_control = [
    "Fecha",
    "Id_Instal",
    "LAeq_dia",
    "Cobertura_corregida_%",
    "Dia_valido"
]

nulos_control = (
    df_ruido_temp[
        variables_control
    ]
    .isna()
    .sum()
)

print("\n" + "=" * 80)
print("CONTROL DE VALORES NULOS")
print("=" * 80)

display(
    nulos_control
    .to_frame(
        "n_nulos"
    )
)


# ============================================================
# 21. CONTROL DE INFINITOS
# ============================================================

columnas_numericas = (
    df_ruido_temp
    .select_dtypes(
        include=[
            np.number
        ]
    )
    .columns
)

n_inf = (
    np.isinf(
        df_ruido_temp[
            columnas_numericas
        ]
    )
    .sum()
    .sum()
)

print("\n" + "=" * 80)
print("CONTROL DE VALORES INFINITOS")
print("=" * 80)

print(
    f"\n✓ Valores infinitos: "
    f"{n_inf:,}"
)


# ============================================================
# 22. CONTROL FINAL
# ============================================================

print("\n" + "=" * 80)
print("AUDITORÍA 5.2 FINALIZADA")
print("=" * 80)

print(
    f"\n✓ Registros analizados: "
    f"{len(df_ruido_temp):,}"
)

print(
    f"✓ Sensores diferentes: "
    f"{df_ruido_temp['Id_Instal'].nunique():,}"
)

print(
    f"✓ Periodo: "
    f"{fecha_inicio} → {fecha_fin}"
)

print(
    f"✓ Fechas diferentes: "
    f"{n_fechas:,}"
)

print(
    f"✓ Años disponibles: "
    f"{len(anios_disponibles)}"
)

print(
    f"✓ Duplicados Fecha + Id_Instal: "
    f"{duplicados_clave:,}"
)

print(
    f"✓ Valores infinitos: "
    f"{n_inf:,}"
)

print(
    "✓ No se ha realizado ninguna imputación."
)

print(
    "✓ No se ha realizado todavía "
    "ninguna asignación espacial."
)

print(
    "✓ Dataset preparado para la "
    "geolocalización de los sensores acústicos."
)

5.2 — AUDITORÍA TEMPORAL Y COBERTURA DE LA RED ACÚSTICA

COBERTURA TEMPORAL GENERAL

✓ Fecha inicial: 2018-01-01 00:00:00
✓ Fecha final: 2024-12-31 00:00:00
✓ Fechas diferentes: 2,557
✓ Fechas inválidas: 0

COBERTURA POR AÑO


,anio,registros,sensores,dias,laeq_media,laeq_mediana
0,2018,34735,174,365,65.734570,65.78
1,2019,42413,162,365,65.243719,65.53
2,2020,39795,160,366,63.151870,63.24
3,2021,42713,186,365,64.003397,63.96
4,2022,46154,241,365,64.537876,64.28
5,2023,53987,229,365,64.647105,64.41
6,2024,57154,244,366,64.279635,64.00



CONTROL DEL PERIODO OBJETIVO 2018–2024

✓ Años disponibles: [2018, 2019, 2020, 2021, 2022, 2023, 2024]
✓ Existen observaciones para todos los años 2018–2024.

UNICIDAD TEMPORAL POR SENSOR

✓ Duplicados Fecha + Id_Instal: 0

DISTRIBUCIÓN DE DÍAS DISPONIBLES POR SENSOR


,n_dias
count,628.000000
mean,504.699045
std,625.662912
min,1.000000
25%,88.000000
50%,256.500000
75%,702.750000
max,2541.000000



CONTINUIDAD TEMPORAL DE LOS SENSORES

✓ Sensores totales: 628
✓ Sensores con ≥ 30 días: 575
✓ Sensores con ≥ 90 días: 462
✓ Sensores con ≥ 180 días: 360
✓ Sensores con ≥ 365 días: 235

COBERTURA DE LAS MEDICIONES


...


COBERTURA DEL CALENDARIO POR AÑO


,anio,dias_con_datos,dias_teoricos,cobertura_calendario_pct
0,2018,365,365,100.0
1,2019,365,365,100.0
2,2020,366,366,100.0
3,2021,365,365,100.0
4,2022,365,365,100.0
5,2023,365,365,100.0
6,2024,366,366,100.0



CONTROL DE VALORES NULOS


,n_nulos
Fecha,0
Id_Instal,0
LAeq_dia,0
Cobertura_corregida_%,0
Dia_valido,0



CONTROL DE VALORES INFINITOS

✓ Valores infinitos: 0

AUDITORÍA 5.2 FINALIZADA

✓ Registros analizados: 316,951
✓ Sensores diferentes: 628
✓ Periodo: 2018-01-01 00:00:00 → 2024-12-31 00:00:00
✓ Fechas diferentes: 2,557
✓ Años disponibles: 7
✓ Duplicados Fecha + Id_Instal: 0
✓ Valores infinitos: 0
✓ No se ha realizado ninguna imputación.
✓ No se ha realizado todavía ninguna asignación espacial.
✓ Dataset preparado para la geolocalización de los sensores acústicos.


### Resultado — 5.2. Auditoría temporal y cobertura de la red acústica

La auditoría temporal confirma que el dataset de contaminación acústica presenta observaciones durante la totalidad del periodo de estudio **2018–2024**, comprendido entre el 1 de enero de 2018 y el 31 de diciembre de 2024.

Se identifican **2.557 fechas diferentes**, correspondientes a la totalidad de los días naturales del periodo. La cobertura del calendario alcanza el **100 % en todos los años**, incluyendo los 366 días de los años bisiestos 2020 y 2024.

El dataset contiene **316.951 registros correspondientes a 628 instalaciones acústicas diferentes**, sin fechas inválidas y sin duplicados en la clave `Fecha + Id_Instal`.

La disponibilidad de los sensores presenta una continuidad temporal heterogénea. De las 628 instalaciones:

- 575 disponen de al menos 30 días de observaciones;
- 462 disponen de al menos 90 días;
- 360 disponen de al menos 180 días;
- 235 disponen de al menos 365 días.

El número de sensores con observaciones varía a lo largo del periodo, desde 174 instalaciones en 2018 hasta 244 en 2024, alcanzándose el mayor número de sensores activos en este último año.

La calidad general de las observaciones es elevada. La variable `Cobertura_corregida_%` presenta una mediana del **100 %**, mientras que **310.960 registros (98,11 %)** están clasificados como días válidos y 5.991 registros (1,89 %) como no válidos.

Se identifica un cambio en la resolución temporal de las fuentes: los registros correspondientes al periodo 2018–2023 proceden de observaciones con resolución original de **1 hora**, mientras que los datos de 2024 proceden de observaciones con resolución de **1 minuto**.

El indicador `LAeq_dia` presenta una media global de **64,49 dB** y una mediana de **64,40 dB**. El rango observado se extiende entre 10,91 y 124,06 dB, por lo que los valores extremos deberán mantenerse bajo control durante las etapas posteriores de construcción y validación de los indicadores espaciales.

No se detectan valores nulos en las variables principales analizadas ni valores infinitos.

> **Conclusión:** el dataset acústico proporciona cobertura diaria completa para el periodo 2018–2024 y una red espacial amplia de sensores. La elevada validez y cobertura de las observaciones permiten continuar con su geolocalización y posterior construcción de indicadores acústicos para las estaciones del dataset maestro, manteniendo explícitamente la heterogeneidad temporal de la red de monitorización.

## 5.3. Geolocalización y validación espacial de los sensores acústicos

Una vez validada la cobertura temporal de la red acústica, se incorpora la información espacial procedente del catálogo de equipos de monitorización.

La vinculación se realiza mediante el identificador común `Id_Instal`, relacionando cada instalación presente en el dataset diario de ruido con sus coordenadas geográficas y atributos básicos de localización.

Las coordenadas originales, expresadas en **WGS84 (EPSG:4326)**, se transforman al sistema proyectado **ETRS89 / UTM zona 31N (EPSG:25831)**, utilizado como referencia espacial común en el proyecto.

Durante esta etapa se comprueba:

- la unicidad de `Id_Instal` en el catálogo espacial;
- la correspondencia entre los sensores presentes en las mediciones y el catálogo;
- la disponibilidad de coordenadas para los sensores utilizados;
- la validez de las coordenadas geográficas;
- la correcta transformación a coordenadas proyectadas `X_ETRS89` y `Y_ETRS89`;
- la conservación del número original de registros tras la incorporación de la información espacial.

En esta fase no se realiza todavía ninguna asignación entre sensores acústicos y estaciones de contaminación atmosférica.

> **Objetivo:** disponer de todas las observaciones acústicas correctamente geolocalizadas y expresadas en un sistema de coordenadas métrico común antes de construir los indicadores espaciales de ruido asociados a las estaciones del dataset maestro.

In [ ]:
# ============================================================
# 5.3. GEOLOCALIZACIÓN Y VALIDACIÓN ESPACIAL
# DE LOS SENSORES ACÚSTICOS
# ============================================================

import pandas as pd
import numpy as np
import geopandas as gpd


print("=" * 80)
print("5.3 — GEOLOCALIZACIÓN Y VALIDACIÓN ESPACIAL")
print("=" * 80)


# ============================================================
# 1. COMPROBAR OBJETOS NECESARIOS
# ============================================================

objetos_necesarios = [
    "df_ruido_temp",
    "df_estaciones_ruido"
]

faltantes = [
    nombre
    for nombre in objetos_necesarios
    if nombre not in globals()
]

if faltantes:
    raise NameError(
        "Faltan objetos necesarios para 5.3:\n"
        f"{faltantes}\n\n"
        "Ejecuta primero las celdas 5.1 y 5.2."
    )


# ============================================================
# 2. COPIAS DE TRABAJO
# ============================================================

ruido_geo = df_ruido_temp.copy()
catalogo_geo = df_estaciones_ruido.copy()


# ============================================================
# 3. NORMALIZAR IDENTIFICADORES
# ============================================================

ruido_geo["Id_Instal"] = (
    ruido_geo["Id_Instal"]
    .astype(str)
    .str.strip()
)

catalogo_geo["Id_Instal"] = (
    catalogo_geo["Id_Instal"]
    .astype(str)
    .str.strip()
)


# ============================================================
# 4. CONTROL DE UNICIDAD DEL CATÁLOGO
# ============================================================

duplicados_catalogo = (
    catalogo_geo
    .duplicated(
        subset=["Id_Instal"]
    )
    .sum()
)

print("\n" + "=" * 80)
print("CONTROL DEL CATÁLOGO")
print("=" * 80)

print(
    f"\n✓ Registros catálogo: "
    f"{len(catalogo_geo):,}"
)

print(
    f"✓ Id_Instal únicos: "
    f"{catalogo_geo['Id_Instal'].nunique():,}"
)

print(
    f"✓ Duplicados Id_Instal: "
    f"{duplicados_catalogo:,}"
)

if duplicados_catalogo > 0:
    raise ValueError(
        "El catálogo contiene Id_Instal duplicados."
    )


# ============================================================
# 5. IDENTIFICAR SENSORES CON Y SIN GEOLOCALIZACIÓN
# ============================================================

ids_ruido = set(
    ruido_geo["Id_Instal"]
    .dropna()
    .unique()
)

ids_catalogo = set(
    catalogo_geo["Id_Instal"]
    .dropna()
    .unique()
)

sensores_con_catalogo = (
    ids_ruido
    .intersection(ids_catalogo)
)

sensores_sin_catalogo = (
    ids_ruido
    -
    ids_catalogo
)

print("\n" + "=" * 80)
print("CORRESPONDENCIA ESPACIAL")
print("=" * 80)

print(
    f"\n✓ Sensores en las series temporales: "
    f"{len(ids_ruido):,}"
)

print(
    f"✓ Sensores con catálogo espacial: "
    f"{len(sensores_con_catalogo):,}"
)

print(
    f"✓ Sensores sin catálogo espacial: "
    f"{len(sensores_sin_catalogo):,}"
)


# ============================================================
# 6. CREAR CATÁLOGO DE SENSORES GEOLOCALIZABLES
# ============================================================

catalogo_validos = (
    catalogo_geo[
        catalogo_geo["Id_Instal"]
        .isin(sensores_con_catalogo)
    ]
    .copy()
)


# ============================================================
# 7. CONTROL DE COORDENADAS
# ============================================================

catalogo_validos["Latitud"] = pd.to_numeric(
    catalogo_validos["Latitud"],
    errors="coerce"
)

catalogo_validos["Longitud"] = pd.to_numeric(
    catalogo_validos["Longitud"],
    errors="coerce"
)

coordenadas_invalidas = (
    catalogo_validos[
        [
            "Latitud",
            "Longitud"
        ]
    ]
    .isna()
    .any(axis=1)
    .sum()
)

print("\n" + "=" * 80)
print("CONTROL DE COORDENADAS")
print("=" * 80)

print(
    f"\n✓ Sensores con coordenadas: "
    f"{len(catalogo_validos) - coordenadas_invalidas:,}"
)

print(
    f"✓ Coordenadas incompletas: "
    f"{coordenadas_invalidas:,}"
)

if coordenadas_invalidas > 0:
    raise ValueError(
        "Existen sensores con coordenadas "
        "geográficas incompletas."
    )


# ============================================================
# 8. VALIDACIÓN DE RANGO GEOGRÁFICO
# ============================================================

coordenadas_fuera_rango = (
    (catalogo_validos["Latitud"] < -90)
    |
    (catalogo_validos["Latitud"] > 90)
    |
    (catalogo_validos["Longitud"] < -180)
    |
    (catalogo_validos["Longitud"] > 180)
).sum()

print(
    f"✓ Coordenadas fuera de rango: "
    f"{coordenadas_fuera_rango:,}"
)

if coordenadas_fuera_rango > 0:
    raise ValueError(
        "Existen coordenadas geográficas fuera de rango."
    )


# ============================================================
# 9. CREAR GEODATAFRAME EN WGS84
# ============================================================

gdf_sensores_ruido = gpd.GeoDataFrame(
    catalogo_validos.copy(),
    geometry=gpd.points_from_xy(
        catalogo_validos["Longitud"],
        catalogo_validos["Latitud"]
    ),
    crs="EPSG:4326"
)


# ============================================================
# 10. TRANSFORMAR A ETRS89 / UTM 31N
# ============================================================

gdf_sensores_ruido = (
    gdf_sensores_ruido
    .to_crs("EPSG:25831")
)


# ============================================================
# 11. GENERAR COORDENADAS PROYECTADAS
# ============================================================

gdf_sensores_ruido["X_ETRS89"] = (
    gdf_sensores_ruido.geometry.x
)

gdf_sensores_ruido["Y_ETRS89"] = (
    gdf_sensores_ruido.geometry.y
)


# ============================================================
# 12. VALIDAR GEOMETRÍAS
# ============================================================

geometrias_invalidas = (
    ~gdf_sensores_ruido.geometry.is_valid
).sum()

geometrias_nulas = (
    gdf_sensores_ruido.geometry.isna()
).sum()

print("\n" + "=" * 80)
print("VALIDACIÓN ESPACIAL")
print("=" * 80)

print(
    f"\n✓ Sensores geolocalizados: "
    f"{len(gdf_sensores_ruido):,}"
)

print(
    f"✓ Geometrías inválidas: "
    f"{geometrias_invalidas:,}"
)

print(
    f"✓ Geometrías nulas: "
    f"{geometrias_nulas:,}"
)

print(
    f"✓ CRS: "
    f"{gdf_sensores_ruido.crs}"
)

if geometrias_invalidas > 0:
    raise ValueError(
        "Existen geometrías inválidas."
    )

if geometrias_nulas > 0:
    raise ValueError(
        "Existen geometrías nulas."
    )


# ============================================================
# 13. SENSOR TEMPORAL + INFORMACIÓN ESPACIAL
# ============================================================

df_ruido_geo = (
    ruido_geo
    .merge(
        gdf_sensores_ruido[
            [
                "Id_Instal",
                "Latitud",
                "Longitud",
                "X_ETRS89",
                "Y_ETRS89"
            ]
        ],
        on="Id_Instal",
        how="left",
        validate="many_to_one"
    )
)


# ============================================================
# 14. DIAGNÓSTICO DE SENSORES SIN GEOLOCALIZACIÓN
# ============================================================

ids_sin_geo = sorted(
    sensores_sin_catalogo
)

df_sin_geo = (
    df_ruido_geo[
        df_ruido_geo["Id_Instal"]
        .isin(ids_sin_geo)
    ]
    .copy()
)

df_sin_geo["Fecha"] = pd.to_datetime(
    df_sin_geo["Fecha"],
    errors="coerce"
)

df_sin_geo["anio"] = (
    df_sin_geo["Fecha"].dt.year
)


# ============================================================
# 15. RESUMEN DEL DIAGNÓSTICO
# ============================================================

print("\n" + "=" * 80)
print("DIAGNÓSTICO DE SENSORES SIN GEOLOCALIZACIÓN")
print("=" * 80)

print(
    f"\n✓ Sensores sin geolocalización: "
    f"{len(ids_sin_geo):,}"
)

print(
    f"✓ Registros afectados: "
    f"{len(df_sin_geo):,}"
)

print(
    f"✓ Registros totales conservados: "
    f"{len(df_ruido_geo):,}"
)


# ============================================================
# 16. DISTRIBUCIÓN ANUAL DE LOS SENSORES SIN GEO
# ============================================================

if len(df_sin_geo) > 0:

    diagnostico_anual = (
        df_sin_geo
        .groupby("anio")
        .agg(
            registros=(
                "Id_Instal",
                "size"
            ),
            sensores=(
                "Id_Instal",
                "nunique"
            ),
            dias=(
                "Fecha",
                "nunique"
            )
        )
        .reset_index()
    )

    print("\n" + "=" * 80)
    print("DISTRIBUCIÓN POR AÑO")
    print("=" * 80)

    display(
        diagnostico_anual
    )


# ============================================================
# 17. OBJETO PARA LAS ETAPAS POSTERIORES
# ============================================================

sensores_ruido_validos = (
    gdf_sensores_ruido
    .copy()
)


# ============================================================
# 18. RESUMEN FINAL
# ============================================================

print("\n" + "=" * 80)
print("RESUMEN FINAL — CELDA 5.3")
print("=" * 80)

print(
    f"\nSensores temporales : "
    f"{len(ids_ruido):,}"
)

print(
    f"Sensores geolocalizados : "
    f"{len(sensores_ruido_validos):,}"
)

print(
    f"Sensores sin geolocalización : "
    f"{len(sensores_sin_catalogo):,}"
)

print(
    f"Registros originales : "
    f"{len(ruido_geo):,}"
)

print(
    f"Registros conservados : "
    f"{len(df_ruido_geo):,}"
)

print(
    f"CRS : "
    f"{sensores_ruido_validos.crs}"
)

print("\n✓ Correspondencia espacial validada.")
print("✓ Sensores geolocalizados correctamente.")
print("✓ Coordenadas transformadas a EPSG:25831.")
print("✓ Sensores sin geolocalización identificados.")
print("✓ Registros sin geolocalización conservados.")
print("✓ df_ruido_geo preparado.")
print("✓ sensores_ruido_validos preparado.")
print("✓ CELDA 5.3 COMPLETADA.")

5.3 — GEOLOCALIZACIÓN Y VALIDACIÓN ESPACIAL

CONTROL DEL CATÁLOGO

✓ Registros catálogo: 989
✓ Id_Instal únicos: 989
✓ Duplicados Id_Instal: 0

CORRESPONDENCIA ESPACIAL

✓ Sensores en las series temporales: 628
✓ Sensores con catálogo espacial: 571
✓ Sensores sin catálogo espacial: 57

CONTROL DE COORDENADAS

✓ Sensores con coordenadas: 571
✓ Coordenadas incompletas: 0
✓ Coordenadas fuera de rango: 0

VALIDACIÓN ESPACIAL

✓ Sensores geolocalizados: 571
✓ Geometrías inválidas: 0
✓ Geometrías nulas: 0
✓ CRS: EPSG:25831

DIAGNÓSTICO DE SENSORES SIN GEOLOCALIZACIÓN

✓ Sensores sin geolocalización: 57
✓ Registros afectados: 12,263
✓ Registros totales conservados: 316,951

DISTRIBUCIÓN POR AÑO


,anio,registros,sensores,dias
0,2018,1675,12,346
1,2019,2659,22,365
2,2020,2389,14,366
3,2021,1828,17,364
4,2022,1685,13,363
5,2023,2027,13,365



RESUMEN FINAL — CELDA 5.3

Sensores temporales : 628
Sensores geolocalizados : 571
Sensores sin geolocalización : 57
Registros originales : 316,951
Registros conservados : 316,951
CRS : EPSG:25831

✓ Correspondencia espacial validada.
✓ Sensores geolocalizados correctamente.
✓ Coordenadas transformadas a EPSG:25831.
✓ Sensores sin geolocalización identificados.
✓ Registros sin geolocalización conservados.
✓ df_ruido_geo preparado.
✓ sensores_ruido_validos preparado.
✓ CELDA 5.3 COMPLETADA.


### Resultado — 5.3. Geolocalización y validación espacial de los sensores acústicos

La integración entre las observaciones temporales de contaminación acústica y el catálogo oficial de instalaciones conserva íntegramente los **316.951 registros** del dataset diario, sin introducir duplicados en la clave `Fecha + Id_Instal`.

De los **628 sensores acústicos** presentes en las series temporales, **571 disponen de correspondencia directa con el catálogo oficial de instalaciones** y pueden ser geolocalizados. Los **57 sensores restantes** no presentan correspondencia con ningún `Id_Instal` del catálogo disponible.

La ausencia de correspondencia espacial afecta a **12.263 registros**, aproximadamente el **3,9 % del dataset acústico**. Estos registros se conservan en el dataset, pero permanecen sin coordenadas y no serán utilizados para la construcción de indicadores espaciales mientras no exista una localización verificable.

El análisis específico de estos sensores muestra que todos los casos sin geolocalización pertenecen al periodo **2018–2023** y proceden de la fuente histórica con resolución original de **1 hora**. No se detectan sensores sin correspondencia asociados a la fuente de 2024.

La distribución temporal de los sensores no geolocalizados no se concentra en un único año, sino que aparecen observaciones a lo largo de todo el periodo 2018–2023. Algunos identificadores presentan además series temporales prolongadas, por lo que sus observaciones se mantienen explícitamente en el dataset en lugar de ser eliminadas durante esta fase.

Para los **571 sensores geolocalizados**, las coordenadas presentan valores coherentes con el ámbito territorial de Barcelona. Las coordenadas originales en **WGS84 (EPSG:4326)** se transforman correctamente al sistema proyectado **ETRS89 / UTM zona 31N (EPSG:25831)**, permitiendo realizar posteriormente cálculos métricos de distancia.

No se detectan sensores con más de una localización, valores infinitos ni alteraciones del número original de registros durante el proceso de geolocalización.

La ausencia de coordenadas de los 57 sensores no se corrige mediante imputación ni mediante asignaciones espaciales aproximadas, preservando así la trazabilidad y evitando introducir localizaciones no verificadas.

> **Conclusión:** se dispone de una red de **571 sensores acústicos con geolocalización verificable** para la construcción de los indicadores espaciales de ruido. Los 57 sensores sin correspondencia en el catálogo oficial se mantienen identificados y conservados en el dataset original, pero quedan excluidos de los cálculos espaciales mientras no exista información oficial suficiente para determinar su localización.

## 5.4. Relación espacial entre estaciones de contaminación y sensores acústicos

Una vez validada la geolocalización de la red acústica, se analiza la relación espacial entre los sensores de ruido con coordenadas verificables y las estaciones físicas de contaminación atmosférica utilizadas en el dataset maestro.

Para cada estación se calculan las distancias euclidianas a los sensores acústicos en el sistema proyectado **ETRS89 / UTM zona 31N (EPSG:25831)**.

A partir de estas distancias se caracteriza la disponibilidad de sensores dentro de tres radios de influencia:

- **500 m**
- **750 m**
- **1000 m**

También se identifica el sensor acústico más próximo a cada estación y su distancia correspondiente.

Esta etapa tiene carácter exclusivamente espacial. Todavía no se incorporan las mediciones `LAeq_dia`, no se realiza ninguna interpolación temporal y no se modifica el dataset maestro.

Los sensores sin geolocalización verificable identificados en la etapa anterior permanecen excluidos únicamente de estos cálculos espaciales.

> **Objetivo:** caracterizar la densidad y proximidad de la red acústica alrededor de cada estación de contaminación antes de definir el método de agregación espacial del indicador diario de ruido.

In [ ]:
# ============================================================
# 5.4. RELACIÓN ESPACIAL ENTRE ESTACIONES DE CONTAMINACIÓN
# Y SENSORES ACÚSTICOS
# ============================================================

import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path


# ============================================================
# 1. PARÁMETROS
# ============================================================

RADIOS = [
    500,
    750,
    1000
]


print("=" * 80)
print("5.4 — RELACIÓN ESPACIAL ESTACIONES ↔ SENSORES ACÚSTICOS")
print("=" * 80)


# ============================================================
# 2. COMPROBAR VARIABLES GENERADAS EN 5.3
# ============================================================

if "gdf_sensores_ruido" not in globals():
    raise NameError(
        "No existe gdf_sensores_ruido. "
        "Debe ejecutarse previamente la celda 5.3."
    )


# ============================================================
# 3. PREPARAR SENSORES ACÚSTICOS GEOLOCALIZADOS
# ============================================================

sensores_ruido_validos = (
    gdf_sensores_ruido[
        gdf_sensores_ruido[
            [
                "X_ETRS89",
                "Y_ETRS89"
            ]
        ]
        .notna()
        .all(axis=1)
    ]
    .copy()
)

sensores_ruido_validos = (
    sensores_ruido_validos
    .drop_duplicates(
        subset=["Id_Instal"]
    )
    .reset_index(drop=True)
)


n_sensores_totales = (
    gdf_sensores_ruido[
        "Id_Instal"
    ]
    .nunique()
)

n_sensores_validos = (
    sensores_ruido_validos[
        "Id_Instal"
    ]
    .nunique()
)

n_sensores_excluidos = (
    n_sensores_totales
    -
    n_sensores_validos
)


print(
    f"\n✓ Sensores acústicos totales: "
    f"{n_sensores_totales:,}"
)

print(
    f"✓ Sensores acústicos geolocalizados: "
    f"{n_sensores_validos:,}"
)

print(
    f"✓ Sensores excluidos del cálculo espacial: "
    f"{n_sensores_excluidos:,}"
)


# ============================================================
# 4. DEFINIR RUTA DEL CHECKPOINT 03
# ============================================================

ruta_base = Path(
    "/content/drive/MyDrive/TFM/"
    "11_Machine_Learning_Temporal"
)

ruta_maestro_03 = (
    ruta_base /
    "INTEGRACION" /
    "03_maestro_contaminacion_meteorologia_trafico.csv"
)


# ============================================================
# 5. COMPROBAR EXISTENCIA DEL CHECKPOINT 03
# ============================================================

if not ruta_maestro_03.exists():

    raise FileNotFoundError(
        "No se encuentra el Checkpoint 03:\n"
        f"{ruta_maestro_03}"
    )


# ============================================================
# 6. CARGAR CHECKPOINT 03
# ============================================================

df_maestro_03 = pd.read_csv(
    ruta_maestro_03,
    low_memory=False
)


print("\n" + "=" * 80)
print("CHECKPOINT 03")
print("=" * 80)

print(
    f"\n✓ Filas: "
    f"{len(df_maestro_03):,}"
)

print(
    f"✓ Columnas: "
    f"{df_maestro_03.shape[1]}"
)


# ============================================================
# 7. COMPROBAR VARIABLES ESPACIALES DEL MAESTRO
# ============================================================

columnas_estaciones_necesarias = [
    "estacion_fisica",
    "estacion_geo",
    "X_ETRS89",
    "Y_ETRS89"
]

faltantes_estaciones = [
    columna
    for columna in columnas_estaciones_necesarias
    if columna not in df_maestro_03.columns
]

if faltantes_estaciones:

    raise KeyError(
        "Faltan variables necesarias en el Checkpoint 03:\n"
        f"{faltantes_estaciones}"
    )


# ============================================================
# 8. RECUPERAR LAS LOCALIZACIONES FÍSICAS
# ============================================================

df_estaciones_cont = (
    df_maestro_03[
        columnas_estaciones_necesarias
    ]
    .drop_duplicates()
    .copy()
)


# ============================================================
# 9. COMPROBAR SI estacion_geo TIENE UNA POSICIÓN ÚNICA
# ============================================================

control_posiciones = (
    df_estaciones_cont
    .groupby("estacion_geo")
    .agg(
        n_x=(
            "X_ETRS89",
            "nunique"
        ),
        n_y=(
            "Y_ETRS89",
            "nunique"
        )
    )
    .reset_index()
)

posiciones_problematicas = (
    control_posiciones[
        (control_posiciones["n_x"] > 1)
        |
        (control_posiciones["n_y"] > 1)
    ]
)


if len(posiciones_problematicas) > 0:

    print(
        "\n⚠ Estaciones con más "
        "de una posición espacial:"
    )

    display(
        posiciones_problematicas
    )

    raise ValueError(
        "Alguna estacion_geo presenta "
        "más de una localización espacial."
    )


# ============================================================
# 10. CONSERVAR UNA FILA POR estacion_geo
# ============================================================

df_estaciones_cont = (
    df_estaciones_cont
    .drop_duplicates(
        subset=["estacion_geo"]
    )
    .reset_index(drop=True)
)


# ============================================================
# 11. CONVERTIR COORDENADAS A NUMÉRICO
# ============================================================

df_estaciones_cont[
    "X_ETRS89"
] = pd.to_numeric(
    df_estaciones_cont[
        "X_ETRS89"
    ],
    errors="coerce"
)

df_estaciones_cont[
    "Y_ETRS89"
] = pd.to_numeric(
    df_estaciones_cont[
        "Y_ETRS89"
    ],
    errors="coerce"
)


# ============================================================
# 12. CONTROL DE COORDENADAS NULAS
# ============================================================

nulos_estaciones = (
    df_estaciones_cont[
        [
            "X_ETRS89",
            "Y_ETRS89"
        ]
    ]
    .isna()
    .any(axis=1)
    .sum()
)


print("\n" + "=" * 80)
print("ESTACIONES FÍSICAS DE CONTAMINACIÓN")
print("=" * 80)

print(
    f"\n✓ Localizaciones físicas recuperadas: "
    f"{len(df_estaciones_cont):,}"
)

print(
    f"✓ estacion_geo diferentes: "
    f"{df_estaciones_cont['estacion_geo'].nunique():,}"
)

print(
    f"✓ Estaciones sin coordenadas completas: "
    f"{nulos_estaciones:,}"
)

display(
    df_estaciones_cont
    .sort_values(
        "estacion_geo"
    )
)


if nulos_estaciones > 0:

    raise ValueError(
        "Existen estaciones de contaminación "
        "sin coordenadas válidas."
    )


if (
    df_estaciones_cont[
        "estacion_geo"
    ]
    .nunique()
    != 8
):

    raise ValueError(
        "Se esperaban 8 localizaciones físicas "
        "de contaminación."
    )


# ============================================================
# 13. CREAR GEODATAFRAME DE ESTACIONES
# ============================================================

gdf_estaciones_cont = gpd.GeoDataFrame(
    df_estaciones_cont,
    geometry=gpd.points_from_xy(
        df_estaciones_cont[
            "X_ETRS89"
        ],
        df_estaciones_cont[
            "Y_ETRS89"
        ]
    ),
    crs="EPSG:25831"
)


# ============================================================
# 14. COMPROBAR CRS DE LOS SENSORES
# ============================================================

if sensores_ruido_validos.crs is None:

    raise ValueError(
        "El GeoDataFrame de sensores acústicos "
        "no tiene CRS definido."
    )

if (
    sensores_ruido_validos.crs
    !=
    gdf_estaciones_cont.crs
):

    sensores_ruido_validos = (
        sensores_ruido_validos
        .to_crs(
            gdf_estaciones_cont.crs
        )
    )


print("\n" + "=" * 80)
print("SISTEMA DE REFERENCIA")
print("=" * 80)

print(
    f"\n✓ CRS estaciones: "
    f"{gdf_estaciones_cont.crs}"
)

print(
    f"✓ CRS sensores: "
    f"{sensores_ruido_validos.crs}"
)


# ============================================================
# 15. PREPARAR TABLA DE ESTACIONES
# ============================================================

estaciones_calc = (
    gdf_estaciones_cont[
        [
            "estacion_fisica",
            "estacion_geo",
            "X_ETRS89",
            "Y_ETRS89"
        ]
    ]
    .rename(
        columns={
            "X_ETRS89":
                "X_estacion",

            "Y_ETRS89":
                "Y_estacion"
        }
    )
    .copy()
)


# ============================================================
# 16. PREPARAR TABLA DE SENSORES
# ============================================================

sensores_calc = (
    sensores_ruido_validos[
        [
            "Id_Instal",
            "X_ETRS89",
            "Y_ETRS89"
        ]
    ]
    .rename(
        columns={
            "X_ETRS89":
                "X_sensor",

            "Y_ETRS89":
                "Y_sensor"
        }
    )
    .copy()
)


# ============================================================
# 17. CONSTRUIR PRODUCTO CARTESIANO ESTACIÓN ↔ SENSOR
# ============================================================

estaciones_calc[
    "_key"
] = 1

sensores_calc[
    "_key"
] = 1


df_dist_ruido = (
    estaciones_calc
    .merge(
        sensores_calc,
        on="_key",
        how="inner"
    )
    .drop(
        columns="_key"
    )
)


# ============================================================
# 18. CALCULAR DISTANCIA EUCLIDIANA
# ============================================================

df_dist_ruido[
    "distancia_m"
] = np.sqrt(
    (
        df_dist_ruido[
            "X_sensor"
        ]
        -
        df_dist_ruido[
            "X_estacion"
        ]
    ) ** 2
    +
    (
        df_dist_ruido[
            "Y_sensor"
        ]
        -
        df_dist_ruido[
            "Y_estacion"
        ]
    ) ** 2
)


# ============================================================
# 19. CONTROL DE LA MATRIZ ESPACIAL
# ============================================================

pares_esperados = (
    8
    *
    n_sensores_validos
)

pares_obtenidos = (
    len(
        df_dist_ruido
    )
)


print("\n" + "=" * 80)
print("MATRIZ ESPACIAL ESTACIÓN ↔ SENSOR")
print("=" * 80)

print(
    f"\n✓ Estaciones: "
    f"{df_dist_ruido['estacion_geo'].nunique():,}"
)

print(
    f"✓ Sensores: "
    f"{df_dist_ruido['Id_Instal'].nunique():,}"
)

print(
    f"✓ Pares esperados: "
    f"{pares_esperados:,}"
)

print(
    f"✓ Pares calculados: "
    f"{pares_obtenidos:,}"
)


if pares_esperados != pares_obtenidos:

    raise ValueError(
        "El número de pares estación-sensor "
        "no coincide con el esperado."
    )


# ============================================================
# 20. SENSOR MÁS PRÓXIMO A CADA ESTACIÓN
# ============================================================

idx_min = (
    df_dist_ruido
    .groupby(
        "estacion_geo"
    )[
        "distancia_m"
    ]
    .idxmin()
)


sensor_mas_cercano = (
    df_dist_ruido
    .loc[
        idx_min,
        [
            "estacion_fisica",
            "estacion_geo",
            "Id_Instal",
            "distancia_m"
        ]
    ]
    .rename(
        columns={
            "Id_Instal":
                "sensor_mas_cercano",

            "distancia_m":
                "dist_sensor_mas_cercano_m"
        }
    )
    .sort_values(
        "estacion_geo"
    )
    .reset_index(
        drop=True
    )
)


print("\n" + "=" * 80)
print("SENSOR ACÚSTICO MÁS PRÓXIMO")
print("=" * 80)

display(
    sensor_mas_cercano
)


# ============================================================
# 21. CONTAR SENSORES EN CADA RADIO
# ============================================================

resumen_radios = (
    df_estaciones_cont[
        [
            "estacion_fisica",
            "estacion_geo"
        ]
    ]
    .copy()
)


for radio in RADIOS:

    conteo = (
        df_dist_ruido[
            df_dist_ruido[
                "distancia_m"
            ] <= radio
        ]
        .groupby(
            [
                "estacion_fisica",
                "estacion_geo"
            ]
        )[
            "Id_Instal"
        ]
        .nunique()
        .reset_index(
            name=
            f"n_sensores_{radio}m"
        )
    )

    resumen_radios = pd.merge(
        resumen_radios,
        conteo,

        on=[
            "estacion_fisica",
            "estacion_geo"
        ],

        how="left",
        validate="one_to_one"
    )

    resumen_radios[
        f"n_sensores_{radio}m"
    ] = (
        resumen_radios[
            f"n_sensores_{radio}m"
        ]
        .fillna(0)
        .astype(int)
    )


# ============================================================
# 22. INCORPORAR SENSOR MÁS CERCANO
# ============================================================

resumen_espacial_ruido = pd.merge(
    resumen_radios,
    sensor_mas_cercano,

    on=[
        "estacion_fisica",
        "estacion_geo"
    ],

    how="left",
    validate="one_to_one"
)


resumen_espacial_ruido = (
    resumen_espacial_ruido
    .sort_values(
        "estacion_geo"
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 80)
print("COBERTURA ESPACIAL POR ESTACIÓN")
print("=" * 80)

display(
    resumen_espacial_ruido
)


# ============================================================
# 23. ESTADÍSTICOS DE DISTANCIA DENTRO DE 1.000 m
# ============================================================

df_distancias_1000 = (
    df_dist_ruido[
        df_dist_ruido[
            "distancia_m"
        ] <= 1000
    ]
    .copy()
)


distancias_1000 = (
    df_distancias_1000
    .groupby(
        "estacion_geo"
    )[
        "distancia_m"
    ]
    .agg(
        n_sensores="count",
        distancia_min_m="min",
        distancia_mediana_m="median",
        distancia_media_m="mean",
        distancia_max_m="max"
    )
    .reset_index()
)


print("\n" + "=" * 80)
print("DISTANCIAS DE SENSORES DENTRO DE 1.000 m")
print("=" * 80)

display(
    distancias_1000
)


# ============================================================
# 24. CONTROL DE COBERTURA POR RADIO
# ============================================================

print("\n" + "=" * 80)
print("CONTROL DE COBERTURA ESPACIAL")
print("=" * 80)


resumen_cobertura_radios = []

for radio in RADIOS:

    columna = (
        f"n_sensores_{radio}m"
    )

    estaciones_con_sensor = (
        resumen_espacial_ruido[
            columna
        ] > 0
    ).sum()

    estaciones_sin_sensor = (
        resumen_espacial_ruido[
            columna
        ] == 0
    ).sum()

    resumen_cobertura_radios.append({
        "radio_m":
            radio,

        "estaciones_con_sensor":
            estaciones_con_sensor,

        "estaciones_sin_sensor":
            estaciones_sin_sensor,

        "cobertura_pct":
            round(
                estaciones_con_sensor
                /
                8
                *
                100,
                2
            )
    })


df_cobertura_radios = pd.DataFrame(
    resumen_cobertura_radios
)

display(
    df_cobertura_radios
)


# ============================================================
# 25. DISTRIBUCIÓN GLOBAL DEL NÚMERO DE SENSORES
# ============================================================

resumen_global_radios = []

for radio in RADIOS:

    columna = (
        f"n_sensores_{radio}m"
    )

    resumen_global_radios.append({
        "radio_m":
            radio,

        "min_sensores":
            resumen_espacial_ruido[
                columna
            ].min(),

        "mediana_sensores":
            resumen_espacial_ruido[
                columna
            ].median(),

        "media_sensores":
            resumen_espacial_ruido[
                columna
            ].mean(),

        "max_sensores":
            resumen_espacial_ruido[
                columna
            ].max()
    })


df_resumen_global_radios = pd.DataFrame(
    resumen_global_radios
)


print("\n" + "=" * 80)
print("DISTRIBUCIÓN GLOBAL DE SENSORES POR RADIO")
print("=" * 80)

display(
    df_resumen_global_radios
)


# ============================================================
# 26. CINCO SENSORES MÁS PRÓXIMOS POR ESTACIÓN
# ============================================================

df_5_sensores = (
    df_dist_ruido
    .sort_values(
        [
            "estacion_geo",
            "distancia_m"
        ]
    )
    .groupby(
        "estacion_geo",
        group_keys=False
    )
    .head(5)
    .copy()
)


df_5_sensores[
    "orden_vecino"
] = (
    df_5_sensores
    .groupby(
        "estacion_geo"
    )
    .cumcount()
    + 1
)


print("\n" + "=" * 80)
print("CINCO SENSORES MÁS PRÓXIMOS POR ESTACIÓN")
print("=" * 80)

display(
    df_5_sensores[
        [
            "estacion_geo",
            "orden_vecino",
            "Id_Instal",
            "distancia_m"
        ]
    ]
)


# ============================================================
# 27. DISTRIBUCIÓN DE DISTANCIAS DE LOS CINCO VECINOS
# ============================================================

resumen_vecinos = (
    df_5_sensores
    .groupby(
        "orden_vecino"
    )[
        "distancia_m"
    ]
    .agg(
        minimo_m="min",
        mediana_m="median",
        media_m="mean",
        maximo_m="max"
    )
    .reset_index()
)


print("\n" + "=" * 80)
print("DISTANCIAS A LOS PRIMEROS CINCO SENSORES")
print("=" * 80)

display(
    resumen_vecinos
)


# ============================================================
# 28. CONTROL DE DUPLICADOS DE LOS PARES
# ============================================================

duplicados_pares = (
    df_dist_ruido
    .duplicated(
        subset=[
            "estacion_geo",
            "Id_Instal"
        ]
    )
    .sum()
)


# ============================================================
# 29. CONTROL DE DISTANCIAS INVÁLIDAS
# ============================================================

distancias_invalidas = (
    (
        ~np.isfinite(
            df_dist_ruido[
                "distancia_m"
            ]
        )
    )
    |
    (
        df_dist_ruido[
            "distancia_m"
        ] < 0
    )
).sum()


print("\n" + "=" * 80)
print("CONTROL DE INTEGRIDAD ESPACIAL")
print("=" * 80)

print(
    f"\n✓ Duplicados estación + sensor: "
    f"{duplicados_pares:,}"
)

print(
    f"✓ Distancias inválidas: "
    f"{distancias_invalidas:,}"
)


if duplicados_pares > 0:

    raise ValueError(
        "Existen pares estación-sensor duplicados."
    )


if distancias_invalidas > 0:

    raise ValueError(
        "Existen distancias espaciales inválidas."
    )


# ============================================================
# 30. RESUMEN FINAL
# ============================================================

print("\n" + "=" * 80)
print("RELACIÓN ESPACIAL 5.4 FINALIZADA")
print("=" * 80)

print(
    f"\n✓ Estaciones analizadas: "
    f"{df_estaciones_cont['estacion_geo'].nunique():,}"
)

print(
    f"✓ Sensores acústicos geolocalizados: "
    f"{n_sensores_validos:,}"
)

print(
    f"✓ Sensores excluidos por falta de geolocalización: "
    f"{n_sensores_excluidos:,}"
)

print(
    f"✓ Pares estación-sensor calculados: "
    f"{pares_obtenidos:,}"
)

print(
    f"✓ Pares esperados: "
    f"{pares_esperados:,}"
)

print(
    f"✓ Estaciones con ≥1 sensor a 500 m: "
    f"{(resumen_espacial_ruido['n_sensores_500m'] > 0).sum():,}/8"
)

print(
    f"✓ Estaciones con ≥1 sensor a 750 m: "
    f"{(resumen_espacial_ruido['n_sensores_750m'] > 0).sum():,}/8"
)

print(
    f"✓ Estaciones con ≥1 sensor a 1000 m: "
    f"{(resumen_espacial_ruido['n_sensores_1000m'] > 0).sum():,}/8"
)

print(
    f"✓ Duplicados estación + sensor: "
    f"{duplicados_pares:,}"
)

print(
    f"✓ Distancias inválidas: "
    f"{distancias_invalidas:,}"
)

print(
    "✓ No se han agregado todavía "
    "las mediciones LAeq_dia."
)

print(
    "✓ No se ha realizado ninguna imputación."
)

print(
    "✓ No se ha modificado el dataset maestro."
)

print(
    "\n✓ Estructura preparada para seleccionar "
    "la estrategia espacial del indicador acústico."
)

5.4 — RELACIÓN ESPACIAL ESTACIONES ↔ SENSORES ACÚSTICOS

✓ Sensores acústicos totales: 571
✓ Sensores acústicos geolocalizados: 571
✓ Sensores excluidos del cálculo espacial: 0

CHECKPOINT 03

✓ Filas: 43,642
✓ Columnas: 59

ESTACIONES FÍSICAS DE CONTAMINACIÓN

✓ Localizaciones físicas recuperadas: 8
✓ estacion_geo diferentes: 8
✓ Estaciones sin coordenadas completas: 0


,estacion_fisica,estacion_geo,X_ETRS89,Y_ETRS89
0,Ciutadella,Ciutadella,432059.474731,4.581971e+06
1,Eixample,Eixample,429249.000364,4.581876e+06
2,Gràcia,Gràcia,429230.091303,4.583364e+06
7,Observ Fabra,Observatori Fabra,426786.192307,4.585575e+06
3,Palau Reial,Palau Reial,426032.466989,4.582152e+06
4,Poblenou,Poblenou,433507.035285,4.583901e+06
5,Sants_hist,Sants,427486.280382,4.581205e+06
6,Vall Hebron,Vall d'Hebron,428791.895985,4.586410e+06



SISTEMA DE REFERENCIA

✓ CRS estaciones: EPSG:25831
✓ CRS sensores: EPSG:25831

MATRIZ ESPACIAL ESTACIÓN ↔ SENSOR

✓ Estaciones: 8
✓ Sensores: 571
✓ Pares esperados: 4,568
✓ Pares calculados: 4,568

SENSOR ACÚSTICO MÁS PRÓXIMO


,estacion_fisica,estacion_geo,sensor_mas_cercano,dist_sensor_mas_cercano_m
0,Ciutadella,Ciutadella,8986,391.178550
1,Eixample,Eixample,3807,148.788130
2,Gràcia,Gràcia,9806,193.270297
3,Observ Fabra,Observatori Fabra,3470,1084.145703
4,Palau Reial,Palau Reial,4586,55.016006
5,Poblenou,Poblenou,3469,378.663087
6,Sants_hist,Sants,9026,178.631877
7,Vall Hebron,Vall d'Hebron,4866,563.805498



COBERTURA ESPACIAL POR ESTACIÓN


...


CINCO SENSORES MÁS PRÓXIMOS POR ESTACIÓN


,estacion_geo,orden_vecino,Id_Instal,distancia_m
487,Ciutadella,1,8986,391.178550
543,Ciutadella,2,9667,391.178550
559,Ciutadella,3,9827,391.178550
490,Ciutadella,4,8989,396.086317
491,Ciutadella,5,8990,396.086317
730,Eixample,1,3807,148.788130
885,Eixample,2,6748,281.594652
1011,Eixample,3,8346,281.594652
896,Eixample,4,6759,299.253310
876,Eixample,5,6688,361.354289



DISTANCIAS A LOS PRIMEROS CINCO SENSORES


,orden_vecino,minimo_m,mediana_m,media_m,maximo_m
0,1,55.016006,285.966692,374.187393,1084.145703
1,2,210.631512,408.551711,544.410614,1456.227932
2,3,281.594652,421.447685,654.637707,1592.138329
3,4,289.668483,443.898697,673.445413,1609.995631
4,5,306.702007,535.283799,765.881134,1726.691742



CONTROL DE INTEGRIDAD ESPACIAL

✓ Duplicados estación + sensor: 0
✓ Distancias inválidas: 0

RELACIÓN ESPACIAL 5.4 FINALIZADA

✓ Estaciones analizadas: 8
✓ Sensores acústicos geolocalizados: 571
✓ Sensores excluidos por falta de geolocalización: 0
✓ Pares estación-sensor calculados: 4,568
✓ Pares esperados: 4,568
✓ Estaciones con ≥1 sensor a 500 m: 6/8
✓ Estaciones con ≥1 sensor a 750 m: 7/8
✓ Estaciones con ≥1 sensor a 1000 m: 7/8
✓ Duplicados estación + sensor: 0
✓ Distancias inválidas: 0
✓ No se han agregado todavía las mediciones LAeq_dia.
✓ No se ha realizado ninguna imputación.
✓ No se ha modificado el dataset maestro.

✓ Estructura preparada para seleccionar la estrategia espacial del indicador acústico.


### Resultado — 5.4. Relación espacial entre estaciones de contaminación y sensores acústicos

La caracterización espacial confirma una red acústica amplia pero heterogéneamente distribuida alrededor de las estaciones de contaminación atmosférica.

Se analizan **571 sensores acústicos geolocalizados** y las **8 estaciones físicas** del dataset maestro, generándose exactamente **4.568 pares estación-sensor**, sin duplicados ni distancias inválidas.

La cobertura espacial varía según el radio considerado:

- **500 m:** 6 de 8 estaciones disponen de al menos un sensor;
- **750 m:** 7 de 8 estaciones presentan cobertura;
- **1.000 m:** 7 de 8 estaciones presentan cobertura.

La única estación sin sensores dentro de 1.000 m es **Observatori Fabra**, cuyo sensor acústico más próximo se localiza aproximadamente a **1.084 m**.

La densidad de sensores presenta diferencias importantes entre estaciones. Dentro de un radio de 1.000 m se identifican desde **2 sensores en Vall d'Hebron** y **4 en Palau Reial**, hasta **61 en Eixample** y **101 en Gràcia**.

Estas diferencias desaconsejan utilizar exclusivamente una media aritmética de las observaciones acústicas dentro de un radio fijo, ya que sensores situados a distancias muy diferentes recibirían el mismo peso.

Se plantea, por tanto, construir indicadores acústicos multiescala mediante ponderación inversa por distancia (IDW) para radios de **500, 750 y 1.000 m**, conservando adicionalmente el número de sensores disponibles y la distancia al sensor más próximo.

Para Observatori Fabra se evaluará de forma explícita la utilización del sensor acústico más cercano, situado ligeramente por encima del umbral de 1.000 m, evitando perder completamente la representación acústica de esta estación por una diferencia espacial marginal.

> **Conclusión:** la estructura espacial de la red justifica una estrategia acústica multiescala y ponderada por distancia, manteniendo un tratamiento específico para Observatori Fabra debido a su menor densidad de sensores próximos.

## 5.5. Construcción de indicadores diarios de contaminación acústica

Una vez caracterizada la relación espacial entre la red acústica y las estaciones de contaminación atmosférica, se construyen indicadores diarios de exposición al ruido con granularidad `fecha + estacion_geo`.

El nivel equivalente de presión sonora `LAeq` se expresa en **decibelios**, una escala logarítmica. Por este motivo, la combinación de diferentes sensores no se realiza directamente mediante una media aritmética de los valores en dB.

Las observaciones se transforman previamente a una magnitud energética proporcional mediante:

\[
E = 10^{LAeq/10}
\]

Los indicadores espaciales se calculan sobre esta escala y posteriormente se transforman de nuevo a decibelios.

Se consideran exclusivamente observaciones acústicas clasificadas como `Dia_valido = True` y correspondientes a sensores con geolocalización verificable.

A partir de la caracterización espacial previa se adoptan cuatro radios diagnósticos:

- **500 m**
- **750 m**
- **1.000 m**
- **1.500 m**

Los tres primeros representan escalas progresivamente más amplias del entorno inmediato de las estaciones. El radio de **1.500 m** se incorpora como escala adicional de diagnóstico para evaluar si permite mejorar la cobertura de las estaciones con menor densidad de sensores, evitando recurrir directamente a observaciones situadas a varios kilómetros.

Para cada combinación `fecha + estacion_geo` y radio se calculan:

- `LAeq_idw_*`: nivel acústico equivalente ponderado mediante distancia inversa;
- `LAeq_energetico_*`: nivel equivalente obtenido mediante agregación energética;
- `n_sensores_obs_*`: número de sensores con observación diaria válida;
- `dist_sensor_min_*`: distancia al sensor válido más próximo dentro del radio;
- `cobertura_media_sensores_*`: cobertura media de las observaciones utilizadas.

Adicionalmente se conserva `LAeq_sensor_mas_cercano` y su distancia correspondiente, independientemente del radio. Este indicador se utiliza únicamente como referencia diagnóstica para analizar situaciones de baja cobertura espacial y no se adopta automáticamente como variable acústica final.

No se realiza ninguna imputación temporal ni espacial. Cuando no existe ninguna observación válida dentro de un radio determinado, el indicador correspondiente permanece como `NaN`.

> **Objetivo:** construir indicadores acústicos diarios multiescala, ponderados espacialmente y compatibles con la naturaleza logarítmica del `LAeq`, para seleccionar posteriormente la escala que ofrezca el mejor equilibrio entre proximidad espacial, disponibilidad temporal y representatividad acústica.

In [ ]:
# ============================================================
# 5.5. CONSTRUCCIÓN DE INDICADORES DIARIOS
# DE CONTAMINACIÓN ACÚSTICA
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 1. PARÁMETROS
# ============================================================

RADIOS = [
    500,
    750,
    1000,
    1500
]

print("=" * 80)
print("5.5 — CONSTRUCCIÓN DE INDICADORES DIARIOS DE RUIDO")
print("=" * 80)


# ============================================================
# 2. COMPROBAR OBJETOS NECESARIOS
# ============================================================

objetos_necesarios = [
    "df_ruido_geo",
    "df_dist_ruido"
]

faltantes_objetos = [
    nombre
    for nombre in objetos_necesarios
    if nombre not in globals()
]

if faltantes_objetos:
    raise NameError(
        "Faltan objetos generados en las etapas anteriores:\n"
        f"{faltantes_objetos}"
    )


# ============================================================
# 3. PREPARAR DATASET TEMPORAL DE RUIDO
# ============================================================

df_ruido_ind = (
    df_ruido_geo
    .copy()
)

df_ruido_ind["Fecha"] = pd.to_datetime(
    df_ruido_ind["Fecha"],
    errors="coerce"
)

df_ruido_ind["LAeq_dia"] = pd.to_numeric(
    df_ruido_ind["LAeq_dia"],
    errors="coerce"
)


# ============================================================
# 4. NORMALIZAR Id_Instal
# ============================================================

df_ruido_ind["Id_Instal"] = (
    df_ruido_ind["Id_Instal"]
    .astype(str)
    .str.strip()
)

df_dist_ruido = (
    df_dist_ruido
    .copy()
)

df_dist_ruido["Id_Instal"] = (
    df_dist_ruido["Id_Instal"]
    .astype(str)
    .str.strip()
)


# ============================================================
# 5. CONTROL PREVIO DE LAS MEDICIONES
# ============================================================

fechas_invalidas = (
    df_ruido_ind["Fecha"]
    .isna()
    .sum()
)

laeq_nulos = (
    df_ruido_ind["LAeq_dia"]
    .isna()
    .sum()
)

print("\n" + "=" * 80)
print("CONTROL PREVIO DE LAS MEDICIONES")
print("=" * 80)

print(
    f"\n✓ Registros originales: "
    f"{len(df_ruido_ind):,}"
)

print(
    f"✓ Fechas inválidas: "
    f"{fechas_invalidas:,}"
)

print(
    f"✓ LAeq_dia nulos: "
    f"{laeq_nulos:,}"
)

if fechas_invalidas > 0:
    raise ValueError(
        "Existen fechas inválidas."
    )


# ============================================================
# 6. SELECCIONAR ÚNICAMENTE DÍAS VÁLIDOS
# ============================================================

df_ruido_valido = (
    df_ruido_ind[
        df_ruido_ind[
            "Dia_valido"
        ] == True
    ]
    .copy()
)

n_validos = len(
    df_ruido_valido
)

n_no_validos = (
    len(df_ruido_ind)
    -
    n_validos
)

print("\n" + "=" * 80)
print("SELECCIÓN DE OBSERVACIONES VÁLIDAS")
print("=" * 80)

print(
    f"\n✓ Registros válidos utilizados: "
    f"{n_validos:,}"
)

print(
    f"✓ Registros no válidos excluidos "
    f"del cálculo: "
    f"{n_no_validos:,}"
)

print(
    f"✓ Porcentaje utilizado: "
    f"{n_validos / len(df_ruido_ind) * 100:.2f} %"
)


# ============================================================
# 7. EXCLUIR SENSORES SIN GEOLOCALIZACIÓN
# ============================================================

df_ruido_valido_geo = (
    df_ruido_valido[
        df_ruido_valido[
            [
                "X_ETRS89",
                "Y_ETRS89"
            ]
        ]
        .notna()
        .all(axis=1)
    ]
    .copy()
)

registros_validos_sin_geo = (
    len(df_ruido_valido)
    -
    len(df_ruido_valido_geo)
)

sensores_validos_geo = (
    df_ruido_valido_geo[
        "Id_Instal"
    ]
    .nunique()
)

print("\n" + "=" * 80)
print("COBERTURA ESPACIAL DE LAS OBSERVACIONES VÁLIDAS")
print("=" * 80)

print(
    f"\n✓ Registros válidos geolocalizados: "
    f"{len(df_ruido_valido_geo):,}"
)

print(
    f"✓ Registros válidos sin geolocalización: "
    f"{registros_validos_sin_geo:,}"
)

print(
    f"✓ Sensores geolocalizados con "
    f"observaciones válidas: "
    f"{sensores_validos_geo:,}"
)


# ============================================================
# 8. TRANSFORMACIÓN ENERGÉTICA DEL LAeq
# ============================================================

df_ruido_valido_geo[
    "energia_acustica"
] = (
    10 ** (
        df_ruido_valido_geo[
            "LAeq_dia"
        ] / 10.0
    )
)


# ============================================================
# 9. PREPARAR RELACIONES ESPACIALES
# ============================================================

relaciones_ruido = (
    df_dist_ruido[
        [
            "estacion_geo",
            "Id_Instal",
            "distancia_m"
        ]
    ]
    .copy()
)

duplicados_relaciones = (
    relaciones_ruido
    .duplicated(
        subset=[
            "estacion_geo",
            "Id_Instal"
        ]
    )
    .sum()
)

if duplicados_relaciones > 0:
    raise ValueError(
        "Existen relaciones estación-sensor duplicadas."
    )


# ============================================================
# 10. RELACIONAR MEDICIONES DIARIAS CON ESTACIONES
# ============================================================

df_ruido_rel = pd.merge(
    df_ruido_valido_geo[
        [
            "Fecha",
            "Id_Instal",
            "LAeq_dia",
            "energia_acustica",
            "Cobertura_corregida_%"
        ]
    ],
    relaciones_ruido,
    on="Id_Instal",
    how="inner",
    validate="many_to_many"
)


print("\n" + "=" * 80)
print("RELACIONES DIARIAS ESTACIÓN ↔ SENSOR")
print("=" * 80)

print(
    f"\n✓ Contribuciones diarias calculadas: "
    f"{len(df_ruido_rel):,}"
)

print(
    f"✓ Estaciones representadas: "
    f"{df_ruido_rel['estacion_geo'].nunique()}"
)

print(
    f"✓ Sensores utilizados: "
    f"{df_ruido_rel['Id_Instal'].nunique():,}"
)


# ============================================================
# 11. PESO INVERSO A LA DISTANCIA
# ============================================================

df_ruido_rel[
    "peso_idw"
] = (
    1.0 /
    df_ruido_rel[
        "distancia_m"
    ].clip(lower=1.0)
)

df_ruido_rel[
    "energia_x_peso"
] = (
    df_ruido_rel[
        "energia_acustica"
    ]
    *
    df_ruido_rel[
        "peso_idw"
    ]
)


# ============================================================
# 12. FUNCIÓN DE AGREGACIÓN POR RADIO
# ============================================================

def construir_indicadores_ruido_radio(
    df,
    radio
):

    temp = (
        df[
            df[
                "distancia_m"
            ] <= radio
        ]
        .copy()
    )

    if temp.empty:
        return pd.DataFrame(
            columns=[
                "Fecha",
                "estacion_geo"
            ]
        )

    agregado = (
        temp
        .groupby(
            [
                "Fecha",
                "estacion_geo"
            ]
        )
        .agg(
            n_sensores_obs=(
                "Id_Instal",
                "nunique"
            ),

            distancia_min=(
                "distancia_m",
                "min"
            ),

            energia_media=(
                "energia_acustica",
                "mean"
            ),

            suma_energia_peso=(
                "energia_x_peso",
                "sum"
            ),

            suma_peso=(
                "peso_idw",
                "sum"
            ),

            cobertura_media_sensores=(
                "Cobertura_corregida_%",
                "mean"
            )
        )
        .reset_index()
    )

    # --------------------------------------------------------
    # Agregación energética sin ponderación espacial
    # --------------------------------------------------------

    agregado[
        f"LAeq_energetico_{radio}m"
    ] = (
        10
        *
        np.log10(
            agregado[
                "energia_media"
            ]
        )
    )

    # --------------------------------------------------------
    # Agregación energética ponderada por distancia
    # --------------------------------------------------------

    agregado[
        f"LAeq_idw_{radio}m"
    ] = (
        10
        *
        np.log10(
            agregado[
                "suma_energia_peso"
            ]
            /
            agregado[
                "suma_peso"
            ]
        )
    )

    agregado = agregado.rename(
        columns={
            "n_sensores_obs":
                f"n_sensores_obs_{radio}m",

            "distancia_min":
                f"dist_sensor_min_{radio}m",

            "cobertura_media_sensores":
                f"cobertura_media_sensores_{radio}m"
        }
    )

    agregado = agregado.drop(
        columns=[
            "energia_media",
            "suma_energia_peso",
            "suma_peso"
        ]
    )

    return agregado


# ============================================================
# 13. CALCULAR INDICADORES MULTIESCALA
# ============================================================

indicadores_ruido_radio = {}

for radio in RADIOS:

    indicadores_ruido_radio[
        radio
    ] = (
        construir_indicadores_ruido_radio(
            df_ruido_rel,
            radio
        )
    )

    print(
        f"\n✓ Radio {radio} m: "
        f"{len(indicadores_ruido_radio[radio]):,} "
        f"combinaciones fecha-estación"
    )


# ============================================================
# 14. UNIFICAR INDICADORES MULTIESCALA
# ============================================================

df_indicadores_ruido = None

for radio in RADIOS:

    temp = (
        indicadores_ruido_radio[
            radio
        ]
    )

    if df_indicadores_ruido is None:

        df_indicadores_ruido = (
            temp.copy()
        )

    else:

        df_indicadores_ruido = pd.merge(
            df_indicadores_ruido,
            temp,
            on=[
                "Fecha",
                "estacion_geo"
            ],
            how="outer",
            validate="one_to_one"
        )


# ============================================================
# 15. SENSOR VÁLIDO MÁS PRÓXIMO DE CADA DÍA
# ============================================================

idx_nearest_diario = (
    df_ruido_rel
    .groupby(
        [
            "Fecha",
            "estacion_geo"
        ]
    )[
        "distancia_m"
    ]
    .idxmin()
)

df_ruido_nearest = (
    df_ruido_rel
    .loc[
        idx_nearest_diario,
        [
            "Fecha",
            "estacion_geo",
            "Id_Instal",
            "LAeq_dia",
            "distancia_m",
            "Cobertura_corregida_%"
        ]
    ]
    .rename(
        columns={
            "Id_Instal":
                "Id_sensor_mas_cercano",

            "LAeq_dia":
                "LAeq_sensor_mas_cercano",

            "distancia_m":
                "dist_sensor_mas_cercano_m",

            "Cobertura_corregida_%":
                "cobertura_sensor_mas_cercano"
        }
    )
    .reset_index(drop=True)
)


# ============================================================
# 16. INCORPORAR SENSOR MÁS CERCANO
# ============================================================

df_indicadores_ruido = pd.merge(
    df_indicadores_ruido,
    df_ruido_nearest,
    on=[
        "Fecha",
        "estacion_geo"
    ],
    how="outer",
    validate="one_to_one"
)


# ============================================================
# 17. HOMOGENEIZAR NOMBRE DE FECHA
# ============================================================

df_indicadores_ruido = (
    df_indicadores_ruido
    .rename(
        columns={
            "Fecha": "fecha"
        }
    )
)


# ============================================================
# 18. AÑADIR AÑO
# ============================================================

df_indicadores_ruido[
    "anio"
] = (
    df_indicadores_ruido[
        "fecha"
    ]
    .dt.year
    .astype("Int64")
)


# ============================================================
# 19. ORDENAR
# ============================================================

df_indicadores_ruido = (
    df_indicadores_ruido
    .sort_values(
        [
            "fecha",
            "estacion_geo"
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# 20. CONTROL DE UNICIDAD
# ============================================================

duplicados_indicadores = (
    df_indicadores_ruido
    .duplicated(
        subset=[
            "fecha",
            "estacion_geo"
        ]
    )
    .sum()
)


print("\n" + "=" * 80)
print("CONTROL DEL DATASET DE INDICADORES")
print("=" * 80)

print(
    f"\n✓ Filas: "
    f"{len(df_indicadores_ruido):,}"
)

print(
    f"✓ Columnas: "
    f"{df_indicadores_ruido.shape[1]}"
)

print(
    f"✓ Duplicados fecha + estacion_geo: "
    f"{duplicados_indicadores:,}"
)

if duplicados_indicadores > 0:
    raise ValueError(
        "Existen duplicados en fecha + estacion_geo."
    )


# ============================================================
# 21. COBERTURA GLOBAL POR RADIO
# ============================================================

resumen_cobertura_ruido = []

for radio in RADIOS:

    columna = (
        f"LAeq_idw_{radio}m"
    )

    validos = (
        df_indicadores_ruido[
            columna
        ]
        .notna()
        .sum()
    )

    total = len(
        df_indicadores_ruido
    )

    resumen_cobertura_ruido.append({
        "radio_m":
            radio,

        "registros_validos":
            validos,

        "registros_totales":
            total,

        "registros_sin_dato":
            total - validos,

        "cobertura_pct":
            round(
                validos
                /
                total
                *
                100,
                2
            )
    })


df_cobertura_ruido = pd.DataFrame(
    resumen_cobertura_ruido
)


print("\n" + "=" * 80)
print("COBERTURA DE LOS INDICADORES ACÚSTICOS")
print("=" * 80)

display(
    df_cobertura_ruido
)


# ============================================================
# 22. COBERTURA POR ESTACIÓN Y RADIO
# ============================================================

resumen_estacion_radio = []

for estacion in sorted(
    df_indicadores_ruido[
        "estacion_geo"
    ].unique()
):

    df_est = (
        df_indicadores_ruido[
            df_indicadores_ruido[
                "estacion_geo"
            ] == estacion
        ]
    )

    for radio in RADIOS:

        columna = (
            f"LAeq_idw_{radio}m"
        )

        n_validos_est = (
            df_est[
                columna
            ]
            .notna()
            .sum()
        )

        total_est = len(
            df_est
        )

        resumen_estacion_radio.append({
            "estacion_geo":
                estacion,

            "radio_m":
                radio,

            "dias_validos":
                n_validos_est,

            "dias_totales":
                total_est,

            "cobertura_pct":
                round(
                    n_validos_est
                    /
                    total_est
                    *
                    100,
                    2
                )
        })


df_cobertura_estacion_radio = pd.DataFrame(
    resumen_estacion_radio
)


print("\n" + "=" * 80)
print("COBERTURA POR ESTACIÓN Y RADIO")
print("=" * 80)

display(
    df_cobertura_estacion_radio
)


# ============================================================
# 23. CONTROL ESPECÍFICO DE OBSERVATORI FABRA
# ============================================================

fabra = (
    df_indicadores_ruido[
        df_indicadores_ruido[
            "estacion_geo"
        ] == "Observatori Fabra"
    ]
)

if len(fabra) > 0:

    print("\n" + "=" * 80)
    print("CONTROL ESPECÍFICO — OBSERVATORI FABRA")
    print("=" * 80)

    print(
        f"\n✓ Días totales: "
        f"{len(fabra):,}"
    )

    for radio in RADIOS:

        print(
            f"✓ Días con IDW {radio} m: "
            f"{fabra[f'LAeq_idw_{radio}m'].notna().sum():,}"
        )

    print(
        f"✓ Días con sensor más cercano: "
        f"{fabra['LAeq_sensor_mas_cercano'].notna().sum():,}"
    )

    print(
        f"✓ Distancia mediana al sensor "
        f"más cercano disponible: "
        f"{fabra['dist_sensor_mas_cercano_m'].median():.2f} m"
    )

    print(
        f"✓ Distancia mínima observada: "
        f"{fabra['dist_sensor_mas_cercano_m'].min():.2f} m"
    )

    print(
        f"✓ Distancia máxima observada: "
        f"{fabra['dist_sensor_mas_cercano_m'].max():.2f} m"
    )


# ============================================================
# 24. ESTADÍSTICOS PRINCIPALES
# ============================================================

columnas_principales = [
    "LAeq_idw_500m",
    "LAeq_idw_750m",
    "LAeq_idw_1000m",
    "LAeq_idw_1500m",
    "LAeq_energetico_1000m",
    "LAeq_energetico_1500m",
    "n_sensores_obs_1000m",
    "n_sensores_obs_1500m",
    "dist_sensor_min_1000m",
    "dist_sensor_min_1500m",
    "LAeq_sensor_mas_cercano",
    "dist_sensor_mas_cercano_m"
]

columnas_principales = [
    columna
    for columna in columnas_principales
    if columna in df_indicadores_ruido.columns
]


print("\n" + "=" * 80)
print("ESTADÍSTICOS DE LOS INDICADORES PRINCIPALES")
print("=" * 80)

display(
    df_indicadores_ruido[
        columnas_principales
    ]
    .describe()
    .T
)


# ============================================================
# 25. RANGO DE LOS INDICADORES ACÚSTICOS
# ============================================================

print("\n" + "=" * 80)
print("RANGO DE LOS INDICADORES ACÚSTICOS")
print("=" * 80)

for columna in [
    "LAeq_idw_500m",
    "LAeq_idw_750m",
    "LAeq_idw_1000m",
    "LAeq_idw_1500m",
    "LAeq_sensor_mas_cercano"
]:

    if columna in df_indicadores_ruido.columns:

        print(
            f"\n{columna}: "
            f"{df_indicadores_ruido[columna].min():.2f} "
            f"→ "
            f"{df_indicadores_ruido[columna].max():.2f} dB"
        )


# ============================================================
# 26. CONTROL DE VALORES INFINITOS
# ============================================================

columnas_numericas = (
    df_indicadores_ruido
    .select_dtypes(
        include=[
            np.number
        ]
    )
    .columns
)

n_inf = (
    np.isinf(
        df_indicadores_ruido[
            columnas_numericas
        ]
    )
    .sum()
    .sum()
)


# ============================================================
# 27. CONTROL FINAL
# ============================================================

print("\n" + "=" * 80)
print("CONSTRUCCIÓN DE INDICADORES 5.5 FINALIZADA")
print("=" * 80)

print(
    f"\n✓ Filas: "
    f"{len(df_indicadores_ruido):,}"
)

print(
    f"✓ Estaciones: "
    f"{df_indicadores_ruido['estacion_geo'].nunique()}"
)

print(
    f"✓ Fechas diferentes: "
    f"{df_indicadores_ruido['fecha'].nunique():,}"
)

print(
    f"✓ Duplicados fecha + estacion_geo: "
    f"{duplicados_indicadores:,}"
)

print(
    f"✓ Valores infinitos: "
    f"{n_inf:,}"
)

print(
    "✓ Solo se han utilizado días acústicos válidos."
)

print(
    "✓ La agregación de LAeq se ha realizado "
    "en escala energética."
)

print(
    "✓ Se conservan indicadores multiescala "
    "500 / 750 / 1000 / 1500 m."
)

print(
    "✓ Se conserva el sensor válido más cercano "
    "como variable diagnóstica."
)

print(
    "✓ No se ha realizado ninguna imputación."
)

print(
    "✓ No se ha modificado el dataset maestro."
)

print(
    "\n✓ Indicadores preparados para "
    "auditoría temporal y selección final."
)

5.5 — CONSTRUCCIÓN DE INDICADORES DIARIOS DE RUIDO

CONTROL PREVIO DE LAS MEDICIONES

✓ Registros originales: 316,951
✓ Fechas inválidas: 0
✓ LAeq_dia nulos: 0

SELECCIÓN DE OBSERVACIONES VÁLIDAS

✓ Registros válidos utilizados: 310,960
✓ Registros no válidos excluidos del cálculo: 5,991
✓ Porcentaje utilizado: 98.11 %

COBERTURA ESPACIAL DE LAS OBSERVACIONES VÁLIDAS

✓ Registros válidos geolocalizados: 298,940
✓ Registros válidos sin geolocalización: 12,020
✓ Sensores geolocalizados con observaciones válidas: 564

RELACIONES DIARIAS ESTACIÓN ↔ SENSOR

✓ Contribuciones diarias calculadas: 2,391,520
✓ Estaciones representadas: 8
✓ Sensores utilizados: 564

✓ Radio 500 m: 12,174 combinaciones fecha-estación

✓ Radio 750 m: 14,228 combinaciones fecha-estación

✓ Radio 1000 m: 14,411 combinaciones fecha-estación

✓ Radio 1500 m: 19,182 combinaciones fecha-estación

CONTROL DEL DATASET DE INDICADORES

✓ Filas: 20,456
✓ Columnas: 27
✓ Duplicados fecha + estacion_geo: 0

COBERTURA DE LOS INDI

,radio_m,registros_validos,registros_totales,registros_sin_dato,cobertura_pct
0,500,12174,20456,8282,59.51
1,750,14228,20456,6228,69.55
2,1000,14411,20456,6045,70.45
3,1500,19182,20456,1274,93.77



COBERTURA POR ESTACIÓN Y RADIO


,estacion_geo,radio_m,dias_validos,dias_totales,cobertura_pct
0,Ciutadella,500,2272,2557,88.85
1,Ciutadella,750,2426,2557,94.88
2,Ciutadella,1000,2426,2557,94.88
3,Ciutadella,1500,2556,2557,99.96
4,Eixample,500,2267,2557,88.66
5,Eixample,750,2490,2557,97.38
6,Eixample,1000,2542,2557,99.41
7,Eixample,1500,2555,2557,99.92
8,Gràcia,500,2547,2557,99.61
9,Gràcia,750,2547,2557,99.61



CONTROL ESPECÍFICO — OBSERVATORI FABRA

✓ Días totales: 2,557
✓ Días con IDW 500 m: 0
✓ Días con IDW 750 m: 0
✓ Días con IDW 1000 m: 0
✓ Días con IDW 1500 m: 1,554
✓ Días con sensor más cercano: 2,557
✓ Distancia mediana al sensor más cercano disponible: 1456.23 m
✓ Distancia mínima observada: 1084.15 m
✓ Distancia máxima observada: 6103.85 m

ESTADÍSTICOS DE LOS INDICADORES PRINCIPALES


,count,mean,std,min,25%,50%,75%,max
LAeq_idw_500m,12174.0,65.158975,3.377320,52.560000,62.939496,65.220000,67.120000,92.033105
LAeq_idw_750m,14228.0,65.124944,3.393567,51.569230,62.959614,64.859183,67.136371,87.616333
LAeq_idw_1000m,14411.0,65.027344,3.298554,52.560000,62.968721,64.788069,66.913822,86.968892
LAeq_idw_1500m,19182.0,65.341263,3.292308,51.600000,63.156586,65.245906,67.290447,96.392108
LAeq_energetico_1000m,14411.0,64.803664,3.308617,52.560000,62.800396,64.580000,66.572408,85.434103
LAeq_energetico_1500m,19182.0,65.417758,3.278748,51.600000,63.195299,65.451218,67.379904,99.299464
n_sensores_obs_1000m,14411.0,10.006176,8.304269,1.000000,4.000000,7.000000,12.000000,30.000000
n_sensores_obs_1500m,19182.0,15.208685,14.743738,1.000000,3.000000,11.000000,26.000000,55.000000
dist_sensor_min_1000m,14411.0,356.053092,144.853366,55.016006,210.631512,378.663087,420.277203,991.432765
dist_sensor_min_1500m,19182.0,579.789939,415.723319,55.016006,299.253310,417.865894,839.779852,1492.479214



RANGO DE LOS INDICADORES ACÚSTICOS

LAeq_idw_500m: 52.56 → 92.03 dB

LAeq_idw_750m: 51.57 → 87.62 dB

LAeq_idw_1000m: 52.56 → 86.97 dB

LAeq_idw_1500m: 51.60 → 96.39 dB

LAeq_sensor_mas_cercano: 42.13 → 94.93 dB

CONSTRUCCIÓN DE INDICADORES 5.5 FINALIZADA

✓ Filas: 20,456
✓ Estaciones: 8
✓ Fechas diferentes: 2,557
✓ Duplicados fecha + estacion_geo: 0
✓ Valores infinitos: 0
✓ Solo se han utilizado días acústicos válidos.
✓ La agregación de LAeq se ha realizado en escala energética.
✓ Se conservan indicadores multiescala 500 / 750 / 1000 / 1500 m.
✓ Se conserva el sensor válido más cercano como variable diagnóstica.
✓ No se ha realizado ninguna imputación.
✓ No se ha modificado el dataset maestro.

✓ Indicadores preparados para auditoría temporal y selección final.


### Resultados — 5.5. Construcción de indicadores diarios de contaminación acústica

La construcción de los indicadores acústicos se realizó a partir de **316.951 registros diarios**, de los cuales **310.960 (98,11 %)** cumplían los criterios de validez definidos previamente. Tras excluir exclusivamente las observaciones correspondientes a sensores sin geolocalización verificable, se utilizaron **298.940 registros válidos y georreferenciados**, procedentes de **564 sensores acústicos**.

Los niveles `LAeq_dia` se transformaron a escala energética antes de realizar las agregaciones espaciales, evitando promediar directamente valores expresados en decibelios. A partir de esta transformación se calcularon indicadores ponderados por distancia inversa (IDW) para radios de **500, 750, 1.000 y 1.500 m**.

La ampliación progresiva del radio produce una mejora considerable de la cobertura temporal:

| Radio | Registros válidos | Cobertura |
|---:|---:|---:|
| 500 m | 12.174 | 59,51 % |
| 750 m | 14.228 | 69,55 % |
| 1.000 m | 14.411 | 70,45 % |
| 1.500 m | 19.182 | **93,77 %** |

El radio de **1.500 m** supone, por tanto, una mejora sustancial respecto al de 1.000 m, reduciendo los registros sin representación acústica de **6.045 a 1.274**.

La mejora resulta especialmente relevante en las estaciones con menor densidad de sensores próximos. **Palau Reial** aumenta su cobertura del **10,56 % a 1.000 m al 92,33 % a 1.500 m**, mientras que **Vall d'Hebron** pasa del **60,50 % al 98,04 %**. En el resto de estaciones urbanas, la cobertura a 1.500 m se aproxima al 100 %.

**Observatori Fabra** constituye el principal caso diferencial. No dispone de observaciones dentro de los primeros 1.000 m, pero la ampliación a 1.500 m permite representar **1.554 de los 2.557 días (60,77 %)**. El sensor disponible más próximo presenta una distancia mínima de aproximadamente **1.084 m** y una mediana de **1.456 m**.

Se conserva adicionalmente el valor del sensor válido más próximo como variable diagnóstica. Aunque este procedimiento proporciona cobertura para todas las combinaciones fecha-estación, no se adopta directamente como indicador definitivo, ya que en determinados días la distancia al sensor disponible puede superar ampliamente el entorno local de la estación, alcanzando aproximadamente **6,1 km**.

Los indicadores IDW presentan valores centrales estables entre escalas. La mediana de `LAeq_idw_1000m` es aproximadamente **64,79 dB**, frente a **65,25 dB** para `LAeq_idw_1500m`. Esta proximidad, junto con el importante incremento de cobertura, justifica conservar el radio de 1.500 m como candidato para la selección posterior del indicador acústico principal.

El dataset resultante contiene **20.456 combinaciones `fecha + estacion_geo`**, correspondientes a **2.557 fechas y 8 estaciones**, sin duplicados ni valores infinitos. No se ha realizado ninguna imputación y los valores ausentes se conservan como información explícita sobre la falta de cobertura acústica.

> **Conclusión:** la escala de 1.500 m ofrece el mejor compromiso preliminar entre proximidad espacial y cobertura temporal, alcanzando un **93,77 % de cobertura global**. No obstante, su adopción como indicador acústico definitivo queda condicionada a la auditoría temporal por estación y año de la siguiente etapa.

## 5.6. Auditoría temporal y selección del indicador acústico principal

Una vez construidos los indicadores acústicos multiescala, se evalúa su comportamiento temporal y espacial antes de seleccionar la variable que se incorporará al dataset maestro.

La etapa anterior mostró que la ampliación progresiva del radio de análisis incrementa sustancialmente la disponibilidad de información acústica. Sin embargo, la selección del indicador definitivo no debe basarse únicamente en la cobertura global, ya que la disponibilidad de sensores puede variar entre estaciones y años.

Por este motivo, se realiza una auditoría específica de los indicadores:

- `LAeq_idw_500m`
- `LAeq_idw_750m`
- `LAeq_idw_1000m`
- `LAeq_idw_1500m`

La evaluación considera:

- cobertura global por año y radio;
- cobertura por estación, año y radio;
- comparación directa entre las escalas de 1.000 y 1.500 m;
- correlación entre ambos indicadores cuando existen observaciones simultáneas;
- magnitud de las diferencias entre ambas escalas;
- número de sensores utilizados;
- distancia al sensor válido más próximo;
- estabilidad temporal del indicador de 1.500 m;
- comportamiento específico de las estaciones con menor cobertura.

La comparación entre 1.000 y 1.500 m permite comprobar si el incremento de cobertura obtenido mediante el radio más amplio mantiene una señal acústica coherente con la escala espacial más local.

La auditoría presta especial atención a **Observatori Fabra, Palau Reial y Vall d'Hebron**, debido a la menor disponibilidad de sensores próximos identificada en las etapas anteriores.

La selección final se realiza considerando conjuntamente cobertura, coherencia entre escalas y representatividad espacial. No se amplían artificialmente los radios para obtener cobertura completa y no se realiza ninguna imputación de los valores ausentes.

Los `NaN` se mantienen como representación explícita de ausencia de información acústica suficientemente próxima a la estación en una fecha determinada.

> **Objetivo:** seleccionar y documentar el indicador acústico principal que ofrezca el mejor equilibrio entre cobertura temporal, proximidad espacial y estabilidad de la señal, preservando las limitaciones reales de la red de sensores.

In [ ]:
# ============================================================
# 5.6. AUDITORÍA TEMPORAL Y SELECCIÓN
# DEL INDICADOR ACÚSTICO PRINCIPAL
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 1. PARÁMETROS
# ============================================================

RADIOS = [
    500,
    750,
    1000,
    1500
]

ANIOS_OBJETIVO = list(
    range(2018, 2025)
)

print("=" * 80)
print("5.6 — AUDITORÍA TEMPORAL Y SELECCIÓN DEL INDICADOR ACÚSTICO")
print("=" * 80)


# ============================================================
# 2. COMPROBAR DATASET GENERADO EN 5.5
# ============================================================

if "df_indicadores_ruido" not in globals():

    raise NameError(
        "No existe df_indicadores_ruido. "
        "Debe ejecutarse previamente la etapa 5.5."
    )


df_ruido_audit = (
    df_indicadores_ruido
    .copy()
)


# ============================================================
# 3. NORMALIZAR FECHA Y AÑO
# ============================================================

df_ruido_audit[
    "fecha"
] = pd.to_datetime(
    df_ruido_audit[
        "fecha"
    ],
    errors="coerce"
)

if (
    df_ruido_audit[
        "fecha"
    ]
    .isna()
    .any()
):

    raise ValueError(
        "Existen fechas inválidas "
        "en df_indicadores_ruido."
    )


df_ruido_audit[
    "anio"
] = (
    df_ruido_audit[
        "fecha"
    ]
    .dt.year
    .astype(int)
)


# ============================================================
# 4. COMPROBAR VARIABLES NECESARIAS
# ============================================================

columnas_necesarias = [
    "fecha",
    "estacion_geo"
]

for radio in RADIOS:

    columnas_necesarias.extend([
        f"LAeq_idw_{radio}m",
        f"n_sensores_obs_{radio}m",
        f"dist_sensor_min_{radio}m"
    ])


faltantes = [
    columna
    for columna in columnas_necesarias
    if columna not in df_ruido_audit.columns
]

if faltantes:

    raise KeyError(
        "Faltan variables necesarias "
        "para la auditoría:\n"
        f"{faltantes}"
    )


# ============================================================
# 5. CONTROL GENERAL
# ============================================================

duplicados = (
    df_ruido_audit
    .duplicated(
        subset=[
            "fecha",
            "estacion_geo"
        ]
    )
    .sum()
)


print("\n" + "=" * 80)
print("CONTROL GENERAL")
print("=" * 80)

print(
    f"\n✓ Filas: "
    f"{len(df_ruido_audit):,}"
)

print(
    f"✓ Estaciones: "
    f"{df_ruido_audit['estacion_geo'].nunique()}"
)

print(
    f"✓ Fechas diferentes: "
    f"{df_ruido_audit['fecha'].nunique():,}"
)

print(
    f"✓ Periodo: "
    f"{df_ruido_audit['fecha'].min().date()} "
    f"→ "
    f"{df_ruido_audit['fecha'].max().date()}"
)

print(
    f"✓ Duplicados fecha + estacion_geo: "
    f"{duplicados:,}"
)

if duplicados > 0:

    raise ValueError(
        "Existen duplicados en la clave "
        "fecha + estacion_geo."
    )


# ============================================================
# 6. COMPROBAR AÑOS DISPONIBLES
# ============================================================

anios_disponibles = sorted(
    df_ruido_audit[
        "anio"
    ]
    .unique()
    .tolist()
)

print("\n" + "=" * 80)
print("COBERTURA DEL PERIODO")
print("=" * 80)

print(
    f"\n✓ Años disponibles: "
    f"{anios_disponibles}"
)

anios_faltantes = [
    anio
    for anio in ANIOS_OBJETIVO
    if anio not in anios_disponibles
]

if anios_faltantes:

    print(
        f"⚠ Años ausentes: "
        f"{anios_faltantes}"
    )

else:

    print(
        "✓ Existen registros para todos "
        "los años 2018–2024."
    )


# ============================================================
# 7. COBERTURA GLOBAL POR AÑO Y RADIO
# ============================================================

registros_cobertura_anual = []

for anio in ANIOS_OBJETIVO:

    temp = (
        df_ruido_audit[
            df_ruido_audit[
                "anio"
            ] == anio
        ]
    )

    total = len(
        temp
    )

    for radio in RADIOS:

        columna = (
            f"LAeq_idw_{radio}m"
        )

        validos = (
            temp[
                columna
            ]
            .notna()
            .sum()
        )

        cobertura = (
            validos
            /
            total
            *
            100
            if total > 0
            else np.nan
        )

        registros_cobertura_anual.append({
            "anio":
                anio,

            "radio_m":
                radio,

            "registros_totales":
                total,

            "registros_validos":
                validos,

            "registros_sin_dato":
                total - validos,

            "cobertura_pct":
                round(
                    cobertura,
                    2
                )
        })


df_cobertura_anual_ruido = pd.DataFrame(
    registros_cobertura_anual
)


print("\n" + "=" * 80)
print("COBERTURA GLOBAL POR AÑO Y RADIO")
print("=" * 80)

display(
    df_cobertura_anual_ruido
)


# ============================================================
# 8. RESUMEN ANUAL DE COBERTURA
# ============================================================

pivot_cobertura_anual = (
    df_cobertura_anual_ruido
    .pivot(
        index="anio",
        columns="radio_m",
        values="cobertura_pct"
    )
    .rename(
        columns={
            500:
                "cobertura_500m_pct",

            750:
                "cobertura_750m_pct",

            1000:
                "cobertura_1000m_pct",

            1500:
                "cobertura_1500m_pct"
        }
    )
    .reset_index()
)


print("\n" + "=" * 80)
print("RESUMEN ANUAL DE COBERTURA")
print("=" * 80)

display(
    pivot_cobertura_anual
)


# ============================================================
# 9. COBERTURA POR ESTACIÓN, AÑO Y RADIO
# ============================================================

registros_estacion_anio = []

estaciones = sorted(
    df_ruido_audit[
        "estacion_geo"
    ]
    .dropna()
    .unique()
)


for estacion in estaciones:

    for anio in ANIOS_OBJETIVO:

        temp = (
            df_ruido_audit[
                (
                    df_ruido_audit[
                        "estacion_geo"
                    ] == estacion
                )
                &
                (
                    df_ruido_audit[
                        "anio"
                    ] == anio
                )
            ]
        )

        total = len(
            temp
        )

        for radio in RADIOS:

            columna = (
                f"LAeq_idw_{radio}m"
            )

            validos = (
                temp[
                    columna
                ]
                .notna()
                .sum()
            )

            cobertura = (
                validos
                /
                total
                *
                100
                if total > 0
                else np.nan
            )

            registros_estacion_anio.append({
                "estacion_geo":
                    estacion,

                "anio":
                    anio,

                "radio_m":
                    radio,

                "dias_totales":
                    total,

                "dias_validos":
                    validos,

                "dias_sin_dato":
                    total - validos,

                "cobertura_pct":
                    round(
                        cobertura,
                        2
                    )
            })


df_cobertura_estacion_anio = pd.DataFrame(
    registros_estacion_anio
)


print("\n" + "=" * 80)
print("COBERTURA POR ESTACIÓN, AÑO Y RADIO")
print("=" * 80)

display(
    df_cobertura_estacion_anio
)


# ============================================================
# 10. COBERTURA 1500 m — ESTACIÓN × AÑO
# ============================================================

tabla_1500 = (
    df_cobertura_estacion_anio[
        df_cobertura_estacion_anio[
            "radio_m"
        ] == 1500
    ]
    .pivot(
        index="estacion_geo",
        columns="anio",
        values="cobertura_pct"
    )
    .reset_index()
)


print("\n" + "=" * 80)
print("COBERTURA IDW 1500 m — ESTACIÓN × AÑO")
print("=" * 80)

display(
    tabla_1500
)


# ============================================================
# 11. COBERTURA 1000 m — ESTACIÓN × AÑO
# ============================================================

tabla_1000 = (
    df_cobertura_estacion_anio[
        df_cobertura_estacion_anio[
            "radio_m"
        ] == 1000
    ]
    .pivot(
        index="estacion_geo",
        columns="anio",
        values="cobertura_pct"
    )
    .reset_index()
)


print("\n" + "=" * 80)
print("COBERTURA IDW 1000 m — ESTACIÓN × AÑO")
print("=" * 80)

display(
    tabla_1000
)


# ============================================================
# 12. COMPARACIÓN SIMULTÁNEA 1000 vs 1500 m
# ============================================================

comparacion = (
    df_ruido_audit[
        [
            "fecha",
            "estacion_geo",
            "LAeq_idw_1000m",
            "LAeq_idw_1500m"
        ]
    ]
    .dropna(
        subset=[
            "LAeq_idw_1000m",
            "LAeq_idw_1500m"
        ]
    )
    .copy()
)


comparacion[
    "dif_1500_1000_db"
] = (
    comparacion[
        "LAeq_idw_1500m"
    ]
    -
    comparacion[
        "LAeq_idw_1000m"
    ]
)

comparacion[
    "dif_abs_1500_1000_db"
] = (
    comparacion[
        "dif_1500_1000_db"
    ]
    .abs()
)


print("\n" + "=" * 80)
print("COMPARACIÓN IDW 1000 m vs 1500 m")
print("=" * 80)

print(
    f"\n✓ Observaciones comparables: "
    f"{len(comparacion):,}"
)

print(
    f"✓ Diferencia media 1500 − 1000: "
    f"{comparacion['dif_1500_1000_db'].mean():.3f} dB"
)

print(
    f"✓ Diferencia mediana: "
    f"{comparacion['dif_1500_1000_db'].median():.3f} dB"
)

print(
    f"✓ Diferencia absoluta media: "
    f"{comparacion['dif_abs_1500_1000_db'].mean():.3f} dB"
)

print(
    f"✓ Diferencia absoluta mediana: "
    f"{comparacion['dif_abs_1500_1000_db'].median():.3f} dB"
)


# ============================================================
# 13. CORRELACIÓN GLOBAL 1000 vs 1500 m
# ============================================================

if len(comparacion) > 1:

    correlacion_global = (
        comparacion[
            [
                "LAeq_idw_1000m",
                "LAeq_idw_1500m"
            ]
        ]
        .corr()
        .iloc[0, 1]
    )

else:

    correlacion_global = np.nan


print(
    f"✓ Correlación global: "
    f"{correlacion_global:.4f}"
)


# ============================================================
# 14. COMPARACIÓN 1000 vs 1500 m POR ESTACIÓN
# ============================================================

comparacion_estaciones = []

for estacion in estaciones:

    temp = (
        comparacion[
            comparacion[
                "estacion_geo"
            ] == estacion
        ]
    )

    if len(temp) >= 2:

        correlacion = (
            temp[
                [
                    "LAeq_idw_1000m",
                    "LAeq_idw_1500m"
                ]
            ]
            .corr()
            .iloc[0, 1]
        )

    else:

        correlacion = np.nan

    comparacion_estaciones.append({
        "estacion_geo":
            estacion,

        "n_comparables":
            len(temp),

        "correlacion_1000_1500":
            correlacion,

        "dif_media_db":
            temp[
                "dif_1500_1000_db"
            ].mean(),

        "dif_mediana_db":
            temp[
                "dif_1500_1000_db"
            ].median(),

        "dif_abs_media_db":
            temp[
                "dif_abs_1500_1000_db"
            ].mean(),

        "dif_abs_mediana_db":
            temp[
                "dif_abs_1500_1000_db"
            ].median()
    })


df_comparacion_estaciones = pd.DataFrame(
    comparacion_estaciones
)


print("\n" + "=" * 80)
print("COMPARACIÓN 1000 vs 1500 m POR ESTACIÓN")
print("=" * 80)

display(
    df_comparacion_estaciones
)


# ============================================================
# 15. SIMILITUD ENTRE ESCALAS
# ============================================================

umbrales_db = [
    0.5,
    1.0,
    2.0,
    3.0
]

control_diferencias = []

for umbral in umbrales_db:

    porcentaje = (
        (
            comparacion[
                "dif_abs_1500_1000_db"
            ] <= umbral
        )
        .mean()
        *
        100
    )

    control_diferencias.append({
        "umbral_db":
            umbral,

        "pct_observaciones":
            round(
                porcentaje,
                2
            )
    })


df_control_diferencias = pd.DataFrame(
    control_diferencias
)


print("\n" + "=" * 80)
print("SIMILITUD ENTRE ESCALAS 1000 Y 1500 m")
print("=" * 80)

display(
    df_control_diferencias
)


# ============================================================
# 16. NÚMERO DE SENSORES Y DISTANCIAS A 1500 m
# ============================================================

resumen_sensores_1500 = (
    df_ruido_audit
    .groupby(
        "estacion_geo"
    )
    .agg(
        dias_con_ruido_1500=(
            "LAeq_idw_1500m",
            "count"
        ),

        sensores_mediana_1500=(
            "n_sensores_obs_1500m",
            "median"
        ),

        sensores_media_1500=(
            "n_sensores_obs_1500m",
            "mean"
        ),

        sensores_min_1500=(
            "n_sensores_obs_1500m",
            "min"
        ),

        sensores_max_1500=(
            "n_sensores_obs_1500m",
            "max"
        ),

        distancia_min_mediana_1500=(
            "dist_sensor_min_1500m",
            "median"
        ),

        distancia_min_media_1500=(
            "dist_sensor_min_1500m",
            "mean"
        )
    )
    .reset_index()
)


print("\n" + "=" * 80)
print("SENSORES UTILIZADOS POR ESTACIÓN — 1500 m")
print("=" * 80)

display(
    resumen_sensores_1500
)


# ============================================================
# 17. ESTABILIDAD ANUAL DEL LAeq IDW 1500 m
# ============================================================

resumen_anual_1500 = (
    df_ruido_audit
    .groupby(
        "anio"
    )
    .agg(
        registros_validos=(
            "LAeq_idw_1500m",
            "count"
        ),

        laeq_media=(
            "LAeq_idw_1500m",
            "mean"
        ),

        laeq_mediana=(
            "LAeq_idw_1500m",
            "median"
        ),

        laeq_std=(
            "LAeq_idw_1500m",
            "std"
        )
    )
    .reset_index()
)


print("\n" + "=" * 80)
print("ESTABILIDAD ANUAL — LAeq IDW 1500 m")
print("=" * 80)

display(
    resumen_anual_1500
)


# ============================================================
# 18. ESTABILIDAD POR ESTACIÓN Y AÑO — 1500 m
# ============================================================

resumen_estacion_anual_1500 = (
    df_ruido_audit
    .groupby(
        [
            "estacion_geo",
            "anio"
        ]
    )
    .agg(
        dias_validos=(
            "LAeq_idw_1500m",
            "count"
        ),

        laeq_media=(
            "LAeq_idw_1500m",
            "mean"
        ),

        laeq_mediana=(
            "LAeq_idw_1500m",
            "median"
        ),

        laeq_std=(
            "LAeq_idw_1500m",
            "std"
        ),

        sensores_mediana=(
            "n_sensores_obs_1500m",
            "median"
        ),

        distancia_min_mediana=(
            "dist_sensor_min_1500m",
            "median"
        )
    )
    .reset_index()
)


print("\n" + "=" * 80)
print("ESTABILIDAD POR ESTACIÓN Y AÑO — 1500 m")
print("=" * 80)

display(
    resumen_estacion_anual_1500
)


# ============================================================
# 19. ESTACIONES CON MENOR COBERTURA
# ============================================================

estaciones_criticas = [
    "Observatori Fabra",
    "Palau Reial",
    "Vall d'Hebron"
]

control_criticas = (
    df_cobertura_estacion_anio[
        (
            df_cobertura_estacion_anio[
                "estacion_geo"
            ]
            .isin(
                estaciones_criticas
            )
        )
        &
        (
            df_cobertura_estacion_anio[
                "radio_m"
            ] == 1500
        )
    ]
    .copy()
)


print("\n" + "=" * 80)
print("ESTACIONES CRÍTICAS — COBERTURA 1500 m")
print("=" * 80)

display(
    control_criticas
)


# ============================================================
# 20. COBERTURA GLOBAL DEL CANDIDATO 1500 m
# ============================================================

total_registros = len(
    df_ruido_audit
)

validos_1500 = (
    df_ruido_audit[
        "LAeq_idw_1500m"
    ]
    .notna()
    .sum()
)

sin_dato_1500 = (
    total_registros
    -
    validos_1500
)

cobertura_1500 = (
    validos_1500
    /
    total_registros
    *
    100
)


# ============================================================
# 21. COBERTURA GLOBAL POR ESTACIÓN — 1500 m
# ============================================================

cobertura_estacion_1500 = (
    df_ruido_audit
    .groupby(
        "estacion_geo"
    )[
        "LAeq_idw_1500m"
    ]
    .apply(
        lambda x:
            x.notna().mean() * 100
    )
    .reset_index(
        name="cobertura_1500_pct"
    )
)


print("\n" + "=" * 80)
print("COBERTURA GLOBAL POR ESTACIÓN — 1500 m")
print("=" * 80)

display(
    cobertura_estacion_1500
)


# ============================================================
# 22. IDENTIFICAR LA COBERTURA MÍNIMA
# ============================================================

idx_min_cobertura = (
    cobertura_estacion_1500[
        "cobertura_1500_pct"
    ]
    .idxmin()
)

estacion_cobertura_minima = (
    cobertura_estacion_1500
    .loc[
        idx_min_cobertura,
        "estacion_geo"
    ]
)

cobertura_minima_estacion = (
    cobertura_estacion_1500
    .loc[
        idx_min_cobertura,
        "cobertura_1500_pct"
    ]
)


print("\n" + "=" * 80)
print("RESUMEN DEL CANDIDATO LAeq_idw_1500m")
print("=" * 80)

print(
    f"\n✓ Registros totales: "
    f"{total_registros:,}"
)

print(
    f"✓ Registros válidos: "
    f"{validos_1500:,}"
)

print(
    f"✓ Registros sin dato: "
    f"{sin_dato_1500:,}"
)

print(
    f"✓ Cobertura global: "
    f"{cobertura_1500:.2f} %"
)

print(
    f"✓ Estación con menor cobertura: "
    f"{estacion_cobertura_minima}"
)

print(
    f"✓ Cobertura mínima entre estaciones: "
    f"{cobertura_minima_estacion:.2f} %"
)


# ============================================================
# 23. CONTROL ESPECÍFICO — OBSERVATORI FABRA
# ============================================================

fabra_1500 = (
    df_cobertura_estacion_anio[
        (
            df_cobertura_estacion_anio[
                "estacion_geo"
            ] == "Observatori Fabra"
        )
        &
        (
            df_cobertura_estacion_anio[
                "radio_m"
            ] == 1500
        )
    ]
    [
        [
            "anio",
            "dias_totales",
            "dias_validos",
            "dias_sin_dato",
            "cobertura_pct"
        ]
    ]
    .copy()
)


print("\n" + "=" * 80)
print("OBSERVATORI FABRA — COBERTURA TEMPORAL 1500 m")
print("=" * 80)

display(
    fabra_1500
)


# ============================================================
# 24. CONTROL DE VALORES INFINITOS
# ============================================================

columnas_numericas = (
    df_ruido_audit
    .select_dtypes(
        include=[
            np.number
        ]
    )
    .columns
)

n_inf = (
    np.isinf(
        df_ruido_audit[
            columnas_numericas
        ]
    )
    .sum()
    .sum()
)


# ============================================================
# 25. RESUMEN DE LA AUDITORÍA
# ============================================================

print("\n" + "=" * 80)
print("AUDITORÍA 5.6 FINALIZADA")
print("=" * 80)

print(
    f"\n✓ Periodo auditado: "
    f"{df_ruido_audit['fecha'].min().date()} "
    f"→ "
    f"{df_ruido_audit['fecha'].max().date()}"
)

print(
    f"✓ Años auditados: "
    f"{len(anios_disponibles)}"
)

print(
    f"✓ Estaciones auditadas: "
    f"{len(estaciones)}"
)

print(
    f"✓ Cobertura global IDW 1500 m: "
    f"{cobertura_1500:.2f} %"
)

print(
    f"✓ Correlación IDW 1000 vs 1500 m: "
    f"{correlacion_global:.4f}"
)

print(
    f"✓ Diferencia absoluta mediana "
    f"1000 vs 1500 m: "
    f"{comparacion['dif_abs_1500_1000_db'].median():.3f} dB"
)

print(
    f"✓ Valores infinitos: "
    f"{n_inf:,}"
)

print(
    "✓ No se ha realizado ninguna imputación."
)

print(
    "✓ El dataset maestro no ha sido modificado."
)


# ============================================================
# 26. SELECCIÓN DEL INDICADOR ACÚSTICO PRINCIPAL
# ============================================================

INDICADOR_RUIDO_PRINCIPAL = (
    "LAeq_idw_1500m"
)


print("\n" + "=" * 80)
print("SELECCIÓN DEL INDICADOR ACÚSTICO PRINCIPAL")
print("=" * 80)

print(
    f"\n✓ Indicador seleccionado: "
    f"{INDICADOR_RUIDO_PRINCIPAL}"
)

print(
    f"✓ Cobertura global: "
    f"{cobertura_1500:.2f} %"
)

print(
    f"✓ Correlación con IDW 1000 m "
    f"en observaciones comparables: "
    f"{correlacion_global:.4f}"
)

print(
    f"✓ Diferencia absoluta mediana "
    f"respecto a IDW 1000 m: "
    f"{comparacion['dif_abs_1500_1000_db'].median():.3f} dB"
)

print(
    "✓ El radio de 1500 m mejora sustancialmente "
    "la cobertura manteniendo una señal coherente "
    "con la escala de 1000 m."
)

print(
    "✓ Los valores ausentes se conservan como NaN."
)

print(
    "✓ No se amplía el radio para forzar cobertura "
    "en Observatori Fabra."
)

print(
    "✓ La discontinuidad temporal de Observatori "
    "Fabra se conserva y queda documentada."
)

print(
    "✓ No se utiliza el sensor más cercano sin "
    "límite espacial como sustitución automática."
)

print(
    "✓ No se ha realizado ninguna imputación."
)

print(
    "\n✓ Indicador acústico principal seleccionado "
    "y preparado para la integración."
)

5.6 — AUDITORÍA TEMPORAL Y SELECCIÓN DEL INDICADOR ACÚSTICO

CONTROL GENERAL

✓ Filas: 20,456
✓ Estaciones: 8
✓ Fechas diferentes: 2,557
✓ Periodo: 2018-01-01 → 2024-12-31
✓ Duplicados fecha + estacion_geo: 0

COBERTURA DEL PERIODO

✓ Años disponibles: [2018, 2019, 2020, 2021, 2022, 2023, 2024]
✓ Existen registros para todos los años 2018–2024.

COBERTURA GLOBAL POR AÑO Y RADIO


,anio,radio_m,registros_totales,registros_validos,registros_sin_dato,cobertura_pct
0,2018,500,2920,1425,1495,48.80
1,2018,750,2920,1748,1172,59.86
2,2018,1000,2920,1797,1123,61.54
3,2018,1500,2920,2827,93,96.82
4,2019,500,2920,1740,1180,59.59
5,2019,750,2920,1793,1127,61.40
6,2019,1000,2920,1794,1126,61.44
7,2019,1500,2920,2781,139,95.24
8,2020,500,2928,1863,1065,63.63
9,2020,750,2928,2000,928,68.31



RESUMEN ANUAL DE COBERTURA


radio_m,anio,cobertura_500m_pct,cobertura_750m_pct,cobertura_1000m_pct,cobertura_1500m_pct
0,2018,48.80,59.86,61.54,96.82
1,2019,59.59,61.40,61.44,95.24
2,2020,63.63,68.31,68.68,98.50
3,2021,62.02,73.32,75.75,99.45
4,2022,59.73,74.52,76.27,91.92
5,2023,62.23,74.79,74.79,87.29
6,2024,60.59,74.66,74.66,87.19



COBERTURA POR ESTACIÓN, AÑO Y RADIO


...


COBERTURA GLOBAL POR ESTACIÓN — 1500 m


,estacion_geo,cobertura_1500_pct
0,Ciutadella,99.960892
1,Eixample,99.921783
2,Gràcia,99.726242
3,Observatori Fabra,60.774345
4,Palau Reial,92.334767
5,Poblenou,99.687133
6,Sants,99.726242
7,Vall d'Hebron,98.044583



RESUMEN DEL CANDIDATO LAeq_idw_1500m

✓ Registros totales: 20,456
✓ Registros válidos: 19,182
✓ Registros sin dato: 1,274
✓ Cobertura global: 93.77 %
✓ Estación con menor cobertura: Observatori Fabra
✓ Cobertura mínima entre estaciones: 60.77 %

OBSERVATORI FABRA — COBERTURA TEMPORAL 1500 m


,anio,dias_totales,dias_validos,dias_sin_dato,cobertura_pct
87,2018,365,354,11,96.99
91,2019,365,347,18,95.07
95,2020,366,353,13,96.45
99,2021,365,361,4,98.90
103,2022,365,139,226,38.08
107,2023,365,0,365,0.00
111,2024,366,0,366,0.00



AUDITORÍA 5.6 FINALIZADA

✓ Periodo auditado: 2018-01-01 → 2024-12-31
✓ Años auditados: 7
✓ Estaciones auditadas: 8
✓ Cobertura global IDW 1500 m: 93.77 %
✓ Correlación IDW 1000 vs 1500 m: 0.8882
✓ Diferencia absoluta mediana 1000 vs 1500 m: 0.522 dB
✓ Valores infinitos: 0
✓ No se ha realizado ninguna imputación.
✓ El dataset maestro no ha sido modificado.

SELECCIÓN DEL INDICADOR ACÚSTICO PRINCIPAL

✓ Indicador seleccionado: LAeq_idw_1500m
✓ Cobertura global: 93.77 %
✓ Correlación con IDW 1000 m en observaciones comparables: 0.8882
✓ Diferencia absoluta mediana respecto a IDW 1000 m: 0.522 dB
✓ El radio de 1500 m mejora sustancialmente la cobertura manteniendo una señal coherente con la escala de 1000 m.
✓ Los valores ausentes se conservan como NaN.
✓ No se amplía el radio para forzar cobertura en Observatori Fabra.
✓ La discontinuidad temporal de Observatori Fabra se conserva y queda documentada.
✓ No se utiliza el sensor más cercano sin límite espacial como sustitución automática.


### Resultados — 5.6. Auditoría temporal y selección del indicador acústico principal

La auditoría se realizó sobre **20.456 combinaciones fecha–estación**, correspondientes a **8 estaciones de calidad del aire** y **2.557 fechas diferentes** comprendidas entre 2018 y 2024. No se detectaron duplicados en la clave `fecha + estacion_geo`, confirmándose la integridad temporal del dataset de indicadores acústicos.

La comparación de las cuatro escalas espaciales consideradas mostró un incremento claro de cobertura al ampliar el radio de búsqueda. El indicador calculado a **1.500 m** presentó las mayores coberturas durante todo el periodo analizado:

| Año | 500 m | 750 m | 1.000 m | 1.500 m |
|---:|---:|---:|---:|---:|
| 2018 | 48,80 % | 59,86 % | 61,54 % | **96,82 %** |
| 2019 | 59,59 % | 61,40 % | 61,44 % | **95,24 %** |
| 2020 | 63,63 % | 68,31 % | 68,68 % | **98,50 %** |
| 2021 | 62,02 % | 73,32 % | 75,75 % | **99,45 %** |
| 2022 | 59,73 % | 74,52 % | 76,27 % | **91,92 %** |
| 2023 | 62,23 % | 74,79 % | 74,79 % | **87,29 %** |
| 2024 | 60,59 % | 74,66 % | 74,66 % | **87,19 %** |

La ampliación de 1.000 a 1.500 m produce, por tanto, una mejora sustancial de la disponibilidad de información acústica.

Para comprobar que este incremento de cobertura no alterase de forma significativa la señal, se compararon directamente ambos indicadores en las **14.411 observaciones con información simultánea**. La correlación entre `LAeq_idw_1000m` y `LAeq_idw_1500m` fue de **0,8882**, mientras que la diferencia absoluta mediana entre ambos indicadores fue de únicamente **0,522 dB**.

Además, el **75,30 %** de las observaciones presentó diferencias iguales o inferiores a 1 dB, el **93,65 %** diferencias inferiores a 2 dB y el **97,15 %** diferencias inferiores a 3 dB. Estos resultados indican que la ampliación del radio incrementa considerablemente la cobertura manteniendo una señal acústica globalmente coherente con la obtenida a 1.000 m.

La cobertura del indicador de 1.500 m es prácticamente completa en la mayoría de las estaciones. Ciutadella, Eixample, Gràcia, Poblenou y Sants presentan coberturas globales próximas al 100 %, mientras que Vall d'Hebron alcanza el **98,04 %** y Palau Reial el **92,33 %**.

La principal limitación se concentra en **Observatori Fabra**, cuya cobertura global a 1.500 m es del **60,77 %**. La estación mantiene coberturas superiores al 95 % entre 2018 y 2021, desciende al **38,08 % en 2022** y carece de observaciones dentro del radio establecido durante **2023 y 2024**.

Esta discontinuidad no se ha corregido mediante ampliaciones adicionales del radio ni mediante la asignación automática del sensor disponible más próximo, ya que ello reduciría la representatividad espacial del indicador. Tampoco se ha realizado ninguna imputación. Los valores ausentes se mantienen como `NaN`, preservando explícitamente las limitaciones reales de la red acústica.

En el conjunto del periodo, `LAeq_idw_1500m` dispone de **19.182 registros válidos de 20.456**, lo que representa una **cobertura global del 93,77 %**.

A partir de estos resultados se selecciona **`LAeq_idw_1500m` como indicador acústico principal** para su posterior integración en el dataset maestro. La elección proporciona el mejor equilibrio entre cobertura temporal, proximidad espacial y estabilidad de la señal, evitando introducir imputaciones o asignaciones espaciales artificiales.

> **Decisión metodológica:** se adopta `LAeq_idw_1500m` como variable principal de exposición acústica. Los valores ausentes se conservan como `NaN` y la discontinuidad temporal observada en Observatori Fabra queda explícitamente documentada como una limitación de cobertura de la red.

### Resultados — 5.6. Auditoría temporal y selección del indicador acústico principal

La auditoría se realizó sobre **20.456 combinaciones fecha–estación**, correspondientes a **8 estaciones de calidad del aire** y **2.557 fechas diferentes** comprendidas entre 2018 y 2024. No se detectaron duplicados en la clave `fecha + estacion_geo`, confirmándose la integridad temporal del dataset de indicadores acústicos.

La comparación de las cuatro escalas espaciales consideradas mostró un incremento claro de cobertura al ampliar el radio de búsqueda. El indicador calculado a **1.500 m** presentó las mayores coberturas durante todo el periodo analizado:

| Año | 500 m | 750 m | 1.000 m | 1.500 m |
|---:|---:|---:|---:|---:|
| 2018 | 48,80 % | 59,86 % | 61,54 % | **96,82 %** |
| 2019 | 59,59 % | 61,40 % | 61,44 % | **95,24 %** |
| 2020 | 63,63 % | 68,31 % | 68,68 % | **98,50 %** |
| 2021 | 62,02 % | 73,32 % | 75,75 % | **99,45 %** |
| 2022 | 59,73 % | 74,52 % | 76,27 % | **91,92 %** |
| 2023 | 62,23 % | 74,79 % | 74,79 % | **87,29 %** |
| 2024 | 60,59 % | 74,66 % | 74,66 % | **87,19 %** |

La ampliación de 1.000 a 1.500 m produce, por tanto, una mejora sustancial de la disponibilidad de información acústica.

Para comprobar que este incremento de cobertura no alterase de forma significativa la señal, se compararon directamente ambos indicadores en las **14.411 observaciones con información simultánea**. La correlación entre `LAeq_idw_1000m` y `LAeq_idw_1500m` fue de **0,8882**, mientras que la diferencia absoluta mediana entre ambos indicadores fue de únicamente **0,522 dB**.

Además, el **75,30 %** de las observaciones presentó diferencias iguales o inferiores a 1 dB, el **93,65 %** diferencias inferiores a 2 dB y el **97,15 %** diferencias inferiores a 3 dB. Estos resultados indican que la ampliación del radio incrementa considerablemente la cobertura manteniendo una señal acústica globalmente coherente con la obtenida a 1.000 m.

La cobertura del indicador de 1.500 m es prácticamente completa en la mayoría de las estaciones. Ciutadella, Eixample, Gràcia, Poblenou y Sants presentan coberturas globales próximas al 100 %, mientras que Vall d'Hebron alcanza el **98,04 %** y Palau Reial el **92,33 %**.

La principal limitación se concentra en **Observatori Fabra**, cuya cobertura global a 1.500 m es del **60,77 %**. La estación mantiene coberturas superiores al 95 % entre 2018 y 2021, desciende al **38,08 % en 2022** y carece de observaciones dentro del radio establecido durante **2023 y 2024**.

Esta discontinuidad no se ha corregido mediante ampliaciones adicionales del radio ni mediante la asignación automática del sensor disponible más próximo, ya que ello reduciría la representatividad espacial del indicador. Tampoco se ha realizado ninguna imputación. Los valores ausentes se mantienen como `NaN`, preservando explícitamente las limitaciones reales de la red acústica.

En el conjunto del periodo, `LAeq_idw_1500m` dispone de **19.182 registros válidos de 20.456**, lo que representa una **cobertura global del 93,77 %**.

A partir de estos resultados se selecciona **`LAeq_idw_1500m` como indicador acústico principal** para su posterior integración en el dataset maestro. La elección proporciona el mejor equilibrio entre cobertura temporal, proximidad espacial y estabilidad de la señal, evitando introducir imputaciones o asignaciones espaciales artificiales.

> **Decisión metodológica:** se adopta `LAeq_idw_1500m` como variable principal de exposición acústica. Los valores ausentes se conservan como `NaN` y la discontinuidad temporal observada en Observatori Fabra queda explícitamente documentada como una limitación de cobertura de la red.

In [ ]:
# ============================================================
# 5.7. INTEGRACIÓN DEL BLOQUE ACÚSTICO
# CHECKPOINT 04
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. RUTAS
# ============================================================

ruta_base = Path(
    "/content/drive/MyDrive/TFM/"
    "11_Machine_Learning_Temporal"
)

ruta_integracion = (
    ruta_base /
    "INTEGRACION"
)

ruta_maestro_03 = (
    ruta_integracion /
    "03_maestro_contaminacion_meteorologia_trafico.csv"
)

ruta_maestro_04 = (
    ruta_integracion /
    "04_maestro_contaminacion_meteorologia_trafico_ruido.csv"
)


# ============================================================
# 2. COMPROBAR EXISTENCIA DEL CHECKPOINT 03
# ============================================================

if not ruta_maestro_03.exists():

    raise FileNotFoundError(
        "No se encuentra el Checkpoint 03:\n"
        f"{ruta_maestro_03}"
    )


# ============================================================
# 3. COMPROBAR DATASET ACÚSTICO EN MEMORIA
# ============================================================

if "df_indicadores_ruido" not in globals():

    raise NameError(
        "No existe df_indicadores_ruido. "
        "Debe ejecutarse previamente la etapa 5.5."
    )


# ============================================================
# 4. CARGAR CHECKPOINT 03
# ============================================================

df_maestro_03 = pd.read_csv(
    ruta_maestro_03,
    low_memory=False
)

df_maestro_03[
    "fecha"
] = pd.to_datetime(
    df_maestro_03[
        "fecha"
    ],
    errors="coerce"
)


# ============================================================
# 5. PREPARAR DATASET ACÚSTICO
# ============================================================

variables_ruido = [
    "fecha",
    "estacion_geo",
    "LAeq_idw_1500m",
    "n_sensores_obs_1500m",
    "dist_sensor_min_1500m",
    "cobertura_media_sensores_1500m"
]

variables_faltantes_ruido = [
    columna
    for columna in variables_ruido
    if columna not in df_indicadores_ruido.columns
]

if variables_faltantes_ruido:

    raise KeyError(
        "Faltan variables acústicas necesarias:\n"
        f"{variables_faltantes_ruido}"
    )


df_ruido_preintegrado = (
    df_indicadores_ruido[
        variables_ruido
    ]
    .copy()
)

df_ruido_preintegrado[
    "fecha"
] = pd.to_datetime(
    df_ruido_preintegrado[
        "fecha"
    ],
    errors="coerce"
)


# ============================================================
# 6. CONTROL INICIAL
# ============================================================

print("=" * 80)
print("5.7 — INTEGRACIÓN DEL BLOQUE ACÚSTICO — CHECKPOINT 04")
print("=" * 80)

print("\nCHECKPOINT 03")

print(
    f"✓ Filas: "
    f"{len(df_maestro_03):,}"
)

print(
    f"✓ Columnas: "
    f"{df_maestro_03.shape[1]}"
)


print("\nDATASET ACÚSTICO PREINTEGRADO")

print(
    f"✓ Filas: "
    f"{len(df_ruido_preintegrado):,}"
)

print(
    f"✓ Columnas: "
    f"{df_ruido_preintegrado.shape[1]}"
)

print(
    f"✓ Estaciones: "
    f"{df_ruido_preintegrado['estacion_geo'].nunique()}"
)

print(
    f"✓ Fechas diferentes: "
    f"{df_ruido_preintegrado['fecha'].nunique():,}"
)


# ============================================================
# 7. CONTROL DE FECHAS INVÁLIDAS
# ============================================================

fechas_invalidas_maestro = (
    df_maestro_03[
        "fecha"
    ]
    .isna()
    .sum()
)

fechas_invalidas_ruido = (
    df_ruido_preintegrado[
        "fecha"
    ]
    .isna()
    .sum()
)

print("\n" + "=" * 80)
print("CONTROL DE FECHAS")
print("=" * 80)

print(
    f"\n✓ Fechas inválidas en maestro: "
    f"{fechas_invalidas_maestro:,}"
)

print(
    f"✓ Fechas inválidas en ruido: "
    f"{fechas_invalidas_ruido:,}"
)

if (
    fechas_invalidas_maestro > 0
    or
    fechas_invalidas_ruido > 0
):

    raise ValueError(
        "Existen fechas inválidas antes de la integración."
    )


# ============================================================
# 8. UNICIDAD DEL DATASET ACÚSTICO
# ============================================================

claves_merge = [
    "fecha",
    "estacion_geo"
]

duplicados_ruido = (
    df_ruido_preintegrado
    .duplicated(
        subset=claves_merge
    )
    .sum()
)

print("\n" + "=" * 80)
print("UNICIDAD DEL DATASET ACÚSTICO")
print("=" * 80)

print(
    f"\n✓ Duplicados fecha + estacion_geo: "
    f"{duplicados_ruido:,}"
)

if duplicados_ruido > 0:

    raise ValueError(
        "El dataset acústico presenta duplicados "
        "en fecha + estacion_geo."
    )


# ============================================================
# 9. CONTROL DE LA CLAVE PRINCIPAL DEL MAESTRO
# ============================================================

clave_maestro = [
    "fecha",
    "estacion_fisica",
    "contaminant"
]

faltantes_clave_maestro = [
    columna
    for columna in clave_maestro
    if columna not in df_maestro_03.columns
]

if faltantes_clave_maestro:

    raise KeyError(
        "Faltan columnas de la clave principal del maestro:\n"
        f"{faltantes_clave_maestro}"
    )


duplicados_maestro_antes = (
    df_maestro_03
    .duplicated(
        subset=clave_maestro
    )
    .sum()
)

filas_antes = len(
    df_maestro_03
)


print("\n" + "=" * 80)
print("CONTROL DEL MAESTRO ANTES DE INTEGRAR")
print("=" * 80)

print(
    f"\n✓ Filas: "
    f"{filas_antes:,}"
)

print(
    f"✓ Duplicados clave principal: "
    f"{duplicados_maestro_antes:,}"
)

if duplicados_maestro_antes > 0:

    raise ValueError(
        "El Checkpoint 03 presenta duplicados."
    )


# ============================================================
# 10. CONTROL DE COLUMNAS COMUNES
# ============================================================

columnas_comunes = (
    set(
        df_maestro_03.columns
    )
    &
    set(
        df_ruido_preintegrado.columns
    )
)

columnas_comunes_no_clave = sorted(
    columnas_comunes
    -
    set(claves_merge)
)


print("\n" + "=" * 80)
print("CONTROL DE COLUMNAS COMUNES")
print("=" * 80)

print(
    f"\n✓ Columnas comunes totales: "
    f"{len(columnas_comunes)}"
)

print(
    f"✓ Columnas comunes fuera de la clave: "
    f"{len(columnas_comunes_no_clave)}"
)

if columnas_comunes_no_clave:

    print(
        "\nColumnas comunes no esperadas:"
    )

    for columna in columnas_comunes_no_clave:
        print(
            f"  • {columna}"
        )

    raise ValueError(
        "Existen columnas comunes fuera "
        "de fecha + estacion_geo."
    )


# ============================================================
# 11. COBERTURA DE LA CLAVE DE INTEGRACIÓN
# ============================================================

claves_maestro = (
    df_maestro_03[
        claves_merge
    ]
    .drop_duplicates()
)

claves_ruido = (
    df_ruido_preintegrado[
        claves_merge
    ]
    .drop_duplicates()
)


control_cobertura = pd.merge(
    claves_maestro,
    claves_ruido.assign(
        disponible_ruido=True
    ),
    on=claves_merge,
    how="left",
    validate="one_to_one"
)

control_cobertura[
    "disponible_ruido"
] = (
    control_cobertura[
        "disponible_ruido"
    ]
    .fillna(False)
    .astype(bool)
)


n_claves_maestro = len(
    control_cobertura
)

n_claves_con_ruido = (
    control_cobertura[
        "disponible_ruido"
    ]
    .sum()
)

n_claves_sin_ruido = (
    n_claves_maestro
    -
    n_claves_con_ruido
)


print("\n" + "=" * 80)
print("COBERTURA DE LA CLAVE DE INTEGRACIÓN")
print("=" * 80)

print(
    f"\n✓ Combinaciones fecha-estación del maestro: "
    f"{n_claves_maestro:,}"
)

print(
    f"✓ Con correspondencia en dataset acústico: "
    f"{n_claves_con_ruido:,}"
)

print(
    f"✓ Sin correspondencia: "
    f"{n_claves_sin_ruido:,}"
)

print(
    f"✓ Cobertura de la clave: "
    f"{n_claves_con_ruido / n_claves_maestro * 100:.2f} %"
)


# ============================================================
# 12. VARIABLES ACÚSTICAS A INCORPORAR
# ============================================================

variables_ruido_nuevas = [
    columna
    for columna in df_ruido_preintegrado.columns
    if columna not in claves_merge
]


print("\n" + "=" * 80)
print("VARIABLES ACÚSTICAS A INCORPORAR")
print("=" * 80)

print(
    f"\n✓ Variables nuevas: "
    f"{len(variables_ruido_nuevas)}"
)

for columna in variables_ruido_nuevas:
    print(
        f"  • {columna}"
    )


# ============================================================
# 13. INTEGRACIÓN MANY-TO-ONE
# ============================================================

df_maestro_04 = pd.merge(
    df_maestro_03,
    df_ruido_preintegrado,
    on=claves_merge,
    how="left",
    validate="many_to_one"
)


# ============================================================
# 14. CONTROL DE DIMENSIONES
# ============================================================

filas_despues = len(
    df_maestro_04
)

columnas_antes = (
    df_maestro_03.shape[1]
)

columnas_despues = (
    df_maestro_04.shape[1]
)

columnas_esperadas = (
    columnas_antes
    +
    len(
        variables_ruido_nuevas
    )
)


print("\n" + "=" * 80)
print("CONTROL DE DIMENSIONES TRAS EL MERGE")
print("=" * 80)

print(
    f"\n✓ Filas antes: "
    f"{filas_antes:,}"
)

print(
    f"✓ Filas después: "
    f"{filas_despues:,}"
)

print(
    f"✓ Columnas antes: "
    f"{columnas_antes}"
)

print(
    f"✓ Variables acústicas añadidas: "
    f"{len(variables_ruido_nuevas)}"
)

print(
    f"✓ Columnas esperadas: "
    f"{columnas_esperadas}"
)

print(
    f"✓ Columnas obtenidas: "
    f"{columnas_despues}"
)


if filas_antes != filas_despues:

    raise ValueError(
        "La integración ha modificado "
        "el número de registros del maestro."
    )


if columnas_despues != columnas_esperadas:

    raise ValueError(
        "El número de columnas obtenido "
        "no coincide con el esperado."
    )


# ============================================================
# 15. CONTROL DE DUPLICADOS TRAS EL MERGE
# ============================================================

duplicados_maestro_despues = (
    df_maestro_04
    .duplicated(
        subset=clave_maestro
    )
    .sum()
)


print("\n" + "=" * 80)
print("CONTROL DE DUPLICADOS TRAS EL MERGE")
print("=" * 80)

print(
    f"\n✓ Duplicados antes: "
    f"{duplicados_maestro_antes:,}"
)

print(
    f"✓ Duplicados después: "
    f"{duplicados_maestro_despues:,}"
)


if (
    duplicados_maestro_despues
    !=
    duplicados_maestro_antes
):

    raise ValueError(
        "La integración ha introducido "
        "duplicados en el maestro."
    )


# ============================================================
# 16. COBERTURA DEL INDICADOR PRINCIPAL
# ============================================================

col_ruido_principal = (
    "LAeq_idw_1500m"
)

n_validos_ruido = (
    df_maestro_04[
        col_ruido_principal
    ]
    .notna()
    .sum()
)

n_nulos_ruido = (
    df_maestro_04[
        col_ruido_principal
    ]
    .isna()
    .sum()
)

cobertura_ruido_maestro = (
    n_validos_ruido
    /
    len(
        df_maestro_04
    )
    *
    100
)


print("\n" + "=" * 80)
print("COBERTURA DEL INDICADOR ACÚSTICO PRINCIPAL")
print("=" * 80)

print(
    f"\n✓ Registros con LAeq_idw_1500m: "
    f"{n_validos_ruido:,}"
)

print(
    f"✓ Registros sin LAeq_idw_1500m: "
    f"{n_nulos_ruido:,}"
)

print(
    f"✓ Cobertura en el maestro: "
    f"{cobertura_ruido_maestro:.2f} %"
)


# ============================================================
# 17. COBERTURA POR ESTACIÓN
# ============================================================

cobertura_estacion = (
    df_maestro_04
    .groupby(
        "estacion_geo"
    )
    .agg(
        registros=(
            "fecha",
            "size"
        ),

        registros_con_ruido=(
            col_ruido_principal,
            "count"
        )
    )
    .reset_index()
)


cobertura_estacion[
    "registros_sin_ruido"
] = (
    cobertura_estacion[
        "registros"
    ]
    -
    cobertura_estacion[
        "registros_con_ruido"
    ]
)

cobertura_estacion[
    "cobertura_pct"
] = (
    cobertura_estacion[
        "registros_con_ruido"
    ]
    /
    cobertura_estacion[
        "registros"
    ]
    *
    100
).round(2)


print("\n" + "=" * 80)
print("COBERTURA DE RUIDO POR ESTACIÓN")
print("=" * 80)

display(
    cobertura_estacion
)


# ============================================================
# 18. COBERTURA POR ESTACIÓN Y AÑO
# ============================================================

df_maestro_04[
    "anio_control_ruido"
] = (
    df_maestro_04[
        "fecha"
    ]
    .dt.year
)


cobertura_estacion_anio = (
    df_maestro_04
    .groupby(
        [
            "estacion_geo",
            "anio_control_ruido"
        ]
    )
    .agg(
        registros=(
            "fecha",
            "size"
        ),

        registros_con_ruido=(
            col_ruido_principal,
            "count"
        )
    )
    .reset_index()
)


cobertura_estacion_anio[
    "cobertura_pct"
] = (
    cobertura_estacion_anio[
        "registros_con_ruido"
    ]
    /
    cobertura_estacion_anio[
        "registros"
    ]
    *
    100
).round(2)


print("\n" + "=" * 80)
print("COBERTURA DE RUIDO POR ESTACIÓN Y AÑO")
print("=" * 80)

display(
    cobertura_estacion_anio
)


# ============================================================
# 19. CONTROL ESPECÍFICO — OBSERVATORI FABRA
# ============================================================

fabra_control = (
    cobertura_estacion_anio[
        cobertura_estacion_anio[
            "estacion_geo"
        ] == "Observatori Fabra"
    ]
    .copy()
)


print("\n" + "=" * 80)
print("CONTROL ESPECÍFICO — OBSERVATORI FABRA")
print("=" * 80)

display(
    fabra_control
)


# ============================================================
# 20. ELIMINAR VARIABLE AUXILIAR DE CONTROL
# ============================================================

df_maestro_04 = (
    df_maestro_04
    .drop(
        columns=[
            "anio_control_ruido"
        ]
    )
)


# ============================================================
# 21. CONTROL DE VALORES INFINITOS
# ============================================================

columnas_numericas = (
    df_maestro_04
    .select_dtypes(
        include=[
            np.number
        ]
    )
    .columns
)

n_inf = (
    np.isinf(
        df_maestro_04[
            columnas_numericas
        ]
    )
    .sum()
    .sum()
)


print("\n" + "=" * 80)
print("CONTROL DE VALORES INFINITOS")
print("=" * 80)

print(
    f"\n✓ Valores infinitos: "
    f"{n_inf:,}"
)

if n_inf > 0:

    raise ValueError(
        "Se han detectado valores infinitos."
    )


# ============================================================
# 22. ORDENAR EL MAESTRO
# ============================================================

columnas_orden = [
    columna
    for columna in [
        "fecha",
        "estacion_geo",
        "estacion_fisica",
        "contaminant"
    ]
    if columna in df_maestro_04.columns
]


df_maestro_04 = (
    df_maestro_04
    .sort_values(
        columnas_orden
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 23. EXPORTAR CHECKPOINT 04
# ============================================================

ruta_integracion.mkdir(
    parents=True,
    exist_ok=True
)


df_maestro_04.to_csv(
    ruta_maestro_04,
    index=False,
    encoding="utf-8"
)


# ============================================================
# 24. RECARGAR CHECKPOINT 04
# ============================================================

df_control_04 = pd.read_csv(
    ruta_maestro_04,
    parse_dates=[
        "fecha"
    ],
    low_memory=False
)


# ============================================================
# 25. VERIFICAR INTEGRIDAD DE LA EXPORTACIÓN
# ============================================================

filas_recarga = len(
    df_control_04
)

columnas_recarga = (
    df_control_04.shape[1]
)

duplicados_recarga = (
    df_control_04
    .duplicated(
        subset=clave_maestro
    )
    .sum()
)

nulos_ruido_recarga = (
    df_control_04[
        col_ruido_principal
    ]
    .isna()
    .sum()
)


print("\n" + "=" * 80)
print("VERIFICACIÓN DEL CHECKPOINT 04")
print("=" * 80)

print(
    f"\n✓ Filas exportadas: "
    f"{len(df_maestro_04):,}"
)

print(
    f"✓ Filas recargadas: "
    f"{filas_recarga:,}"
)

print(
    f"✓ Columnas exportadas: "
    f"{df_maestro_04.shape[1]}"
)

print(
    f"✓ Columnas recargadas: "
    f"{columnas_recarga}"
)

print(
    f"✓ Duplicados tras recarga: "
    f"{duplicados_recarga:,}"
)

print(
    f"✓ NaN de ruido antes de exportar: "
    f"{n_nulos_ruido:,}"
)

print(
    f"✓ NaN de ruido tras recarga: "
    f"{nulos_ruido_recarga:,}"
)


if filas_recarga != len(
    df_maestro_04
):

    raise ValueError(
        "El número de filas ha cambiado "
        "durante la exportación."
    )


if columnas_recarga != df_maestro_04.shape[1]:

    raise ValueError(
        "El número de columnas ha cambiado "
        "durante la exportación."
    )


if duplicados_recarga != duplicados_maestro_despues:

    raise ValueError(
        "La recarga ha alterado "
        "la unicidad del maestro."
    )


if nulos_ruido_recarga != n_nulos_ruido:

    raise ValueError(
        "La exportación ha alterado "
        "los NaN del indicador acústico."
    )


# ============================================================
# 26. TAMAÑO DEL ARCHIVO
# ============================================================

tamano_mb = (
    ruta_maestro_04.stat().st_size
    /
    (1024 ** 2)
)


# ============================================================
# 27. RESUMEN FINAL
# ============================================================

print("\n" + "=" * 80)
print("CHECKPOINT 04 — CONTAMINACIÓN ACÚSTICA")
print("=" * 80)

print(
    f"\n✓ Registros conservados: "
    f"{filas_recarga:,}"
)

print(
    f"✓ Variables disponibles: "
    f"{columnas_recarga}"
)

print(
    f"✓ Variables acústicas añadidas: "
    f"{len(variables_ruido_nuevas)}"
)

print(
    f"✓ Localizaciones físicas: "
    f"{df_control_04['estacion_geo'].nunique()}"
)

print(
    f"✓ Contaminantes: "
    f"{df_control_04['contaminant'].nunique()}"
)

print(
    f"✓ Duplicados de la clave principal: "
    f"{duplicados_recarga:,}"
)

print(
    f"✓ Cobertura LAeq_idw_1500m: "
    f"{df_control_04[col_ruido_principal].notna().mean() * 100:.2f} %"
)

print(
    f"✓ NaN conservados en LAeq_idw_1500m: "
    f"{nulos_ruido_recarga:,}"
)

print(
    f"✓ Valores infinitos: "
    f"{n_inf:,}"
)

print(
    f"✓ Tamaño del archivo: "
    f"{tamano_mb:.2f} MB"
)

print(
    "✓ Sin pérdida ni multiplicación de registros."
)

print(
    "✓ Checkpoint 03 conservado sin modificaciones."
)

print(
    "✓ No se ha realizado ninguna imputación acústica."
)

print(
    "✓ La discontinuidad de Observatori Fabra "
    "permanece explícitamente representada."
)

print(
    "✓ Integridad del Checkpoint 04 "
    "verificada tras la exportación."
)

print(
    "\n✓ Archivo maestro generado:"
)

print(
    ruta_maestro_04
)

5.7 — INTEGRACIÓN DEL BLOQUE ACÚSTICO — CHECKPOINT 04

CHECKPOINT 03
✓ Filas: 43,642
✓ Columnas: 59

DATASET ACÚSTICO PREINTEGRADO
✓ Filas: 20,456
✓ Columnas: 6
✓ Estaciones: 8
✓ Fechas diferentes: 2,557

CONTROL DE FECHAS

✓ Fechas inválidas en maestro: 0
✓ Fechas inválidas en ruido: 0

UNICIDAD DEL DATASET ACÚSTICO

✓ Duplicados fecha + estacion_geo: 0

CONTROL DEL MAESTRO ANTES DE INTEGRAR

✓ Filas: 43,642
✓ Duplicados clave principal: 0

CONTROL DE COLUMNAS COMUNES

✓ Columnas comunes totales: 2
✓ Columnas comunes fuera de la clave: 0

COBERTURA DE LA CLAVE DE INTEGRACIÓN

✓ Combinaciones fecha-estación del maestro: 15,417
✓ Con correspondencia en dataset acústico: 15,417
✓ Sin correspondencia: 0
✓ Cobertura de la clave: 100.00 %

VARIABLES ACÚSTICAS A INCORPORAR

✓ Variables nuevas: 4
  • LAeq_idw_1500m
  • n_sensores_obs_1500m
  • dist_sensor_min_1500m
  • cobertura_media_sensores_1500m

CONTROL DE DIMENSIONES TRAS EL MERGE

✓ Filas antes: 43,642
✓ Filas después: 43,642
✓ Columna

,estacion_geo,registros,registros_con_ruido,registros_sin_ruido,cobertura_pct
0,Ciutadella,4428,4426,2,99.95
1,Eixample,6975,6969,6,99.91
2,Gràcia,6120,6112,8,99.87
3,Observatori Fabra,440,413,27,93.86
4,Palau Reial,7359,7201,158,97.85
5,Poblenou,5675,5651,24,99.58
6,Sants,6163,6157,6,99.90
7,Vall d'Hebron,6482,6369,113,98.26



COBERTURA DE RUIDO POR ESTACIÓN Y AÑO


,estacion_geo,anio_control_ruido,registros,registros_con_ruido,cobertura_pct
0,Ciutadella,2018,337,337,100.00
1,Ciutadella,2019,490,490,100.00
2,Ciutadella,2020,704,704,100.00
3,Ciutadella,2021,705,705,100.00
4,Ciutadella,2022,730,728,99.73
5,Ciutadella,2023,730,730,100.00
6,Ciutadella,2024,732,732,100.00
7,Eixample,2018,397,397,100.00
8,Eixample,2019,721,719,99.72
9,Eixample,2020,1049,1049,100.00



CONTROL ESPECÍFICO — OBSERVATORI FABRA


,estacion_geo,anio_control_ruido,registros,registros_con_ruido,cobertura_pct
21,Observatori Fabra,2018,355,328,92.39
22,Observatori Fabra,2019,85,85,100.00



CONTROL DE VALORES INFINITOS

✓ Valores infinitos: 0

VERIFICACIÓN DEL CHECKPOINT 04

✓ Filas exportadas: 43,642
✓ Filas recargadas: 43,642
✓ Columnas exportadas: 63
✓ Columnas recargadas: 63
✓ Duplicados tras recarga: 0
✓ NaN de ruido antes de exportar: 344
✓ NaN de ruido tras recarga: 344

CHECKPOINT 04 — CONTAMINACIÓN ACÚSTICA

✓ Registros conservados: 43,642
✓ Variables disponibles: 63
✓ Variables acústicas añadidas: 4
✓ Localizaciones físicas: 8
✓ Contaminantes: 4
✓ Duplicados de la clave principal: 0
✓ Cobertura LAeq_idw_1500m: 99.21 %
✓ NaN conservados en LAeq_idw_1500m: 344
✓ Valores infinitos: 0
✓ Tamaño del archivo: 24.25 MB
✓ Sin pérdida ni multiplicación de registros.
✓ Checkpoint 03 conservado sin modificaciones.
✓ No se ha realizado ninguna imputación acústica.
✓ La discontinuidad de Observatori Fabra permanece explícitamente representada.
✓ Integridad del Checkpoint 04 verificada tras la exportación.

✓ Archivo maestro generado:
/content/drive/MyDrive/TFM/11_Machine_Lea

### Resultados — 5.7. Integración del bloque acústico y generación del Checkpoint 04

La integración del bloque de contaminación acústica se realiza sobre el **Checkpoint 03**, compuesto por **43.642 registros y 59 variables**.

El dataset acústico utilizado para la integración contiene **20.456 combinaciones únicas `fecha + estacion_geo`**, correspondientes a las 8 estaciones y a las 2.557 fechas del periodo de estudio.

La correspondencia entre las claves del maestro y el dataset acústico es completa: las **15.417 combinaciones fecha-estación presentes en el maestro disponen de correspondencia en el dataset acústico**, alcanzándose una cobertura de clave del **100 %**.

Se incorporan cuatro variables:

- `LAeq_idw_1500m`;
- `n_sensores_obs_1500m`;
- `dist_sensor_min_1500m`;
- `cobertura_media_sensores_1500m`.

La integración conserva exactamente los **43.642 registros originales** y amplía el dataset de **59 a 63 variables**, sin introducir duplicados en la clave principal `fecha + estacion_fisica + contaminant`.

El indicador acústico principal `LAeq_idw_1500m` está disponible en **43.298 registros**, lo que representa una cobertura del **99,21 %** dentro del maestro. Los **344 registros sin información acústica** permanecen como `NaN`, sin realizar ninguna imputación.

La cobertura por estación es elevada en todo el conjunto del maestro. Ciutadella, Eixample, Gràcia, Poblenou y Sants presentan coberturas próximas al 100 %, mientras que Vall d'Hebron alcanza el **98,26 %**, Palau Reial el **97,85 %** y Observatori Fabra el **93,86 %** dentro de los registros efectivamente presentes en el maestro.

La menor cobertura observada en determinadas estaciones y periodos se conserva explícitamente como característica de disponibilidad de la red acústica y no se corrige mediante ampliación adicional del radio ni mediante imputación.

No se detectan valores infinitos y la exportación del Checkpoint 04 mantiene íntegramente las dimensiones, unicidad y valores ausentes del dataset.

Tras la recarga del archivo se verifican nuevamente **43.642 registros, 63 variables, 0 duplicados y 344 valores ausentes en `LAeq_idw_1500m`**.

> **Conclusión:** el bloque de contaminación acústica queda integrado correctamente en el dataset maestro, manteniendo la estructura original, la trazabilidad de los valores ausentes y las limitaciones reales de cobertura. El resultado constituye el **Checkpoint 04** del proceso de integración.

# 6 · INTEGRACIÓN DEL TRANSPORTE AÉREO

El cuarto bloque explicativo incorpora la **actividad aérea diaria** como componente adicional de la movilidad y de la actividad metropolitana.

La información previamente depurada se audita para comprobar su estructura, cobertura temporal y coherencia interna. Posteriormente se selecciona un conjunto reducido de indicadores capaces de representar tanto la intensidad como la composición de la actividad aérea.

La integración se realiza a escala diaria mediante la variable `fecha`, manteniendo la granularidad del dataset maestro definida por las observaciones de contaminación atmosférica.

La fuente aérea no dispone de información para la totalidad del periodo de estudio. Esta limitación temporal se conserva explícitamente mediante valores ausentes, evitando interpretar la ausencia de datos como ausencia de actividad.

El bloque finaliza con la generación del **Checkpoint 05**.

> **Objetivo del bloque:** incorporar la variabilidad diaria de la actividad aérea al dataset maestro mediante un conjunto reducido e interpretable de indicadores, preservando la cobertura temporal real de la fuente.

## 6.1. Carga y auditoría estructural del dataset diario de transporte aéreo

Se inicia el bloque de transporte aéreo mediante la carga y auditoría del dataset `df_vuelos_diario_limpio.csv`, previamente generado durante la fase de limpieza y almacenado en la carpeta de preintegración.

En esta etapa no se realiza ninguna transformación adicional ni integración con el dataset maestro. El objetivo es verificar la estructura real del archivo antes de definir los indicadores de actividad aérea que se utilizarán posteriormente.

La auditoría comprende:

- dimensiones del dataset;
- nombres y tipos de las variables;
- identificación de la columna temporal;
- periodo disponible;
- número de fechas y años representados;
- granularidad temporal;
- duplicados;
- valores ausentes;
- variables numéricas disponibles;
- rangos y estadísticos básicos;
- presencia de valores negativos o infinitos;
- número de registros por fecha y por año.

Esta revisión permitirá determinar si el dataset presenta una observación única por día o si mantiene una desagregación adicional —por ejemplo, por compañía, zona u otra categoría— que deba ser considerada antes de construir el indicador diario definitivo.

> **Objetivo:** caracterizar y validar la estructura del dataset diario de vuelos antes de realizar cualquier agregación, selección de variables o integración con el dataset maestro.

In [ ]:
# ============================================================
# 6.1. CARGA Y AUDITORÍA ESTRUCTURAL
# DEL DATASET DIARIO DE TRANSPORTE AÉREO
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. RUTAS
# ============================================================

ruta_base = Path(
    "/content/drive/MyDrive/TFM/"
    "11_Machine_Learning_Temporal"
)

ruta_vuelos = (
    ruta_base /
    "PREINTEGRACION" /
    "05_Transporte_Aereo" /
    "df_vuelos_diario_limpio.csv"
)


# ============================================================
# 2. COMPROBAR EXISTENCIA DEL ARCHIVO
# ============================================================

if not ruta_vuelos.exists():

    raise FileNotFoundError(
        "No se encuentra el archivo:\n"
        f"{ruta_vuelos}"
    )


# ============================================================
# 3. CARGA
# ============================================================

df_vuelos = pd.read_csv(
    ruta_vuelos,
    low_memory=False
)


print("=" * 80)
print("6.1 — CARGA Y AUDITORÍA ESTRUCTURAL DEL TRANSPORTE AÉREO")
print("=" * 80)

print(
    f"\n✓ Archivo cargado:\n"
    f"{ruta_vuelos}"
)


# ============================================================
# 4. DIMENSIONES
# ============================================================

print("\n" + "=" * 80)
print("DIMENSIONES")
print("=" * 80)

print(
    f"\n✓ Filas: "
    f"{len(df_vuelos):,}"
)

print(
    f"✓ Columnas: "
    f"{df_vuelos.shape[1]}"
)


# ============================================================
# 5. VARIABLES DISPONIBLES
# ============================================================

print("\n" + "=" * 80)
print("VARIABLES DISPONIBLES")
print("=" * 80)

for i, columna in enumerate(
    df_vuelos.columns,
    start=1
):
    print(
        f"{i:02d}. {columna}"
    )


# ============================================================
# 6. TIPOS DE DATOS
# ============================================================

print("\n" + "=" * 80)
print("TIPOS DE DATOS")
print("=" * 80)

print(
    df_vuelos.dtypes
)


# ============================================================
# 7. IDENTIFICAR COLUMNA DE FECHA
# ============================================================

candidatas_fecha = [
    "fecha",
    "Fecha",
    "FECHA",
    "date",
    "Date",
    "DATE"
]

col_fecha = next(
    (
        columna
        for columna in candidatas_fecha
        if columna in df_vuelos.columns
    ),
    None
)


if col_fecha is None:

    raise KeyError(
        "No se ha identificado automáticamente "
        "la columna de fecha.\n"
        f"Columnas disponibles: {df_vuelos.columns.tolist()}"
    )


print("\n" + "=" * 80)
print("VARIABLE TEMPORAL")
print("=" * 80)

print(
    f"\n✓ Columna temporal identificada: "
    f"{col_fecha}"
)


# ============================================================
# 8. CONVERTIR FECHA
# ============================================================

df_vuelos[
    col_fecha
] = pd.to_datetime(
    df_vuelos[
        col_fecha
    ],
    errors="coerce"
)


fechas_invalidas = (
    df_vuelos[
        col_fecha
    ]
    .isna()
    .sum()
)


print(
    f"✓ Fechas inválidas: "
    f"{fechas_invalidas:,}"
)


if fechas_invalidas > 0:

    raise ValueError(
        "Existen fechas inválidas en el dataset de vuelos."
    )


# ============================================================
# 9. PERIODO TEMPORAL
# ============================================================

fecha_min = (
    df_vuelos[
        col_fecha
    ]
    .min()
)

fecha_max = (
    df_vuelos[
        col_fecha
    ]
    .max()
)

n_fechas = (
    df_vuelos[
        col_fecha
    ]
    .nunique()
)


print("\n" + "=" * 80)
print("COBERTURA TEMPORAL GENERAL")
print("=" * 80)

print(
    f"\n✓ Fecha inicial: "
    f"{fecha_min}"
)

print(
    f"✓ Fecha final: "
    f"{fecha_max}"
)

print(
    f"✓ Fechas diferentes: "
    f"{n_fechas:,}"
)


# ============================================================
# 10. CREAR AÑO AUXILIAR
# ============================================================

df_vuelos[
    "_anio_control"
] = (
    df_vuelos[
        col_fecha
    ]
    .dt.year
)


anios_disponibles = sorted(
    df_vuelos[
        "_anio_control"
    ]
    .dropna()
    .unique()
    .tolist()
)


print(
    f"✓ Años disponibles: "
    f"{anios_disponibles}"
)


# ============================================================
# 11. REGISTROS POR AÑO
# ============================================================

registros_anio = (
    df_vuelos
    .groupby(
        "_anio_control"
    )
    .agg(
        registros=(
            col_fecha,
            "size"
        ),
        dias=(
            col_fecha,
            "nunique"
        )
    )
    .reset_index()
    .rename(
        columns={
            "_anio_control":
                "anio"
        }
    )
)


print("\n" + "=" * 80)
print("REGISTROS Y DÍAS POR AÑO")
print("=" * 80)

display(
    registros_anio
)


# ============================================================
# 12. GRANULARIDAD TEMPORAL
# ============================================================

registros_por_fecha = (
    df_vuelos
    .groupby(
        col_fecha
    )
    .size()
)


print("\n" + "=" * 80)
print("GRANULARIDAD TEMPORAL")
print("=" * 80)

print(
    f"\n✓ Mínimo de registros por fecha: "
    f"{registros_por_fecha.min():,}"
)

print(
    f"✓ Mediana de registros por fecha: "
    f"{registros_por_fecha.median():.1f}"
)

print(
    f"✓ Media de registros por fecha: "
    f"{registros_por_fecha.mean():.2f}"
)

print(
    f"✓ Máximo de registros por fecha: "
    f"{registros_por_fecha.max():,}"
)


if (
    registros_por_fecha.max() == 1
    and
    registros_por_fecha.min() == 1
):

    print(
        "✓ El dataset presenta una única "
        "observación por fecha."
    )

else:

    print(
        "⚠ Existen varias observaciones para "
        "al menos algunas fechas."
    )

    print(
        "  La dimensión adicional deberá "
        "identificarse antes de integrar."
    )


# ============================================================
# 13. DUPLICADOS COMPLETOS
# ============================================================

duplicados_completos = (
    df_vuelos
    .drop(
        columns=[
            "_anio_control"
        ]
    )
    .duplicated()
    .sum()
)


print("\n" + "=" * 80)
print("CONTROL DE DUPLICADOS")
print("=" * 80)

print(
    f"\n✓ Duplicados completos: "
    f"{duplicados_completos:,}"
)


# ============================================================
# 14. DUPLICADOS DE FECHA
# ============================================================

duplicados_fecha = (
    df_vuelos
    .duplicated(
        subset=[
            col_fecha
        ]
    )
    .sum()
)


print(
    f"✓ Registros con fecha repetida "
    f"después de la primera aparición: "
    f"{duplicados_fecha:,}"
)


# ============================================================
# 15. COLUMNAS CONSTANTES
# ============================================================

columnas_constantes = [
    columna
    for columna in df_vuelos.columns
    if (
        columna != "_anio_control"
        and
        df_vuelos[
            columna
        ].nunique(
            dropna=False
        ) <= 1
    )
]


print("\n" + "=" * 80)
print("COLUMNAS CONSTANTES")
print("=" * 80)

print(
    f"\n✓ Número de columnas constantes: "
    f"{len(columnas_constantes)}"
)

if columnas_constantes:

    for columna in columnas_constantes:
        print(
            f"  • {columna}"
        )

else:

    print(
        "✓ No se detectan columnas constantes."
    )


# ============================================================
# 16. VALORES AUSENTES
# ============================================================

nulos = (
    df_vuelos
    .drop(
        columns=[
            "_anio_control"
        ]
    )
    .isna()
    .sum()
)

nulos = (
    nulos[
        nulos > 0
    ]
    .sort_values(
        ascending=False
    )
)


print("\n" + "=" * 80)
print("VALORES AUSENTES")
print("=" * 80)

if len(nulos) == 0:

    print(
        "\n✓ No existen valores ausentes."
    )

else:

    print(
        f"\n✓ Variables con valores ausentes: "
        f"{len(nulos)}"
    )

    tabla_nulos = pd.DataFrame({
        "variable":
            nulos.index,

        "n_nan":
            nulos.values,

        "pct_nan":
            (
                nulos.values
                /
                len(df_vuelos)
                *
                100
            ).round(2)
    })

    display(
        tabla_nulos
    )


# ============================================================
# 17. VARIABLES NUMÉRICAS
# ============================================================

columnas_numericas = (
    df_vuelos
    .drop(
        columns=[
            "_anio_control"
        ]
    )
    .select_dtypes(
        include=[
            np.number
        ]
    )
    .columns
    .tolist()
)


print("\n" + "=" * 80)
print("VARIABLES NUMÉRICAS")
print("=" * 80)

print(
    f"\n✓ Número de variables numéricas: "
    f"{len(columnas_numericas)}"
)

for columna in columnas_numericas:
    print(
        f"  • {columna}"
    )


# ============================================================
# 18. ESTADÍSTICOS DESCRIPTIVOS
# ============================================================

if columnas_numericas:

    resumen_numerico = (
        df_vuelos[
            columnas_numericas
        ]
        .describe()
        .T
    )

    print("\n" + "=" * 80)
    print("ESTADÍSTICOS DE VARIABLES NUMÉRICAS")
    print("=" * 80)

    display(
        resumen_numerico
    )


# ============================================================
# 19. VALORES INFINITOS
# ============================================================

if columnas_numericas:

    n_inf = (
        np.isinf(
            df_vuelos[
                columnas_numericas
            ]
        )
        .sum()
        .sum()
    )

else:

    n_inf = 0


print("\n" + "=" * 80)
print("CONTROL DE VALORES INFINITOS")
print("=" * 80)

print(
    f"\n✓ Valores infinitos: "
    f"{n_inf:,}"
)


# ============================================================
# 20. VALORES NEGATIVOS
# ============================================================

if columnas_numericas:

    negativos = []

    for columna in columnas_numericas:

        n_negativos = (
            df_vuelos[
                columna
            ] < 0
        ).sum()

        if n_negativos > 0:

            negativos.append({
                "variable":
                    columna,

                "n_negativos":
                    int(
                        n_negativos
                    ),

                "minimo":
                    df_vuelos[
                        columna
                    ].min()
            })


    df_negativos = pd.DataFrame(
        negativos
    )


    print("\n" + "=" * 80)
    print("CONTROL DE VALORES NEGATIVOS")
    print("=" * 80)

    if df_negativos.empty:

        print(
            "\n✓ No se detectan valores negativos "
            "en las variables numéricas."
        )

    else:

        print(
            "\n⚠ Se detectan valores negativos:"
        )

        display(
            df_negativos
        )


# ============================================================
# 21. VARIABLES CATEGÓRICAS / TEXTO
# ============================================================

columnas_categoricas = (
    df_vuelos
    .drop(
        columns=[
            "_anio_control"
        ]
    )
    .select_dtypes(
        exclude=[
            np.number,
            "datetime"
        ]
    )
    .columns
    .tolist()
)


print("\n" + "=" * 80)
print("VARIABLES CATEGÓRICAS")
print("=" * 80)

print(
    f"\n✓ Número de variables categóricas: "
    f"{len(columnas_categoricas)}"
)


for columna in columnas_categoricas:

    print(
        f"\n• {columna}: "
        f"{df_vuelos[columna].nunique(dropna=True):,} "
        f"valores diferentes"
    )

    valores = (
        df_vuelos[
            columna
        ]
        .value_counts(
            dropna=False
        )
        .head(10)
    )

    display(
        valores.to_frame(
            name="registros"
        )
    )


# ============================================================
# 22. PRIMERAS OBSERVACIONES
# ============================================================

print("\n" + "=" * 80)
print("MUESTRA DEL DATASET")
print("=" * 80)

display(
    df_vuelos
    .drop(
        columns=[
            "_anio_control"
        ]
    )
    .head(10)
)


# ============================================================
# 23. RESUMEN DE LA AUDITORÍA
# ============================================================

print("\n" + "=" * 80)
print("AUDITORÍA 6.1 FINALIZADA")
print("=" * 80)

print(
    f"\n✓ Registros: "
    f"{len(df_vuelos):,}"
)

print(
    f"✓ Variables originales: "
    f"{df_vuelos.shape[1] - 1}"
)

print(
    f"✓ Fecha inicial: "
    f"{fecha_min.date()}"
)

print(
    f"✓ Fecha final: "
    f"{fecha_max.date()}"
)

print(
    f"✓ Fechas diferentes: "
    f"{n_fechas:,}"
)

print(
    f"✓ Años disponibles: "
    f"{anios_disponibles}"
)

print(
    f"✓ Duplicados completos: "
    f"{duplicados_completos:,}"
)

print(
    f"✓ Valores infinitos: "
    f"{n_inf:,}"
)

print(
    "✓ No se ha realizado ninguna transformación "
    "ni imputación."
)

print(
    "✓ Dataset preparado para analizar "
    "granularidad y cobertura temporal en 6.2."
)


# ============================================================
# 24. ELIMINAR VARIABLE AUXILIAR
# ============================================================

df_vuelos = (
    df_vuelos
    .drop(
        columns=[
            "_anio_control"
        ]
    )
)

6.1 — CARGA Y AUDITORÍA ESTRUCTURAL DEL TRANSPORTE AÉREO

✓ Archivo cargado:
/content/drive/MyDrive/TFM/11_Machine_Learning_Temporal/PREINTEGRACION/05_Transporte_Aereo/df_vuelos_diario_limpio.csv

DIMENSIONES

✓ Filas: 2,011
✓ Columnas: 18

VARIABLES DISPONIBLES
01. Fecha
02. Anio
03. Mes
04. Trimestre
05. Dia_semana
06. Fin_semana
07. Periodo_COVID
08. Es_COVID
09. Ventana_Jul_Dic
10. Vuelos_total
11. Vuelos_Espanya
12. Vuelos_Europa
13. Vuelos_America
14. Vuelos_Asia
15. Vuelos_Africa
16. Vuelos_internacionales
17. Companias_activas
18. Zonas_activas

TIPOS DE DATOS
Fecha                      object
Anio                        int64
Mes                         int64
Trimestre                   int64
Dia_semana                  int64
Fin_semana                  int64
Periodo_COVID              object
Es_COVID                    int64
Ventana_Jul_Dic             int64
Vuelos_total                int64
Vuelos_Espanya            float64
Vuelos_Europa             float64
Vuelos_America   

,anio,registros,dias
0,2019,184,184
1,2020,366,366
2,2021,365,365
3,2022,365,365
4,2023,365,365
5,2024,366,366



GRANULARIDAD TEMPORAL

✓ Mínimo de registros por fecha: 1
✓ Mediana de registros por fecha: 1.0
✓ Media de registros por fecha: 1.00
✓ Máximo de registros por fecha: 1
✓ El dataset presenta una única observación por fecha.

CONTROL DE DUPLICADOS

✓ Duplicados completos: 0
✓ Registros con fecha repetida después de la primera aparición: 0

COLUMNAS CONSTANTES

✓ Número de columnas constantes: 0
✓ No se detectan columnas constantes.

VALORES AUSENTES

✓ No existen valores ausentes.

VARIABLES NUMÉRICAS

✓ Número de variables numéricas: 16
  • Anio
  • Mes
  • Trimestre
  • Dia_semana
  • Fin_semana
  • Es_COVID
  • Ventana_Jul_Dic
  • Vuelos_total
  • Vuelos_Espanya
  • Vuelos_Europa
  • Vuelos_America
  • Vuelos_Asia
  • Vuelos_Africa
  • Vuelos_internacionales
  • Companias_activas
  • Zonas_activas

ESTADÍSTICOS DE VARIABLES NUMÉRICAS


,count,mean,std,min,25%,50%,75%,max
Anio,2011.0,2021.725510,1.602622,2019.0,2020.0,2022.0,2023.0,2024.0
Mes,2011.0,6.793138,3.437715,1.0,4.0,7.0,10.0,12.0
Trimestre,2011.0,2.598707,1.113275,1.0,2.0,3.0,4.0,4.0
Dia_semana,2011.0,2.997514,2.001118,0.0,1.0,3.0,5.0,6.0
Fin_semana,2011.0,0.285430,0.451731,0.0,0.0,0.0,1.0,1.0
Es_COVID,2011.0,0.363501,0.481127,0.0,0.0,0.0,1.0,1.0
Ventana_Jul_Dic,2011.0,0.548981,0.497719,0.0,0.0,1.0,1.0,1.0
Vuelos_total,2011.0,551.496271,244.609170,11.0,406.5,635.0,741.0,924.0
Vuelos_Espanya,2011.0,193.508702,69.128860,6.0,168.0,218.0,242.0,298.0
Vuelos_Europa,2011.0,327.919443,164.342327,2.0,217.5,383.0,457.0,579.0



CONTROL DE VALORES INFINITOS

✓ Valores infinitos: 0

CONTROL DE VALORES NEGATIVOS

✓ No se detectan valores negativos en las variables numéricas.

VARIABLES CATEGÓRICAS

✓ Número de variables categóricas: 1

• Periodo_COVID: 3 valores diferentes


,registros
Periodo_COVID,
Post-COVID,1096
COVID,731
Pre-COVID,184



MUESTRA DEL DATASET


,Fecha,Anio,Mes,Trimestre,Dia_semana,Fin_semana,Periodo_COVID,Es_COVID,Ventana_Jul_Dic,Vuelos_total,Vuelos_Espanya,Vuelos_Europa,Vuelos_America,Vuelos_Asia,Vuelos_Africa,Vuelos_internacionales,Companias_activas,Zonas_activas
0,2019-07-01,2019,7,3,0,0,Pre-COVID,0,1,742,244.0,458.0,10.0,19.0,11.0,498.0,20,5
1,2019-07-02,2019,7,3,1,0,Pre-COVID,0,1,715,246.0,435.0,10.0,17.0,7.0,469.0,20,5
2,2019-07-03,2019,7,3,2,0,Pre-COVID,0,1,757,260.0,458.0,10.0,16.0,13.0,497.0,20,5
3,2019-07-04,2019,7,3,3,0,Pre-COVID,0,1,728,253.0,428.0,10.0,16.0,21.0,475.0,19,5
4,2019-07-05,2019,7,3,4,0,Pre-COVID,0,1,737,250.0,454.0,10.0,17.0,6.0,487.0,19,5
5,2019-07-06,2019,7,3,5,1,Pre-COVID,0,1,651,188.0,421.0,10.0,16.0,16.0,463.0,21,5
6,2019-07-07,2019,7,3,6,1,Pre-COVID,0,1,684,215.0,426.0,10.0,19.0,14.0,469.0,19,5
7,2019-07-08,2019,7,3,0,0,Pre-COVID,0,1,699,231.0,433.0,8.0,18.0,9.0,468.0,20,5
8,2019-07-09,2019,7,3,1,0,Pre-COVID,0,1,707,240.0,430.0,10.0,17.0,10.0,467.0,20,5
9,2019-07-10,2019,7,3,2,0,Pre-COVID,0,1,727,251.0,436.0,10.0,17.0,13.0,476.0,20,5



AUDITORÍA 6.1 FINALIZADA

✓ Registros: 2,011
✓ Variables originales: 18
✓ Fecha inicial: 2019-07-01
✓ Fecha final: 2024-12-31
✓ Fechas diferentes: 2,011
✓ Años disponibles: [2019, 2020, 2021, 2022, 2023, 2024]
✓ Duplicados completos: 0
✓ Valores infinitos: 0
✓ No se ha realizado ninguna transformación ni imputación.
✓ Dataset preparado para analizar granularidad y cobertura temporal en 6.2.


### Resultados — 6.1. Carga y auditoría estructural del dataset diario de transporte aéreo

El dataset `df_vuelos_diario_limpio.csv` contiene **2.011 registros y 18 variables**, correspondientes a observaciones diarias de actividad aérea.

La cobertura temporal se extiende desde el **1 de julio de 2019 hasta el 31 de diciembre de 2024**, con **2.011 fechas diferentes**. La disponibilidad anual es la siguiente:

| Año | Registros | Días disponibles |
|---:|---:|---:|
| 2019 | 184 | 184 |
| 2020 | 366 | 366 |
| 2021 | 365 | 365 |
| 2022 | 365 | 365 |
| 2023 | 365 | 365 |
| 2024 | 366 | 366 |

El año 2019 presenta únicamente información correspondiente al periodo **julio–diciembre**, mientras que los años 2020–2024 disponen de cobertura anual completa. No existen observaciones anteriores al 1 de julio de 2019.

La auditoría de granularidad confirma que el dataset presenta **exactamente una observación por fecha**: el mínimo, la mediana, la media y el máximo de registros diarios son iguales a 1. No se detectan duplicados completos ni fechas repetidas.

El dataset contiene variables temporales y de caracterización del periodo (`Anio`, `Mes`, `Trimestre`, `Dia_semana`, `Fin_semana`, `Periodo_COVID`, `Es_COVID` y `Ventana_Jul_Dic`) junto con indicadores de actividad aérea:

- `Vuelos_total`
- `Vuelos_Espanya`
- `Vuelos_Europa`
- `Vuelos_America`
- `Vuelos_Asia`
- `Vuelos_Africa`
- `Vuelos_internacionales`
- `Companias_activas`
- `Zonas_activas`

No se detectan **valores ausentes, valores infinitos ni valores negativos** en las variables numéricas. Tampoco existen columnas constantes.

`Vuelos_total` presenta valores comprendidos entre **11 y 924 vuelos diarios**, con una mediana de **635 vuelos/día** y una media de aproximadamente **551,5 vuelos/día**, mostrando una variabilidad temporal elevada que deberá analizarse específicamente en relación con los periodos pre-COVID, COVID y post-COVID.

La variable categórica `Periodo_COVID` distingue tres periodos: **Pre-COVID (184 días), COVID (731 días) y Post-COVID (1.096 días)**.

La estructura del dataset resulta, por tanto, adecuada para continuar con el análisis temporal sin necesidad de realizar nuevas agregaciones diarias.

La principal limitación identificada en esta etapa es de carácter temporal: **no existen datos para 2018 ni para el primer semestre de 2019**. Esta ausencia se conservará explícitamente y se evaluará frente al periodo del dataset maestro antes de realizar la integración, sin aplicar imputaciones en esta fase.

> **Conclusión:** el dataset de transporte aéreo presenta una estructura diaria consistente, sin duplicados ni anomalías estructurales, y cobertura continua desde julio de 2019 hasta diciembre de 2024. Queda preparado para la auditoría temporal y de cobertura de la etapa 6.2.

## 6.2. Auditoría temporal, cobertura y coherencia interna del transporte aéreo

Una vez validada la estructura del dataset diario de transporte aéreo, se realiza una auditoría específica de su continuidad temporal y de la coherencia interna de los indicadores disponibles.

El dataset presenta una observación diaria desde julio de 2019 hasta diciembre de 2024. Dado que el periodo general del estudio comprende 2018–2024, resulta necesario diferenciar explícitamente entre:

- días con información aérea disponible;
- días anteriores al inicio de la serie;
- posibles huecos internos dentro del periodo cubierto.

La ausencia de datos anteriores al 1 de julio de 2019 se considera **ausencia de información** y no actividad aérea igual a cero. Por tanto, no se realizará ninguna imputación ni reconstrucción artificial de 2018 o del primer semestre de 2019.

La auditoría incluye:

- continuidad diaria de la serie;
- identificación de posibles fechas ausentes dentro del periodo observado;
- cobertura respecto al periodo completo 2018–2024;
- cobertura anual;
- caracterización de los periodos pre-COVID, COVID y post-COVID;
- evolución anual de la actividad aérea;
- coherencia entre `Vuelos_total` y sus componentes geográficos;
- coherencia de `Vuelos_internacionales`;
- comprobación de que los vuelos nacionales e internacionales reconstruyen el total;
- control de valores ausentes, negativos e infinitos.

Esta etapa permitirá determinar qué variables representan información independiente y cuáles son combinaciones derivadas o redundantes, evitando introducir posteriormente variables perfectamente colineales en el modelo.

> **Objetivo:** verificar la continuidad, cobertura y consistencia interna de la serie diaria de vuelos antes de seleccionar los indicadores de transporte aéreo que se incorporarán al dataset maestro.

In [ ]:
# ============================================================
# 6.2. AUDITORÍA TEMPORAL, COBERTURA
# Y COHERENCIA INTERNA DEL TRANSPORTE AÉREO
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 1. COMPROBAR DATASET DE 6.1
# ============================================================

if "df_vuelos" not in globals():

    raise NameError(
        "No existe df_vuelos. "
        "Debe ejecutarse previamente la etapa 6.1."
    )


df_vuelos_audit = (
    df_vuelos
    .copy()
)


print("=" * 80)
print("6.2 — AUDITORÍA TEMPORAL Y COHERENCIA DEL TRANSPORTE AÉREO")
print("=" * 80)


# ============================================================
# 2. NORMALIZAR FECHA
# ============================================================

if "Fecha" not in df_vuelos_audit.columns:

    raise KeyError(
        "No existe la variable Fecha "
        "en el dataset de vuelos."
    )


df_vuelos_audit[
    "Fecha"
] = pd.to_datetime(
    df_vuelos_audit[
        "Fecha"
    ],
    errors="coerce"
)


fechas_invalidas = (
    df_vuelos_audit[
        "Fecha"
    ]
    .isna()
    .sum()
)


print("\n" + "=" * 80)
print("CONTROL TEMPORAL INICIAL")
print("=" * 80)

print(
    f"\n✓ Registros: "
    f"{len(df_vuelos_audit):,}"
)

print(
    f"✓ Fechas inválidas: "
    f"{fechas_invalidas:,}"
)


if fechas_invalidas > 0:

    raise ValueError(
        "Existen fechas inválidas."
    )


# ============================================================
# 3. CONTROL DE UNICIDAD TEMPORAL
# ============================================================

duplicados_fecha = (
    df_vuelos_audit
    .duplicated(
        subset=[
            "Fecha"
        ]
    )
    .sum()
)


print(
    f"✓ Duplicados de Fecha: "
    f"{duplicados_fecha:,}"
)


if duplicados_fecha > 0:

    raise ValueError(
        "El dataset presenta más de "
        "una observación por fecha."
    )


# ============================================================
# 4. PERIODO OBSERVADO
# ============================================================

fecha_inicio = (
    df_vuelos_audit[
        "Fecha"
    ]
    .min()
)

fecha_fin = (
    df_vuelos_audit[
        "Fecha"
    ]
    .max()
)


print(
    f"✓ Periodo observado: "
    f"{fecha_inicio.date()} "
    f"→ "
    f"{fecha_fin.date()}"
)


# ============================================================
# 5. CONTINUIDAD DENTRO DEL PERIODO OBSERVADO
# ============================================================

calendario_observado = pd.date_range(
    start=fecha_inicio,
    end=fecha_fin,
    freq="D"
)

fechas_observadas = pd.DatetimeIndex(
    df_vuelos_audit[
        "Fecha"
    ]
    .drop_duplicates()
)


fechas_ausentes_internas = (
    calendario_observado
    .difference(
        fechas_observadas
    )
)


print("\n" + "=" * 80)
print("CONTINUIDAD DE LA SERIE OBSERVADA")
print("=" * 80)

print(
    f"\n✓ Días esperados entre "
    f"{fecha_inicio.date()} y {fecha_fin.date()}: "
    f"{len(calendario_observado):,}"
)

print(
    f"✓ Días observados: "
    f"{len(fechas_observadas):,}"
)

print(
    f"✓ Huecos internos: "
    f"{len(fechas_ausentes_internas):,}"
)


if len(fechas_ausentes_internas) == 0:

    print(
        "✓ La serie es completamente continua "
        "dentro del periodo observado."
    )

else:

    print(
        "\n⚠ Fechas ausentes dentro "
        "del periodo observado:"
    )

    display(
        pd.DataFrame({
            "Fecha_ausente":
                fechas_ausentes_internas
        })
    )


# ============================================================
# 6. COBERTURA DEL PERIODO OBJETIVO 2018–2024
# ============================================================

inicio_objetivo = pd.Timestamp(
    "2018-01-01"
)

fin_objetivo = pd.Timestamp(
    "2024-12-31"
)

calendario_objetivo = pd.date_range(
    start=inicio_objetivo,
    end=fin_objetivo,
    freq="D"
)

dias_objetivo = len(
    calendario_objetivo
)

dias_con_datos = (
    fechas_observadas
    .intersection(
        calendario_objetivo
    )
    .size
)

dias_sin_datos = (
    dias_objetivo
    -
    dias_con_datos
)

cobertura_periodo = (
    dias_con_datos
    /
    dias_objetivo
    *
    100
)


print("\n" + "=" * 80)
print("COBERTURA DEL PERIODO OBJETIVO 2018–2024")
print("=" * 80)

print(
    f"\n✓ Días del periodo objetivo: "
    f"{dias_objetivo:,}"
)

print(
    f"✓ Días con datos de vuelos: "
    f"{dias_con_datos:,}"
)

print(
    f"✓ Días sin datos de vuelos: "
    f"{dias_sin_datos:,}"
)

print(
    f"✓ Cobertura temporal global: "
    f"{cobertura_periodo:.2f} %"
)

print(
    "✓ Los días anteriores al inicio de la serie "
    "se consideran ausencia de información, no cero vuelos."
)


# ============================================================
# 7. COBERTURA POR AÑO
# ============================================================

registros_cobertura_anual = []

for anio in range(
    2018,
    2025
):

    inicio_anio = pd.Timestamp(
        f"{anio}-01-01"
    )

    fin_anio = pd.Timestamp(
        f"{anio}-12-31"
    )

    calendario_anio = pd.date_range(
        start=inicio_anio,
        end=fin_anio,
        freq="D"
    )

    fechas_anio = (
        fechas_observadas[
            fechas_observadas.year == anio
        ]
    )

    dias_esperados = len(
        calendario_anio
    )

    dias_disponibles = len(
        fechas_anio
    )

    cobertura = (
        dias_disponibles
        /
        dias_esperados
        *
        100
    )

    registros_cobertura_anual.append({
        "anio":
            anio,

        "dias_esperados":
            dias_esperados,

        "dias_disponibles":
            dias_disponibles,

        "dias_sin_datos":
            dias_esperados
            -
            dias_disponibles,

        "cobertura_pct":
            round(
                cobertura,
                2
            )
    })


df_cobertura_anual_vuelos = pd.DataFrame(
    registros_cobertura_anual
)


print("\n" + "=" * 80)
print("COBERTURA TEMPORAL POR AÑO")
print("=" * 80)

display(
    df_cobertura_anual_vuelos
)


# ============================================================
# 8. VARIABLES DE ACTIVIDAD NECESARIAS
# ============================================================

variables_actividad = [
    "Vuelos_total",
    "Vuelos_Espanya",
    "Vuelos_Europa",
    "Vuelos_America",
    "Vuelos_Asia",
    "Vuelos_Africa",
    "Vuelos_internacionales",
    "Companias_activas",
    "Zonas_activas"
]


variables_faltantes = [
    variable
    for variable in variables_actividad
    if variable not in df_vuelos_audit.columns
]


if variables_faltantes:

    raise KeyError(
        "Faltan variables necesarias:\n"
        f"{variables_faltantes}"
    )


# ============================================================
# 9. CONTROL DE NaN, INFINITOS Y NEGATIVOS
# ============================================================

n_nan_actividad = (
    df_vuelos_audit[
        variables_actividad
    ]
    .isna()
    .sum()
    .sum()
)

n_inf_actividad = (
    np.isinf(
        df_vuelos_audit[
            variables_actividad
        ]
    )
    .sum()
    .sum()
)

n_negativos_actividad = (
    (
        df_vuelos_audit[
            variables_actividad
        ] < 0
    )
    .sum()
    .sum()
)


print("\n" + "=" * 80)
print("CONTROL DE VARIABLES DE ACTIVIDAD")
print("=" * 80)

print(
    f"\n✓ Valores ausentes: "
    f"{n_nan_actividad:,}"
)

print(
    f"✓ Valores infinitos: "
    f"{n_inf_actividad:,}"
)

print(
    f"✓ Valores negativos: "
    f"{n_negativos_actividad:,}"
)


# ============================================================
# 10. COHERENCIA DE LOS COMPONENTES GEOGRÁFICOS
# ============================================================

componentes_geograficos = [
    "Vuelos_Espanya",
    "Vuelos_Europa",
    "Vuelos_America",
    "Vuelos_Asia",
    "Vuelos_Africa"
]


df_vuelos_audit[
    "_suma_componentes"
] = (
    df_vuelos_audit[
        componentes_geograficos
    ]
    .sum(
        axis=1
    )
)


df_vuelos_audit[
    "_dif_total_componentes"
] = (
    df_vuelos_audit[
        "Vuelos_total"
    ]
    -
    df_vuelos_audit[
        "_suma_componentes"
    ]
)


n_incoherentes_total = (
    ~np.isclose(
        df_vuelos_audit[
            "Vuelos_total"
        ],
        df_vuelos_audit[
            "_suma_componentes"
        ],
        rtol=0,
        atol=1e-9
    )
).sum()


print("\n" + "=" * 80)
print("COHERENCIA DE VUELOS_TOTAL")
print("=" * 80)

print(
    f"\n✓ Registros evaluados: "
    f"{len(df_vuelos_audit):,}"
)

print(
    f"✓ Registros donde total = suma geográfica: "
    f"{len(df_vuelos_audit) - n_incoherentes_total:,}"
)

print(
    f"✓ Registros con discrepancia: "
    f"{n_incoherentes_total:,}"
)

print(
    f"✓ Diferencia mínima: "
    f"{df_vuelos_audit['_dif_total_componentes'].min():.3f}"
)

print(
    f"✓ Diferencia máxima: "
    f"{df_vuelos_audit['_dif_total_componentes'].max():.3f}"
)


# ============================================================
# 11. COHERENCIA DE VUELOS INTERNACIONALES
# ============================================================

componentes_internacionales = [
    "Vuelos_Europa",
    "Vuelos_America",
    "Vuelos_Asia",
    "Vuelos_Africa"
]


df_vuelos_audit[
    "_suma_internacionales"
] = (
    df_vuelos_audit[
        componentes_internacionales
    ]
    .sum(
        axis=1
    )
)


df_vuelos_audit[
    "_dif_internacionales"
] = (
    df_vuelos_audit[
        "Vuelos_internacionales"
    ]
    -
    df_vuelos_audit[
        "_suma_internacionales"
    ]
)


n_incoherentes_internacionales = (
    ~np.isclose(
        df_vuelos_audit[
            "Vuelos_internacionales"
        ],
        df_vuelos_audit[
            "_suma_internacionales"
        ],
        rtol=0,
        atol=1e-9
    )
).sum()


print("\n" + "=" * 80)
print("COHERENCIA DE VUELOS_INTERNACIONALES")
print("=" * 80)

print(
    f"\n✓ Registros donde internacionales = "
    f"Europa + América + Asia + África: "
    f"{len(df_vuelos_audit) - n_incoherentes_internacionales:,}"
)

print(
    f"✓ Registros con discrepancia: "
    f"{n_incoherentes_internacionales:,}"
)

print(
    f"✓ Diferencia mínima: "
    f"{df_vuelos_audit['_dif_internacionales'].min():.3f}"
)

print(
    f"✓ Diferencia máxima: "
    f"{df_vuelos_audit['_dif_internacionales'].max():.3f}"
)


# ============================================================
# 12. COHERENCIA NACIONAL + INTERNACIONAL = TOTAL
# ============================================================

df_vuelos_audit[
    "_total_nacional_internacional"
] = (
    df_vuelos_audit[
        "Vuelos_Espanya"
    ]
    +
    df_vuelos_audit[
        "Vuelos_internacionales"
    ]
)


df_vuelos_audit[
    "_dif_nacional_internacional"
] = (
    df_vuelos_audit[
        "Vuelos_total"
    ]
    -
    df_vuelos_audit[
        "_total_nacional_internacional"
    ]
)


n_incoherentes_nacional_internacional = (
    ~np.isclose(
        df_vuelos_audit[
            "Vuelos_total"
        ],
        df_vuelos_audit[
            "_total_nacional_internacional"
        ],
        rtol=0,
        atol=1e-9
    )
).sum()


print("\n" + "=" * 80)
print("COHERENCIA NACIONAL + INTERNACIONAL")
print("=" * 80)

print(
    f"\n✓ Registros donde "
    f"España + internacionales = total: "
    f"{len(df_vuelos_audit) - n_incoherentes_nacional_internacional:,}"
)

print(
    f"✓ Registros con discrepancia: "
    f"{n_incoherentes_nacional_internacional:,}"
)

print(
    f"✓ Diferencia mínima: "
    f"{df_vuelos_audit['_dif_nacional_internacional'].min():.3f}"
)

print(
    f"✓ Diferencia máxima: "
    f"{df_vuelos_audit['_dif_nacional_internacional'].max():.3f}"
)


# ============================================================
# 13. RESUMEN ANUAL DE ACTIVIDAD
# ============================================================

df_vuelos_audit[
    "_anio_control"
] = (
    df_vuelos_audit[
        "Fecha"
    ]
    .dt.year
)


resumen_anual = (
    df_vuelos_audit
    .groupby(
        "_anio_control"
    )
    .agg(
        dias=(
            "Fecha",
            "nunique"
        ),

        vuelos_total=(
            "Vuelos_total",
            "sum"
        ),

        vuelos_media_dia=(
            "Vuelos_total",
            "mean"
        ),

        vuelos_mediana_dia=(
            "Vuelos_total",
            "median"
        ),

        vuelos_min_dia=(
            "Vuelos_total",
            "min"
        ),

        vuelos_max_dia=(
            "Vuelos_total",
            "max"
        ),

        vuelos_espanya=(
            "Vuelos_Espanya",
            "sum"
        ),

        vuelos_internacionales=(
            "Vuelos_internacionales",
            "sum"
        ),

        companias_media=(
            "Companias_activas",
            "mean"
        ),

        zonas_media=(
            "Zonas_activas",
            "mean"
        )
    )
    .reset_index()
    .rename(
        columns={
            "_anio_control":
                "anio"
        }
    )
)


print("\n" + "=" * 80)
print("ACTIVIDAD AÉREA POR AÑO")
print("=" * 80)

display(
    resumen_anual
)


# ============================================================
# 14. RESUMEN POR PERIODO COVID
# ============================================================

if "Periodo_COVID" in df_vuelos_audit.columns:

    resumen_covid = (
        df_vuelos_audit
        .groupby(
            "Periodo_COVID",
            dropna=False
        )
        .agg(
            dias=(
                "Fecha",
                "nunique"
            ),

            vuelos_total=(
                "Vuelos_total",
                "sum"
            ),

            vuelos_media_dia=(
                "Vuelos_total",
                "mean"
            ),

            vuelos_mediana_dia=(
                "Vuelos_total",
                "median"
            ),

            vuelos_min_dia=(
                "Vuelos_total",
                "min"
            ),

            vuelos_max_dia=(
                "Vuelos_total",
                "max"
            ),

            companias_media=(
                "Companias_activas",
                "mean"
            ),

            zonas_media=(
                "Zonas_activas",
                "mean"
            )
        )
        .reset_index()
    )


    print("\n" + "=" * 80)
    print("ACTIVIDAD AÉREA POR PERIODO COVID")
    print("=" * 80)

    display(
        resumen_covid
    )


# ============================================================
# 15. PESO NACIONAL / INTERNACIONAL
# ============================================================

df_vuelos_audit[
    "_pct_espanya"
] = np.where(
    df_vuelos_audit[
        "Vuelos_total"
    ] > 0,

    df_vuelos_audit[
        "Vuelos_Espanya"
    ]
    /
    df_vuelos_audit[
        "Vuelos_total"
    ]
    *
    100,

    np.nan
)


df_vuelos_audit[
    "_pct_internacional"
] = np.where(
    df_vuelos_audit[
        "Vuelos_total"
    ] > 0,

    df_vuelos_audit[
        "Vuelos_internacionales"
    ]
    /
    df_vuelos_audit[
        "Vuelos_total"
    ]
    *
    100,

    np.nan
)


print("\n" + "=" * 80)
print("COMPOSICIÓN DE LA ACTIVIDAD AÉREA")
print("=" * 80)

print(
    f"\n✓ Peso medio de vuelos España: "
    f"{df_vuelos_audit['_pct_espanya'].mean():.2f} %"
)

print(
    f"✓ Peso medio de vuelos internacionales: "
    f"{df_vuelos_audit['_pct_internacional'].mean():.2f} %"
)


# ============================================================
# 16. CORRELACIÓN ENTRE VARIABLES DE ACTIVIDAD
# ============================================================

variables_correlacion = [
    "Vuelos_total",
    "Vuelos_Espanya",
    "Vuelos_Europa",
    "Vuelos_America",
    "Vuelos_Asia",
    "Vuelos_Africa",
    "Vuelos_internacionales",
    "Companias_activas",
    "Zonas_activas"
]


matriz_correlacion = (
    df_vuelos_audit[
        variables_correlacion
    ]
    .corr()
)


print("\n" + "=" * 80)
print("CORRELACIÓN ENTRE VARIABLES DE ACTIVIDAD")
print("=" * 80)

display(
    matriz_correlacion.round(3)
)


# ============================================================
# 17. IDENTIFICAR CORRELACIONES MUY ALTAS
# ============================================================

pares_correlacion = []

for i, variable_1 in enumerate(
    variables_correlacion
):

    for variable_2 in variables_correlacion[
        i + 1:
    ]:

        correlacion = (
            matriz_correlacion
            .loc[
                variable_1,
                variable_2
            ]
        )

        if abs(
            correlacion
        ) >= 0.95:

            pares_correlacion.append({
                "variable_1":
                    variable_1,

                "variable_2":
                    variable_2,

                "correlacion":
                    correlacion
            })


df_correlaciones_altas = pd.DataFrame(
    pares_correlacion
)


print("\n" + "=" * 80)
print("PARES CON |CORRELACIÓN| ≥ 0.95")
print("=" * 80)

if df_correlaciones_altas.empty:

    print(
        "\n✓ No existen pares con "
        "|correlación| ≥ 0.95."
    )

else:

    display(
        df_correlaciones_altas
        .sort_values(
            "correlacion",
            key=abs,
            ascending=False
        )
    )


# ============================================================
# 18. CONTROL FINAL DE VARIABLES AUXILIARES
# ============================================================

columnas_auxiliares = [
    "_suma_componentes",
    "_dif_total_componentes",
    "_suma_internacionales",
    "_dif_internacionales",
    "_total_nacional_internacional",
    "_dif_nacional_internacional",
    "_anio_control",
    "_pct_espanya",
    "_pct_internacional"
]


# ============================================================
# 19. RESUMEN FINAL
# ============================================================

print("\n" + "=" * 80)
print("AUDITORÍA 6.2 FINALIZADA")
print("=" * 80)

print(
    f"\n✓ Periodo observado: "
    f"{fecha_inicio.date()} "
    f"→ "
    f"{fecha_fin.date()}"
)

print(
    f"✓ Días observados: "
    f"{len(fechas_observadas):,}"
)

print(
    f"✓ Huecos internos: "
    f"{len(fechas_ausentes_internas):,}"
)

print(
    f"✓ Cobertura respecto a 2018–2024: "
    f"{cobertura_periodo:.2f} %"
)

print(
    f"✓ Discrepancias Vuelos_total "
    f"vs suma geográfica: "
    f"{n_incoherentes_total:,}"
)

print(
    f"✓ Discrepancias Vuelos_internacionales "
    f"vs suma internacional: "
    f"{n_incoherentes_internacionales:,}"
)

print(
    f"✓ Discrepancias España + internacionales "
    f"vs total: "
    f"{n_incoherentes_nacional_internacional:,}"
)

print(
    f"✓ Valores ausentes en variables de actividad: "
    f"{n_nan_actividad:,}"
)

print(
    f"✓ Valores infinitos: "
    f"{n_inf_actividad:,}"
)

print(
    f"✓ Valores negativos: "
    f"{n_negativos_actividad:,}"
)

print(
    "✓ Los días anteriores al 01/07/2019 "
    "se mantienen como ausencia de información."
)

print(
    "✓ No se ha realizado ninguna imputación."
)

print(
    "\n✓ Dataset preparado para seleccionar "
    "los indicadores de transporte aéreo en 6.3."
)


# ============================================================
# 20. RESTAURAR DATASET SIN VARIABLES AUXILIARES
# ============================================================

df_vuelos_audit = (
    df_vuelos_audit
    .drop(
        columns=[
            columna
            for columna in columnas_auxiliares
            if columna in df_vuelos_audit.columns
        ]
    )
)

6.2 — AUDITORÍA TEMPORAL Y COHERENCIA DEL TRANSPORTE AÉREO

CONTROL TEMPORAL INICIAL

✓ Registros: 2,011
✓ Fechas inválidas: 0
✓ Duplicados de Fecha: 0
✓ Periodo observado: 2019-07-01 → 2024-12-31

CONTINUIDAD DE LA SERIE OBSERVADA

✓ Días esperados entre 2019-07-01 y 2024-12-31: 2,011
✓ Días observados: 2,011
✓ Huecos internos: 0
✓ La serie es completamente continua dentro del periodo observado.

COBERTURA DEL PERIODO OBJETIVO 2018–2024

✓ Días del periodo objetivo: 2,557
✓ Días con datos de vuelos: 2,011
✓ Días sin datos de vuelos: 546
✓ Cobertura temporal global: 78.65 %
✓ Los días anteriores al inicio de la serie se consideran ausencia de información, no cero vuelos.

COBERTURA TEMPORAL POR AÑO


,anio,dias_esperados,dias_disponibles,dias_sin_datos,cobertura_pct
0,2018,365,0,365,0.00
1,2019,365,184,181,50.41
2,2020,366,366,0,100.00
3,2021,365,365,0,100.00
4,2022,365,365,0,100.00
5,2023,365,365,0,100.00
6,2024,366,366,0,100.00



CONTROL DE VARIABLES DE ACTIVIDAD

✓ Valores ausentes: 0
✓ Valores infinitos: 0
✓ Valores negativos: 0

COHERENCIA DE VUELOS_TOTAL

✓ Registros evaluados: 2,011
✓ Registros donde total = suma geográfica: 2,011
✓ Registros con discrepancia: 0
✓ Diferencia mínima: 0.000
✓ Diferencia máxima: 0.000

COHERENCIA DE VUELOS_INTERNACIONALES

✓ Registros donde internacionales = Europa + América + Asia + África: 2,011
✓ Registros con discrepancia: 0
✓ Diferencia mínima: 0.000
✓ Diferencia máxima: 0.000

COHERENCIA NACIONAL + INTERNACIONAL

✓ Registros donde España + internacionales = total: 2,011
✓ Registros con discrepancia: 0
✓ Diferencia mínima: 0.000
✓ Diferencia máxima: 0.000

ACTIVIDAD AÉREA POR AÑO


,anio,dias,vuelos_total,vuelos_media_dia,vuelos_mediana_dia,vuelos_min_dia,vuelos_max_dia,vuelos_espanya,vuelos_internacionales,companias_media,zonas_media
0,2019,184,123358,670.423913,693.5,343,799,41252.0,82106.0,19.836957,5.000000
1,2020,366,93212,254.677596,212.0,11,762,40463.0,52749.0,14.155738,3.617486
2,2021,365,126363,346.200000,389.0,45,638,57302.0,69061.0,17.679452,4.553425
3,2022,365,227976,624.591781,671.0,244,791,78028.0,149948.0,22.134247,4.978082
4,2023,365,258362,707.841096,731.0,504,833,85320.0,173042.0,23.887671,5.000000
5,2024,366,279788,764.448087,783.0,521,924,86781.0,193007.0,24.398907,5.000000



ACTIVIDAD AÉREA POR PERIODO COVID


...


COMPOSICIÓN DE LA ACTIVIDAD AÉREA

✓ Peso medio de vuelos España: 39.60 %
✓ Peso medio de vuelos internacionales: 60.40 %

CORRELACIÓN ENTRE VARIABLES DE ACTIVIDAD


,Vuelos_total,Vuelos_Espanya,Vuelos_Europa,Vuelos_America,Vuelos_Asia,Vuelos_Africa,Vuelos_internacionales,Companias_activas,Zonas_activas
Vuelos_total,1.000,0.954,0.994,0.880,0.860,0.815,0.993,0.896,0.744
Vuelos_Espanya,0.954,1.000,0.918,0.759,0.778,0.718,0.914,0.895,0.750
Vuelos_Europa,0.994,0.918,1.000,0.897,0.862,0.819,0.999,0.876,0.717
Vuelos_America,0.880,0.759,0.897,1.000,0.781,0.766,0.905,0.781,0.655
Vuelos_Asia,0.860,0.778,0.862,0.781,1.000,0.769,0.871,0.756,0.733
Vuelos_Africa,0.815,0.718,0.819,0.766,0.769,1.000,0.833,0.719,0.685
Vuelos_internacionales,0.993,0.914,0.999,0.905,0.871,0.833,1.000,0.875,0.724
Companias_activas,0.896,0.895,0.876,0.781,0.756,0.719,0.875,1.000,0.777
Zonas_activas,0.744,0.750,0.717,0.655,0.733,0.685,0.724,0.777,1.000



PARES CON |CORRELACIÓN| ≥ 0.95


,variable_1,variable_2,correlacion
3,Vuelos_Europa,Vuelos_internacionales,0.999457
1,Vuelos_total,Vuelos_Europa,0.994240
2,Vuelos_total,Vuelos_internacionales,0.993387
0,Vuelos_total,Vuelos_Espanya,0.954356



AUDITORÍA 6.2 FINALIZADA

✓ Periodo observado: 2019-07-01 → 2024-12-31
✓ Días observados: 2,011
✓ Huecos internos: 0
✓ Cobertura respecto a 2018–2024: 78.65 %
✓ Discrepancias Vuelos_total vs suma geográfica: 0
✓ Discrepancias Vuelos_internacionales vs suma internacional: 0
✓ Discrepancias España + internacionales vs total: 0
✓ Valores ausentes en variables de actividad: 0
✓ Valores infinitos: 0
✓ Valores negativos: 0
✓ Los días anteriores al 01/07/2019 se mantienen como ausencia de información.
✓ No se ha realizado ninguna imputación.

✓ Dataset preparado para seleccionar los indicadores de transporte aéreo en 6.3.


### Resultados — 6.2. Auditoría temporal, cobertura y coherencia interna del transporte aéreo

La auditoría temporal confirma que el dataset de transporte aéreo contiene **2.011 observaciones diarias**, correspondientes al periodo comprendido entre el **1 de julio de 2019 y el 31 de diciembre de 2024**.

La serie presenta una **continuidad temporal completa dentro del periodo observado**: los 2.011 días esperados están presentes en el dataset y no se detecta ningún hueco interno ni ninguna fecha duplicada.

Respecto al periodo global del estudio, comprendido entre 2018 y 2024, la información aérea está disponible para **2.011 de los 2.557 días**, lo que representa una cobertura temporal del **78,65 %**. Los **546 días sin información** corresponden exclusivamente al periodo anterior al inicio de la serie.

La cobertura anual es:

| Año | Días esperados | Días disponibles | Días sin datos | Cobertura |
|---:|---:|---:|---:|---:|
| 2018 | 365 | 0 | 365 | 0,00 % |
| 2019 | 365 | 184 | 181 | 50,41 % |
| 2020 | 366 | 366 | 0 | 100,00 % |
| 2021 | 365 | 365 | 0 | 100,00 % |
| 2022 | 365 | 365 | 0 | 100,00 % |
| 2023 | 365 | 365 | 0 | 100,00 % |
| 2024 | 366 | 366 | 0 | 100,00 % |

Por tanto, la ausencia de información se concentra en **2018 y en el primer semestre de 2019**. Estos días se consideran ausencia de información y no actividad aérea igual a cero, por lo que se conservarán como `NaN` durante la integración y no se realizará ninguna imputación.

La auditoría de las variables de actividad no detecta **valores ausentes, infinitos ni negativos**.

Asimismo, se verifica una coherencia matemática exacta entre los diferentes indicadores de actividad aérea:

- `Vuelos_total` coincide en los 2.011 registros con la suma de `Vuelos_Espanya`, `Vuelos_Europa`, `Vuelos_America`, `Vuelos_Asia` y `Vuelos_Africa`.
- `Vuelos_internacionales` coincide exactamente con la suma de Europa, América, Asia y África.
- `Vuelos_Espanya + Vuelos_internacionales` reproduce exactamente `Vuelos_total`.

No se detecta ninguna discrepancia en estas relaciones.

La evolución temporal refleja claramente la alteración de la actividad aérea durante la pandemia. La media diaria pasa de aproximadamente **670,4 vuelos/día en el periodo pre-COVID** a **300,4 vuelos/día durante el periodo COVID**, aumentando posteriormente hasta aproximadamente **699,0 vuelos/día en el periodo post-COVID**.

Por años, la media diaria evoluciona desde **254,7 vuelos/día en 2020** hasta **764,4 vuelos/día en 2024**, superándose en los últimos años los niveles observados en el segundo semestre de 2019.

En términos de composición, los vuelos nacionales representan de media aproximadamente el **39,60 %** de la actividad diaria, mientras que los vuelos internacionales representan el **60,40 %**.

El análisis de correlaciones muestra una elevada asociación entre varios indicadores de actividad. Destacan:

- `Vuelos_Europa` y `Vuelos_internacionales`: **r = 0,999**.
- `Vuelos_total` y `Vuelos_Europa`: **r = 0,994**.
- `Vuelos_total` y `Vuelos_internacionales`: **r = 0,993**.
- `Vuelos_total` y `Vuelos_Espanya`: **r = 0,954**.

Estas relaciones, junto con las identidades matemáticas verificadas, evidencian una fuerte redundancia entre varias de las variables disponibles. En consecuencia, no resulta conveniente incorporar simultáneamente el total y todos sus componentes como variables independientes del modelo.

> **Conclusión:** la serie aérea presenta una estructura temporal continua y consistente desde julio de 2019 hasta diciembre de 2024, sin anomalías internas ni incoherencias entre sus componentes. La principal limitación es la ausencia de información en 2018 y durante el primer semestre de 2019, que se conservará explícitamente como `NaN`. La elevada redundancia entre los indicadores disponibles deberá considerarse en la selección de variables de la etapa 6.3.

## 6.3. Selección y construcción de los indicadores de transporte aéreo

La auditoría realizada en la etapa anterior confirma que las variables de actividad aérea presentan una elevada redundancia estructural.

`Vuelos_total` se obtiene exactamente como suma de los componentes geográficos, mientras que `Vuelos_internacionales` corresponde a la suma de Europa, América, Asia y África. Además, varias de estas variables presentan correlaciones próximas a la unidad.

Por este motivo, no se incorporan simultáneamente al dataset maestro todos los componentes absolutos de actividad aérea.

Se selecciona `Vuelos_total` como **indicador principal de intensidad de tráfico aéreo**, al representar de forma directa y fácilmente interpretable el volumen diario de actividad.

Como variables complementarias se consideran:

- `Vuelos_internacionales_pct`: porcentaje diario de vuelos internacionales respecto al total, utilizado como indicador de composición de la actividad;
- `Companias_activas`: número diario de compañías con actividad;
- `Zonas_activas`: número de zonas geográficas con actividad aérea.

El porcentaje internacional se utiliza en lugar del número absoluto de vuelos internacionales para reducir la redundancia con `Vuelos_total` y representar una dimensión diferente: la composición geográfica de la actividad.

Las variables absolutas correspondientes a España, Europa, América, Asia y África se conservan en el dataset original para análisis exploratorio y visualización, pero no se seleccionan como indicadores principales de integración.

En esta etapa se comprueba:

- validez matemática del porcentaje internacional;
- rango de los indicadores;
- valores ausentes, infinitos y negativos;
- comportamiento por año;
- comportamiento durante los periodos pre-COVID, COVID y post-COVID;
- correlación entre los indicadores finalmente seleccionados.

No se realiza ninguna imputación temporal. La ausencia de información anterior al 1 de julio de 2019 continuará representándose explícitamente en las siguientes etapas.

> **Objetivo:** obtener un conjunto reducido, interpretable y no matemáticamente redundante de indicadores diarios de transporte aéreo para su posterior preintegración con el dataset maestro.

In [ ]:
# ============================================================
# 6.3. SELECCIÓN Y CONSTRUCCIÓN DE INDICADORES
# DE TRANSPORTE AÉREO
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 1. COMPROBAR DATASET PROCEDENTE DE 6.2
# ============================================================

if "df_vuelos_audit" not in globals():

    raise NameError(
        "No existe df_vuelos_audit. "
        "Debe ejecutarse previamente la etapa 6.2."
    )


df_vuelos_indicadores = (
    df_vuelos_audit
    .copy()
)


print("=" * 80)
print("6.3 — SELECCIÓN DE INDICADORES DE TRANSPORTE AÉREO")
print("=" * 80)


# ============================================================
# 2. VARIABLES NECESARIAS
# ============================================================

variables_necesarias = [
    "Fecha",
    "Vuelos_total",
    "Vuelos_internacionales",
    "Companias_activas",
    "Zonas_activas"
]


variables_faltantes = [
    variable
    for variable in variables_necesarias
    if variable not in df_vuelos_indicadores.columns
]


if variables_faltantes:

    raise KeyError(
        "Faltan variables necesarias:\n"
        f"{variables_faltantes}"
    )


# ============================================================
# 3. NORMALIZAR FECHA
# ============================================================

df_vuelos_indicadores[
    "Fecha"
] = pd.to_datetime(
    df_vuelos_indicadores[
        "Fecha"
    ],
    errors="coerce"
)


fechas_invalidas = (
    df_vuelos_indicadores[
        "Fecha"
    ]
    .isna()
    .sum()
)


print("\n" + "=" * 80)
print("CONTROL INICIAL")
print("=" * 80)

print(
    f"\n✓ Registros: "
    f"{len(df_vuelos_indicadores):,}"
)

print(
    f"✓ Fechas inválidas: "
    f"{fechas_invalidas:,}"
)


if fechas_invalidas > 0:

    raise ValueError(
        "Existen fechas inválidas."
    )


# ============================================================
# 4. CONTROL DE UNICIDAD
# ============================================================

duplicados_fecha = (
    df_vuelos_indicadores
    .duplicated(
        subset=["Fecha"]
    )
    .sum()
)


print(
    f"✓ Duplicados de Fecha: "
    f"{duplicados_fecha:,}"
)


if duplicados_fecha > 0:

    raise ValueError(
        "Existen fechas duplicadas."
    )


# ============================================================
# 5. CONSTRUIR PORCENTAJE DE VUELOS INTERNACIONALES
# ============================================================

df_vuelos_indicadores[
    "Vuelos_internacionales_pct"
] = np.where(
    df_vuelos_indicadores[
        "Vuelos_total"
    ] > 0,

    (
        df_vuelos_indicadores[
            "Vuelos_internacionales"
        ]
        /
        df_vuelos_indicadores[
            "Vuelos_total"
        ]
        *
        100
    ),

    np.nan
)


# ============================================================
# 6. SELECCIÓN FINAL DE INDICADORES
# ============================================================

variables_finales = [
    "Fecha",
    "Vuelos_total",
    "Vuelos_internacionales_pct",
    "Companias_activas",
    "Zonas_activas"
]


df_vuelos_modelo = (
    df_vuelos_indicadores[
        variables_finales
    ]
    .copy()
)


print("\n" + "=" * 80)
print("INDICADORES SELECCIONADOS")
print("=" * 80)

for variable in variables_finales[1:]:

    print(
        f"✓ {variable}"
    )


print(
    "\n✓ Indicador principal: Vuelos_total"
)


# ============================================================
# 7. CONTROL DEL PORCENTAJE INTERNACIONAL
# ============================================================

pct_min = (
    df_vuelos_modelo[
        "Vuelos_internacionales_pct"
    ]
    .min()
)

pct_max = (
    df_vuelos_modelo[
        "Vuelos_internacionales_pct"
    ]
    .max()
)

pct_media = (
    df_vuelos_modelo[
        "Vuelos_internacionales_pct"
    ]
    .mean()
)

pct_mediana = (
    df_vuelos_modelo[
        "Vuelos_internacionales_pct"
    ]
    .median()
)


fuera_rango = (
    (
        df_vuelos_modelo[
            "Vuelos_internacionales_pct"
        ] < 0
    )
    |
    (
        df_vuelos_modelo[
            "Vuelos_internacionales_pct"
        ] > 100
    )
).sum()


print("\n" + "=" * 80)
print("CONTROL DE VUELOS_INTERNACIONALES_PCT")
print("=" * 80)

print(
    f"\n✓ Mínimo: "
    f"{pct_min:.2f} %"
)

print(
    f"✓ Máximo: "
    f"{pct_max:.2f} %"
)

print(
    f"✓ Media: "
    f"{pct_media:.2f} %"
)

print(
    f"✓ Mediana: "
    f"{pct_mediana:.2f} %"
)

print(
    f"✓ Valores fuera del rango 0–100 %: "
    f"{fuera_rango:,}"
)


if fuera_rango > 0:

    raise ValueError(
        "Existen porcentajes internacionales "
        "fuera del rango 0–100 %."
    )


# ============================================================
# 8. CONTROL DE NaN
# ============================================================

tabla_nan = pd.DataFrame({
    "variable":
        variables_finales[1:],

    "n_nan":
        [
            df_vuelos_modelo[
                variable
            ].isna().sum()

            for variable
            in variables_finales[1:]
        ]
})


tabla_nan[
    "pct_nan"
] = (
    tabla_nan[
        "n_nan"
    ]
    /
    len(df_vuelos_modelo)
    *
    100
).round(2)


print("\n" + "=" * 80)
print("VALORES AUSENTES")
print("=" * 80)

display(
    tabla_nan
)


# ============================================================
# 9. CONTROL DE INFINITOS
# ============================================================

variables_numericas = (
    variables_finales[1:]
)


n_inf = (
    np.isinf(
        df_vuelos_modelo[
            variables_numericas
        ]
    )
    .sum()
    .sum()
)


print("\n" + "=" * 80)
print("VALORES INFINITOS")
print("=" * 80)

print(
    f"\n✓ Valores infinitos: "
    f"{n_inf:,}"
)


if n_inf > 0:

    raise ValueError(
        "Existen valores infinitos."
    )


# ============================================================
# 10. CONTROL DE VALORES NEGATIVOS
# ============================================================

negativos = []

for variable in variables_numericas:

    n_negativos = (
        df_vuelos_modelo[
            variable
        ] < 0
    ).sum()

    if n_negativos > 0:

        negativos.append({
            "variable":
                variable,

            "n_negativos":
                int(n_negativos),

            "minimo":
                df_vuelos_modelo[
                    variable
                ].min()
        })


df_negativos = pd.DataFrame(
    negativos
)


print("\n" + "=" * 80)
print("CONTROL DE VALORES NEGATIVOS")
print("=" * 80)

if df_negativos.empty:

    print(
        "\n✓ No se detectan valores negativos."
    )

else:

    display(
        df_negativos
    )


# ============================================================
# 11. ESTADÍSTICOS DE LOS INDICADORES
# ============================================================

resumen_indicadores = (
    df_vuelos_modelo[
        variables_numericas
    ]
    .describe()
    .T
)


print("\n" + "=" * 80)
print("ESTADÍSTICOS DE LOS INDICADORES SELECCIONADOS")
print("=" * 80)

display(
    resumen_indicadores
)


# ============================================================
# 12. CORRELACIÓN ENTRE INDICADORES SELECCIONADOS
# ============================================================

matriz_correlacion_final = (
    df_vuelos_modelo[
        variables_numericas
    ]
    .corr()
)


print("\n" + "=" * 80)
print("CORRELACIÓN ENTRE INDICADORES SELECCIONADOS")
print("=" * 80)

display(
    matriz_correlacion_final.round(3)
)


# ============================================================
# 13. IDENTIFICAR CORRELACIONES MUY ALTAS
# ============================================================

pares_altos = []

for i, variable_1 in enumerate(
    variables_numericas
):

    for variable_2 in variables_numericas[
        i + 1:
    ]:

        correlacion = (
            matriz_correlacion_final
            .loc[
                variable_1,
                variable_2
            ]
        )

        if abs(correlacion) >= 0.95:

            pares_altos.append({
                "variable_1":
                    variable_1,

                "variable_2":
                    variable_2,

                "correlacion":
                    correlacion
            })


df_correlaciones_finales_altas = pd.DataFrame(
    pares_altos
)


print("\n" + "=" * 80)
print("PARES SELECCIONADOS CON |CORRELACIÓN| ≥ 0.95")
print("=" * 80)

if df_correlaciones_finales_altas.empty:

    print(
        "\n✓ No existen pares seleccionados "
        "con |correlación| ≥ 0.95."
    )

else:

    display(
        df_correlaciones_finales_altas
    )


# ============================================================
# 14. RESUMEN POR AÑO
# ============================================================

df_vuelos_modelo[
    "_anio_control"
] = (
    df_vuelos_modelo[
        "Fecha"
    ]
    .dt.year
)


resumen_anual_indicadores = (
    df_vuelos_modelo
    .groupby(
        "_anio_control"
    )
    .agg(
        dias=(
            "Fecha",
            "nunique"
        ),

        vuelos_media=(
            "Vuelos_total",
            "mean"
        ),

        vuelos_mediana=(
            "Vuelos_total",
            "median"
        ),

        pct_internacional_media=(
            "Vuelos_internacionales_pct",
            "mean"
        ),

        companias_media=(
            "Companias_activas",
            "mean"
        ),

        zonas_media=(
            "Zonas_activas",
            "mean"
        )
    )
    .reset_index()
    .rename(
        columns={
            "_anio_control":
                "anio"
        }
    )
)


print("\n" + "=" * 80)
print("INDICADORES SELECCIONADOS POR AÑO")
print("=" * 80)

display(
    resumen_anual_indicadores
)


# ============================================================
# 15. CONTROL DE VARIABILIDAD
# ============================================================

variabilidad = []

for variable in variables_numericas:

    media = (
        df_vuelos_modelo[
            variable
        ].mean()
    )

    desviacion = (
        df_vuelos_modelo[
            variable
        ].std()
    )

    if media != 0:

        cv = (
            desviacion
            /
            abs(media)
            *
            100
        )

    else:

        cv = np.nan


    variabilidad.append({
        "variable":
            variable,

        "media":
            media,

        "desviacion_std":
            desviacion,

        "coef_variacion_pct":
            cv
    })


df_variabilidad = pd.DataFrame(
    variabilidad
)


print("\n" + "=" * 80)
print("VARIABILIDAD DE LOS INDICADORES")
print("=" * 80)

display(
    df_variabilidad.round(3)
)


# ============================================================
# 16. RESTAURAR DATASET FINAL
# ============================================================

df_vuelos_modelo = (
    df_vuelos_modelo
    .drop(
        columns=[
            "_anio_control"
        ]
    )
)


df_vuelos_modelo = (
    df_vuelos_modelo
    .sort_values(
        "Fecha"
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 17. CONTROL FINAL
# ============================================================

print("\n" + "=" * 80)
print("SELECCIÓN DE INDICADORES FINALIZADA")
print("=" * 80)

print(
    f"\n✓ Registros conservados: "
    f"{len(df_vuelos_modelo):,}"
)

print(
    f"✓ Variables finales: "
    f"{df_vuelos_modelo.shape[1]}"
)

print(
    f"✓ Fechas diferentes: "
    f"{df_vuelos_modelo['Fecha'].nunique():,}"
)

print(
    f"✓ Duplicados de Fecha: "
    f"{df_vuelos_modelo.duplicated(subset=['Fecha']).sum():,}"
)

print(
    "✓ Indicador principal: "
    "Vuelos_total"
)

print(
    "✓ Indicador de composición: "
    "Vuelos_internacionales_pct"
)

print(
    "✓ Indicadores complementarios: "
    "Companias_activas y Zonas_activas"
)

print(
    "✓ No se ha realizado ninguna imputación."
)

print(
    "✓ Dataset preparado para "
    "preintegración en 6.4."
)

6.3 — SELECCIÓN DE INDICADORES DE TRANSPORTE AÉREO

CONTROL INICIAL

✓ Registros: 2,011
✓ Fechas inválidas: 0
✓ Duplicados de Fecha: 0

INDICADORES SELECCIONADOS
✓ Vuelos_total
✓ Vuelos_internacionales_pct
✓ Companias_activas
✓ Zonas_activas

✓ Indicador principal: Vuelos_total

CONTROL DE VUELOS_INTERNACIONALES_PCT

✓ Mínimo: 7.69 %
✓ Máximo: 73.93 %
✓ Media: 60.40 %
✓ Mediana: 65.40 %
✓ Valores fuera del rango 0–100 %: 0

VALORES AUSENTES


,variable,n_nan,pct_nan
0,Vuelos_total,0,0.0
1,Vuelos_internacionales_pct,0,0.0
2,Companias_activas,0,0.0
3,Zonas_activas,0,0.0



VALORES INFINITOS

✓ Valores infinitos: 0

CONTROL DE VALORES NEGATIVOS

✓ No se detectan valores negativos.

ESTADÍSTICOS DE LOS INDICADORES SELECCIONADOS


,count,mean,std,min,25%,50%,75%,max
Vuelos_total,2011.0,551.496271,244.609170,11.000000,406.500000,635.000000,741.000000,924.00000
Vuelos_internacionales_pct,2011.0,60.401209,11.385094,7.692308,56.638767,65.399738,68.064536,73.92638
Companias_activas,2011.0,20.393834,4.887087,3.000000,19.000000,22.000000,24.000000,25.00000
Zonas_activas,2011.0,4.663352,0.726571,2.000000,5.000000,5.000000,5.000000,5.00000



CORRELACIÓN ENTRE INDICADORES SELECCIONADOS


...


PARES SELECCIONADOS CON |CORRELACIÓN| ≥ 0.95

✓ No existen pares seleccionados con |correlación| ≥ 0.95.

INDICADORES SELECCIONADOS POR AÑO


,anio,dias,vuelos_media,vuelos_mediana,pct_internacional_media,companias_media,zonas_media
0,2019,184,670.423913,693.5,66.421680,19.836957,5.000000
1,2020,366,254.677596,212.0,48.503995,14.155738,3.617486
2,2021,365,346.200000,389.0,49.397335,17.679452,4.553425
3,2022,365,624.591781,671.0,65.381751,22.134247,4.978082
4,2023,365,707.841096,731.0,66.851446,23.887671,5.000000
5,2024,366,764.448087,783.0,68.846003,24.398907,5.000000



VARIABILIDAD DE LOS INDICADORES


,variable,media,desviacion_std,coef_variacion_pct
0,Vuelos_total,551.496,244.609,44.354
1,Vuelos_internacionales_pct,60.401,11.385,18.849
2,Companias_activas,20.394,4.887,23.964
3,Zonas_activas,4.663,0.727,15.580



SELECCIÓN DE INDICADORES FINALIZADA

✓ Registros conservados: 2,011
✓ Variables finales: 5
✓ Fechas diferentes: 2,011
✓ Duplicados de Fecha: 0
✓ Indicador principal: Vuelos_total
✓ Indicador de composición: Vuelos_internacionales_pct
✓ Indicadores complementarios: Companias_activas y Zonas_activas
✓ No se ha realizado ninguna imputación.
✓ Dataset preparado para preintegración en 6.4.


### Resultados — 6.3. Selección y construcción de los indicadores de transporte aéreo

A partir de la auditoría de redundancia realizada previamente se selecciona un conjunto reducido de indicadores destinados a representar diferentes dimensiones de la actividad aérea diaria.

Las variables finalmente seleccionadas son:

- `Vuelos_total`, como indicador principal de intensidad de tráfico aéreo;
- `Vuelos_internacionales_pct`, como indicador de composición internacional de la actividad;
- `Companias_activas`, como indicador complementario de diversidad operativa;
- `Zonas_activas`, como indicador de amplitud geográfica de la actividad.

El indicador `Vuelos_internacionales_pct` se construye como el porcentaje de vuelos internacionales respecto al número total de vuelos diarios. Presenta valores comprendidos entre **7,69 % y 73,93 %**, con una media del **60,40 %** y una mediana del **65,40 %**. No se detectan valores fuera del intervalo teórico 0–100 %.

Los cuatro indicadores presentan una estructura completa dentro del periodo observado: no existen valores ausentes, infinitos ni negativos.

`Vuelos_total` presenta la mayor variabilidad relativa, con un coeficiente de variación del **44,35 %**, seguido de `Companias_activas` (**23,96 %**), `Vuelos_internacionales_pct` (**18,85 %**) y `Zonas_activas` (**15,58 %**).

La matriz de correlaciones confirma que la selección realizada reduce la redundancia existente en el dataset original. Ningún par de indicadores presenta una correlación absoluta igual o superior a 0,95.

Las asociaciones más elevadas corresponden a:

- `Vuelos_total` y `Companias_activas`: **r = 0,896**;
- `Vuelos_total` y `Vuelos_internacionales_pct`: **r = 0,894**;
- `Vuelos_internacionales_pct` y `Companias_activas`: **r = 0,867**.

Aunque estas asociaciones reflejan la relación existente entre las diferentes dimensiones de la actividad aeroportuaria, no constituyen identidades matemáticas como las detectadas entre los indicadores originales.

La evolución anual conserva claramente la alteración temporal asociada a la pandemia y la posterior recuperación de la actividad. La media diaria de `Vuelos_total` pasa de aproximadamente **254,7 vuelos/día en 2020** a **764,4 vuelos/día en 2024**. Paralelamente, el porcentaje medio de vuelos internacionales aumenta desde aproximadamente **48,50 % en 2020** hasta **68,85 % en 2024**.

El dataset resultante contiene **2.011 registros y 5 variables**, correspondientes a `Fecha` y los cuatro indicadores seleccionados. Se mantienen **2.011 fechas diferentes y ningún duplicado temporal**.

No se realiza ninguna imputación y se mantiene la ausencia de información anterior al inicio de la serie.

> **Conclusión:** se obtiene un bloque compacto e interpretable de cuatro indicadores de transporte aéreo, eliminando la redundancia matemática existente entre los componentes originales y conservando diferentes dimensiones de intensidad, composición y diversidad de la actividad. El dataset queda preparado para su preintegración temporal en la etapa 6.4.

## 6.4. Preintegración temporal y exportación del bloque de transporte aéreo

Una vez seleccionados los indicadores de transporte aéreo, se adapta la serie diaria al periodo completo del estudio, comprendido entre **2018 y 2024**.

El dataset original dispone de información continua desde el **1 de julio de 2019 hasta el 31 de diciembre de 2024**. Para garantizar una estructura temporal homogénea con el resto de bloques del proyecto se construye un calendario diario completo entre el 1 de enero de 2018 y el 31 de diciembre de 2024.

Las fechas anteriores al inicio real de la serie se mantienen explícitamente como valores ausentes (`NaN`). No se interpreta la ausencia de datos como ausencia de vuelos ni se aplica ninguna imputación.

Se conservan los cuatro indicadores seleccionados:

- `Vuelos_total`
- `Vuelos_internacionales_pct`
- `Companias_activas`
- `Zonas_activas`

Durante esta etapa se verifica:

- presencia de las 2.557 fechas del periodo 2018–2024;
- unicidad de la fecha;
- mantenimiento exacto de los valores observados desde julio de 2019;
- conservación de los 546 días sin información previa al inicio de la serie;
- ausencia de valores infinitos;
- coherencia temporal tras la expansión del calendario;
- integridad del archivo después de su exportación y recarga.

El resultado se almacena como dataset aéreo preintegrado y queda preparado para su posterior incorporación al dataset maestro.

> **Objetivo:** disponer de una serie diaria de transporte aéreo alineada con el periodo temporal global del proyecto, conservando explícitamente las ausencias de información y sin introducir imputaciones artificiales.

In [ ]:
# ============================================================
# 6.4. PREINTEGRACIÓN TEMPORAL Y EXPORTACIÓN
# DEL BLOQUE DE TRANSPORTE AÉREO
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. COMPROBAR DATASET PROCEDENTE DE 6.3
# ============================================================

if "df_vuelos_modelo" not in globals():
    raise NameError(
        "No existe df_vuelos_modelo. "
        "Debe ejecutarse previamente la etapa 6.3."
    )


df_vuelos_pre = (
    df_vuelos_modelo
    .copy()
)


# ============================================================
# 2. NORMALIZAR FECHA
# ============================================================

df_vuelos_pre[
    "Fecha"
] = pd.to_datetime(
    df_vuelos_pre[
        "Fecha"
    ],
    errors="coerce"
)

fechas_invalidas = (
    df_vuelos_pre[
        "Fecha"
    ]
    .isna()
    .sum()
)

if fechas_invalidas > 0:
    raise ValueError(
        "Existen fechas inválidas."
    )


# ============================================================
# 3. CONTROL DE UNICIDAD
# ============================================================

duplicados_fecha = (
    df_vuelos_pre
    .duplicated(
        subset=["Fecha"]
    )
    .sum()
)

if duplicados_fecha > 0:
    raise ValueError(
        "Existen fechas duplicadas."
    )


# ============================================================
# 4. CREAR CALENDARIO COMPLETO 2018–2024
# ============================================================

fecha_inicio_objetivo = pd.Timestamp(
    "2018-01-01"
)

fecha_fin_objetivo = pd.Timestamp(
    "2024-12-31"
)

calendario = pd.DataFrame({
    "Fecha": pd.date_range(
        start=fecha_inicio_objetivo,
        end=fecha_fin_objetivo,
        freq="D"
    )
})


# ============================================================
# 5. INTEGRAR SERIE OBSERVADA EN EL CALENDARIO
# ============================================================

df_vuelos_preintegrado = pd.merge(
    calendario,
    df_vuelos_pre,
    on="Fecha",
    how="left",
    validate="one_to_one"
)


# ============================================================
# 6. CONTROL DE DIMENSIONES
# ============================================================

print("=" * 80)
print("6.4 — PREINTEGRACIÓN TEMPORAL DEL TRANSPORTE AÉREO")
print("=" * 80)

print(
    f"\n✓ Días del calendario completo: "
    f"{len(calendario):,}"
)

print(
    f"✓ Registros observados originales: "
    f"{len(df_vuelos_pre):,}"
)

print(
    f"✓ Registros preintegrados: "
    f"{len(df_vuelos_preintegrado):,}"
)

print(
    f"✓ Variables disponibles: "
    f"{df_vuelos_preintegrado.shape[1]}"
)


# ============================================================
# 7. CONTROL DE DUPLICADOS
# ============================================================

duplicados_preintegrado = (
    df_vuelos_preintegrado
    .duplicated(
        subset=["Fecha"]
    )
    .sum()
)

print(
    f"✓ Duplicados de Fecha: "
    f"{duplicados_preintegrado:,}"
)

if duplicados_preintegrado > 0:
    raise ValueError(
        "La preintegración ha generado fechas duplicadas."
    )


# ============================================================
# 8. VARIABLES DE ACTIVIDAD
# ============================================================

variables_actividad = [
    "Vuelos_total",
    "Vuelos_internacionales_pct",
    "Companias_activas",
    "Zonas_activas"
]


# ============================================================
# 9. CONTROL DE COBERTURA
# ============================================================

print("\n" + "=" * 80)
print("COBERTURA TEMPORAL DE LOS INDICADORES")
print("=" * 80)

resumen_cobertura = []

for variable in variables_actividad:

    n_validos = (
        df_vuelos_preintegrado[
            variable
        ]
        .notna()
        .sum()
    )

    n_nan = (
        df_vuelos_preintegrado[
            variable
        ]
        .isna()
        .sum()
    )

    cobertura = (
        n_validos
        /
        len(df_vuelos_preintegrado)
        *
        100
    )

    resumen_cobertura.append({
        "variable":
            variable,

        "registros_validos":
            n_validos,

        "registros_nan":
            n_nan,

        "cobertura_pct":
            round(
                cobertura,
                2
            )
    })


df_cobertura_vuelos_pre = pd.DataFrame(
    resumen_cobertura
)

display(
    df_cobertura_vuelos_pre
)


# ============================================================
# 10. COMPROBAR QUE LOS 546 NaN SON LOS ESPERADOS
# ============================================================

n_nan_total = (
    df_vuelos_preintegrado[
        "Vuelos_total"
    ]
    .isna()
    .sum()
)

print("\n" + "=" * 80)
print("CONTROL DE AUSENCIA PREVIA AL INICIO DE LA SERIE")
print("=" * 80)

print(
    f"\n✓ Días sin Vuelos_total: "
    f"{n_nan_total:,}"
)

if n_nan_total != 546:
    print(
        "⚠ El número de días sin datos "
        "no coincide con los 546 esperados."
    )


# ============================================================
# 11. IDENTIFICAR RANGO DE FECHAS SIN DATOS
# ============================================================

fechas_sin_vuelos = (
    df_vuelos_preintegrado.loc[
        df_vuelos_preintegrado[
            "Vuelos_total"
        ].isna(),
        "Fecha"
    ]
)


if len(fechas_sin_vuelos) > 0:

    print(
        f"✓ Primera fecha sin datos: "
        f"{fechas_sin_vuelos.min().date()}"
    )

    print(
        f"✓ Última fecha sin datos: "
        f"{fechas_sin_vuelos.max().date()}"
    )


# ============================================================
# 12. COMPROBAR CONTINUIDAD DESDE 01/07/2019
# ============================================================

periodo_observado = (
    df_vuelos_preintegrado[
        df_vuelos_preintegrado[
            "Fecha"
        ] >= pd.Timestamp(
            "2019-07-01"
        )
    ]
)

huecos_desde_inicio = (
    periodo_observado[
        "Vuelos_total"
    ]
    .isna()
    .sum()
)

print(
    f"✓ Huecos desde 01/07/2019: "
    f"{huecos_desde_inicio:,}"
)

if huecos_desde_inicio > 0:
    raise ValueError(
        "Existen huecos dentro del periodo observado."
    )


# ============================================================
# 13. COBERTURA POR AÑO
# ============================================================

df_vuelos_preintegrado[
    "_anio_control"
] = (
    df_vuelos_preintegrado[
        "Fecha"
    ]
    .dt.year
)

cobertura_anual = (
    df_vuelos_preintegrado
    .groupby(
        "_anio_control"
    )
    .agg(
        dias=(
            "Fecha",
            "size"
        ),

        dias_con_vuelos=(
            "Vuelos_total",
            "count"
        )
    )
    .reset_index()
    .rename(
        columns={
            "_anio_control":
                "anio"
        }
    )
)

cobertura_anual[
    "dias_sin_vuelos"
] = (
    cobertura_anual[
        "dias"
    ]
    -
    cobertura_anual[
        "dias_con_vuelos"
    ]
)

cobertura_anual[
    "cobertura_pct"
] = (
    cobertura_anual[
        "dias_con_vuelos"
    ]
    /
    cobertura_anual[
        "dias"
    ]
    *
    100
).round(2)


print("\n" + "=" * 80)
print("COBERTURA POR AÑO")
print("=" * 80)

display(
    cobertura_anual
)


# ============================================================
# 14. ELIMINAR VARIABLE AUXILIAR
# ============================================================

df_vuelos_preintegrado = (
    df_vuelos_preintegrado
    .drop(
        columns=[
            "_anio_control"
        ]
    )
)


# ============================================================
# 15. CONTROL DE VALORES INFINITOS
# ============================================================

n_inf = (
    np.isinf(
        df_vuelos_preintegrado[
            variables_actividad
        ]
    )
    .sum()
    .sum()
)

print("\n" + "=" * 80)
print("CONTROL DE VALORES INFINITOS")
print("=" * 80)

print(
    f"\n✓ Valores infinitos: "
    f"{n_inf:,}"
)

if n_inf > 0:
    raise ValueError(
        "Existen valores infinitos."
    )


# ============================================================
# 16. CONTROL DE LOS VALORES OBSERVADOS
# ============================================================

print("\n" + "=" * 80)
print("CONTROL DE RANGOS")
print("=" * 80)

for variable in variables_actividad:

    print(
        f"\n{variable}: "
        f"{df_vuelos_preintegrado[variable].min():.2f} "
        f"→ "
        f"{df_vuelos_preintegrado[variable].max():.2f}"
    )


# ============================================================
# 17. RUTAS DE EXPORTACIÓN
# ============================================================

ruta_salida = (
    Path(
        "/content/drive/MyDrive/TFM/"
        "11_Machine_Learning_Temporal/"
        "PREINTEGRACION/05_Transporte_Aereo"
    )
)

ruta_salida.mkdir(
    parents=True,
    exist_ok=True
)

ruta_archivo = (
    ruta_salida /
    "df_vuelos_preintegrado.csv"
)


# ============================================================
# 18. EXPORTAR
# ============================================================

df_vuelos_preintegrado.to_csv(
    ruta_archivo,
    index=False,
    encoding="utf-8"
)


# ============================================================
# 19. RECARGAR Y VALIDAR
# ============================================================

df_control_vuelos_pre = pd.read_csv(
    ruta_archivo,
    parse_dates=[
        "Fecha"
    ],
    low_memory=False
)


filas_recargadas = len(
    df_control_vuelos_pre
)

columnas_recargadas = (
    df_control_vuelos_pre.shape[1]
)

duplicados_recarga = (
    df_control_vuelos_pre
    .duplicated(
        subset=[
            "Fecha"
        ]
    )
    .sum()
)

nan_vuelos_recarga = (
    df_control_vuelos_pre[
        "Vuelos_total"
    ]
    .isna()
    .sum()
)


print("\n" + "=" * 80)
print("CONTROL POSTERIOR A LA EXPORTACIÓN")
print("=" * 80)

print(
    f"\n✓ Filas exportadas: "
    f"{len(df_vuelos_preintegrado):,}"
)

print(
    f"✓ Filas recargadas: "
    f"{filas_recargadas:,}"
)

print(
    f"✓ Columnas exportadas: "
    f"{df_vuelos_preintegrado.shape[1]}"
)

print(
    f"✓ Columnas recargadas: "
    f"{columnas_recargadas}"
)

print(
    f"✓ Duplicados tras recarga: "
    f"{duplicados_recarga:,}"
)

print(
    f"✓ NaN Vuelos_total antes: "
    f"{n_nan_total:,}"
)

print(
    f"✓ NaN Vuelos_total después: "
    f"{nan_vuelos_recarga:,}"
)


# ============================================================
# 20. CONTROLES ESTRICTOS DE INTEGRIDAD
# ============================================================

if filas_recargadas != len(
    df_vuelos_preintegrado
):
    raise ValueError(
        "El número de filas ha cambiado "
        "durante la exportación."
    )

if columnas_recargadas != (
    df_vuelos_preintegrado.shape[1]
):
    raise ValueError(
        "El número de columnas ha cambiado "
        "durante la exportación."
    )

if duplicados_recarga > 0:
    raise ValueError(
        "La exportación ha generado duplicados."
    )

if nan_vuelos_recarga != n_nan_total:
    raise ValueError(
        "La exportación ha alterado "
        "los NaN de Vuelos_total."
    )


# ============================================================
# 21. TAMAÑO DEL ARCHIVO
# ============================================================

tamano_mb = (
    ruta_archivo.stat().st_size
    /
    (1024 ** 2)
)


# ============================================================
# 22. RESUMEN FINAL
# ============================================================

print("\n" + "=" * 80)
print("PREINTEGRACIÓN DE TRANSPORTE AÉREO FINALIZADA")
print("=" * 80)

print(
    f"\n✓ Registros: "
    f"{len(df_control_vuelos_pre):,}"
)

print(
    f"✓ Variables: "
    f"{df_control_vuelos_pre.shape[1]}"
)

print(
    f"✓ Fecha inicial: "
    f"{df_control_vuelos_pre['Fecha'].min().date()}"
)

print(
    f"✓ Fecha final: "
    f"{df_control_vuelos_pre['Fecha'].max().date()}"
)

print(
    f"✓ Días con datos: "
    f"{df_control_vuelos_pre['Vuelos_total'].notna().sum():,}"
)

print(
    f"✓ Días sin datos: "
    f"{df_control_vuelos_pre['Vuelos_total'].isna().sum():,}"
)

print(
    f"✓ Cobertura global: "
    f"{df_control_vuelos_pre['Vuelos_total'].notna().mean() * 100:.2f} %"
)

print(
    f"✓ Duplicados de Fecha: "
    f"{duplicados_recarga:,}"
)

print(
    f"✓ Valores infinitos: "
    f"{n_inf:,}"
)

print(
    f"✓ Tamaño del archivo: "
    f"{tamano_mb:.2f} MB"
)

print(
    "✓ Los días anteriores al 01/07/2019 "
    "permanecen como NaN."
)

print(
    "✓ No se ha realizado ninguna imputación."
)

print(
    "✓ Integridad verificada tras la exportación."
)

print(
    "\n✓ Archivo generado:"
)

print(
    ruta_archivo
)

6.4 — PREINTEGRACIÓN TEMPORAL DEL TRANSPORTE AÉREO

✓ Días del calendario completo: 2,557
✓ Registros observados originales: 2,011
✓ Registros preintegrados: 2,557
✓ Variables disponibles: 5
✓ Duplicados de Fecha: 0

COBERTURA TEMPORAL DE LOS INDICADORES


,variable,registros_validos,registros_nan,cobertura_pct
0,Vuelos_total,2011,546,78.65
1,Vuelos_internacionales_pct,2011,546,78.65
2,Companias_activas,2011,546,78.65
3,Zonas_activas,2011,546,78.65



CONTROL DE AUSENCIA PREVIA AL INICIO DE LA SERIE

✓ Días sin Vuelos_total: 546
✓ Primera fecha sin datos: 2018-01-01
✓ Última fecha sin datos: 2019-06-30
✓ Huecos desde 01/07/2019: 0

COBERTURA POR AÑO


,anio,dias,dias_con_vuelos,dias_sin_vuelos,cobertura_pct
0,2018,365,0,365,0.00
1,2019,365,184,181,50.41
2,2020,366,366,0,100.00
3,2021,365,365,0,100.00
4,2022,365,365,0,100.00
5,2023,365,365,0,100.00
6,2024,366,366,0,100.00



CONTROL DE VALORES INFINITOS

✓ Valores infinitos: 0

CONTROL DE RANGOS

Vuelos_total: 11.00 → 924.00

Vuelos_internacionales_pct: 7.69 → 73.93

Companias_activas: 3.00 → 25.00

Zonas_activas: 2.00 → 5.00

CONTROL POSTERIOR A LA EXPORTACIÓN

✓ Filas exportadas: 2,557
✓ Filas recargadas: 2,557
✓ Columnas exportadas: 5
✓ Columnas recargadas: 5
✓ Duplicados tras recarga: 0
✓ NaN Vuelos_total antes: 546
✓ NaN Vuelos_total después: 546

PREINTEGRACIÓN DE TRANSPORTE AÉREO FINALIZADA

✓ Registros: 2,557
✓ Variables: 5
✓ Fecha inicial: 2018-01-01
✓ Fecha final: 2024-12-31
✓ Días con datos: 2,011
✓ Días sin datos: 546
✓ Cobertura global: 78.65 %
✓ Duplicados de Fecha: 0
✓ Valores infinitos: 0
✓ Tamaño del archivo: 0.09 MB
✓ Los días anteriores al 01/07/2019 permanecen como NaN.
✓ No se ha realizado ninguna imputación.
✓ Integridad verificada tras la exportación.

✓ Archivo generado:
/content/drive/MyDrive/TFM/11_Machine_Learning_Temporal/PREINTEGRACION/05_Transporte_Aereo/df_vuelos_preintegrad

### Resultados — 6.4. Preintegración temporal y exportación del bloque de transporte aéreo

Los indicadores seleccionados de transporte aéreo se adaptan al calendario completo del estudio, comprendido entre el **1 de enero de 2018 y el 31 de diciembre de 2024**.

El dataset preintegrado resultante contiene **2.557 registros diarios y 5 variables**, correspondientes a la fecha y los cuatro indicadores seleccionados:

- `Vuelos_total`
- `Vuelos_internacionales_pct`
- `Companias_activas`
- `Zonas_activas`

La serie observada aporta información para **2.011 días**, mientras que **546 días permanecen sin datos**, lo que representa una cobertura temporal global del **78,65 %**.

Los valores ausentes se concentran exclusivamente entre el **1 de enero de 2018 y el 30 de junio de 2019**. A partir del **1 de julio de 2019** la serie es completamente continua y no presenta ningún hueco interno.

La cobertura anual es del **0 % en 2018**, del **50,41 % en 2019** y del **100 % entre 2020 y 2024**.

No se detectan valores infinitos ni duplicados temporales. Los rangos de los indicadores observados permanecen coherentes con el dataset original:

- `Vuelos_total`: 11–924 vuelos/día;
- `Vuelos_internacionales_pct`: 7,69–73,93 %;
- `Companias_activas`: 3–25;
- `Zonas_activas`: 2–5.

No se realiza ninguna imputación. Los días anteriores al inicio de la serie se mantienen como `NaN`, diferenciando explícitamente ausencia de información de actividad aérea igual a cero.

La exportación del dataset se valida mediante su recarga, conservándose exactamente las **2.557 filas, 5 variables, 0 duplicados y 546 valores ausentes en `Vuelos_total`**.

> **Conclusión:** el bloque aéreo queda temporalmente alineado con el periodo 2018–2024 y preparado para su incorporación al dataset maestro, preservando de forma explícita la ausencia de información previa al 1 de julio de 2019.

## 6.5. Integración del transporte aéreo y generación del Checkpoint 05

Una vez preparado el dataset aéreo diario para el periodo completo 2018–2024, se incorporan los indicadores seleccionados al dataset maestro correspondiente al Checkpoint 04.

La integración se realiza mediante la variable temporal `fecha`.

El bloque aéreo presenta una única observación diaria para el conjunto del ámbito aeroportuario considerado. Por tanto, los indicadores de actividad aérea correspondientes a una fecha determinada se replican sobre las estaciones físicas y contaminantes presentes en el maestro para ese mismo día.

Se incorporan cuatro variables:

- `Vuelos_total`
- `Vuelos_internacionales_pct`
- `Companias_activas`
- `Zonas_activas`

Los valores anteriores al **1 de julio de 2019** permanecen como `NaN`, preservando explícitamente la ausencia de información aérea durante ese periodo.

Durante la integración se comprueba:

- unicidad de la fecha en el dataset aéreo;
- correspondencia temporal entre ambas fuentes;
- conservación exacta del número de registros del maestro;
- ausencia de nuevos duplicados en la clave principal;
- cobertura de los indicadores aéreos;
- distribución de valores ausentes por año;
- ausencia de valores infinitos;
- integridad del archivo tras su exportación y recarga.

El resultado se almacena como **Checkpoint 05**, manteniendo inalterados los checkpoints anteriores.

> **Objetivo:** incorporar la actividad aérea diaria al dataset maestro sin alterar su estructura y conservando explícitamente la limitación temporal asociada a la ausencia de datos anteriores al 1 de julio de 2019.

In [ ]:
# ============================================================
# 6.5. INTEGRACIÓN DEL TRANSPORTE AÉREO
# CHECKPOINT 05
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. RUTAS
# ============================================================

ruta_base = Path(
    "/content/drive/MyDrive/TFM/"
    "11_Machine_Learning_Temporal"
)

ruta_integracion = (
    ruta_base /
    "INTEGRACION"
)

ruta_maestro_04 = (
    ruta_integracion /
    "04_maestro_contaminacion_meteorologia_trafico_ruido.csv"
)

ruta_vuelos_pre = (
    ruta_base /
    "PREINTEGRACION" /
    "05_Transporte_Aereo" /
    "df_vuelos_preintegrado.csv"
)

ruta_maestro_05 = (
    ruta_integracion /
    "05_maestro_contaminacion_meteorologia_trafico_ruido_vuelos.csv"
)


# ============================================================
# 2. COMPROBAR EXISTENCIA DE ARCHIVOS
# ============================================================

if not ruta_maestro_04.exists():

    raise FileNotFoundError(
        "No se encuentra el Checkpoint 04:\n"
        f"{ruta_maestro_04}"
    )


if not ruta_vuelos_pre.exists():

    raise FileNotFoundError(
        "No se encuentra el dataset aéreo preintegrado:\n"
        f"{ruta_vuelos_pre}"
    )


# ============================================================
# 3. CARGAR ARCHIVOS
# ============================================================

df_maestro_04 = pd.read_csv(
    ruta_maestro_04,
    low_memory=False
)

df_vuelos_pre = pd.read_csv(
    ruta_vuelos_pre,
    low_memory=False
)


# ============================================================
# 4. NORMALIZAR FECHAS
# ============================================================

df_maestro_04[
    "fecha"
] = pd.to_datetime(
    df_maestro_04[
        "fecha"
    ],
    errors="coerce"
)


df_vuelos_pre[
    "Fecha"
] = pd.to_datetime(
    df_vuelos_pre[
        "Fecha"
    ],
    errors="coerce"
)


# Homogeneizar nombre temporal
df_vuelos_pre = (
    df_vuelos_pre
    .rename(
        columns={
            "Fecha": "fecha"
        }
    )
)


# ============================================================
# 5. CONTROL INICIAL
# ============================================================

print("=" * 80)
print("6.5 — INTEGRACIÓN DEL TRANSPORTE AÉREO — CHECKPOINT 05")
print("=" * 80)

print("\nCHECKPOINT 04")

print(
    f"✓ Filas: "
    f"{len(df_maestro_04):,}"
)

print(
    f"✓ Columnas: "
    f"{df_maestro_04.shape[1]}"
)


print("\nDATASET AÉREO PREINTEGRADO")

print(
    f"✓ Filas: "
    f"{len(df_vuelos_pre):,}"
)

print(
    f"✓ Columnas: "
    f"{df_vuelos_pre.shape[1]}"
)

print(
    f"✓ Fechas diferentes: "
    f"{df_vuelos_pre['fecha'].nunique():,}"
)


# ============================================================
# 6. CONTROL DE FECHAS INVÁLIDAS
# ============================================================

fechas_invalidas_maestro = (
    df_maestro_04[
        "fecha"
    ]
    .isna()
    .sum()
)

fechas_invalidas_vuelos = (
    df_vuelos_pre[
        "fecha"
    ]
    .isna()
    .sum()
)


print("\n" + "=" * 80)
print("CONTROL DE FECHAS")
print("=" * 80)

print(
    f"\n✓ Fechas inválidas en maestro: "
    f"{fechas_invalidas_maestro:,}"
)

print(
    f"✓ Fechas inválidas en vuelos: "
    f"{fechas_invalidas_vuelos:,}"
)


if (
    fechas_invalidas_maestro > 0
    or
    fechas_invalidas_vuelos > 0
):

    raise ValueError(
        "Existen fechas inválidas antes de la integración."
    )


# ============================================================
# 7. UNICIDAD DEL DATASET AÉREO
# ============================================================

duplicados_vuelos = (
    df_vuelos_pre
    .duplicated(
        subset=["fecha"]
    )
    .sum()
)


print("\n" + "=" * 80)
print("UNICIDAD DEL DATASET AÉREO")
print("=" * 80)

print(
    f"\n✓ Duplicados de fecha: "
    f"{duplicados_vuelos:,}"
)


if duplicados_vuelos > 0:

    raise ValueError(
        "El dataset aéreo presenta fechas duplicadas."
    )


# ============================================================
# 8. CONTROL DEL MAESTRO ANTES DEL MERGE
# ============================================================

clave_maestro = [
    "fecha",
    "estacion_fisica",
    "contaminant"
]


faltantes_clave_maestro = [
    columna
    for columna in clave_maestro
    if columna not in df_maestro_04.columns
]


if faltantes_clave_maestro:

    raise KeyError(
        "Faltan columnas de la clave principal del maestro:\n"
        f"{faltantes_clave_maestro}"
    )


duplicados_maestro_antes = (
    df_maestro_04
    .duplicated(
        subset=clave_maestro
    )
    .sum()
)

filas_antes = len(
    df_maestro_04
)

columnas_antes = (
    df_maestro_04.shape[1]
)


print("\n" + "=" * 80)
print("CONTROL DEL MAESTRO ANTES DE INTEGRAR")
print("=" * 80)

print(
    f"\n✓ Filas: "
    f"{filas_antes:,}"
)

print(
    f"✓ Columnas: "
    f"{columnas_antes}"
)

print(
    f"✓ Duplicados clave principal: "
    f"{duplicados_maestro_antes:,}"
)


if duplicados_maestro_antes > 0:

    raise ValueError(
        "El Checkpoint 04 presenta duplicados."
    )


# ============================================================
# 9. CONTROL DE COLUMNAS COMUNES
# ============================================================

columnas_comunes = (
    set(
        df_maestro_04.columns
    )
    &
    set(
        df_vuelos_pre.columns
    )
)


columnas_comunes_no_clave = sorted(
    columnas_comunes
    -
    {"fecha"}
)


print("\n" + "=" * 80)
print("CONTROL DE COLUMNAS COMUNES")
print("=" * 80)

print(
    f"\n✓ Columnas comunes totales: "
    f"{len(columnas_comunes)}"
)

print(
    f"✓ Columnas comunes fuera de la clave: "
    f"{len(columnas_comunes_no_clave)}"
)


if columnas_comunes_no_clave:

    print(
        "\nColumnas comunes no esperadas:"
    )

    for columna in columnas_comunes_no_clave:

        print(
            f"  • {columna}"
        )

    raise ValueError(
        "Existen columnas comunes fuera de fecha."
    )


# ============================================================
# 10. COBERTURA DE LA CLAVE TEMPORAL
# ============================================================

fechas_maestro = (
    df_maestro_04[
        ["fecha"]
    ]
    .drop_duplicates()
)


fechas_vuelos = (
    df_vuelos_pre[
        ["fecha"]
    ]
    .drop_duplicates()
)


control_cobertura = pd.merge(
    fechas_maestro,
    fechas_vuelos.assign(
        fecha_en_vuelos=True
    ),
    on="fecha",
    how="left",
    validate="one_to_one"
)


control_cobertura[
    "fecha_en_vuelos"
] = (
    control_cobertura[
        "fecha_en_vuelos"
    ]
    .fillna(False)
    .astype(bool)
)


n_fechas_maestro = len(
    control_cobertura
)

n_fechas_con_correspondencia = (
    control_cobertura[
        "fecha_en_vuelos"
    ]
    .sum()
)

n_fechas_sin_correspondencia = (
    n_fechas_maestro
    -
    n_fechas_con_correspondencia
)


print("\n" + "=" * 80)
print("COBERTURA DE LA CLAVE TEMPORAL")
print("=" * 80)

print(
    f"\n✓ Fechas del maestro: "
    f"{n_fechas_maestro:,}"
)

print(
    f"✓ Fechas con correspondencia "
    f"en dataset aéreo: "
    f"{n_fechas_con_correspondencia:,}"
)

print(
    f"✓ Fechas sin correspondencia: "
    f"{n_fechas_sin_correspondencia:,}"
)

print(
    f"✓ Cobertura de la clave temporal: "
    f"{n_fechas_con_correspondencia / n_fechas_maestro * 100:.2f} %"
)


# ============================================================
# 11. VARIABLES AÉREAS A INCORPORAR
# ============================================================

variables_vuelos = [
    "Vuelos_total",
    "Vuelos_internacionales_pct",
    "Companias_activas",
    "Zonas_activas"
]


faltantes_vuelos = [
    variable
    for variable in variables_vuelos
    if variable not in df_vuelos_pre.columns
]


if faltantes_vuelos:

    raise KeyError(
        "Faltan variables aéreas necesarias:\n"
        f"{faltantes_vuelos}"
    )


print("\n" + "=" * 80)
print("VARIABLES AÉREAS A INCORPORAR")
print("=" * 80)

print(
    f"\n✓ Variables nuevas: "
    f"{len(variables_vuelos)}"
)

for variable in variables_vuelos:

    print(
        f"  • {variable}"
    )


# ============================================================
# 12. PREPARAR TABLA DE MERGE
# ============================================================

df_vuelos_merge = (
    df_vuelos_pre[
        [
            "fecha",
            *variables_vuelos
        ]
    ]
    .copy()
)


# ============================================================
# 13. INTEGRACIÓN MANY-TO-ONE
# ============================================================

df_maestro_05 = pd.merge(
    df_maestro_04,
    df_vuelos_merge,
    on="fecha",
    how="left",
    validate="many_to_one"
)


# ============================================================
# 14. CONTROL DE DIMENSIONES
# ============================================================

filas_despues = len(
    df_maestro_05
)

columnas_despues = (
    df_maestro_05.shape[1]
)

columnas_esperadas = (
    columnas_antes
    +
    len(
        variables_vuelos
    )
)


print("\n" + "=" * 80)
print("CONTROL DE DIMENSIONES TRAS EL MERGE")
print("=" * 80)

print(
    f"\n✓ Filas antes: "
    f"{filas_antes:,}"
)

print(
    f"✓ Filas después: "
    f"{filas_despues:,}"
)

print(
    f"✓ Columnas antes: "
    f"{columnas_antes}"
)

print(
    f"✓ Variables aéreas añadidas: "
    f"{len(variables_vuelos)}"
)

print(
    f"✓ Columnas esperadas: "
    f"{columnas_esperadas}"
)

print(
    f"✓ Columnas obtenidas: "
    f"{columnas_despues}"
)


if filas_antes != filas_despues:

    raise ValueError(
        "La integración ha modificado "
        "el número de filas."
    )


if columnas_despues != columnas_esperadas:

    raise ValueError(
        "El número de columnas obtenido "
        "no coincide con el esperado."
    )


# ============================================================
# 15. CONTROL DE DUPLICADOS TRAS EL MERGE
# ============================================================

duplicados_maestro_despues = (
    df_maestro_05
    .duplicated(
        subset=clave_maestro
    )
    .sum()
)


print("\n" + "=" * 80)
print("CONTROL DE DUPLICADOS TRAS EL MERGE")
print("=" * 80)

print(
    f"\n✓ Duplicados antes: "
    f"{duplicados_maestro_antes:,}"
)

print(
    f"✓ Duplicados después: "
    f"{duplicados_maestro_despues:,}"
)


if (
    duplicados_maestro_despues
    !=
    duplicados_maestro_antes
):

    raise ValueError(
        "La integración ha introducido duplicados."
    )


# ============================================================
# 16. COBERTURA DEL INDICADOR PRINCIPAL
# ============================================================

col_vuelos_principal = (
    "Vuelos_total"
)


n_validos_vuelos = (
    df_maestro_05[
        col_vuelos_principal
    ]
    .notna()
    .sum()
)

n_nulos_vuelos = (
    df_maestro_05[
        col_vuelos_principal
    ]
    .isna()
    .sum()
)

cobertura_vuelos_maestro = (
    n_validos_vuelos
    /
    len(
        df_maestro_05
    )
    *
    100
)


print("\n" + "=" * 80)
print("COBERTURA DEL INDICADOR AÉREO PRINCIPAL")
print("=" * 80)

print(
    f"\n✓ Registros con Vuelos_total: "
    f"{n_validos_vuelos:,}"
)

print(
    f"✓ Registros sin Vuelos_total: "
    f"{n_nulos_vuelos:,}"
)

print(
    f"✓ Cobertura en el maestro: "
    f"{cobertura_vuelos_maestro:.2f} %"
)


# ============================================================
# 17. COBERTURA POR AÑO
# ============================================================

df_maestro_05[
    "_anio_control_vuelos"
] = (
    df_maestro_05[
        "fecha"
    ]
    .dt.year
)


cobertura_anual_maestro = (
    df_maestro_05
    .groupby(
        "_anio_control_vuelos"
    )
    .agg(
        registros=(
            "fecha",
            "size"
        ),

        registros_con_vuelos=(
            "Vuelos_total",
            "count"
        )
    )
    .reset_index()
    .rename(
        columns={
            "_anio_control_vuelos":
                "anio"
        }
    )
)


cobertura_anual_maestro[
    "registros_sin_vuelos"
] = (
    cobertura_anual_maestro[
        "registros"
    ]
    -
    cobertura_anual_maestro[
        "registros_con_vuelos"
    ]
)


cobertura_anual_maestro[
    "cobertura_pct"
] = (
    cobertura_anual_maestro[
        "registros_con_vuelos"
    ]
    /
    cobertura_anual_maestro[
        "registros"
    ]
    *
    100
).round(2)


print("\n" + "=" * 80)
print("COBERTURA DE VUELOS POR AÑO EN EL MAESTRO")
print("=" * 80)

display(
    cobertura_anual_maestro
)


# ============================================================
# 18. CONTROL DEL PERIODO SIN INFORMACIÓN
# ============================================================

periodo_sin_info = (
    df_maestro_05[
        df_maestro_05[
            "fecha"
        ] < pd.Timestamp(
            "2019-07-01"
        )
    ]
)


n_periodo_sin_info = len(
    periodo_sin_info
)

n_periodo_sin_info_nan = (
    periodo_sin_info[
        "Vuelos_total"
    ]
    .isna()
    .sum()
)


print("\n" + "=" * 80)
print("CONTROL DEL PERIODO PREVIO AL 01/07/2019")
print("=" * 80)

print(
    f"\n✓ Registros del maestro anteriores "
    f"al 01/07/2019: "
    f"{n_periodo_sin_info:,}"
)

print(
    f"✓ Registros con Vuelos_total = NaN: "
    f"{n_periodo_sin_info_nan:,}"
)


if (
    n_periodo_sin_info
    !=
    n_periodo_sin_info_nan
):

    raise ValueError(
        "Algún registro anterior al 01/07/2019 "
        "presenta información de vuelos inesperada."
    )


# ============================================================
# 19. CONTROL DEL PERIODO CON INFORMACIÓN
# ============================================================

periodo_con_info = (
    df_maestro_05[
        df_maestro_05[
            "fecha"
        ] >= pd.Timestamp(
            "2019-07-01"
        )
    ]
)


n_nan_periodo_observado = (
    periodo_con_info[
        "Vuelos_total"
    ]
    .isna()
    .sum()
)


print("\n" + "=" * 80)
print("CONTROL DEL PERIODO DESDE 01/07/2019")
print("=" * 80)

print(
    f"\n✓ Registros sin Vuelos_total "
    f"desde 01/07/2019: "
    f"{n_nan_periodo_observado:,}"
)


if n_nan_periodo_observado > 0:

    raise ValueError(
        "Existen NaN de vuelos dentro "
        "del periodo observado."
    )


# ============================================================
# 20. ELIMINAR VARIABLE AUXILIAR
# ============================================================

df_maestro_05 = (
    df_maestro_05
    .drop(
        columns=[
            "_anio_control_vuelos"
        ]
    )
)


# ============================================================
# 21. CONTROL DE VALORES INFINITOS
# ============================================================

columnas_numericas = (
    df_maestro_05
    .select_dtypes(
        include=[
            np.number
        ]
    )
    .columns
)


n_inf = (
    np.isinf(
        df_maestro_05[
            columnas_numericas
        ]
    )
    .sum()
    .sum()
)


print("\n" + "=" * 80)
print("CONTROL DE VALORES INFINITOS")
print("=" * 80)

print(
    f"\n✓ Valores infinitos: "
    f"{n_inf:,}"
)


if n_inf > 0:

    raise ValueError(
        "Se han detectado valores infinitos."
    )


# ============================================================
# 22. ORDENAR EL MAESTRO
# ============================================================

columnas_orden = [
    columna
    for columna in [
        "fecha",
        "estacion_geo",
        "estacion_fisica",
        "contaminant"
    ]
    if columna in df_maestro_05.columns
]


df_maestro_05 = (
    df_maestro_05
    .sort_values(
        columnas_orden
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 23. EXPORTAR CHECKPOINT 05
# ============================================================

ruta_integracion.mkdir(
    parents=True,
    exist_ok=True
)


df_maestro_05.to_csv(
    ruta_maestro_05,
    index=False,
    encoding="utf-8"
)


# ============================================================
# 24. RECARGAR CHECKPOINT 05
# ============================================================

df_control_05 = pd.read_csv(
    ruta_maestro_05,
    parse_dates=[
        "fecha"
    ],
    low_memory=False
)


# ============================================================
# 25. VERIFICAR INTEGRIDAD DE LA EXPORTACIÓN
# ============================================================

filas_recarga = len(
    df_control_05
)

columnas_recarga = (
    df_control_05.shape[1]
)

duplicados_recarga = (
    df_control_05
    .duplicated(
        subset=clave_maestro
    )
    .sum()
)

nulos_vuelos_recarga = (
    df_control_05[
        "Vuelos_total"
    ]
    .isna()
    .sum()
)


print("\n" + "=" * 80)
print("VERIFICACIÓN DEL CHECKPOINT 05")
print("=" * 80)

print(
    f"\n✓ Filas exportadas: "
    f"{len(df_maestro_05):,}"
)

print(
    f"✓ Filas recargadas: "
    f"{filas_recarga:,}"
)

print(
    f"✓ Columnas exportadas: "
    f"{df_maestro_05.shape[1]}"
)

print(
    f"✓ Columnas recargadas: "
    f"{columnas_recarga}"
)

print(
    f"✓ Duplicados tras recarga: "
    f"{duplicados_recarga:,}"
)

print(
    f"✓ NaN Vuelos_total antes: "
    f"{n_nulos_vuelos:,}"
)

print(
    f"✓ NaN Vuelos_total después: "
    f"{nulos_vuelos_recarga:,}"
)


if filas_recarga != len(
    df_maestro_05
):

    raise ValueError(
        "El número de filas ha cambiado "
        "durante la exportación."
    )


if columnas_recarga != (
    df_maestro_05.shape[1]
):

    raise ValueError(
        "El número de columnas ha cambiado "
        "durante la exportación."
    )


if duplicados_recarga != (
    duplicados_maestro_despues
):

    raise ValueError(
        "La recarga ha alterado "
        "la unicidad del maestro."
    )


if nulos_vuelos_recarga != (
    n_nulos_vuelos
):

    raise ValueError(
        "La exportación ha alterado "
        "los NaN de Vuelos_total."
    )


# ============================================================
# 26. TAMAÑO DEL ARCHIVO
# ============================================================

tamano_mb = (
    ruta_maestro_05.stat().st_size
    /
    (1024 ** 2)
)


# ============================================================
# 27. RESUMEN FINAL
# ============================================================

print("\n" + "=" * 80)
print("CHECKPOINT 05 — TRANSPORTE AÉREO")
print("=" * 80)

print(
    f"\n✓ Registros conservados: "
    f"{filas_recarga:,}"
)

print(
    f"✓ Variables disponibles: "
    f"{columnas_recarga}"
)

print(
    f"✓ Variables aéreas añadidas: "
    f"{len(variables_vuelos)}"
)

print(
    f"✓ Localizaciones físicas: "
    f"{df_control_05['estacion_geo'].nunique()}"
)

print(
    f"✓ Contaminantes: "
    f"{df_control_05['contaminant'].nunique()}"
)

print(
    f"✓ Duplicados de la clave principal: "
    f"{duplicados_recarga:,}"
)

print(
    f"✓ Cobertura Vuelos_total: "
    f"{df_control_05['Vuelos_total'].notna().mean() * 100:.2f} %"
)

print(
    f"✓ NaN conservados en Vuelos_total: "
    f"{nulos_vuelos_recarga:,}"
)

print(
    f"✓ Valores infinitos: "
    f"{n_inf:,}"
)

print(
    f"✓ Tamaño del archivo: "
    f"{tamano_mb:.2f} MB"
)

print(
    "✓ Sin pérdida ni multiplicación de registros."
)

print(
    "✓ Checkpoint 04 conservado sin modificaciones."
)

print(
    "✓ Los valores anteriores al 01/07/2019 "
    "permanecen como NaN."
)

print(
    "✓ No se ha realizado ninguna imputación aérea."
)

print(
    "✓ Integridad del Checkpoint 05 "
    "verificada tras la exportación."
)

print(
    "\n✓ Archivo maestro generado:"
)

print(
    ruta_maestro_05
)

6.5 — INTEGRACIÓN DEL TRANSPORTE AÉREO — CHECKPOINT 05

CHECKPOINT 04
✓ Filas: 43,642
✓ Columnas: 63

DATASET AÉREO PREINTEGRADO
✓ Filas: 2,557
✓ Columnas: 5
✓ Fechas diferentes: 2,557

CONTROL DE FECHAS

✓ Fechas inválidas en maestro: 0
✓ Fechas inválidas en vuelos: 0

UNICIDAD DEL DATASET AÉREO

✓ Duplicados de fecha: 0

CONTROL DEL MAESTRO ANTES DE INTEGRAR

✓ Filas: 43,642
✓ Columnas: 63
✓ Duplicados clave principal: 0

CONTROL DE COLUMNAS COMUNES

✓ Columnas comunes totales: 1
✓ Columnas comunes fuera de la clave: 0

COBERTURA DE LA CLAVE TEMPORAL

✓ Fechas del maestro: 2,249
✓ Fechas con correspondencia en dataset aéreo: 2,249
✓ Fechas sin correspondencia: 0
✓ Cobertura de la clave temporal: 100.00 %

VARIABLES AÉREAS A INCORPORAR

✓ Variables nuevas: 4
  • Vuelos_total
  • Vuelos_internacionales_pct
  • Companias_activas
  • Zonas_activas

CONTROL DE DIMENSIONES TRAS EL MERGE

✓ Filas antes: 43,642
✓ Filas después: 43,642
✓ Columnas antes: 63
✓ Variables aéreas añadidas: 4
✓ Col

,anio,registros,registros_con_vuelos,registros_sin_vuelos,cobertura_pct
0,2018,2830,0,2830,0.00
1,2019,4311,2390,1921,55.44
2,2020,6894,6894,0,100.00
3,2021,7051,7051,0,100.00
4,2022,7446,7446,0,100.00
5,2023,7634,7634,0,100.00
6,2024,7476,7476,0,100.00



CONTROL DEL PERIODO PREVIO AL 01/07/2019

✓ Registros del maestro anteriores al 01/07/2019: 4,751
✓ Registros con Vuelos_total = NaN: 4,751

CONTROL DEL PERIODO DESDE 01/07/2019

✓ Registros sin Vuelos_total desde 01/07/2019: 0

CONTROL DE VALORES INFINITOS

✓ Valores infinitos: 0

VERIFICACIÓN DEL CHECKPOINT 05

✓ Filas exportadas: 43,642
✓ Filas recargadas: 43,642
✓ Columnas exportadas: 67
✓ Columnas recargadas: 67
✓ Duplicados tras recarga: 0
✓ NaN Vuelos_total antes: 4,751
✓ NaN Vuelos_total después: 4,751

CHECKPOINT 05 — TRANSPORTE AÉREO

✓ Registros conservados: 43,642
✓ Variables disponibles: 67
✓ Variables aéreas añadidas: 4
✓ Localizaciones físicas: 8
✓ Contaminantes: 4
✓ Duplicados de la clave principal: 0
✓ Cobertura Vuelos_total: 89.11 %
✓ NaN conservados en Vuelos_total: 4,751
✓ Valores infinitos: 0
✓ Tamaño del archivo: 25.45 MB
✓ Sin pérdida ni multiplicación de registros.
✓ Checkpoint 04 conservado sin modificaciones.
✓ Los valores anteriores al 01/07/2019 permanecen 

### Resultados — 6.5. Integración del transporte aéreo y generación del Checkpoint 05

La integración del bloque de transporte aéreo se realiza sobre el **Checkpoint 04**, compuesto por **43.642 registros y 63 variables**.

El dataset aéreo preintegrado contiene **2.557 fechas**, correspondientes al calendario completo 2018–2024, y presenta una única observación por día.

La correspondencia temporal entre ambas fuentes es completa para las fechas presentes en el maestro: las **2.249 fechas distintas del Checkpoint 04 disponen de correspondencia en el dataset aéreo**, alcanzándose una cobertura de clave temporal del **100 %**.

Se incorporan cuatro variables:

- `Vuelos_total`
- `Vuelos_internacionales_pct`
- `Companias_activas`
- `Zonas_activas`

La integración conserva exactamente los **43.642 registros originales** y amplía el dataset de **63 a 67 variables**, sin introducir duplicados en la clave principal `fecha + estacion_fisica + contaminant`.

El indicador principal `Vuelos_total` está disponible en **38.891 registros**, lo que representa una cobertura del **89,11 %** dentro del maestro.

Los **4.751 registros sin información aérea** corresponden exclusivamente al periodo anterior al **1 de julio de 2019**. Todos los registros del maestro anteriores a esa fecha mantienen `Vuelos_total = NaN`, mientras que desde el 1 de julio de 2019 no existe ningún registro sin información de vuelos.

La cobertura anual dentro del maestro es del **0 % en 2018**, del **55,44 % en 2019** y del **100 % entre 2020 y 2024**. La diferencia respecto a la cobertura diaria del calendario completo se debe a la distribución real de observaciones del propio dataset maestro.

No se detectan valores infinitos ni alteraciones de la estructura original del dataset.

La exportación del Checkpoint 05 se valida mediante su recarga, conservándose exactamente:

- **43.642 registros**;
- **67 variables**;
- **0 duplicados**;
- **4.751 valores ausentes en `Vuelos_total`**.

No se realiza ninguna imputación y los valores anteriores al inicio real de la serie aérea permanecen como `NaN`.

> **Conclusión:** el bloque de transporte aéreo queda correctamente integrado en el dataset maestro, manteniendo la estructura original y la limitación temporal real de la fuente. El resultado constituye el **Checkpoint 05** del proceso de integración.

# 7 · INTEGRACIÓN DEL TRANSPORTE MARÍTIMO

El último bloque incorporado al dataset maestro corresponde a la **actividad marítima del puerto de Barcelona**.

La fuente marítima proporciona información diaria sobre la intensidad de las operaciones portuarias, las características de los buques y la composición de la actividad según diferentes tipologías.

Antes de su integración se auditan la cobertura temporal y la coherencia interna de la serie, se analizan posibles redundancias entre indicadores y se selecciona un conjunto reducido de variables representativas.

La integración se realiza mediante la variable `fecha`. Se distingue explícitamente entre ausencia de información y ausencia de actividad: cuando no existen movimientos portuarios, los indicadores de conteo pueden tomar valor cero, mientras que determinadas magnitudes medias permanecen como `NaN` al no existir buques sobre los que calcularlas.

Este bloque completa la construcción del dataset y genera el **Checkpoint 06**, correspondiente al maestro multifuente definitivo de la fase de integración.

> **Objetivo del bloque:** incorporar indicadores diarios de actividad portuaria y completar la construcción secuencial del dataset maestro.

## 7.1. Carga y auditoría estructural del dataset marítimo diario

Se carga el dataset diario de actividad portuaria generado durante la fase de preprocesado.

Antes de realizar cualquier transformación o integración se audita su estructura para identificar:

- dimensiones del dataset;
- nombres y tipos de variables;
- variable temporal disponible;
- periodo cubierto;
- número de fechas diferentes;
- posibles fechas inválidas;
- duplicados temporales;
- valores ausentes;
- valores infinitos;
- variables numéricas y categóricas disponibles.

En esta etapa no se modifica ni se imputa ninguna variable.

> **Objetivo:** conocer y validar la estructura real del dataset marítimo limpio antes de definir los indicadores que se incorporarán al modelo.

In [ ]:
# ============================================================
# 7.1. CARGA Y AUDITORÍA ESTRUCTURAL
# DEL TRANSPORTE MARÍTIMO
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. RUTA
# ============================================================

ruta_puerto = Path(
    "/content/drive/MyDrive/TFM/"
    "11_Machine_Learning_Temporal/"
    "PREINTEGRACION/03_Transporte_Maritimo/"
    "df_puerto_diario_limpio.csv"
)


# ============================================================
# 2. COMPROBAR EXISTENCIA
# ============================================================

if not ruta_puerto.exists():

    raise FileNotFoundError(
        "No se encuentra el dataset marítimo:\n"
        f"{ruta_puerto}"
    )


# ============================================================
# 3. CARGAR DATASET
# ============================================================

df_puerto = pd.read_csv(
    ruta_puerto,
    low_memory=False
)


print("=" * 80)
print("7.1 — CARGA Y AUDITORÍA ESTRUCTURAL DEL TRANSPORTE MARÍTIMO")
print("=" * 80)

print(
    f"\n✓ Archivo cargado:\n{ruta_puerto}"
)

print(
    f"\n✓ Filas: "
    f"{len(df_puerto):,}"
)

print(
    f"✓ Columnas: "
    f"{df_puerto.shape[1]}"
)


# ============================================================
# 4. MOSTRAR VARIABLES DISPONIBLES
# ============================================================

print("\n" + "=" * 80)
print("VARIABLES DISPONIBLES")
print("=" * 80)

for i, columna in enumerate(
    df_puerto.columns,
    start=1
):

    print(
        f"{i:02d}. {columna}"
    )


# ============================================================
# 5. TIPOS DE DATOS
# ============================================================

tipos_puerto = pd.DataFrame({
    "variable":
        df_puerto.columns,

    "dtype":
        [
            str(dtype)
            for dtype
            in df_puerto.dtypes
        ]
})


print("\n" + "=" * 80)
print("TIPOS DE DATOS")
print("=" * 80)

display(
    tipos_puerto
)


# ============================================================
# 6. IDENTIFICAR VARIABLE TEMPORAL
# ============================================================

candidatas_fecha = [
    "Fecha",
    "fecha",
    "DATE",
    "Date",
    "date"
]


columnas_fecha = [
    columna
    for columna in candidatas_fecha
    if columna in df_puerto.columns
]


if len(columnas_fecha) == 0:

    raise KeyError(
        "No se ha identificado automáticamente "
        "ninguna variable de fecha.\n"
        f"Columnas disponibles:\n"
        f"{list(df_puerto.columns)}"
    )


if len(columnas_fecha) > 1:

    print(
        "\n⚠ Se han encontrado varias "
        "posibles variables temporales:"
    )

    print(
        columnas_fecha
    )


col_fecha = columnas_fecha[0]


print("\n" + "=" * 80)
print("VARIABLE TEMPORAL")
print("=" * 80)

print(
    f"\n✓ Variable identificada: "
    f"{col_fecha}"
)


# ============================================================
# 7. NORMALIZAR FECHA
# ============================================================

df_puerto[
    col_fecha
] = pd.to_datetime(
    df_puerto[
        col_fecha
    ],
    errors="coerce"
)


fechas_invalidas = (
    df_puerto[
        col_fecha
    ]
    .isna()
    .sum()
)


print(
    f"✓ Fechas inválidas: "
    f"{fechas_invalidas:,}"
)


if fechas_invalidas > 0:

    raise ValueError(
        "Existen fechas inválidas "
        "en el dataset marítimo."
    )


# ============================================================
# 8. COBERTURA TEMPORAL GENERAL
# ============================================================

fecha_inicio = (
    df_puerto[
        col_fecha
    ]
    .min()
)

fecha_fin = (
    df_puerto[
        col_fecha
    ]
    .max()
)

n_fechas = (
    df_puerto[
        col_fecha
    ]
    .nunique()
)


print("\n" + "=" * 80)
print("COBERTURA TEMPORAL GENERAL")
print("=" * 80)

print(
    f"\n✓ Fecha inicial: "
    f"{fecha_inicio}"
)

print(
    f"✓ Fecha final: "
    f"{fecha_fin}"
)

print(
    f"✓ Fechas diferentes: "
    f"{n_fechas:,}"
)


# ============================================================
# 9. DUPLICADOS TEMPORALES
# ============================================================

duplicados_fecha = (
    df_puerto
    .duplicated(
        subset=[
            col_fecha
        ]
    )
    .sum()
)


print("\n" + "=" * 80)
print("UNICIDAD TEMPORAL")
print("=" * 80)

print(
    f"\n✓ Duplicados de fecha: "
    f"{duplicados_fecha:,}"
)


if duplicados_fecha == 0:

    print(
        "✓ El dataset presenta una única "
        "observación por fecha."
    )

else:

    print(
        "⚠ Existen varias observaciones "
        "para alguna fecha."
    )

    print(
        "⚠ Se analizarán antes de realizar "
        "cualquier agregación o integración."
    )


# ============================================================
# 10. VALORES AUSENTES
# ============================================================

tabla_nan = pd.DataFrame({
    "variable":
        df_puerto.columns,

    "n_nan":
        [
            df_puerto[
                columna
            ]
            .isna()
            .sum()

            for columna
            in df_puerto.columns
        ]
})


tabla_nan[
    "pct_nan"
] = (
    tabla_nan[
        "n_nan"
    ]
    /
    len(df_puerto)
    *
    100
).round(2)


tabla_nan = (
    tabla_nan
    .sort_values(
        [
            "pct_nan",
            "variable"
        ],
        ascending=[
            False,
            True
        ]
    )
    .reset_index(
        drop=True
    )
)


print("\n" + "=" * 80)
print("VALORES AUSENTES")
print("=" * 80)

display(
    tabla_nan
)


# ============================================================
# 11. VARIABLES NUMÉRICAS
# ============================================================

columnas_numericas = (
    df_puerto
    .select_dtypes(
        include=[
            np.number
        ]
    )
    .columns
    .tolist()
)


print("\n" + "=" * 80)
print("VARIABLES NUMÉRICAS")
print("=" * 80)

print(
    f"\n✓ Número de variables numéricas: "
    f"{len(columnas_numericas)}"
)

for columna in columnas_numericas:

    print(
        f"  • {columna}"
    )


# ============================================================
# 12. VARIABLES NO NUMÉRICAS
# ============================================================

columnas_no_numericas = [
    columna
    for columna in df_puerto.columns
    if (
        columna not in columnas_numericas
        and
        columna != col_fecha
    )
]


print("\n" + "=" * 80)
print("VARIABLES NO NUMÉRICAS")
print("=" * 80)

print(
    f"\n✓ Número de variables no numéricas "
    f"(excluyendo fecha): "
    f"{len(columnas_no_numericas)}"
)

for columna in columnas_no_numericas:

    print(
        f"  • {columna}"
    )


# ============================================================
# 13. CONTROL DE INFINITOS
# ============================================================

if columnas_numericas:

    n_inf = (
        np.isinf(
            df_puerto[
                columnas_numericas
            ]
        )
        .sum()
        .sum()
    )

else:

    n_inf = 0


print("\n" + "=" * 80)
print("CONTROL DE VALORES INFINITOS")
print("=" * 80)

print(
    f"\n✓ Valores infinitos: "
    f"{n_inf:,}"
)


# ============================================================
# 14. CONTROL DE VALORES NEGATIVOS
# ============================================================

if columnas_numericas:

    tabla_negativos = pd.DataFrame({
        "variable":
            columnas_numericas,

        "n_negativos":
            [
                (
                    df_puerto[
                        columna
                    ] < 0
                ).sum()

                for columna
                in columnas_numericas
            ]
    })

else:

    tabla_negativos = pd.DataFrame(
        columns=[
            "variable",
            "n_negativos"
        ]
    )


print("\n" + "=" * 80)
print("CONTROL DE VALORES NEGATIVOS")
print("=" * 80)

if (
    len(tabla_negativos) == 0
    or
    tabla_negativos[
        "n_negativos"
    ].sum() == 0
):

    print(
        "\n✓ No se detectan valores negativos "
        "en las variables numéricas."
    )

else:

    display(
        tabla_negativos[
            tabla_negativos[
                "n_negativos"
            ] > 0
        ]
    )


# ============================================================
# 15. ESTADÍSTICOS DESCRIPTIVOS
# ============================================================

if columnas_numericas:

    print("\n" + "=" * 80)
    print("ESTADÍSTICOS DE VARIABLES NUMÉRICAS")
    print("=" * 80)

    display(
        df_puerto[
            columnas_numericas
        ]
        .describe()
        .T
    )


# ============================================================
# 16. MUESTRA DEL DATASET
# ============================================================

print("\n" + "=" * 80)
print("MUESTRA DEL DATASET")
print("=" * 80)

display(
    df_puerto.head(10)
)


# ============================================================
# 17. RESUMEN FINAL
# ============================================================

print("\n" + "=" * 80)
print("AUDITORÍA 7.1 FINALIZADA")
print("=" * 80)

print(
    f"\n✓ Registros: "
    f"{len(df_puerto):,}"
)

print(
    f"✓ Variables: "
    f"{df_puerto.shape[1]}"
)

print(
    f"✓ Variable temporal: "
    f"{col_fecha}"
)

print(
    f"✓ Fecha inicial: "
    f"{fecha_inicio.date()}"
)

print(
    f"✓ Fecha final: "
    f"{fecha_fin.date()}"
)

print(
    f"✓ Fechas diferentes: "
    f"{n_fechas:,}"
)

print(
    f"✓ Duplicados de fecha: "
    f"{duplicados_fecha:,}"
)

print(
    f"✓ Variables numéricas: "
    f"{len(columnas_numericas)}"
)

print(
    f"✓ Variables no numéricas "
    f"(sin fecha): "
    f"{len(columnas_no_numericas)}"
)

print(
    f"✓ Valores infinitos: "
    f"{n_inf:,}"
)

print(
    "✓ No se ha realizado ninguna "
    "transformación ni imputación."
)

print(
    "\n✓ Dataset preparado para "
    "auditoría temporal en 7.2."
)

7.1 — CARGA Y AUDITORÍA ESTRUCTURAL DEL TRANSPORTE MARÍTIMO

✓ Archivo cargado:
/content/drive/MyDrive/TFM/11_Machine_Learning_Temporal/PREINTEGRACION/03_Transporte_Maritimo/df_puerto_diario_limpio.csv

✓ Filas: 2,557
✓ Columnas: 17

VARIABLES DISPONIBLES
01. fecha
02. Puerto_movimientos
03. Puerto_n_buques
04. Puerto_eslora_total
05. Puerto_eslora_media
06. Puerto_calado_medio
07. Puerto_llegadas
08. Puerto_salidas
09. Puerto_carga
10. Puerto_ferry_pasajeros
11. Puerto_portacontenedores
12. Puerto_ro_ro
13. Puerto_tanque
14. anio
15. mes
16. dia_semana
17. Periodo_COVID

TIPOS DE DATOS


,variable,dtype
0,fecha,object
1,Puerto_movimientos,float64
2,Puerto_n_buques,float64
3,Puerto_eslora_total,float64
4,Puerto_eslora_media,float64
5,Puerto_calado_medio,float64
6,Puerto_llegadas,float64
7,Puerto_salidas,float64
8,Puerto_carga,float64
9,Puerto_ferry_pasajeros,float64



VARIABLE TEMPORAL

✓ Variable identificada: fecha
✓ Fechas inválidas: 0

COBERTURA TEMPORAL GENERAL

✓ Fecha inicial: 2018-01-01 00:00:00
✓ Fecha final: 2024-12-31 00:00:00
✓ Fechas diferentes: 2,557

UNICIDAD TEMPORAL

✓ Duplicados de fecha: 0
✓ El dataset presenta una única observación por fecha.

VALORES AUSENTES


,variable,n_nan,pct_nan
0,Puerto_calado_medio,1,0.04
1,Puerto_eslora_media,1,0.04
2,Periodo_COVID,0,0.00
3,Puerto_carga,0,0.00
4,Puerto_eslora_total,0,0.00
5,Puerto_ferry_pasajeros,0,0.00
6,Puerto_llegadas,0,0.00
7,Puerto_movimientos,0,0.00
8,Puerto_n_buques,0,0.00
9,Puerto_portacontenedores,0,0.00



VARIABLES NUMÉRICAS

✓ Número de variables numéricas: 15
  • Puerto_movimientos
  • Puerto_n_buques
  • Puerto_eslora_total
  • Puerto_eslora_media
  • Puerto_calado_medio
  • Puerto_llegadas
  • Puerto_salidas
  • Puerto_carga
  • Puerto_ferry_pasajeros
  • Puerto_portacontenedores
  • Puerto_ro_ro
  • Puerto_tanque
  • anio
  • mes
  • dia_semana

VARIABLES NO NUMÉRICAS

✓ Número de variables no numéricas (excluyendo fecha): 1
  • Periodo_COVID

CONTROL DE VALORES INFINITOS

✓ Valores infinitos: 0

CONTROL DE VALORES NEGATIVOS

✓ No se detectan valores negativos en las variables numéricas.

ESTADÍSTICOS DE VARIABLES NUMÉRICAS


,count,mean,std,min,25%,50%,75%,max
Puerto_movimientos,2557.0,54.770825,11.586777,0.000000,47.000000,55.000000,63.000000,89.000000
Puerto_n_buques,2557.0,35.441533,6.600390,0.000000,31.000000,36.000000,40.000000,62.000000
Puerto_eslora_total,2557.0,10269.812104,2214.431977,0.000000,8847.360000,10368.520000,11780.960000,16789.840000
Puerto_eslora_media,2556.0,187.639278,11.294365,150.283125,179.772167,186.943141,195.357619,231.893696
Puerto_calado_medio,2556.0,8.460398,0.468352,6.502619,8.139144,8.427691,8.744667,10.479032
Puerto_llegadas,2557.0,27.384044,6.040450,0.000000,23.000000,28.000000,32.000000,52.000000
Puerto_salidas,2557.0,27.386781,6.329331,0.000000,23.000000,27.000000,32.000000,49.000000
Puerto_carga,2557.0,4.373485,2.859813,0.000000,2.000000,4.000000,6.000000,20.000000
Puerto_ferry_pasajeros,2557.0,19.895190,6.962281,0.000000,15.000000,21.000000,25.000000,42.000000
Puerto_portacontenedores,2557.0,14.923348,4.920533,0.000000,12.000000,15.000000,18.000000,43.000000



MUESTRA DEL DATASET


,fecha,Puerto_movimientos,Puerto_n_buques,Puerto_eslora_total,Puerto_eslora_media,Puerto_calado_medio,Puerto_llegadas,Puerto_salidas,Puerto_carga,Puerto_ferry_pasajeros,Puerto_portacontenedores,Puerto_ro_ro,Puerto_tanque,anio,mes,dia_semana,Periodo_COVID
0,2018-01-01,25.0,22.0,4321.61,172.864400,8.092800,15.0,10.0,6.0,7.0,9.0,2.0,1.0,2018,1,0,pre_covid
1,2018-01-02,57.0,36.0,9705.04,170.263860,8.170175,25.0,32.0,10.0,19.0,17.0,6.0,5.0,2018,1,1,pre_covid
2,2018-01-03,54.0,34.0,10355.79,191.773889,8.862222,24.0,30.0,5.0,19.0,24.0,2.0,4.0,2018,1,2,pre_covid
3,2018-01-04,49.0,33.0,10066.35,205.435714,9.045714,29.0,20.0,6.0,19.0,15.0,6.0,3.0,2018,1,3,pre_covid
4,2018-01-05,65.0,44.0,13305.83,204.705077,9.150154,30.0,35.0,8.0,24.0,25.0,6.0,2.0,2018,1,4,pre_covid
5,2018-01-06,38.0,32.0,7163.95,188.525000,9.075526,25.0,13.0,4.0,9.0,15.0,5.0,5.0,2018,1,5,pre_covid
6,2018-01-07,52.0,36.0,9686.44,186.277692,9.098269,25.0,27.0,5.0,10.0,21.0,5.0,11.0,2018,1,6,pre_covid
7,2018-01-08,55.0,33.0,10738.48,195.245091,8.686364,27.0,28.0,3.0,22.0,20.0,3.0,7.0,2018,1,0,pre_covid
8,2018-01-09,67.0,41.0,12639.02,188.642090,8.823134,31.0,36.0,5.0,19.0,28.0,10.0,5.0,2018,1,1,pre_covid
9,2018-01-10,56.0,36.0,10599.38,189.274643,8.624643,25.0,31.0,3.0,17.0,27.0,5.0,4.0,2018,1,2,pre_covid



AUDITORÍA 7.1 FINALIZADA

✓ Registros: 2,557
✓ Variables: 17
✓ Variable temporal: fecha
✓ Fecha inicial: 2018-01-01
✓ Fecha final: 2024-12-31
✓ Fechas diferentes: 2,557
✓ Duplicados de fecha: 0
✓ Variables numéricas: 15
✓ Variables no numéricas (sin fecha): 1
✓ Valores infinitos: 0
✓ No se ha realizado ninguna transformación ni imputación.

✓ Dataset preparado para auditoría temporal en 7.2.


### Resultados — 7.1. Auditoría estructural del transporte marítimo

El dataset diario de transporte marítimo contiene **2.557 registros y 17 variables**, con una observación por día para el periodo comprendido entre el **1 de enero de 2018 y el 31 de diciembre de 2024**.

La variable temporal se identifica correctamente como `fecha`. No se detectan fechas inválidas ni duplicados temporales, por lo que el dataset presenta una estructura estrictamente diaria y una única observación por fecha.

El conjunto contiene indicadores relacionados con diferentes dimensiones de la actividad portuaria:

- intensidad general de actividad (`Puerto_movimientos`, `Puerto_n_buques`);
- características físicas de los buques (`Puerto_eslora_total`, `Puerto_eslora_media`, `Puerto_calado_medio`);
- dinámica de operaciones (`Puerto_llegadas`, `Puerto_salidas`);
- tipología de tráfico marítimo (`Puerto_carga`, `Puerto_ferry_pasajeros`, `Puerto_portacontenedores`, `Puerto_ro_ro`, `Puerto_tanque`);
- variables temporales auxiliares (`anio`, `mes`, `dia_semana`, `Periodo_COVID`).

Se identifican **15 variables numéricas** y una variable categórica adicional, `Periodo_COVID`.

La estructura de valores ausentes es prácticamente completa. Únicamente se detecta **un valor ausente en `Puerto_eslora_media` y uno en `Puerto_calado_medio`**, equivalentes al **0,04 %** de los registros de cada variable. El resto de las variables presenta cobertura completa.

No se detectan valores infinitos ni valores negativos en las variables numéricas.

Los estadísticos descriptivos muestran una media diaria de aproximadamente **54,77 movimientos** y **35,44 buques**, junto con una media de **27,38 llegadas** y **27,39 salidas** diarias.

En esta etapa no se realiza ninguna transformación ni imputación.

> **Conclusión:** el dataset marítimo presenta una estructura temporal completa y consistente para 2018–2024, sin duplicados ni problemas estructurales relevantes. Los dos valores ausentes detectados en las variables medias deberán analizarse antes de cualquier tratamiento. El dataset queda preparado para la auditoría temporal y de coherencia interna de la etapa 7.2.

## 7.2. Auditoría temporal y coherencia interna del transporte marítimo

Una vez validada la estructura general del dataset marítimo, se analiza su cobertura temporal y la coherencia interna de los principales indicadores de actividad portuaria.

La auditoría comprende:

- cobertura diaria y anual durante 2018–2024;
- identificación de posibles fechas ausentes dentro del calendario;
- análisis específico de los valores ausentes detectados en `Puerto_eslora_media` y `Puerto_calado_medio`;
- comprobación de si dichos valores responden a días sin actividad marítima;
- verificación de la relación entre movimientos, llegadas y salidas;
- análisis de la relación entre número de movimientos y número de buques;
- control de valores negativos e infinitos;
- identificación de días con actividad nula;
- resumen anual de los principales indicadores marítimos.

En esta etapa no se realiza ninguna imputación ni modificación de los datos originales.

> **Objetivo:** verificar que la serie marítima presenta cobertura temporal completa y que sus indicadores mantienen relaciones internas coherentes antes de seleccionar las variables que se incorporarán al modelo.

In [ ]:
# ============================================================
# 7.2. AUDITORÍA TEMPORAL Y COHERENCIA INTERNA
# DEL TRANSPORTE MARÍTIMO
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 1. COMPROBAR DATASET DE 7.1
# ============================================================

if "df_puerto" not in globals():
    raise NameError(
        "No existe df_puerto. "
        "Debe ejecutarse previamente la etapa 7.1."
    )


df_puerto_temp = df_puerto.copy()


# ============================================================
# 2. NORMALIZAR FECHA
# ============================================================

df_puerto_temp["fecha"] = pd.to_datetime(
    df_puerto_temp["fecha"],
    errors="coerce"
)

fechas_invalidas = (
    df_puerto_temp["fecha"]
    .isna()
    .sum()
)

if fechas_invalidas > 0:
    raise ValueError(
        "Existen fechas inválidas."
    )


# ============================================================
# 3. COBERTURA TEMPORAL GENERAL
# ============================================================

fecha_inicio = df_puerto_temp["fecha"].min()
fecha_fin = df_puerto_temp["fecha"].max()

n_fechas = (
    df_puerto_temp["fecha"]
    .nunique()
)

duplicados_fecha = (
    df_puerto_temp
    .duplicated(
        subset=["fecha"]
    )
    .sum()
)


print("=" * 80)
print("7.2 — AUDITORÍA TEMPORAL Y COHERENCIA INTERNA")
print("=" * 80)

print("\n" + "=" * 80)
print("COBERTURA TEMPORAL GENERAL")
print("=" * 80)

print(
    f"\n✓ Fecha inicial: "
    f"{fecha_inicio.date()}"
)

print(
    f"✓ Fecha final: "
    f"{fecha_fin.date()}"
)

print(
    f"✓ Fechas diferentes: "
    f"{n_fechas:,}"
)

print(
    f"✓ Fechas inválidas: "
    f"{fechas_invalidas:,}"
)

print(
    f"✓ Duplicados de fecha: "
    f"{duplicados_fecha:,}"
)


# ============================================================
# 4. COMPROBAR CALENDARIO COMPLETO 2018–2024
# ============================================================

calendario_esperado = pd.date_range(
    start="2018-01-01",
    end="2024-12-31",
    freq="D"
)

fechas_observadas = pd.DatetimeIndex(
    df_puerto_temp["fecha"]
    .dropna()
    .unique()
)

fechas_faltantes = (
    calendario_esperado
    .difference(
        fechas_observadas
    )
)

fechas_fuera_periodo = (
    fechas_observadas
    .difference(
        calendario_esperado
    )
)


print("\n" + "=" * 80)
print("CONTINUIDAD DEL CALENDARIO")
print("=" * 80)

print(
    f"\n✓ Días esperados 2018–2024: "
    f"{len(calendario_esperado):,}"
)

print(
    f"✓ Días observados: "
    f"{len(fechas_observadas):,}"
)

print(
    f"✓ Fechas faltantes: "
    f"{len(fechas_faltantes):,}"
)

print(
    f"✓ Fechas fuera del periodo objetivo: "
    f"{len(fechas_fuera_periodo):,}"
)


if len(fechas_faltantes) > 0:

    print(
        "\n⚠ Primeras fechas faltantes:"
    )

    print(
        list(
            fechas_faltantes[:20]
        )
    )

else:

    print(
        "✓ El calendario 2018–2024 "
        "está completo."
    )


# ============================================================
# 5. COBERTURA POR AÑO
# ============================================================

df_puerto_temp[
    "_anio_control"
] = (
    df_puerto_temp["fecha"]
    .dt.year
)


cobertura_anual = (
    df_puerto_temp
    .groupby(
        "_anio_control"
    )
    .agg(
        registros=(
            "fecha",
            "size"
        ),

        dias=(
            "fecha",
            "nunique"
        ),

        movimientos_validos=(
            "Puerto_movimientos",
            "count"
        ),

        buques_validos=(
            "Puerto_n_buques",
            "count"
        )
    )
    .reset_index()
    .rename(
        columns={
            "_anio_control": "anio"
        }
    )
)


dias_teoricos = {
    2018: 365,
    2019: 365,
    2020: 366,
    2021: 365,
    2022: 365,
    2023: 365,
    2024: 366
}


cobertura_anual[
    "dias_teoricos"
] = (
    cobertura_anual["anio"]
    .map(
        dias_teoricos
    )
)


cobertura_anual[
    "cobertura_pct"
] = (
    cobertura_anual["dias"]
    /
    cobertura_anual["dias_teoricos"]
    *
    100
).round(2)


print("\n" + "=" * 80)
print("COBERTURA POR AÑO")
print("=" * 80)

display(
    cobertura_anual
)


# ============================================================
# 6. LOCALIZAR LOS NaN DETECTADOS EN 7.1
# ============================================================

variables_medias = [
    "Puerto_eslora_media",
    "Puerto_calado_medio"
]


filas_nan_medias = (
    df_puerto_temp.loc[
        df_puerto_temp[
            variables_medias
        ]
        .isna()
        .any(axis=1)
    ]
    .copy()
)


print("\n" + "=" * 80)
print("ANÁLISIS DE LOS VALORES AUSENTES")
print("=" * 80)

print(
    f"\n✓ Filas con algún NaN "
    f"en eslora/calado medios: "
    f"{len(filas_nan_medias):,}"
)


if len(filas_nan_medias) > 0:

    columnas_control_nan = [
        "fecha",
        "Puerto_movimientos",
        "Puerto_n_buques",
        "Puerto_eslora_total",
        "Puerto_eslora_media",
        "Puerto_calado_medio",
        "Puerto_llegadas",
        "Puerto_salidas",
        "Puerto_carga",
        "Puerto_ferry_pasajeros",
        "Puerto_portacontenedores",
        "Puerto_ro_ro",
        "Puerto_tanque"
    ]

    display(
        filas_nan_medias[
            columnas_control_nan
        ]
    )


# ============================================================
# 7. COMPROBAR SI LOS NaN CORRESPONDEN A ACTIVIDAD NULA
# ============================================================

if len(filas_nan_medias) > 0:

    nan_con_cero_buques = (
        filas_nan_medias[
            "Puerto_n_buques"
        ]
        .eq(0)
        .sum()
    )

    nan_con_cero_movimientos = (
        filas_nan_medias[
            "Puerto_movimientos"
        ]
        .eq(0)
        .sum()
    )

else:

    nan_con_cero_buques = 0
    nan_con_cero_movimientos = 0


print(
    f"\n✓ Filas NaN con 0 buques: "
    f"{nan_con_cero_buques:,}"
)

print(
    f"✓ Filas NaN con 0 movimientos: "
    f"{nan_con_cero_movimientos:,}"
)


# ============================================================
# 8. DÍAS CON ACTIVIDAD NULA
# ============================================================

dias_cero_movimientos = (
    df_puerto_temp[
        "Puerto_movimientos"
    ]
    .eq(0)
    .sum()
)

dias_cero_buques = (
    df_puerto_temp[
        "Puerto_n_buques"
    ]
    .eq(0)
    .sum()
)


print("\n" + "=" * 80)
print("DÍAS CON ACTIVIDAD NULA")
print("=" * 80)

print(
    f"\n✓ Días con 0 movimientos: "
    f"{dias_cero_movimientos:,}"
)

print(
    f"✓ Días con 0 buques: "
    f"{dias_cero_buques:,}"
)


if dias_cero_movimientos > 0:

    print(
        "\nDías con 0 movimientos:"
    )

    display(
        df_puerto_temp.loc[
            df_puerto_temp[
                "Puerto_movimientos"
            ].eq(0)
        ]
    )


# ============================================================
# 9. RELACIÓN MOVIMIENTOS = LLEGADAS + SALIDAS
# ============================================================

df_puerto_temp[
    "_movimientos_calculados"
] = (
    df_puerto_temp[
        "Puerto_llegadas"
    ]
    +
    df_puerto_temp[
        "Puerto_salidas"
    ]
)


df_puerto_temp[
    "_dif_movimientos"
] = (
    df_puerto_temp[
        "Puerto_movimientos"
    ]
    -
    df_puerto_temp[
        "_movimientos_calculados"
    ]
)


n_movimientos_no_coinciden = (
    ~np.isclose(
        df_puerto_temp[
            "Puerto_movimientos"
        ],
        df_puerto_temp[
            "_movimientos_calculados"
        ],
        equal_nan=True
    )
).sum()


dif_movimientos_max = (
    df_puerto_temp[
        "_dif_movimientos"
    ]
    .abs()
    .max()
)


print("\n" + "=" * 80)
print("COHERENCIA MOVIMIENTOS ↔ LLEGADAS + SALIDAS")
print("=" * 80)

print(
    f"\n✓ Filas donde no coincide: "
    f"{n_movimientos_no_coinciden:,}"
)

print(
    f"✓ Diferencia absoluta máxima: "
    f"{dif_movimientos_max:.2f}"
)


if n_movimientos_no_coinciden == 0:

    print(
        "✓ Puerto_movimientos = "
        "Puerto_llegadas + Puerto_salidas "
        "en todas las fechas."
    )


# ============================================================
# 10. RELACIÓN MOVIMIENTOS / NÚMERO DE BUQUES
# ============================================================

dias_movimientos_menores_buques = (
    (
        df_puerto_temp[
            "Puerto_movimientos"
        ]
        <
        df_puerto_temp[
            "Puerto_n_buques"
        ]
    )
    .sum()
)


correlacion_mov_buques = (
    df_puerto_temp[
        [
            "Puerto_movimientos",
            "Puerto_n_buques"
        ]
    ]
    .corr()
    .iloc[
        0,
        1
    ]
)


print("\n" + "=" * 80)
print("RELACIÓN MOVIMIENTOS ↔ NÚMERO DE BUQUES")
print("=" * 80)

print(
    f"\n✓ Días con movimientos < nº de buques: "
    f"{dias_movimientos_menores_buques:,}"
)

print(
    f"✓ Correlación movimientos / nº buques: "
    f"{correlacion_mov_buques:.4f}"
)


# ============================================================
# 11. CONTROL DE VARIABLES DE TIPOLOGÍA
# ============================================================

variables_tipologia = [
    "Puerto_carga",
    "Puerto_ferry_pasajeros",
    "Puerto_portacontenedores",
    "Puerto_ro_ro",
    "Puerto_tanque"
]


df_puerto_temp[
    "_tipologias_suma"
] = (
    df_puerto_temp[
        variables_tipologia
    ]
    .sum(axis=1)
)


dias_tipologias_superan_buques = (
    (
        df_puerto_temp[
            "_tipologias_suma"
        ]
        >
        df_puerto_temp[
            "Puerto_n_buques"
        ]
    )
    .sum()
)


print("\n" + "=" * 80)
print("CONTROL DE TIPOLOGÍAS DE BUQUES")
print("=" * 80)

print(
    f"\n✓ Días en que la suma de tipologías "
    f"supera Puerto_n_buques: "
    f"{dias_tipologias_superan_buques:,}"
)

print(
    "ℹ Este control es diagnóstico: "
    "las categorías podrían no ser mutuamente excluyentes."
)


# ============================================================
# 12. CONTROL DE NEGATIVOS
# ============================================================

variables_puerto = [
    "Puerto_movimientos",
    "Puerto_n_buques",
    "Puerto_eslora_total",
    "Puerto_eslora_media",
    "Puerto_calado_medio",
    "Puerto_llegadas",
    "Puerto_salidas",
    *variables_tipologia
]


n_negativos = (
    (
        df_puerto_temp[
            variables_puerto
        ] < 0
    )
    .sum()
    .sum()
)


print("\n" + "=" * 80)
print("CONTROL DE VALORES NEGATIVOS")
print("=" * 80)

print(
    f"\n✓ Valores negativos: "
    f"{n_negativos:,}"
)


# ============================================================
# 13. CONTROL DE INFINITOS
# ============================================================

n_inf = (
    np.isinf(
        df_puerto_temp[
            variables_puerto
        ]
    )
    .sum()
    .sum()
)


print("\n" + "=" * 80)
print("CONTROL DE VALORES INFINITOS")
print("=" * 80)

print(
    f"\n✓ Valores infinitos: "
    f"{n_inf:,}"
)


# ============================================================
# 14. RESUMEN ANUAL
# ============================================================

resumen_anual = (
    df_puerto_temp
    .groupby(
        "_anio_control"
    )
    .agg(
        movimientos_media=(
            "Puerto_movimientos",
            "mean"
        ),

        buques_media=(
            "Puerto_n_buques",
            "mean"
        ),

        eslora_total_media=(
            "Puerto_eslora_total",
            "mean"
        ),

        llegadas_media=(
            "Puerto_llegadas",
            "mean"
        ),

        salidas_media=(
            "Puerto_salidas",
            "mean"
        ),

        portacontenedores_media=(
            "Puerto_portacontenedores",
            "mean"
        ),

        ferry_media=(
            "Puerto_ferry_pasajeros",
            "mean"
        )
    )
    .reset_index()
    .rename(
        columns={
            "_anio_control":
                "anio"
        }
    )
)


print("\n" + "=" * 80)
print("RESUMEN ANUAL DE ACTIVIDAD MARÍTIMA")
print("=" * 80)

display(
    resumen_anual.round(2)
)


# ============================================================
# 15. ELIMINAR VARIABLES AUXILIARES
# ============================================================

df_puerto_temp = (
    df_puerto_temp
    .drop(
        columns=[
            "_anio_control",
            "_movimientos_calculados",
            "_dif_movimientos",
            "_tipologias_suma"
        ]
    )
)


# ============================================================
# 16. RESUMEN FINAL
# ============================================================

print("\n" + "=" * 80)
print("AUDITORÍA 7.2 FINALIZADA")
print("=" * 80)

print(
    f"\n✓ Periodo auditado: "
    f"{fecha_inicio.date()} → "
    f"{fecha_fin.date()}"
)

print(
    f"✓ Días esperados: "
    f"{len(calendario_esperado):,}"
)

print(
    f"✓ Días observados: "
    f"{len(fechas_observadas):,}"
)

print(
    f"✓ Fechas faltantes: "
    f"{len(fechas_faltantes):,}"
)

print(
    f"✓ Duplicados de fecha: "
    f"{duplicados_fecha:,}"
)

print(
    f"✓ Filas con NaN en variables medias: "
    f"{len(filas_nan_medias):,}"
)

print(
    f"✓ Días con 0 movimientos: "
    f"{dias_cero_movimientos:,}"
)

print(
    f"✓ Inconsistencias "
    f"movimientos vs llegadas+salidas: "
    f"{n_movimientos_no_coinciden:,}"
)

print(
    f"✓ Valores negativos: "
    f"{n_negativos:,}"
)

print(
    f"✓ Valores infinitos: "
    f"{n_inf:,}"
)

print(
    "✓ No se ha realizado ninguna imputación."
)

print(
    "\n✓ Dataset preparado para "
    "selección de indicadores en 7.3."
)

7.2 — AUDITORÍA TEMPORAL Y COHERENCIA INTERNA

COBERTURA TEMPORAL GENERAL

✓ Fecha inicial: 2018-01-01
✓ Fecha final: 2024-12-31
✓ Fechas diferentes: 2,557
✓ Fechas inválidas: 0
✓ Duplicados de fecha: 0

CONTINUIDAD DEL CALENDARIO

✓ Días esperados 2018–2024: 2,557
✓ Días observados: 2,557
✓ Fechas faltantes: 0
✓ Fechas fuera del periodo objetivo: 0
✓ El calendario 2018–2024 está completo.

COBERTURA POR AÑO


,anio,registros,dias,movimientos_validos,buques_validos,dias_teoricos,cobertura_pct
0,2018,365,365,365,365,365,100.0
1,2019,365,365,365,365,365,100.0
2,2020,366,366,366,366,366,100.0
3,2021,365,365,365,365,365,100.0
4,2022,365,365,365,365,365,100.0
5,2023,365,365,365,365,365,100.0
6,2024,366,366,366,366,366,100.0



ANÁLISIS DE LOS VALORES AUSENTES

✓ Filas con algún NaN en eslora/calado medios: 1


,fecha,Puerto_movimientos,Puerto_n_buques,Puerto_eslora_total,Puerto_eslora_media,Puerto_calado_medio,Puerto_llegadas,Puerto_salidas,Puerto_carga,Puerto_ferry_pasajeros,Puerto_portacontenedores,Puerto_ro_ro,Puerto_tanque
750,2020-01-21,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0



✓ Filas NaN con 0 buques: 1
✓ Filas NaN con 0 movimientos: 1

DÍAS CON ACTIVIDAD NULA

✓ Días con 0 movimientos: 1
✓ Días con 0 buques: 1

Días con 0 movimientos:


,fecha,Puerto_movimientos,Puerto_n_buques,Puerto_eslora_total,Puerto_eslora_media,Puerto_calado_medio,Puerto_llegadas,Puerto_salidas,Puerto_carga,Puerto_ferry_pasajeros,Puerto_portacontenedores,Puerto_ro_ro,Puerto_tanque,anio,mes,dia_semana,Periodo_COVID,_anio_control
750,2020-01-21,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2020,1,1,covid,2020



COHERENCIA MOVIMIENTOS ↔ LLEGADAS + SALIDAS

✓ Filas donde no coincide: 0
✓ Diferencia absoluta máxima: 0.00
✓ Puerto_movimientos = Puerto_llegadas + Puerto_salidas en todas las fechas.

RELACIÓN MOVIMIENTOS ↔ NÚMERO DE BUQUES

✓ Días con movimientos < nº de buques: 0
✓ Correlación movimientos / nº buques: 0.9265

CONTROL DE TIPOLOGÍAS DE BUQUES

✓ Días en que la suma de tipologías supera Puerto_n_buques: 2,556
ℹ Este control es diagnóstico: las categorías podrían no ser mutuamente excluyentes.

CONTROL DE VALORES NEGATIVOS

✓ Valores negativos: 0

CONTROL DE VALORES INFINITOS

✓ Valores infinitos: 0

RESUMEN ANUAL DE ACTIVIDAD MARÍTIMA


,anio,movimientos_media,buques_media,eslora_total_media,llegadas_media,salidas_media,portacontenedores_media,ferry_media
0,2018,57.75,37.95,10818.97,28.87,28.87,15.71,21.39
1,2019,58.32,37.79,10885.88,29.17,29.15,15.57,21.25
2,2020,45.32,30.01,8168.27,22.66,22.66,14.43,12.52
3,2021,51.05,33.07,9255.91,25.53,25.52,14.81,15.78
4,2022,58.15,36.92,11000.72,29.11,29.04,13.73,23.14
5,2023,57.01,36.32,10974.96,28.50,28.50,15.14,23.24
6,2024,55.82,36.05,10788.32,27.86,27.96,15.07,21.96



AUDITORÍA 7.2 FINALIZADA

✓ Periodo auditado: 2018-01-01 → 2024-12-31
✓ Días esperados: 2,557
✓ Días observados: 2,557
✓ Fechas faltantes: 0
✓ Duplicados de fecha: 0
✓ Filas con NaN en variables medias: 1
✓ Días con 0 movimientos: 1
✓ Inconsistencias movimientos vs llegadas+salidas: 0
✓ Valores negativos: 0
✓ Valores infinitos: 0
✓ No se ha realizado ninguna imputación.

✓ Dataset preparado para selección de indicadores en 7.3.


### Resultados — 7.2. Auditoría temporal y coherencia interna del transporte marítimo

La serie de transporte marítimo presenta una cobertura temporal completa durante todo el periodo de estudio.

Se observan **2.557 días entre el 1 de enero de 2018 y el 31 de diciembre de 2024**, coincidiendo exactamente con los 2.557 días esperados del calendario. No existen fechas faltantes, fechas fuera del periodo objetivo, fechas inválidas ni duplicados temporales.

La cobertura es del **100 % en todos los años entre 2018 y 2024**, incluyendo los años bisiestos 2020 y 2024.

La auditoría de valores ausentes identifica una única fecha con `NaN` en `Puerto_eslora_media` y `Puerto_calado_medio`: **21 de enero de 2020**. Ese día presenta simultáneamente:

- 0 movimientos;
- 0 buques;
- 0 llegadas;
- 0 salidas;
- 0 actividad en las tipologías de buque analizadas.

Por tanto, estos valores ausentes no representan pérdida de información, sino una consecuencia estructural de la ausencia de buques sobre los que calcular una eslora o un calado medios. Se conserva esta situación explícitamente y no se realiza ninguna imputación en esta etapa.

La relación entre los indicadores de movimientos resulta completamente coherente:

`Puerto_movimientos = Puerto_llegadas + Puerto_salidas`

La igualdad se cumple en las **2.557 fechas**, sin ninguna discrepancia. En consecuencia, estas tres variables contienen una redundancia matemática que deberá considerarse durante la selección de indicadores.

`Puerto_movimientos` y `Puerto_n_buques` presentan asimismo una asociación elevada, con una correlación de **r = 0,9265**, aunque no constituyen variables matemáticamente equivalentes.

La suma de las variables de tipología supera `Puerto_n_buques` en 2.556 días. Este resultado indica que dichas variables no deben interpretarse como una partición mutuamente excluyente del número diario de buques, por lo que no se utiliza su suma como control de identidad respecto a `Puerto_n_buques`.

No se detectan valores negativos ni infinitos.

La evolución anual refleja además una alteración clara de la actividad marítima durante el periodo COVID. La media diaria de movimientos disminuye desde **58,32 en 2019 hasta 45,32 en 2020**, recuperándose posteriormente hasta **58,15 en 2022**. El número medio diario de buques presenta un comportamiento similar.

> **Conclusión:** el bloque marítimo presenta una cobertura temporal excelente y una elevada coherencia interna. La única ausencia detectada corresponde a un día sin actividad marítima y no constituye un error de calidad de datos. Se identifica además una redundancia matemática entre movimientos, llegadas y salidas que deberá resolverse en la selección de indicadores de la etapa 7.3.

## 7.3. Análisis de redundancia y selección de indicadores de transporte marítimo

Una vez comprobada la cobertura y coherencia interna de la serie marítima, se analiza la información aportada por los diferentes indicadores con el objetivo de obtener un conjunto reducido e interpretable de variables para su incorporación al dataset maestro.

La auditoría anterior ha demostrado que `Puerto_movimientos` es exactamente igual a la suma de `Puerto_llegadas` y `Puerto_salidas`. Por tanto, estas variables no deben incorporarse simultáneamente como predictores independientes.

Asimismo, `Puerto_movimientos` y `Puerto_n_buques` presentan una asociación elevada, aunque no constituyen variables matemáticamente equivalentes.

En esta etapa se analiza:

- la correlación entre los indicadores cuantitativos;
- la redundancia entre movimientos, llegadas y salidas;
- la relación entre intensidad de actividad y características físicas de los buques;
- el comportamiento de las diferentes tipologías de tráfico marítimo;
- la variabilidad de cada indicador;
- la presencia de variables con variabilidad insuficiente;
- las correlaciones elevadas entre posibles predictores.

La selección final buscará representar dimensiones diferentes de la actividad portuaria, evitando incorporar simultáneamente variables matemáticamente redundantes.

No se realiza ninguna imputación. El único día sin actividad marítima mantiene como `NaN` las variables medias que no pueden definirse en ausencia de buques.

> **Objetivo:** seleccionar un conjunto compacto de indicadores marítimos que represente intensidad, dimensión física y composición de la actividad portuaria, reduciendo redundancias antes de su incorporación al dataset maestro.

In [ ]:
# ============================================================
# 7.3. ANÁLISIS DE REDUNDANCIA Y SELECCIÓN
# DE INDICADORES DE TRANSPORTE MARÍTIMO
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 1. COMPROBAR DATASET PROCEDENTE DE 7.2
# ============================================================

if "df_puerto_temp" not in globals():

    raise NameError(
        "No existe df_puerto_temp. "
        "Debe ejecutarse previamente la etapa 7.2."
    )


df_puerto_sel = (
    df_puerto_temp
    .copy()
)


print("=" * 80)
print("7.3 — ANÁLISIS Y SELECCIÓN DE INDICADORES MARÍTIMOS")
print("=" * 80)


# ============================================================
# 2. VARIABLES CANDIDATAS
# ============================================================

variables_candidatas = [
    "Puerto_movimientos",
    "Puerto_n_buques",
    "Puerto_eslora_total",
    "Puerto_eslora_media",
    "Puerto_calado_medio",
    "Puerto_llegadas",
    "Puerto_salidas",
    "Puerto_carga",
    "Puerto_ferry_pasajeros",
    "Puerto_portacontenedores",
    "Puerto_ro_ro",
    "Puerto_tanque"
]


variables_faltantes = [
    variable
    for variable in variables_candidatas
    if variable not in df_puerto_sel.columns
]


if variables_faltantes:

    raise KeyError(
        "Faltan variables marítimas necesarias:\n"
        f"{variables_faltantes}"
    )


print("\n" + "=" * 80)
print("VARIABLES CANDIDATAS")
print("=" * 80)

print(
    f"\n✓ Variables candidatas: "
    f"{len(variables_candidatas)}"
)

for variable in variables_candidatas:

    print(
        f"  • {variable}"
    )


# ============================================================
# 3. CONTROL DE NaN
# ============================================================

tabla_nan_73 = pd.DataFrame({
    "variable":
        variables_candidatas,

    "n_nan":
        [
            df_puerto_sel[
                variable
            ]
            .isna()
            .sum()

            for variable
            in variables_candidatas
        ]
})


tabla_nan_73[
    "pct_nan"
] = (
    tabla_nan_73[
        "n_nan"
    ]
    /
    len(df_puerto_sel)
    *
    100
).round(3)


print("\n" + "=" * 80)
print("VALORES AUSENTES")
print("=" * 80)

display(
    tabla_nan_73
)


# ============================================================
# 4. ESTADÍSTICOS Y VARIABILIDAD
# ============================================================

resumen_variabilidad = []

for variable in variables_candidatas:

    serie = (
        df_puerto_sel[
            variable
        ]
    )

    media = serie.mean()
    mediana = serie.median()
    desviacion = serie.std()
    minimo = serie.min()
    maximo = serie.max()
    n_unicos = serie.nunique(
        dropna=True
    )

    if (
        pd.notna(media)
        and
        media != 0
    ):

        cv = (
            desviacion
            /
            abs(media)
            *
            100
        )

    else:

        cv = np.nan


    resumen_variabilidad.append({
        "variable":
            variable,

        "media":
            media,

        "mediana":
            mediana,

        "desviacion_std":
            desviacion,

        "minimo":
            minimo,

        "maximo":
            maximo,

        "n_valores_unicos":
            n_unicos,

        "coef_variacion_pct":
            cv
    })


df_variabilidad_puerto = pd.DataFrame(
    resumen_variabilidad
)


print("\n" + "=" * 80)
print("VARIABILIDAD DE LOS INDICADORES")
print("=" * 80)

display(
    df_variabilidad_puerto.round(3)
)


# ============================================================
# 5. VARIABLES CON VARIABILIDAD MUY BAJA
# ============================================================

variables_baja_variabilidad = (
    df_variabilidad_puerto.loc[
        df_variabilidad_puerto[
            "n_valores_unicos"
        ] <= 2,
        [
            "variable",
            "n_valores_unicos",
            "coef_variacion_pct"
        ]
    ]
)


print("\n" + "=" * 80)
print("CONTROL DE VARIABILIDAD MUY BAJA")
print("=" * 80)

if variables_baja_variabilidad.empty:

    print(
        "\n✓ No se detectan variables "
        "prácticamente constantes."
    )

else:

    display(
        variables_baja_variabilidad
    )


# ============================================================
# 6. MATRIZ DE CORRELACIÓN COMPLETA
# ============================================================

matriz_corr_puerto = (
    df_puerto_sel[
        variables_candidatas
    ]
    .corr()
)


print("\n" + "=" * 80)
print("MATRIZ DE CORRELACIÓN — VARIABLES CANDIDATAS")
print("=" * 80)

display(
    matriz_corr_puerto.round(3)
)


# ============================================================
# 7. EXTRAER PARES CON CORRELACIÓN ELEVADA
# ============================================================

pares_correlacion = []

for i, variable_1 in enumerate(
    variables_candidatas
):

    for variable_2 in variables_candidatas[
        i + 1:
    ]:

        correlacion = (
            matriz_corr_puerto.loc[
                variable_1,
                variable_2
            ]
        )

        if (
            pd.notna(correlacion)
            and
            abs(correlacion) >= 0.90
        ):

            pares_correlacion.append({
                "variable_1":
                    variable_1,

                "variable_2":
                    variable_2,

                "correlacion":
                    correlacion
            })


df_corr_altas_puerto = (
    pd.DataFrame(
        pares_correlacion
    )
)


if not df_corr_altas_puerto.empty:

    df_corr_altas_puerto = (
        df_corr_altas_puerto
        .assign(
            correlacion_abs=lambda x:
                x[
                    "correlacion"
                ].abs()
        )
        .sort_values(
            "correlacion_abs",
            ascending=False
        )
        .drop(
            columns=[
                "correlacion_abs"
            ]
        )
        .reset_index(
            drop=True
        )
    )


print("\n" + "=" * 80)
print("PARES CON |CORRELACIÓN| ≥ 0.90")
print("=" * 80)

if df_corr_altas_puerto.empty:

    print(
        "\n✓ No se detectan pares "
        "con |r| ≥ 0.90."
    )

else:

    display(
        df_corr_altas_puerto.round(4)
    )


# ============================================================
# 8. CONFIRMAR REDUNDANCIA MATEMÁTICA
# MOVIMIENTOS = LLEGADAS + SALIDAS
# ============================================================

diferencia_movimientos = (
    df_puerto_sel[
        "Puerto_movimientos"
    ]
    -
    (
        df_puerto_sel[
            "Puerto_llegadas"
        ]
        +
        df_puerto_sel[
            "Puerto_salidas"
        ]
    )
)


n_inconsistencias_mov = (
    ~np.isclose(
        diferencia_movimientos,
        0,
        equal_nan=True
    )
).sum()


print("\n" + "=" * 80)
print("REDUNDANCIA MATEMÁTICA")
print("=" * 80)

print(
    f"\n✓ Inconsistencias en "
    f"movimientos = llegadas + salidas: "
    f"{n_inconsistencias_mov:,}"
)


if n_inconsistencias_mov == 0:

    print(
        "✓ Se confirma redundancia exacta."
    )

    print(
        "✓ Llegadas y salidas no se "
        "seleccionarán junto con movimientos."
    )


# ============================================================
# 9. EQUILIBRIO LLEGADAS / SALIDAS
# ============================================================

df_puerto_sel[
    "_balance_llegadas_salidas"
] = (
    df_puerto_sel[
        "Puerto_llegadas"
    ]
    -
    df_puerto_sel[
        "Puerto_salidas"
    ]
)


balance_media = (
    df_puerto_sel[
        "_balance_llegadas_salidas"
    ]
    .mean()
)

balance_abs_media = (
    df_puerto_sel[
        "_balance_llegadas_salidas"
    ]
    .abs()
    .mean()
)

balance_max = (
    df_puerto_sel[
        "_balance_llegadas_salidas"
    ]
    .abs()
    .max()
)


print("\n" + "=" * 80)
print("BALANCE LLEGADAS / SALIDAS")
print("=" * 80)

print(
    f"\n✓ Balance medio "
    f"(llegadas - salidas): "
    f"{balance_media:.3f}"
)

print(
    f"✓ Diferencia absoluta media: "
    f"{balance_abs_media:.3f}"
)

print(
    f"✓ Diferencia absoluta máxima: "
    f"{balance_max:.3f}"
)


# ============================================================
# 10. RELACIÓN ESLORA TOTAL / Nº BUQUES
# ============================================================

df_puerto_sel[
    "_eslora_media_calculada"
] = np.where(
    df_puerto_sel[
        "Puerto_n_buques"
    ] > 0,

    (
        df_puerto_sel[
            "Puerto_eslora_total"
        ]
        /
        df_puerto_sel[
            "Puerto_n_buques"
        ]
    ),

    np.nan
)


diferencia_eslora_media = (
    df_puerto_sel[
        "Puerto_eslora_media"
    ]
    -
    df_puerto_sel[
        "_eslora_media_calculada"
    ]
)


n_eslora_no_coincide = (
    ~np.isclose(
        df_puerto_sel[
            "Puerto_eslora_media"
        ],
        df_puerto_sel[
            "_eslora_media_calculada"
        ],
        rtol=1e-5,
        atol=1e-5,
        equal_nan=True
    )
).sum()


print("\n" + "=" * 80)
print("RELACIÓN ESLORA TOTAL / ESLORA MEDIA")
print("=" * 80)

print(
    f"\n✓ Filas donde Puerto_eslora_media "
    f"no coincide con eslora_total / n_buques: "
    f"{n_eslora_no_coincide:,}"
)


if n_eslora_no_coincide == 0:

    print(
        "✓ Puerto_eslora_media es derivable "
        "de Puerto_eslora_total y Puerto_n_buques."
    )


# ============================================================
# 11. ANALIZAR TIPOLOGÍAS
# ============================================================

variables_tipologia = [
    "Puerto_carga",
    "Puerto_ferry_pasajeros",
    "Puerto_portacontenedores",
    "Puerto_ro_ro",
    "Puerto_tanque"
]


resumen_tipologias = (
    df_puerto_sel[
        variables_tipologia
    ]
    .agg(
        [
            "mean",
            "median",
            "std",
            "min",
            "max"
        ]
    )
    .T
)


resumen_tipologias[
    "pct_dias_cero"
] = [
    (
        df_puerto_sel[
            variable
        ].eq(0).mean()
        *
        100
    )

    for variable
    in variables_tipologia
]


print("\n" + "=" * 80)
print("COMPORTAMIENTO DE LAS TIPOLOGÍAS")
print("=" * 80)

display(
    resumen_tipologias.round(3)
)


# ============================================================
# 12. CORRELACIÓN DE TIPOLOGÍAS CON ACTIVIDAD GENERAL
# ============================================================

corr_tipologias_actividad = []

for variable in variables_tipologia:

    corr_mov = (
        df_puerto_sel[
            [
                variable,
                "Puerto_movimientos"
            ]
        ]
        .corr()
        .iloc[
            0,
            1
        ]
    )

    corr_buques = (
        df_puerto_sel[
            [
                variable,
                "Puerto_n_buques"
            ]
        ]
        .corr()
        .iloc[
            0,
            1
        ]
    )

    corr_tipologias_actividad.append({
        "variable":
            variable,

        "corr_movimientos":
            corr_mov,

        "corr_n_buques":
            corr_buques
    })


df_corr_tipologias = pd.DataFrame(
    corr_tipologias_actividad
)


print("\n" + "=" * 80)
print("TIPOLOGÍAS ↔ ACTIVIDAD GENERAL")
print("=" * 80)

display(
    df_corr_tipologias.round(4)
)


# ============================================================
# 13. PROPUESTA INICIAL DE INDICADORES
# ============================================================
#
# Todavía NO eliminamos físicamente las variables del dataset.
#
# La propuesta busca representar:
#
# 1. intensidad de actividad;
# 2. dimensión física de los buques;
# 3. composición del tráfico.
#
# La selección definitiva se valida con los resultados
# obtenidos en esta misma celda.
# ============================================================

variables_propuesta = [
    "Puerto_movimientos",
    "Puerto_eslora_media",
    "Puerto_calado_medio",
    "Puerto_ferry_pasajeros",
    "Puerto_portacontenedores",
    "Puerto_tanque"
]


print("\n" + "=" * 80)
print("PROPUESTA INICIAL DE INDICADORES")
print("=" * 80)

for variable in variables_propuesta:

    print(
        f"  • {variable}"
    )


# ============================================================
# 14. MATRIZ DE CORRELACIÓN DE LA PROPUESTA
# ============================================================

matriz_corr_propuesta = (
    df_puerto_sel[
        variables_propuesta
    ]
    .corr()
)


print("\n" + "=" * 80)
print("CORRELACIÓN — PROPUESTA INICIAL")
print("=" * 80)

display(
    matriz_corr_propuesta.round(3)
)


# ============================================================
# 15. PARES MUY CORRELACIONADOS EN LA PROPUESTA
# ============================================================

pares_propuesta = []

for i, variable_1 in enumerate(
    variables_propuesta
):

    for variable_2 in variables_propuesta[
        i + 1:
    ]:

        correlacion = (
            matriz_corr_propuesta.loc[
                variable_1,
                variable_2
            ]
        )

        if (
            pd.notna(correlacion)
            and
            abs(correlacion) >= 0.95
        ):

            pares_propuesta.append({
                "variable_1":
                    variable_1,

                "variable_2":
                    variable_2,

                "correlacion":
                    correlacion
            })


df_corr_propuesta_altas = pd.DataFrame(
    pares_propuesta
)


print("\n" + "=" * 80)
print("REDUNDANCIA EN LA PROPUESTA — |r| ≥ 0.95")
print("=" * 80)

if df_corr_propuesta_altas.empty:

    print(
        "\n✓ Ningún par de la propuesta "
        "presenta |r| ≥ 0.95."
    )

else:

    display(
        df_corr_propuesta_altas.round(4)
    )


# ============================================================
# 16. CREAR DATASET CANDIDATO
# ============================================================

df_puerto_modelo = (
    df_puerto_sel[
        [
            "fecha",
            *variables_propuesta
        ]
    ]
    .copy()
)


# ============================================================
# 17. CONTROL FINAL DEL DATASET CANDIDATO
# ============================================================

duplicados_final = (
    df_puerto_modelo
    .duplicated(
        subset=[
            "fecha"
        ]
    )
    .sum()
)


print("\n" + "=" * 80)
print("CONTROL DEL DATASET CANDIDATO")
print("=" * 80)

print(
    f"\n✓ Registros: "
    f"{len(df_puerto_modelo):,}"
)

print(
    f"✓ Variables: "
    f"{df_puerto_modelo.shape[1]}"
)

print(
    f"✓ Fechas diferentes: "
    f"{df_puerto_modelo['fecha'].nunique():,}"
)

print(
    f"✓ Duplicados de fecha: "
    f"{duplicados_final:,}"
)


print("\nNaN por variable:")

display(
    pd.DataFrame({
        "variable":
            variables_propuesta,

        "n_nan":
            [
                df_puerto_modelo[
                    variable
                ]
                .isna()
                .sum()

                for variable
                in variables_propuesta
            ]
    })
)


# ============================================================
# 18. LIMPIAR VARIABLES AUXILIARES
# ============================================================

df_puerto_sel = (
    df_puerto_sel
    .drop(
        columns=[
            "_balance_llegadas_salidas",
            "_eslora_media_calculada"
        ]
    )
)


# ============================================================
# 19. RESUMEN FINAL
# ============================================================

print("\n" + "=" * 80)
print("ANÁLISIS 7.3 FINALIZADO")
print("=" * 80)

print(
    f"\n✓ Variables candidatas analizadas: "
    f"{len(variables_candidatas)}"
)

print(
    f"✓ Variables de la propuesta inicial: "
    f"{len(variables_propuesta)}"
)

print(
    "✓ Redundancia movimientos / "
    "llegadas / salidas evaluada."
)

print(
    "✓ Relación eslora total / "
    "eslora media evaluada."
)

print(
    "✓ Tipologías de buque evaluadas."
)

print(
    "✓ Correlaciones de la propuesta "
    "calculadas."
)

print(
    "✓ No se ha realizado ninguna imputación."
)

print(
    "\n✓ Resultados preparados para "
    "cerrar la selección definitiva."
)


7.3 — ANÁLISIS Y SELECCIÓN DE INDICADORES MARÍTIMOS

VARIABLES CANDIDATAS

✓ Variables candidatas: 12
  • Puerto_movimientos
  • Puerto_n_buques
  • Puerto_eslora_total
  • Puerto_eslora_media
  • Puerto_calado_medio
  • Puerto_llegadas
  • Puerto_salidas
  • Puerto_carga
  • Puerto_ferry_pasajeros
  • Puerto_portacontenedores
  • Puerto_ro_ro
  • Puerto_tanque

VALORES AUSENTES


,variable,n_nan,pct_nan
0,Puerto_movimientos,0,0.000
1,Puerto_n_buques,0,0.000
2,Puerto_eslora_total,0,0.000
3,Puerto_eslora_media,1,0.039
4,Puerto_calado_medio,1,0.039
5,Puerto_llegadas,0,0.000
6,Puerto_salidas,0,0.000
7,Puerto_carga,0,0.000
8,Puerto_ferry_pasajeros,0,0.000
9,Puerto_portacontenedores,0,0.000



VARIABILIDAD DE LOS INDICADORES


,variable,media,mediana,desviacion_std,minimo,maximo,n_valores_unicos,coef_variacion_pct
0,Puerto_movimientos,54.771,55.000,11.587,0.000,89.000,72,21.155
1,Puerto_n_buques,35.442,36.000,6.600,0.000,62.000,47,18.623
2,Puerto_eslora_total,10269.812,10368.520,2214.432,0.000,16789.840,2550,21.563
3,Puerto_eslora_media,187.639,186.943,11.294,150.283,231.894,2553,6.019
4,Puerto_calado_medio,8.460,8.428,0.468,6.503,10.479,2537,5.536
5,Puerto_llegadas,27.384,28.000,6.040,0.000,52.000,41,22.058
6,Puerto_salidas,27.387,27.000,6.329,0.000,49.000,44,23.111
7,Puerto_carga,4.373,4.000,2.860,0.000,20.000,20,65.390
8,Puerto_ferry_pasajeros,19.895,21.000,6.962,0.000,42.000,42,34.995
9,Puerto_portacontenedores,14.923,15.000,4.921,0.000,43.000,35,32.972



CONTROL DE VARIABILIDAD MUY BAJA

✓ No se detectan variables prácticamente constantes.

MATRIZ DE CORRELACIÓN — VARIABLES CANDIDATAS


...


PROPUESTA INICIAL DE INDICADORES
  • Puerto_movimientos
  • Puerto_eslora_media
  • Puerto_calado_medio
  • Puerto_ferry_pasajeros
  • Puerto_portacontenedores
  • Puerto_tanque

CORRELACIÓN — PROPUESTA INICIAL


,Puerto_movimientos,Puerto_eslora_media,Puerto_calado_medio,Puerto_ferry_pasajeros,Puerto_portacontenedores,Puerto_tanque
Puerto_movimientos,1.000,-0.056,-0.191,0.678,0.531,0.444
Puerto_eslora_media,-0.056,1.000,0.484,0.273,-0.016,-0.253
Puerto_calado_medio,-0.191,0.484,1.000,-0.458,0.253,0.106
Puerto_ferry_pasajeros,0.678,0.273,-0.458,1.000,0.051,0.062
Puerto_portacontenedores,0.531,-0.016,0.253,0.051,1.000,0.027
Puerto_tanque,0.444,-0.253,0.106,0.062,0.027,1.000



REDUNDANCIA EN LA PROPUESTA — |r| ≥ 0.95

✓ Ningún par de la propuesta presenta |r| ≥ 0.95.

CONTROL DEL DATASET CANDIDATO

✓ Registros: 2,557
✓ Variables: 7
✓ Fechas diferentes: 2,557
✓ Duplicados de fecha: 0

NaN por variable:


,variable,n_nan
0,Puerto_movimientos,0
1,Puerto_eslora_media,1
2,Puerto_calado_medio,1
3,Puerto_ferry_pasajeros,0
4,Puerto_portacontenedores,0
5,Puerto_tanque,0



ANÁLISIS 7.3 FINALIZADO

✓ Variables candidatas analizadas: 12
✓ Variables de la propuesta inicial: 6
✓ Redundancia movimientos / llegadas / salidas evaluada.
✓ Relación eslora total / eslora media evaluada.
✓ Tipologías de buque evaluadas.
✓ Correlaciones de la propuesta calculadas.
✓ No se ha realizado ninguna imputación.

✓ Resultados preparados para cerrar la selección definitiva.


### Resultados — 7.3. Análisis de redundancia y selección de indicadores marítimos

Se analizan **12 indicadores candidatos** de actividad marítima con el objetivo de reducir la dimensionalidad del bloque y evitar la incorporación de variables redundantes al dataset maestro.

Todas las variables presentan variabilidad suficiente y no se detectan indicadores prácticamente constantes.

El análisis confirma una elevada redundancia dentro del grupo de variables asociadas a la intensidad general de actividad. `Puerto_movimientos` presenta correlaciones elevadas con `Puerto_eslora_total` (r = 0,9619) y `Puerto_n_buques` (r = 0,9265).

Además, se confirma la identidad matemática:

`Puerto_movimientos = Puerto_llegadas + Puerto_salidas`

en la totalidad de los registros. En consecuencia, se selecciona `Puerto_movimientos` como indicador sintético de intensidad y se excluyen `Puerto_llegadas` y `Puerto_salidas` de la selección final.

`Puerto_eslora_media` no resulta directamente reproducible mediante el cociente entre `Puerto_eslora_total` y `Puerto_n_buques`, por lo que se conserva como indicador independiente de las características físicas de los buques. `Puerto_calado_medio` se mantiene asimismo como indicador complementario de dimensión y características operativas.

Las diferentes tipologías presentan patrones de asociación diferenciados respecto a la actividad general. Para mantener un bloque compacto e interpretable se seleccionan tres categorías representativas: tráfico de ferris de pasajeros, portacontenedores y buques tanque.

La selección definitiva queda constituida por seis indicadores:

- `Puerto_movimientos`: intensidad global de actividad portuaria;
- `Puerto_eslora_media`: dimensión media de los buques;
- `Puerto_calado_medio`: característica física y operativa;
- `Puerto_ferry_pasajeros`: actividad asociada al tráfico de pasajeros;
- `Puerto_portacontenedores`: actividad asociada al tráfico de contenedores;
- `Puerto_tanque`: actividad asociada a buques tanque.

La matriz de correlación del conjunto seleccionado no presenta ningún par con una correlación absoluta igual o superior a 0,95, reduciendo de forma sustancial la redundancia presente en el conjunto original.

El dataset candidato resultante contiene **2.557 registros diarios y 7 variables**, incluyendo la fecha, sin duplicados temporales.

Únicamente permanecen dos valores ausentes: uno en `Puerto_eslora_media` y otro en `Puerto_calado_medio`. Ambos corresponden al mismo día sin actividad marítima identificado previamente y se mantienen sin imputar en esta etapa.

> **Conclusión:** el bloque marítimo se reduce de 12 indicadores candidatos a 6 variables complementarias que representan intensidad, características físicas y composición de la actividad portuaria. La selección mantiene la información relevante del bloque al tiempo que reduce redundancias antes de su integración con el dataset maestro.

## 7.4. Preintegración temporal y exportación del bloque de transporte marítimo

Una vez seleccionados los indicadores marítimos definitivos, se prepara el dataset para su incorporación al dataset maestro.

A diferencia del bloque aéreo, la serie marítima presenta cobertura temporal completa durante todo el periodo de estudio, desde el 1 de enero de 2018 hasta el 31 de diciembre de 2024. Por tanto, no es necesario reconstruir ni ampliar el calendario.

Se conservan los seis indicadores seleccionados:

- `Puerto_movimientos`
- `Puerto_eslora_media`
- `Puerto_calado_medio`
- `Puerto_ferry_pasajeros`
- `Puerto_portacontenedores`
- `Puerto_tanque`

El dataset mantiene una única observación por fecha y cobertura completa en la variable principal de intensidad.

Los únicos valores ausentes corresponden a `Puerto_eslora_media` y `Puerto_calado_medio` en el único día sin actividad marítima del periodo. Estos valores se conservan como `NaN`, ya que no representan pérdida de información sino la imposibilidad de calcular una media en ausencia de buques.

Durante esta etapa se verifica:

- presencia de las 2.557 fechas del periodo 2018–2024;
- unicidad temporal;
- cobertura de cada indicador;
- conservación de los valores ausentes estructurales;
- ausencia de valores infinitos y negativos;
- integridad del dataset tras su exportación y recarga.

> **Objetivo:** generar un dataset marítimo preintegrado, compacto y temporalmente completo, listo para su incorporación al Checkpoint 05.

In [ ]:
# ============================================================
# 7.4. PREINTEGRACIÓN TEMPORAL Y EXPORTACIÓN
# DEL BLOQUE DE TRANSPORTE MARÍTIMO
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. COMPROBAR DATASET PROCEDENTE DE 7.3
# ============================================================

if "df_puerto_modelo" not in globals():

    raise NameError(
        "No existe df_puerto_modelo. "
        "Debe ejecutarse previamente la etapa 7.3."
    )


df_puerto_preintegrado = (
    df_puerto_modelo
    .copy()
)


# ============================================================
# 2. NORMALIZAR FECHA
# ============================================================

df_puerto_preintegrado[
    "fecha"
] = pd.to_datetime(
    df_puerto_preintegrado[
        "fecha"
    ],
    errors="coerce"
)


fechas_invalidas = (
    df_puerto_preintegrado[
        "fecha"
    ]
    .isna()
    .sum()
)


if fechas_invalidas > 0:

    raise ValueError(
        "Existen fechas inválidas."
    )


# ============================================================
# 3. VARIABLES MARÍTIMAS SELECCIONADAS
# ============================================================

variables_puerto_finales = [
    "Puerto_movimientos",
    "Puerto_eslora_media",
    "Puerto_calado_medio",
    "Puerto_ferry_pasajeros",
    "Puerto_portacontenedores",
    "Puerto_tanque"
]


variables_faltantes = [
    variable
    for variable in variables_puerto_finales
    if variable not in df_puerto_preintegrado.columns
]


if variables_faltantes:

    raise KeyError(
        "Faltan variables marítimas seleccionadas:\n"
        f"{variables_faltantes}"
    )


# ============================================================
# 4. CONTROL INICIAL
# ============================================================

print("=" * 80)
print("7.4 — PREINTEGRACIÓN DEL TRANSPORTE MARÍTIMO")
print("=" * 80)

print(
    f"\n✓ Registros: "
    f"{len(df_puerto_preintegrado):,}"
)

print(
    f"✓ Variables: "
    f"{df_puerto_preintegrado.shape[1]}"
)

print(
    f"✓ Fecha inicial: "
    f"{df_puerto_preintegrado['fecha'].min().date()}"
)

print(
    f"✓ Fecha final: "
    f"{df_puerto_preintegrado['fecha'].max().date()}"
)

print(
    f"✓ Fechas diferentes: "
    f"{df_puerto_preintegrado['fecha'].nunique():,}"
)


# ============================================================
# 5. CONTROL DE DUPLICADOS
# ============================================================

duplicados_fecha = (
    df_puerto_preintegrado
    .duplicated(
        subset=[
            "fecha"
        ]
    )
    .sum()
)


print(
    f"✓ Duplicados de fecha: "
    f"{duplicados_fecha:,}"
)


if duplicados_fecha > 0:

    raise ValueError(
        "Existen fechas duplicadas."
    )


# ============================================================
# 6. COMPROBAR CALENDARIO COMPLETO 2018–2024
# ============================================================

calendario_esperado = pd.date_range(
    start="2018-01-01",
    end="2024-12-31",
    freq="D"
)


fechas_observadas = pd.DatetimeIndex(
    df_puerto_preintegrado[
        "fecha"
    ]
    .unique()
)


fechas_faltantes = (
    calendario_esperado
    .difference(
        fechas_observadas
    )
)


print("\n" + "=" * 80)
print("CONTROL DEL CALENDARIO")
print("=" * 80)

print(
    f"\n✓ Días esperados: "
    f"{len(calendario_esperado):,}"
)

print(
    f"✓ Días observados: "
    f"{len(fechas_observadas):,}"
)

print(
    f"✓ Fechas faltantes: "
    f"{len(fechas_faltantes):,}"
)


if len(fechas_faltantes) > 0:

    raise ValueError(
        "El dataset marítimo no cubre "
        "todo el calendario 2018–2024."
    )


# ============================================================
# 7. COBERTURA POR VARIABLE
# ============================================================

resumen_cobertura = []

for variable in variables_puerto_finales:

    n_validos = (
        df_puerto_preintegrado[
            variable
        ]
        .notna()
        .sum()
    )

    n_nan = (
        df_puerto_preintegrado[
            variable
        ]
        .isna()
        .sum()
    )

    cobertura = (
        n_validos
        /
        len(df_puerto_preintegrado)
        *
        100
    )

    resumen_cobertura.append({
        "variable":
            variable,

        "registros_validos":
            n_validos,

        "registros_nan":
            n_nan,

        "cobertura_pct":
            round(
                cobertura,
                3
            )
    })


df_cobertura_puerto_pre = pd.DataFrame(
    resumen_cobertura
)


print("\n" + "=" * 80)
print("COBERTURA DE LOS INDICADORES MARÍTIMOS")
print("=" * 80)

display(
    df_cobertura_puerto_pre
)


# ============================================================
# 8. CONTROL DEL DÍA SIN ACTIVIDAD
# ============================================================

filas_nan = (
    df_puerto_preintegrado.loc[
        df_puerto_preintegrado[
            [
                "Puerto_eslora_media",
                "Puerto_calado_medio"
            ]
        ]
        .isna()
        .any(axis=1)
    ]
    .copy()
)


print("\n" + "=" * 80)
print("CONTROL DE NaN ESTRUCTURALES")
print("=" * 80)

print(
    f"\n✓ Filas con NaN en eslora/calado medios: "
    f"{len(filas_nan):,}"
)


if len(filas_nan) > 0:

    display(
        filas_nan
    )


# ============================================================
# 9. CONTROL DE VALORES INFINITOS
# ============================================================

n_inf = (
    np.isinf(
        df_puerto_preintegrado[
            variables_puerto_finales
        ]
    )
    .sum()
    .sum()
)


print("\n" + "=" * 80)
print("CONTROL DE VALORES INFINITOS")
print("=" * 80)

print(
    f"\n✓ Valores infinitos: "
    f"{n_inf:,}"
)


if n_inf > 0:

    raise ValueError(
        "Existen valores infinitos."
    )


# ============================================================
# 10. CONTROL DE VALORES NEGATIVOS
# ============================================================

n_negativos = (
    (
        df_puerto_preintegrado[
            variables_puerto_finales
        ] < 0
    )
    .sum()
    .sum()
)


print("\n" + "=" * 80)
print("CONTROL DE VALORES NEGATIVOS")
print("=" * 80)

print(
    f"\n✓ Valores negativos: "
    f"{n_negativos:,}"
)


if n_negativos > 0:

    raise ValueError(
        "Existen valores negativos."
    )


# ============================================================
# 11. COBERTURA POR AÑO
# ============================================================

df_puerto_preintegrado[
    "_anio_control"
] = (
    df_puerto_preintegrado[
        "fecha"
    ]
    .dt.year
)


cobertura_anual = (
    df_puerto_preintegrado
    .groupby(
        "_anio_control"
    )
    .agg(
        dias=(
            "fecha",
            "size"
        ),

        dias_con_movimientos=(
            "Puerto_movimientos",
            "count"
        )
    )
    .reset_index()
    .rename(
        columns={
            "_anio_control":
                "anio"
        }
    )
)


cobertura_anual[
    "dias_sin_movimientos"
] = (
    cobertura_anual[
        "dias"
    ]
    -
    cobertura_anual[
        "dias_con_movimientos"
    ]
)


cobertura_anual[
    "cobertura_pct"
] = (
    cobertura_anual[
        "dias_con_movimientos"
    ]
    /
    cobertura_anual[
        "dias"
    ]
    *
    100
).round(2)


print("\n" + "=" * 80)
print("COBERTURA MARÍTIMA POR AÑO")
print("=" * 80)

display(
    cobertura_anual
)


# ============================================================
# 12. ELIMINAR VARIABLE AUXILIAR
# ============================================================

df_puerto_preintegrado = (
    df_puerto_preintegrado
    .drop(
        columns=[
            "_anio_control"
        ]
    )
)


# ============================================================
# 13. ORDENAR TEMPORALMENTE
# ============================================================

df_puerto_preintegrado = (
    df_puerto_preintegrado
    .sort_values(
        "fecha"
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 14. RUTA DE EXPORTACIÓN
# ============================================================

ruta_salida = Path(
    "/content/drive/MyDrive/TFM/"
    "11_Machine_Learning_Temporal/"
    "PREINTEGRACION/03_Transporte_Maritimo"
)


ruta_salida.mkdir(
    parents=True,
    exist_ok=True
)


ruta_archivo = (
    ruta_salida /
    "df_puerto_preintegrado.csv"
)


# ============================================================
# 15. EXPORTAR
# ============================================================

df_puerto_preintegrado.to_csv(
    ruta_archivo,
    index=False,
    encoding="utf-8"
)


# ============================================================
# 16. RECARGAR
# ============================================================

df_control_puerto_pre = pd.read_csv(
    ruta_archivo,
    parse_dates=[
        "fecha"
    ],
    low_memory=False
)


# ============================================================
# 17. CONTROL POSTERIOR A LA EXPORTACIÓN
# ============================================================

filas_recarga = len(
    df_control_puerto_pre
)

columnas_recarga = (
    df_control_puerto_pre.shape[1]
)

duplicados_recarga = (
    df_control_puerto_pre
    .duplicated(
        subset=[
            "fecha"
        ]
    )
    .sum()
)


nan_eslora_antes = (
    df_puerto_preintegrado[
        "Puerto_eslora_media"
    ]
    .isna()
    .sum()
)

nan_eslora_despues = (
    df_control_puerto_pre[
        "Puerto_eslora_media"
    ]
    .isna()
    .sum()
)


nan_calado_antes = (
    df_puerto_preintegrado[
        "Puerto_calado_medio"
    ]
    .isna()
    .sum()
)

nan_calado_despues = (
    df_control_puerto_pre[
        "Puerto_calado_medio"
    ]
    .isna()
    .sum()
)


print("\n" + "=" * 80)
print("CONTROL POSTERIOR A LA EXPORTACIÓN")
print("=" * 80)

print(
    f"\n✓ Filas exportadas: "
    f"{len(df_puerto_preintegrado):,}"
)

print(
    f"✓ Filas recargadas: "
    f"{filas_recarga:,}"
)

print(
    f"✓ Columnas exportadas: "
    f"{df_puerto_preintegrado.shape[1]}"
)

print(
    f"✓ Columnas recargadas: "
    f"{columnas_recarga}"
)

print(
    f"✓ Duplicados tras recarga: "
    f"{duplicados_recarga:,}"
)

print(
    f"✓ NaN eslora antes: "
    f"{nan_eslora_antes:,}"
)

print(
    f"✓ NaN eslora después: "
    f"{nan_eslora_despues:,}"
)

print(
    f"✓ NaN calado antes: "
    f"{nan_calado_antes:,}"
)

print(
    f"✓ NaN calado después: "
    f"{nan_calado_despues:,}"
)


# ============================================================
# 18. CONTROLES ESTRICTOS
# ============================================================

if filas_recarga != len(
    df_puerto_preintegrado
):

    raise ValueError(
        "El número de filas ha cambiado "
        "durante la exportación."
    )


if columnas_recarga != (
    df_puerto_preintegrado.shape[1]
):

    raise ValueError(
        "El número de columnas ha cambiado "
        "durante la exportación."
    )


if duplicados_recarga > 0:

    raise ValueError(
        "La exportación ha generado duplicados."
    )


if (
    nan_eslora_antes
    !=
    nan_eslora_despues
):

    raise ValueError(
        "La exportación ha alterado "
        "los NaN de eslora media."
    )


if (
    nan_calado_antes
    !=
    nan_calado_despues
):

    raise ValueError(
        "La exportación ha alterado "
        "los NaN de calado medio."
    )


# ============================================================
# 19. TAMAÑO DEL ARCHIVO
# ============================================================

tamano_mb = (
    ruta_archivo.stat().st_size
    /
    (1024 ** 2)
)


# ============================================================
# 20. RESUMEN FINAL
# ============================================================

print("\n" + "=" * 80)
print("PREINTEGRACIÓN MARÍTIMA FINALIZADA")
print("=" * 80)

print(
    f"\n✓ Registros: "
    f"{len(df_control_puerto_pre):,}"
)

print(
    f"✓ Variables: "
    f"{df_control_puerto_pre.shape[1]}"
)

print(
    f"✓ Fecha inicial: "
    f"{df_control_puerto_pre['fecha'].min().date()}"
)

print(
    f"✓ Fecha final: "
    f"{df_control_puerto_pre['fecha'].max().date()}"
)

print(
    f"✓ Fechas diferentes: "
    f"{df_control_puerto_pre['fecha'].nunique():,}"
)

print(
    f"✓ Cobertura Puerto_movimientos: "
    f"{df_control_puerto_pre['Puerto_movimientos'].notna().mean() * 100:.2f} %"
)

print(
    f"✓ NaN Puerto_eslora_media: "
    f"{df_control_puerto_pre['Puerto_eslora_media'].isna().sum():,}"
)

print(
    f"✓ NaN Puerto_calado_medio: "
    f"{df_control_puerto_pre['Puerto_calado_medio'].isna().sum():,}"
)

print(
    f"✓ Duplicados de fecha: "
    f"{duplicados_recarga:,}"
)

print(
    f"✓ Valores infinitos: "
    f"{n_inf:,}"
)

print(
    f"✓ Valores negativos: "
    f"{n_negativos:,}"
)

print(
    f"✓ Tamaño del archivo: "
    f"{tamano_mb:.2f} MB"
)

print(
    "✓ Cobertura temporal completa 2018–2024."
)

print(
    "✓ Los NaN estructurales del día sin actividad "
    "se mantienen sin imputación."
)

print(
    "✓ Integridad verificada tras la exportación."
)

print(
    "\n✓ Archivo generado:"
)

print(
    ruta_archivo
)

7.4 — PREINTEGRACIÓN DEL TRANSPORTE MARÍTIMO

✓ Registros: 2,557
✓ Variables: 7
✓ Fecha inicial: 2018-01-01
✓ Fecha final: 2024-12-31
✓ Fechas diferentes: 2,557
✓ Duplicados de fecha: 0

CONTROL DEL CALENDARIO

✓ Días esperados: 2,557
✓ Días observados: 2,557
✓ Fechas faltantes: 0

COBERTURA DE LOS INDICADORES MARÍTIMOS


,variable,registros_validos,registros_nan,cobertura_pct
0,Puerto_movimientos,2557,0,100.000
1,Puerto_eslora_media,2556,1,99.961
2,Puerto_calado_medio,2556,1,99.961
3,Puerto_ferry_pasajeros,2557,0,100.000
4,Puerto_portacontenedores,2557,0,100.000
5,Puerto_tanque,2557,0,100.000



CONTROL DE NaN ESTRUCTURALES

✓ Filas con NaN en eslora/calado medios: 1


,fecha,Puerto_movimientos,Puerto_eslora_media,Puerto_calado_medio,Puerto_ferry_pasajeros,Puerto_portacontenedores,Puerto_tanque
750,2020-01-21,0.0,NaN,NaN,0.0,0.0,0.0



CONTROL DE VALORES INFINITOS

✓ Valores infinitos: 0

CONTROL DE VALORES NEGATIVOS

✓ Valores negativos: 0

COBERTURA MARÍTIMA POR AÑO


,anio,dias,dias_con_movimientos,dias_sin_movimientos,cobertura_pct
0,2018,365,365,0,100.0
1,2019,365,365,0,100.0
2,2020,366,366,0,100.0
3,2021,365,365,0,100.0
4,2022,365,365,0,100.0
5,2023,365,365,0,100.0
6,2024,366,366,0,100.0



CONTROL POSTERIOR A LA EXPORTACIÓN

✓ Filas exportadas: 2,557
✓ Filas recargadas: 2,557
✓ Columnas exportadas: 7
✓ Columnas recargadas: 7
✓ Duplicados tras recarga: 0
✓ NaN eslora antes: 1
✓ NaN eslora después: 1
✓ NaN calado antes: 1
✓ NaN calado después: 1

PREINTEGRACIÓN MARÍTIMA FINALIZADA

✓ Registros: 2,557
✓ Variables: 7
✓ Fecha inicial: 2018-01-01
✓ Fecha final: 2024-12-31
✓ Fechas diferentes: 2,557
✓ Cobertura Puerto_movimientos: 100.00 %
✓ NaN Puerto_eslora_media: 1
✓ NaN Puerto_calado_medio: 1
✓ Duplicados de fecha: 0
✓ Valores infinitos: 0
✓ Valores negativos: 0
✓ Tamaño del archivo: 0.16 MB
✓ Cobertura temporal completa 2018–2024.
✓ Los NaN estructurales del día sin actividad se mantienen sin imputación.
✓ Integridad verificada tras la exportación.

✓ Archivo generado:
/content/drive/MyDrive/TFM/11_Machine_Learning_Temporal/PREINTEGRACION/03_Transporte_Maritimo/df_puerto_preintegrado.csv


### Resultados — 7.4. Preintegración temporal y exportación del bloque de transporte marítimo

El dataset marítimo preintegrado contiene **2.557 registros diarios y 7 variables**, correspondientes a la fecha y los seis indicadores seleccionados.

La cobertura temporal es completa para el periodo comprendido entre el **1 de enero de 2018 y el 31 de diciembre de 2024**, sin fechas faltantes ni duplicados.

Los indicadores seleccionados presentan las siguientes coberturas:

- `Puerto_movimientos`: 100,00 %;
- `Puerto_eslora_media`: 99,961 %;
- `Puerto_calado_medio`: 99,961 %;
- `Puerto_ferry_pasajeros`: 100,00 %;
- `Puerto_portacontenedores`: 100,00 %;
- `Puerto_tanque`: 100,00 %.

Los únicos valores ausentes corresponden a `Puerto_eslora_media` y `Puerto_calado_medio` en el día **21 de enero de 2020**, fecha en la que no se registró actividad marítima. Estos valores se mantienen como `NaN`, ya que representan una ausencia estructural de la media y no una pérdida de información.

No se detectan valores infinitos ni negativos.

La cobertura anual de `Puerto_movimientos` es del **100 % en todos los años entre 2018 y 2024**.

La exportación se valida mediante recarga del archivo, conservándose exactamente las **2.557 filas, 7 variables, 0 duplicados y los dos NaN estructurales previamente identificados**.

> **Conclusión:** el bloque marítimo queda preintegrado con cobertura temporal completa, sin anomalías estructurales y preservando los valores ausentes que derivan del único día sin actividad. El dataset queda preparado para su incorporación al Checkpoint 05.

## 7.5. Integración del transporte marítimo y generación del Checkpoint 06

Una vez preparado el dataset marítimo preintegrado, se incorporan los seis indicadores seleccionados al dataset maestro correspondiente al Checkpoint 05.

La integración se realiza mediante la variable temporal `fecha`.

El bloque marítimo presenta una única observación diaria y cobertura completa durante el periodo 2018–2024. Por tanto, los indicadores correspondientes a una fecha determinada se replican sobre todas las estaciones físicas y contaminantes presentes en el maestro para ese mismo día.

Se incorporan las siguientes variables:

- `Puerto_movimientos`
- `Puerto_eslora_media`
- `Puerto_calado_medio`
- `Puerto_ferry_pasajeros`
- `Puerto_portacontenedores`
- `Puerto_tanque`

Los únicos valores ausentes corresponden a `Puerto_eslora_media` y `Puerto_calado_medio` en el único día sin actividad marítima del periodo. Estos valores se mantienen como `NaN` y no se realiza ninguna imputación.

Durante la integración se comprueba:

- unicidad de la fecha en el dataset marítimo;
- correspondencia temporal entre ambas fuentes;
- conservación exacta del número de registros del maestro;
- ausencia de nuevos duplicados en la clave principal;
- cobertura de los indicadores marítimos;
- conservación de los valores ausentes estructurales;
- ausencia de valores infinitos;
- integridad del archivo tras su exportación y recarga.

El resultado se almacena como **Checkpoint 06**, manteniendo inalterados los checkpoints anteriores.

> **Objetivo:** incorporar la actividad marítima diaria al dataset maestro preservando su estructura, cobertura temporal y limitaciones reales, generando el checkpoint final de la fase principal de integración temporal.

In [ ]:
# ============================================================
# 7.5. INTEGRACIÓN DEL TRANSPORTE MARÍTIMO
# CHECKPOINT 06
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. RUTAS
# ============================================================

ruta_base = Path(
    "/content/drive/MyDrive/TFM/"
    "11_Machine_Learning_Temporal"
)

ruta_integracion = (
    ruta_base /
    "INTEGRACION"
)

ruta_maestro_05 = (
    ruta_integracion /
    "05_maestro_contaminacion_meteorologia_trafico_ruido_vuelos.csv"
)

ruta_puerto_pre = (
    ruta_base /
    "PREINTEGRACION" /
    "03_Transporte_Maritimo" /
    "df_puerto_preintegrado.csv"
)

ruta_maestro_06 = (
    ruta_integracion /
    "06_maestro_contaminacion_meteorologia_trafico_ruido_vuelos_puerto.csv"
)


# ============================================================
# 2. COMPROBAR EXISTENCIA DE ARCHIVOS
# ============================================================

if not ruta_maestro_05.exists():

    raise FileNotFoundError(
        "No se encuentra el Checkpoint 05:\n"
        f"{ruta_maestro_05}"
    )


if not ruta_puerto_pre.exists():

    raise FileNotFoundError(
        "No se encuentra el dataset marítimo preintegrado:\n"
        f"{ruta_puerto_pre}"
    )


# ============================================================
# 3. CARGAR ARCHIVOS
# ============================================================

df_maestro_05 = pd.read_csv(
    ruta_maestro_05,
    low_memory=False
)

df_puerto_pre = pd.read_csv(
    ruta_puerto_pre,
    low_memory=False
)


# ============================================================
# 4. NORMALIZAR FECHAS
# ============================================================

df_maestro_05[
    "fecha"
] = pd.to_datetime(
    df_maestro_05[
        "fecha"
    ],
    errors="coerce"
)


df_puerto_pre[
    "fecha"
] = pd.to_datetime(
    df_puerto_pre[
        "fecha"
    ],
    errors="coerce"
)


# ============================================================
# 5. CONTROL INICIAL
# ============================================================

print("=" * 80)
print("7.5 — INTEGRACIÓN DEL TRANSPORTE MARÍTIMO — CHECKPOINT 06")
print("=" * 80)

print("\nCHECKPOINT 05")

print(
    f"✓ Filas: "
    f"{len(df_maestro_05):,}"
)

print(
    f"✓ Columnas: "
    f"{df_maestro_05.shape[1]}"
)


print("\nDATASET MARÍTIMO PREINTEGRADO")

print(
    f"✓ Filas: "
    f"{len(df_puerto_pre):,}"
)

print(
    f"✓ Columnas: "
    f"{df_puerto_pre.shape[1]}"
)

print(
    f"✓ Fechas diferentes: "
    f"{df_puerto_pre['fecha'].nunique():,}"
)


# ============================================================
# 6. CONTROL DE FECHAS INVÁLIDAS
# ============================================================

fechas_invalidas_maestro = (
    df_maestro_05[
        "fecha"
    ]
    .isna()
    .sum()
)

fechas_invalidas_puerto = (
    df_puerto_pre[
        "fecha"
    ]
    .isna()
    .sum()
)


print("\n" + "=" * 80)
print("CONTROL DE FECHAS")
print("=" * 80)

print(
    f"\n✓ Fechas inválidas en maestro: "
    f"{fechas_invalidas_maestro:,}"
)

print(
    f"✓ Fechas inválidas en marítimo: "
    f"{fechas_invalidas_puerto:,}"
)


if (
    fechas_invalidas_maestro > 0
    or
    fechas_invalidas_puerto > 0
):

    raise ValueError(
        "Existen fechas inválidas antes de la integración."
    )


# ============================================================
# 7. UNICIDAD DEL DATASET MARÍTIMO
# ============================================================

duplicados_puerto = (
    df_puerto_pre
    .duplicated(
        subset=[
            "fecha"
        ]
    )
    .sum()
)


print("\n" + "=" * 80)
print("UNICIDAD DEL DATASET MARÍTIMO")
print("=" * 80)

print(
    f"\n✓ Duplicados de fecha: "
    f"{duplicados_puerto:,}"
)


if duplicados_puerto > 0:

    raise ValueError(
        "El dataset marítimo presenta fechas duplicadas."
    )


# ============================================================
# 8. CONTROL DEL MAESTRO ANTES DEL MERGE
# ============================================================

clave_maestro = [
    "fecha",
    "estacion_fisica",
    "contaminant"
]


faltantes_clave_maestro = [
    columna
    for columna in clave_maestro
    if columna not in df_maestro_05.columns
]


if faltantes_clave_maestro:

    raise KeyError(
        "Faltan columnas de la clave principal del maestro:\n"
        f"{faltantes_clave_maestro}"
    )


duplicados_maestro_antes = (
    df_maestro_05
    .duplicated(
        subset=clave_maestro
    )
    .sum()
)

filas_antes = len(
    df_maestro_05
)

columnas_antes = (
    df_maestro_05.shape[1]
)


print("\n" + "=" * 80)
print("CONTROL DEL MAESTRO ANTES DE INTEGRAR")
print("=" * 80)

print(
    f"\n✓ Filas: "
    f"{filas_antes:,}"
)

print(
    f"✓ Columnas: "
    f"{columnas_antes}"
)

print(
    f"✓ Duplicados clave principal: "
    f"{duplicados_maestro_antes:,}"
)


if duplicados_maestro_antes > 0:

    raise ValueError(
        "El Checkpoint 05 presenta duplicados."
    )


# ============================================================
# 9. VARIABLES MARÍTIMAS A INCORPORAR
# ============================================================

variables_puerto = [
    "Puerto_movimientos",
    "Puerto_eslora_media",
    "Puerto_calado_medio",
    "Puerto_ferry_pasajeros",
    "Puerto_portacontenedores",
    "Puerto_tanque"
]


faltantes_puerto = [
    variable
    for variable in variables_puerto
    if variable not in df_puerto_pre.columns
]


if faltantes_puerto:

    raise KeyError(
        "Faltan variables marítimas necesarias:\n"
        f"{faltantes_puerto}"
    )


print("\n" + "=" * 80)
print("VARIABLES MARÍTIMAS A INCORPORAR")
print("=" * 80)

print(
    f"\n✓ Variables nuevas: "
    f"{len(variables_puerto)}"
)

for variable in variables_puerto:

    print(
        f"  • {variable}"
    )


# ============================================================
# 10. CONTROL DE COLUMNAS COMUNES
# ============================================================

columnas_comunes = (
    set(
        df_maestro_05.columns
    )
    &
    set(
        df_puerto_pre.columns
    )
)


columnas_comunes_no_clave = sorted(
    columnas_comunes
    -
    {"fecha"}
)


print("\n" + "=" * 80)
print("CONTROL DE COLUMNAS COMUNES")
print("=" * 80)

print(
    f"\n✓ Columnas comunes totales: "
    f"{len(columnas_comunes)}"
)

print(
    f"✓ Columnas comunes fuera de la clave: "
    f"{len(columnas_comunes_no_clave)}"
)


if columnas_comunes_no_clave:

    print(
        "\nColumnas comunes no esperadas:"
    )

    for columna in columnas_comunes_no_clave:

        print(
            f"  • {columna}"
        )

    raise ValueError(
        "Existen columnas comunes fuera de fecha."
    )


# ============================================================
# 11. COBERTURA DE LA CLAVE TEMPORAL
# ============================================================

fechas_maestro = (
    df_maestro_05[
        ["fecha"]
    ]
    .drop_duplicates()
)


fechas_puerto = (
    df_puerto_pre[
        ["fecha"]
    ]
    .drop_duplicates()
)


control_cobertura = pd.merge(
    fechas_maestro,
    fechas_puerto.assign(
        fecha_en_puerto=True
    ),
    on="fecha",
    how="left",
    validate="one_to_one"
)


control_cobertura[
    "fecha_en_puerto"
] = (
    control_cobertura[
        "fecha_en_puerto"
    ]
    .fillna(False)
    .astype(bool)
)


n_fechas_maestro = len(
    control_cobertura
)

n_fechas_con_puerto = (
    control_cobertura[
        "fecha_en_puerto"
    ]
    .sum()
)

n_fechas_sin_puerto = (
    n_fechas_maestro
    -
    n_fechas_con_puerto
)


print("\n" + "=" * 80)
print("COBERTURA DE LA CLAVE TEMPORAL")
print("=" * 80)

print(
    f"\n✓ Fechas del maestro: "
    f"{n_fechas_maestro:,}"
)

print(
    f"✓ Fechas con correspondencia marítima: "
    f"{n_fechas_con_puerto:,}"
)

print(
    f"✓ Fechas sin correspondencia marítima: "
    f"{n_fechas_sin_puerto:,}"
)

print(
    f"✓ Cobertura de la clave temporal: "
    f"{n_fechas_con_puerto / n_fechas_maestro * 100:.2f} %"
)


# ============================================================
# 12. PREPARAR TABLA DE MERGE
# ============================================================

df_puerto_merge = (
    df_puerto_pre[
        [
            "fecha",
            *variables_puerto
        ]
    ]
    .copy()
)


# ============================================================
# 13. INTEGRACIÓN MANY-TO-ONE
# ============================================================

df_maestro_06 = pd.merge(
    df_maestro_05,
    df_puerto_merge,
    on="fecha",
    how="left",
    validate="many_to_one"
)


# ============================================================
# 14. CONTROL DE DIMENSIONES
# ============================================================

filas_despues = len(
    df_maestro_06
)

columnas_despues = (
    df_maestro_06.shape[1]
)

columnas_esperadas = (
    columnas_antes
    +
    len(
        variables_puerto
    )
)


print("\n" + "=" * 80)
print("CONTROL DE DIMENSIONES TRAS EL MERGE")
print("=" * 80)

print(
    f"\n✓ Filas antes: "
    f"{filas_antes:,}"
)

print(
    f"✓ Filas después: "
    f"{filas_despues:,}"
)

print(
    f"✓ Columnas antes: "
    f"{columnas_antes}"
)

print(
    f"✓ Variables marítimas añadidas: "
    f"{len(variables_puerto)}"
)

print(
    f"✓ Columnas esperadas: "
    f"{columnas_esperadas}"
)

print(
    f"✓ Columnas obtenidas: "
    f"{columnas_despues}"
)


if filas_antes != filas_despues:

    raise ValueError(
        "La integración ha modificado "
        "el número de filas."
    )


if columnas_despues != columnas_esperadas:

    raise ValueError(
        "El número de columnas obtenido "
        "no coincide con el esperado."
    )


# ============================================================
# 15. CONTROL DE DUPLICADOS TRAS EL MERGE
# ============================================================

duplicados_maestro_despues = (
    df_maestro_06
    .duplicated(
        subset=clave_maestro
    )
    .sum()
)


print("\n" + "=" * 80)
print("CONTROL DE DUPLICADOS TRAS EL MERGE")
print("=" * 80)

print(
    f"\n✓ Duplicados antes: "
    f"{duplicados_maestro_antes:,}"
)

print(
    f"✓ Duplicados después: "
    f"{duplicados_maestro_despues:,}"
)


if (
    duplicados_maestro_despues
    !=
    duplicados_maestro_antes
):

    raise ValueError(
        "La integración ha introducido duplicados."
    )


# ============================================================
# 16. COBERTURA DE VARIABLES MARÍTIMAS
# ============================================================

resumen_cobertura_maestro = []

for variable in variables_puerto:

    n_validos = (
        df_maestro_06[
            variable
        ]
        .notna()
        .sum()
    )

    n_nan = (
        df_maestro_06[
            variable
        ]
        .isna()
        .sum()
    )

    cobertura = (
        n_validos
        /
        len(df_maestro_06)
        *
        100
    )

    resumen_cobertura_maestro.append({
        "variable":
            variable,

        "registros_validos":
            n_validos,

        "registros_nan":
            n_nan,

        "cobertura_pct":
            round(
                cobertura,
                3
            )
    })


df_cobertura_puerto_maestro = pd.DataFrame(
    resumen_cobertura_maestro
)


print("\n" + "=" * 80)
print("COBERTURA DE VARIABLES MARÍTIMAS EN EL MAESTRO")
print("=" * 80)

display(
    df_cobertura_puerto_maestro
)


# ============================================================
# 17. CONTROL ESPECÍFICO DEL DÍA SIN ACTIVIDAD
# ============================================================

fecha_sin_actividad = pd.Timestamp(
    "2020-01-21"
)


control_dia_cero = (
    df_maestro_06[
        df_maestro_06[
            "fecha"
        ] == fecha_sin_actividad
    ]
    [
        [
            "fecha",
            "estacion_geo",
            "contaminant",
            *variables_puerto
        ]
    ]
    .copy()
)


print("\n" + "=" * 80)
print("CONTROL DEL DÍA SIN ACTIVIDAD — 2020-01-21")
print("=" * 80)

print(
    f"\n✓ Registros del maestro ese día: "
    f"{len(control_dia_cero):,}"
)

print(
    f"✓ Registros con Puerto_movimientos = 0: "
    f"{control_dia_cero['Puerto_movimientos'].eq(0).sum():,}"
)

print(
    f"✓ NaN Puerto_eslora_media: "
    f"{control_dia_cero['Puerto_eslora_media'].isna().sum():,}"
)

print(
    f"✓ NaN Puerto_calado_medio: "
    f"{control_dia_cero['Puerto_calado_medio'].isna().sum():,}"
)


# ============================================================
# 18. CONTROL DE VALORES INFINITOS
# ============================================================

columnas_numericas = (
    df_maestro_06
    .select_dtypes(
        include=[
            np.number
        ]
    )
    .columns
)


n_inf = (
    np.isinf(
        df_maestro_06[
            columnas_numericas
        ]
    )
    .sum()
    .sum()
)


print("\n" + "=" * 80)
print("CONTROL DE VALORES INFINITOS")
print("=" * 80)

print(
    f"\n✓ Valores infinitos: "
    f"{n_inf:,}"
)


if n_inf > 0:

    raise ValueError(
        "Se han detectado valores infinitos."
    )


# ============================================================
# 19. ORDENAR EL MAESTRO
# ============================================================

columnas_orden = [
    columna
    for columna in [
        "fecha",
        "estacion_geo",
        "estacion_fisica",
        "contaminant"
    ]
    if columna in df_maestro_06.columns
]


df_maestro_06 = (
    df_maestro_06
    .sort_values(
        columnas_orden
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 20. EXPORTAR CHECKPOINT 06
# ============================================================

ruta_integracion.mkdir(
    parents=True,
    exist_ok=True
)


df_maestro_06.to_csv(
    ruta_maestro_06,
    index=False,
    encoding="utf-8"
)


# ============================================================
# 21. RECARGAR CHECKPOINT 06
# ============================================================

df_control_06 = pd.read_csv(
    ruta_maestro_06,
    parse_dates=[
        "fecha"
    ],
    low_memory=False
)


# ============================================================
# 22. VERIFICAR INTEGRIDAD DE LA EXPORTACIÓN
# ============================================================

filas_recarga = len(
    df_control_06
)

columnas_recarga = (
    df_control_06.shape[1]
)

duplicados_recarga = (
    df_control_06
    .duplicated(
        subset=clave_maestro
    )
    .sum()
)


nan_eslora_antes = (
    df_maestro_06[
        "Puerto_eslora_media"
    ]
    .isna()
    .sum()
)

nan_eslora_despues = (
    df_control_06[
        "Puerto_eslora_media"
    ]
    .isna()
    .sum()
)


nan_calado_antes = (
    df_maestro_06[
        "Puerto_calado_medio"
    ]
    .isna()
    .sum()
)

nan_calado_despues = (
    df_control_06[
        "Puerto_calado_medio"
    ]
    .isna()
    .sum()
)


print("\n" + "=" * 80)
print("VERIFICACIÓN DEL CHECKPOINT 06")
print("=" * 80)

print(
    f"\n✓ Filas exportadas: "
    f"{len(df_maestro_06):,}"
)

print(
    f"✓ Filas recargadas: "
    f"{filas_recarga:,}"
)

print(
    f"✓ Columnas exportadas: "
    f"{df_maestro_06.shape[1]}"
)

print(
    f"✓ Columnas recargadas: "
    f"{columnas_recarga}"
)

print(
    f"✓ Duplicados tras recarga: "
    f"{duplicados_recarga:,}"
)

print(
    f"✓ NaN eslora antes: "
    f"{nan_eslora_antes:,}"
)

print(
    f"✓ NaN eslora después: "
    f"{nan_eslora_despues:,}"
)

print(
    f"✓ NaN calado antes: "
    f"{nan_calado_antes:,}"
)

print(
    f"✓ NaN calado después: "
    f"{nan_calado_despues:,}"
)


if filas_recarga != len(
    df_maestro_06
):

    raise ValueError(
        "El número de filas ha cambiado "
        "durante la exportación."
    )


if columnas_recarga != (
    df_maestro_06.shape[1]
):

    raise ValueError(
        "El número de columnas ha cambiado "
        "durante la exportación."
    )


if duplicados_recarga != (
    duplicados_maestro_despues
):

    raise ValueError(
        "La recarga ha alterado "
        "la unicidad del maestro."
    )


if nan_eslora_antes != nan_eslora_despues:

    raise ValueError(
        "La exportación ha alterado "
        "los NaN de eslora media."
    )


if nan_calado_antes != nan_calado_despues:

    raise ValueError(
        "La exportación ha alterado "
        "los NaN de calado medio."
    )


# ============================================================
# 23. TAMAÑO DEL ARCHIVO
# ============================================================

tamano_mb = (
    ruta_maestro_06.stat().st_size
    /
    (1024 ** 2)
)


# ============================================================
# 24. RESUMEN FINAL
# ============================================================

print("\n" + "=" * 80)
print("CHECKPOINT 06 — TRANSPORTE MARÍTIMO")
print("=" * 80)

print(
    f"\n✓ Registros conservados: "
    f"{filas_recarga:,}"
)

print(
    f"✓ Variables disponibles: "
    f"{columnas_recarga}"
)

print(
    f"✓ Variables marítimas añadidas: "
    f"{len(variables_puerto)}"
)

print(
    f"✓ Localizaciones físicas: "
    f"{df_control_06['estacion_geo'].nunique()}"
)

print(
    f"✓ Contaminantes: "
    f"{df_control_06['contaminant'].nunique()}"
)

print(
    f"✓ Duplicados de la clave principal: "
    f"{duplicados_recarga:,}"
)

print(
    f"✓ Cobertura Puerto_movimientos: "
    f"{df_control_06['Puerto_movimientos'].notna().mean() * 100:.2f} %"
)

print(
    f"✓ NaN Puerto_eslora_media: "
    f"{nan_eslora_despues:,}"
)

print(
    f"✓ NaN Puerto_calado_medio: "
    f"{nan_calado_despues:,}"
)

print(
    f"✓ Valores infinitos: "
    f"{n_inf:,}"
)

print(
    f"✓ Tamaño del archivo: "
    f"{tamano_mb:.2f} MB"
)

print(
    "✓ Sin pérdida ni multiplicación de registros."
)

print(
    "✓ Checkpoint 05 conservado sin modificaciones."
)

print(
    "✓ Cobertura marítima temporal completa."
)

print(
    "✓ Los NaN estructurales del día sin actividad "
    "permanecen sin imputación."
)

print(
    "✓ Integridad del Checkpoint 06 "
    "verificada tras la exportación."
)

print(
    "\n✓ Archivo maestro generado:"
)

print(
    ruta_maestro_06
)

7.5 — INTEGRACIÓN DEL TRANSPORTE MARÍTIMO — CHECKPOINT 06

CHECKPOINT 05
✓ Filas: 43,642
✓ Columnas: 67

DATASET MARÍTIMO PREINTEGRADO
✓ Filas: 2,557
✓ Columnas: 7
✓ Fechas diferentes: 2,557

CONTROL DE FECHAS

✓ Fechas inválidas en maestro: 0
✓ Fechas inválidas en marítimo: 0

UNICIDAD DEL DATASET MARÍTIMO

✓ Duplicados de fecha: 0

CONTROL DEL MAESTRO ANTES DE INTEGRAR

✓ Filas: 43,642
✓ Columnas: 67
✓ Duplicados clave principal: 0

VARIABLES MARÍTIMAS A INCORPORAR

✓ Variables nuevas: 6
  • Puerto_movimientos
  • Puerto_eslora_media
  • Puerto_calado_medio
  • Puerto_ferry_pasajeros
  • Puerto_portacontenedores
  • Puerto_tanque

CONTROL DE COLUMNAS COMUNES

✓ Columnas comunes totales: 1
✓ Columnas comunes fuera de la clave: 0

COBERTURA DE LA CLAVE TEMPORAL

✓ Fechas del maestro: 2,249
✓ Fechas con correspondencia marítima: 2,249
✓ Fechas sin correspondencia marítima: 0
✓ Cobertura de la clave temporal: 100.00 %

CONTROL DE DIMENSIONES TRAS EL MERGE

✓ Filas antes: 43,642
✓ Filas d

,variable,registros_validos,registros_nan,cobertura_pct
0,Puerto_movimientos,43642,0,100.000
1,Puerto_eslora_media,43622,20,99.954
2,Puerto_calado_medio,43622,20,99.954
3,Puerto_ferry_pasajeros,43642,0,100.000
4,Puerto_portacontenedores,43642,0,100.000
5,Puerto_tanque,43642,0,100.000



CONTROL DEL DÍA SIN ACTIVIDAD — 2020-01-21

✓ Registros del maestro ese día: 20
✓ Registros con Puerto_movimientos = 0: 20
✓ NaN Puerto_eslora_media: 20
✓ NaN Puerto_calado_medio: 20

CONTROL DE VALORES INFINITOS

✓ Valores infinitos: 0

VERIFICACIÓN DEL CHECKPOINT 06

✓ Filas exportadas: 43,642
✓ Filas recargadas: 43,642
✓ Columnas exportadas: 73
✓ Columnas recargadas: 73
✓ Duplicados tras recarga: 0
✓ NaN eslora antes: 20
✓ NaN eslora después: 20
✓ NaN calado antes: 20
✓ NaN calado después: 20

CHECKPOINT 06 — TRANSPORTE MARÍTIMO

✓ Registros conservados: 43,642
✓ Variables disponibles: 73
✓ Variables marítimas añadidas: 6
✓ Localizaciones físicas: 8
✓ Contaminantes: 4
✓ Duplicados de la clave principal: 0
✓ Cobertura Puerto_movimientos: 100.00 %
✓ NaN Puerto_eslora_media: 20
✓ NaN Puerto_calado_medio: 20
✓ Valores infinitos: 0
✓ Tamaño del archivo: 27.66 MB
✓ Sin pérdida ni multiplicación de registros.
✓ Checkpoint 05 conservado sin modificaciones.
✓ Cobertura marítima temporal com

### Resultados — 7.5. Integración del transporte marítimo y generación del Checkpoint 06

La integración del bloque marítimo se realiza sobre el **Checkpoint 05**, compuesto por **43.642 registros y 67 variables**.

El dataset marítimo preintegrado contiene **2.557 fechas**, correspondientes al periodo completo 2018–2024, con una única observación por día.

La correspondencia temporal entre ambas fuentes es completa: las **2.249 fechas presentes en el maestro** disponen de correspondencia en el dataset marítimo, alcanzándose una cobertura de clave temporal del **100 %**.

Se incorporan seis variables:

- `Puerto_movimientos`
- `Puerto_eslora_media`
- `Puerto_calado_medio`
- `Puerto_ferry_pasajeros`
- `Puerto_portacontenedores`
- `Puerto_tanque`

La integración conserva exactamente los **43.642 registros originales** y amplía el dataset de **67 a 73 variables**, sin introducir duplicados en la clave principal `fecha + estacion_fisica + contaminant`.

`Puerto_movimientos`, `Puerto_ferry_pasajeros`, `Puerto_portacontenedores` y `Puerto_tanque` presentan una cobertura del **100 %** en el maestro.

`Puerto_eslora_media` y `Puerto_calado_medio` presentan una cobertura del **99,954 %**, con **20 valores ausentes cada una**.

Estos 20 registros corresponden exclusivamente al **21 de enero de 2020**, único día del periodo sin actividad marítima. En esa fecha los 20 registros del maestro presentan `Puerto_movimientos = 0`, mientras que las variables medias de eslora y calado permanecen como `NaN`, al no existir buques sobre los que calcular dichos valores.

Estos valores ausentes se conservan como información estructural y no se realiza ninguna imputación.

No se detectan valores infinitos ni modificaciones en la estructura original del maestro.

La exportación del Checkpoint 06 se valida mediante su recarga, conservándose exactamente:

- **43.642 registros**;
- **73 variables**;
- **0 duplicados**;
- **20 NaN en `Puerto_eslora_media`**;
- **20 NaN en `Puerto_calado_medio`**;
- **0 valores infinitos**.

> **Conclusión:** el bloque de transporte marítimo queda correctamente integrado en el dataset maestro, con cobertura temporal completa y preservando los únicos valores ausentes estructurales asociados al día sin actividad portuaria. El resultado constituye el **Checkpoint 06** de la fase de integración temporal.

# 8 · AUDITORÍA GLOBAL DEL DATASET MAESTRO INTEGRADO

Una vez completadas todas las integraciones se realiza una **auditoría técnica global del Checkpoint 06**.

Esta etapa no constituye un análisis exploratorio de datos. Su finalidad es certificar que el proceso de integración ha conservado correctamente la estructura, granularidad y trazabilidad del dataset y documentar las limitaciones de cobertura que deberán considerarse posteriormente.

La auditoría revisa la integridad de la clave principal, cobertura temporal, estaciones y localizaciones geográficas, contaminantes, valores ausentes, columnas duplicadas o vacías, valores infinitos y cobertura de los diferentes bloques incorporados.

No se realiza ninguna imputación, eliminación de registros, tratamiento de valores extremos ni selección de variables.

La superación de esta auditoría permite considerar el **Checkpoint 06 como dataset maestro definitivo de la fase ETL de integración**.

> **Objetivo del bloque:** certificar la integridad estructural, temporal y espacial del maestro antes de iniciar las fases de visualización, análisis exploratorio y modelado.

## 8.1. Auditoría estructural y de cobertura del dataset maestro integrado

Se realiza una auditoría técnica global del **Checkpoint 06**, correspondiente al dataset maestro obtenido tras la integración sucesiva de contaminación atmosférica, meteorología, tráfico rodado, contaminación acústica, actividad aérea y actividad marítima.

El objetivo de esta etapa es verificar que el proceso de integración ha conservado correctamente la estructura del dataset y documentar las limitaciones de cobertura existentes antes de iniciar las fases de visualización y modelado.

La auditoría comprende:

- comprobación de las dimensiones finales del dataset;
- revisión de los nombres y tipos de las variables;
- detección de columnas duplicadas o completamente vacías;
- validación de la variable temporal y del periodo disponible;
- comprobación de la cobertura anual;
- verificación de la unicidad de la clave `fecha + estacion_fisica + contaminant`;
- revisión de las estaciones físicas y contaminantes presentes;
- análisis de la cobertura por combinación estación–contaminante;
- inventario global de valores ausentes;
- detección de valores infinitos;
- identificación de las variables pertenecientes a cada bloque de información;
- cálculo de la cobertura global de los diferentes bloques;
- comprobación específica de las limitaciones temporales previamente documentadas en vuelos, ruido y transporte marítimo.

Los valores ausentes no se consideran automáticamente errores de integración. Se mantiene la distinción entre ausencia de información y ausencia estructural derivada de las características de cada fuente. En particular, se conservan las limitaciones temporales ya identificadas durante las etapas de preintegración e integración.

Esta etapa tiene exclusivamente carácter de **control de calidad e integridad**. No se realizan imputaciones, eliminación de registros, tratamiento de valores extremos, análisis de correlaciones, selección de variables ni transformaciones destinadas al aprendizaje automático.

> **Objetivo:** certificar la integridad estructural y temporal del Checkpoint 06, cuantificar la cobertura final de las fuentes integradas y dejar documentadas las limitaciones que deberán considerarse posteriormente en la visualización, el análisis exploratorio y el modelado.

In [ ]:
# ============================================================
# 8.1. AUDITORÍA ESTRUCTURAL Y DE COBERTURA
# DEL DATASET MAESTRO INTEGRADO
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. RUTA DEL CHECKPOINT 06
# ============================================================

ruta_maestro_06 = Path(
    "/content/drive/MyDrive/TFM/"
    "11_Machine_Learning_Temporal/"
    "INTEGRACION/"
    "06_maestro_contaminacion_meteorologia_trafico_ruido_vuelos_puerto.csv"
)


# ============================================================
# 2. COMPROBAR EXISTENCIA
# ============================================================

if not ruta_maestro_06.exists():

    raise FileNotFoundError(
        "No se encuentra el Checkpoint 06:\n"
        f"{ruta_maestro_06}"
    )


# ============================================================
# 3. CARGA DEL DATASET
# ============================================================

df_maestro_final = pd.read_csv(
    ruta_maestro_06,
    low_memory=False
)


if "fecha" not in df_maestro_final.columns:

    raise KeyError(
        "No existe la variable 'fecha' "
        "en el Checkpoint 06."
    )


df_maestro_final[
    "fecha"
] = pd.to_datetime(
    df_maestro_final[
        "fecha"
    ],
    errors="coerce"
)


print("=" * 80)
print("8.1 — AUDITORÍA GLOBAL DEL DATASET MAESTRO INTEGRADO")
print("=" * 80)


# ============================================================
# 4. DIMENSIONES GENERALES
# ============================================================

n_filas = len(
    df_maestro_final
)

n_columnas = (
    df_maestro_final.shape[1]
)


print("\n" + "=" * 80)
print("ESTRUCTURA GENERAL")
print("=" * 80)

print(
    f"\n✓ Filas: "
    f"{n_filas:,}"
)

print(
    f"✓ Columnas: "
    f"{n_columnas}"
)

print(
    f"✓ Memoria aproximada: "
    f"{df_maestro_final.memory_usage(deep=True).sum() / (1024 ** 2):.2f} MB"
)


# ============================================================
# 5. LISTADO DE VARIABLES
# ============================================================

print("\n" + "=" * 80)
print("VARIABLES DEL DATASET")
print("=" * 80)

for i, columna in enumerate(
    df_maestro_final.columns,
    start=1
):

    print(
        f"{i:02d}. {columna}"
    )


# ============================================================
# 6. COLUMNAS DUPLICADAS
# ============================================================

columnas_duplicadas = (
    df_maestro_final.columns[
        df_maestro_final.columns.duplicated()
    ]
    .tolist()
)


print("\n" + "=" * 80)
print("CONTROL DE COLUMNAS DUPLICADAS")
print("=" * 80)

print(
    f"\n✓ Columnas duplicadas: "
    f"{len(columnas_duplicadas)}"
)


if columnas_duplicadas:

    for columna in columnas_duplicadas:

        print(
            f"  • {columna}"
        )


# ============================================================
# 7. TIPOS DE DATOS
# ============================================================

tabla_tipos = (
    df_maestro_final
    .dtypes
    .astype(str)
    .value_counts()
    .rename_axis(
        "dtype"
    )
    .reset_index(
        name="n_variables"
    )
)


print("\n" + "=" * 80)
print("TIPOS DE DATOS")
print("=" * 80)

display(
    tabla_tipos
)


# ============================================================
# 8. CONTROL DE FECHAS
# ============================================================

fechas_invalidas = (
    df_maestro_final[
        "fecha"
    ]
    .isna()
    .sum()
)


print("\n" + "=" * 80)
print("CONTROL DE FECHA")
print("=" * 80)

print(
    f"\n✓ Fechas inválidas: "
    f"{fechas_invalidas:,}"
)


if fechas_invalidas > 0:

    raise ValueError(
        "Existen fechas inválidas "
        "en el maestro final."
    )


# ============================================================
# 9. COBERTURA TEMPORAL
# ============================================================

fecha_inicio = (
    df_maestro_final[
        "fecha"
    ]
    .min()
)

fecha_fin = (
    df_maestro_final[
        "fecha"
    ]
    .max()
)

n_fechas = (
    df_maestro_final[
        "fecha"
    ]
    .nunique()
)

anios_presentes = sorted(
    df_maestro_final[
        "fecha"
    ]
    .dt.year
    .unique()
    .tolist()
)


print("\n" + "=" * 80)
print("COBERTURA TEMPORAL")
print("=" * 80)

print(
    f"\n✓ Fecha inicial: "
    f"{fecha_inicio.date()}"
)

print(
    f"✓ Fecha final: "
    f"{fecha_fin.date()}"
)

print(
    f"✓ Fechas diferentes: "
    f"{n_fechas:,}"
)

print(
    f"✓ Años presentes: "
    f"{anios_presentes}"
)


# ============================================================
# 10. COBERTURA TEMPORAL POR AÑO
# ============================================================

cobertura_anual = (
    df_maestro_final
    .assign(
        anio_control=
            df_maestro_final[
                "fecha"
            ]
            .dt.year
    )
    .groupby(
        "anio_control"
    )
    .agg(
        registros=(
            "fecha",
            "size"
        ),

        dias=(
            "fecha",
            "nunique"
        )
    )
    .reset_index()
    .rename(
        columns={
            "anio_control":
                "anio"
        }
    )
)


print("\n" + "=" * 80)
print("COBERTURA TEMPORAL POR AÑO")
print("=" * 80)

display(
    cobertura_anual
)


# ============================================================
# 11. CLAVE PRINCIPAL
# ============================================================

clave_principal = [
    "fecha",
    "estacion_fisica",
    "contaminant"
]


faltantes_clave = [
    columna
    for columna in clave_principal
    if columna not in df_maestro_final.columns
]


if faltantes_clave:

    raise KeyError(
        "Faltan variables de la clave principal:\n"
        f"{faltantes_clave}"
    )


duplicados_clave = (
    df_maestro_final
    .duplicated(
        subset=clave_principal
    )
    .sum()
)


nan_clave = (
    df_maestro_final[
        clave_principal
    ]
    .isna()
    .sum()
)


print("\n" + "=" * 80)
print("INTEGRIDAD DE LA CLAVE PRINCIPAL")
print("=" * 80)

print(
    f"\n✓ Duplicados "
    f"fecha + estacion_fisica + contaminant: "
    f"{duplicados_clave:,}"
)

print(
    "\nNaN en la clave:"
)

display(
    nan_clave
    .rename(
        "n_nan"
    )
    .to_frame()
)


# ============================================================
# 12. ESTACIONES, LOCALIZACIONES Y CONTAMINANTES
# ============================================================

n_estaciones_fisicas = (
    df_maestro_final[
        "estacion_fisica"
    ]
    .nunique()
)

n_contaminantes = (
    df_maestro_final[
        "contaminant"
    ]
    .nunique()
)


if "estacion_geo" in df_maestro_final.columns:

    n_estaciones_geo = (
        df_maestro_final[
            "estacion_geo"
        ]
        .nunique()
    )

else:

    n_estaciones_geo = np.nan


print("\n" + "=" * 80)
print("ESTRUCTURA ESPACIAL Y CONTAMINANTES")
print("=" * 80)

print(
    f"\n✓ Estaciones físicas: "
    f"{n_estaciones_fisicas}"
)

if pd.notna(n_estaciones_geo):

    print(
        f"✓ Localizaciones geográficas: "
        f"{int(n_estaciones_geo)}"
    )

    print(
        "✓ La diferencia entre estaciones físicas "
        "y geográficas responde a cambios históricos "
        "de códigos/identificadores."
    )


print(
    f"✓ Contaminantes: "
    f"{n_contaminantes}"
)

print(
    "\nContaminantes presentes:"
)

print(
    sorted(
        df_maestro_final[
            "contaminant"
        ]
        .dropna()
        .unique()
        .tolist()
    )
)


# ============================================================
# 13. COBERTURA POR ESTACIÓN Y CONTAMINANTE
# ============================================================

tabla_estacion_contaminante = (
    df_maestro_final
    .groupby(
        [
            "estacion_fisica",
            "contaminant"
        ]
    )
    .agg(
        registros=(
            "fecha",
            "size"
        ),

        dias=(
            "fecha",
            "nunique"
        ),

        fecha_inicio=(
            "fecha",
            "min"
        ),

        fecha_fin=(
            "fecha",
            "max"
        )
    )
    .reset_index()
)


print("\n" + "=" * 80)
print("COBERTURA POR ESTACIÓN Y CONTAMINANTE")
print("=" * 80)

display(
    tabla_estacion_contaminante
)


# ============================================================
# 14. COLUMNAS COMPLETAMENTE VACÍAS
# ============================================================

columnas_vacias = [
    columna
    for columna in df_maestro_final.columns
    if df_maestro_final[
        columna
    ]
    .isna()
    .all()
]


print("\n" + "=" * 80)
print("COLUMNAS COMPLETAMENTE VACÍAS")
print("=" * 80)

print(
    f"\n✓ Columnas 100 % NaN: "
    f"{len(columnas_vacias)}"
)


if columnas_vacias:

    for columna in columnas_vacias:

        print(
            f"  • {columna}"
        )


# ============================================================
# 15. INVENTARIO GLOBAL DE NaN
# ============================================================

tabla_nan_global = pd.DataFrame({
    "variable":
        df_maestro_final.columns,

    "n_nan":
        [
            df_maestro_final[
                columna
            ]
            .isna()
            .sum()

            for columna
            in df_maestro_final.columns
        ]
})


tabla_nan_global[
    "pct_nan"
] = (
    tabla_nan_global[
        "n_nan"
    ]
    /
    n_filas
    *
    100
).round(3)


tabla_nan_global = (
    tabla_nan_global
    .sort_values(
        [
            "pct_nan",
            "n_nan"
        ],
        ascending=[
            False,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)


tabla_nan_presentes = (
    tabla_nan_global[
        tabla_nan_global[
            "n_nan"
        ] > 0
    ]
    .copy()
)


print("\n" + "=" * 80)
print("INVENTARIO GLOBAL DE VALORES AUSENTES")
print("=" * 80)

print(
    f"\n✓ Variables con algún NaN: "
    f"{len(tabla_nan_presentes)}"
)

print(
    f"✓ Variables sin NaN: "
    f"{n_columnas - len(tabla_nan_presentes)}"
)

display(
    tabla_nan_presentes
)


# ============================================================
# 16. CONTROL DE VALORES INFINITOS
# ============================================================

columnas_numericas = (
    df_maestro_final
    .select_dtypes(
        include=[
            np.number
        ]
    )
    .columns
    .tolist()
)


tabla_inf = pd.DataFrame({
    "variable":
        columnas_numericas,

    "n_inf":
        [
            np.isinf(
                df_maestro_final[
                    columna
                ]
            )
            .sum()

            for columna
            in columnas_numericas
        ]
})


n_inf_total = (
    tabla_inf[
        "n_inf"
    ]
    .sum()
)


print("\n" + "=" * 80)
print("CONTROL DE VALORES INFINITOS")
print("=" * 80)

print(
    f"\n✓ Valores infinitos totales: "
    f"{n_inf_total:,}"
)


if n_inf_total > 0:

    display(
        tabla_inf[
            tabla_inf[
                "n_inf"
            ] > 0
        ]
    )


# ============================================================
# 17. DEFINICIÓN CORRECTA DE BLOQUES
# ============================================================

variables_meteorologia = [
    "codigo_meteo",
    "estacion_meteo_asignada",
    "distancia_meteo_m",
    "distancia_meteo_km",
    "TM",
    "TN",
    "TX",
    "HRM",
    "HRN",
    "HRX",
    "PM",
    "PN",
    "PX",
    "PPT",
    "RS24h",
    "DVM10",
    "DVVX10",
    "VVM10",
    "VVX10"
]


variables_meteorologia = [
    variable
    for variable in variables_meteorologia
    if variable in df_maestro_final.columns
]


variables_trafico = [
    columna
    for columna in df_maestro_final.columns
    if (
        columna.startswith(
            (
                "n_aforos_",
                "trafico_",
                "cobertura_aforos_",
                "dist_aforo_"
            )
        )
    )
]


variables_ruido = [
    columna
    for columna in df_maestro_final.columns
    if (
        columna.startswith(
            "LAeq_"
        )
        or
        columna.startswith(
            "n_sensores_"
        )
        or
        columna.startswith(
            "dist_sensor_"
        )
        or
        columna.startswith(
            "cobertura_media_sensores_"
        )
    )
]


variables_vuelos = [
    columna
    for columna in df_maestro_final.columns
    if columna.startswith(
        (
            "Vuelos_",
            "Companias_",
            "Zonas_"
        )
    )
]


variables_puerto = [
    columna
    for columna in df_maestro_final.columns
    if columna.startswith(
        "Puerto_"
    )
]


bloques = {

    "Meteorología":
        variables_meteorologia,

    "Tráfico":
        variables_trafico,

    "Ruido":
        variables_ruido,

    "Vuelos":
        variables_vuelos,

    "Puerto":
        variables_puerto
}


# ============================================================
# 18. VARIABLES DETECTADAS POR BLOQUE
# ============================================================

print("\n" + "=" * 80)
print("VARIABLES DETECTADAS POR BLOQUE")
print("=" * 80)


for bloque, variables in bloques.items():

    print(
        f"\n{bloque}: "
        f"{len(variables)} variables"
    )

    for variable in variables:

        print(
            f"  • {variable}"
        )


# ============================================================
# 19. COBERTURA RESUMIDA POR BLOQUE
# ============================================================

resumen_bloques = []


for bloque, variables in bloques.items():

    if len(variables) == 0:

        resumen_bloques.append({
            "bloque":
                bloque,

            "n_variables":
                0,

            "cobertura_media_pct":
                np.nan,

            "cobertura_min_pct":
                np.nan,

            "cobertura_max_pct":
                np.nan
        })

        continue


    coberturas = (
        df_maestro_final[
            variables
        ]
        .notna()
        .mean()
        *
        100
    )


    resumen_bloques.append({
        "bloque":
            bloque,

        "n_variables":
            len(
                variables
            ),

        "cobertura_media_pct":
            coberturas.mean(),

        "cobertura_min_pct":
            coberturas.min(),

        "cobertura_max_pct":
            coberturas.max()
    })


df_resumen_bloques = pd.DataFrame(
    resumen_bloques
)


print("\n" + "=" * 80)
print("COBERTURA GLOBAL POR BLOQUE")
print("=" * 80)

display(
    df_resumen_bloques.round(2)
)


# ============================================================
# 20. CONTROL ESPECÍFICO DE VUELOS
# ============================================================

if "Vuelos_total" in df_maestro_final.columns:

    fecha_inicio_vuelos = pd.Timestamp(
        "2019-07-01"
    )


    mascara_antes_vuelos = (
        df_maestro_final[
            "fecha"
        ] < fecha_inicio_vuelos
    )


    mascara_despues_vuelos = (
        df_maestro_final[
            "fecha"
        ] >= fecha_inicio_vuelos
    )


    registros_antes = (
        mascara_antes_vuelos
        .sum()
    )


    nan_vuelos_antes = (
        df_maestro_final.loc[
            mascara_antes_vuelos,
            "Vuelos_total"
        ]
        .isna()
        .sum()
    )


    nan_vuelos_despues = (
        df_maestro_final.loc[
            mascara_despues_vuelos,
            "Vuelos_total"
        ]
        .isna()
        .sum()
    )


    print("\n" + "=" * 80)
    print("CONTROL DE LA LIMITACIÓN TEMPORAL DE VUELOS")
    print("=" * 80)

    print(
        f"\n✓ Registros anteriores al 01/07/2019: "
        f"{registros_antes:,}"
    )

    print(
        f"✓ NaN Vuelos_total antes del 01/07/2019: "
        f"{nan_vuelos_antes:,}"
    )

    print(
        f"✓ NaN Vuelos_total desde el 01/07/2019: "
        f"{nan_vuelos_despues:,}"
    )


# ============================================================
# 21. CONTROL ESPECÍFICO DE RUIDO
# ============================================================

if variables_ruido:

    print("\n" + "=" * 80)
    print("CONTROL FINAL DEL BLOQUE ACÚSTICO")
    print("=" * 80)


    for variable in variables_ruido:

        n_nan = (
            df_maestro_final[
                variable
            ]
            .isna()
            .sum()
        )

        cobertura = (
            df_maestro_final[
                variable
            ]
            .notna()
            .mean()
            *
            100
        )


        print(
            f"✓ {variable}: "
            f"{n_nan:,} NaN — "
            f"{cobertura:.2f} % cobertura"
        )


# ============================================================
# 22. CONTROL ESPECÍFICO DEL PUERTO
# ============================================================

if variables_puerto:

    print("\n" + "=" * 80)
    print("CONTROL FINAL DEL BLOQUE MARÍTIMO")
    print("=" * 80)


    for variable in variables_puerto:

        n_nan = (
            df_maestro_final[
                variable
            ]
            .isna()
            .sum()
        )

        cobertura = (
            df_maestro_final[
                variable
            ]
            .notna()
            .mean()
            *
            100
        )


        print(
            f"✓ {variable}: "
            f"{n_nan:,} NaN — "
            f"{cobertura:.2f} % cobertura"
        )


# ============================================================
# 23. RESUMEN GENERAL DE COBERTURA
# ============================================================

n_cobertura_completa = (
    tabla_nan_global[
        "n_nan"
    ]
    .eq(0)
    .sum()
)


n_cobertura_parcial = (
    (
        tabla_nan_global[
            "n_nan"
        ] > 0
    )
    &
    (
        tabla_nan_global[
            "n_nan"
        ] < n_filas
    )
).sum()


n_cobertura_nula = (
    tabla_nan_global[
        "n_nan"
    ]
    .eq(
        n_filas
    )
    .sum()
)


print("\n" + "=" * 80)
print("RESUMEN DE COBERTURA GLOBAL")
print("=" * 80)

print(
    f"\n✓ Variables con cobertura completa: "
    f"{n_cobertura_completa}"
)

print(
    f"✓ Variables con cobertura parcial: "
    f"{n_cobertura_parcial}"
)

print(
    f"✓ Variables completamente vacías: "
    f"{n_cobertura_nula}"
)


# ============================================================
# 24. CONTROLES ESTRICTOS DE INTEGRIDAD
# ============================================================

errores_criticos = []


if n_filas != 43642:

    errores_criticos.append(
        f"Número inesperado de filas: "
        f"{n_filas:,}"
    )


if n_columnas != 73:

    errores_criticos.append(
        f"Número inesperado de columnas: "
        f"{n_columnas}"
    )


if duplicados_clave > 0:

    errores_criticos.append(
        f"Duplicados en clave principal: "
        f"{duplicados_clave:,}"
    )


if columnas_duplicadas:

    errores_criticos.append(
        "Existen nombres de columnas duplicados."
    )


if n_inf_total > 0:

    errores_criticos.append(
        f"Existen {n_inf_total:,} "
        "valores infinitos."
    )


if len(columnas_vacias) > 0:

    errores_criticos.append(
        f"Existen {len(columnas_vacias)} "
        "columnas completamente vacías."
    )


if nan_clave.sum() > 0:

    errores_criticos.append(
        "Existen valores ausentes "
        "en la clave principal."
    )


# ============================================================
# 25. RESULTADO DE LA AUDITORÍA
# ============================================================

print("\n" + "=" * 80)
print("RESULTADO DE LA AUDITORÍA GLOBAL")
print("=" * 80)


if len(errores_criticos) == 0:

    print(
        "\n✓ AUDITORÍA ESTRUCTURAL SUPERADA."
    )

    print(
        "✓ No se detectan errores críticos "
        "de integración."
    )


else:

    print(
        "\n⚠ SE HAN DETECTADO INCIDENCIAS:"
    )

    for error in errores_criticos:

        print(
            f"  • {error}"
        )


# ============================================================
# 26. RESUMEN FINAL
# ============================================================

print("\n" + "=" * 80)
print("CHECKPOINT 06 — RESUMEN FINAL")
print("=" * 80)

print(
    f"\n✓ Registros: "
    f"{n_filas:,}"
)

print(
    f"✓ Variables: "
    f"{n_columnas}"
)

print(
    f"✓ Periodo: "
    f"{fecha_inicio.date()} "
    f"→ "
    f"{fecha_fin.date()}"
)

print(
    f"✓ Fechas diferentes: "
    f"{n_fechas:,}"
)

print(
    f"✓ Estaciones físicas: "
    f"{n_estaciones_fisicas}"
)

if pd.notna(n_estaciones_geo):

    print(
        f"✓ Localizaciones geográficas: "
        f"{int(n_estaciones_geo)}"
    )

print(
    f"✓ Contaminantes: "
    f"{n_contaminantes}"
)

print(
    f"✓ Duplicados clave principal: "
    f"{duplicados_clave:,}"
)

print(
    f"✓ Columnas duplicadas: "
    f"{len(columnas_duplicadas)}"
)

print(
    f"✓ Columnas completamente vacías: "
    f"{len(columnas_vacias)}"
)

print(
    f"✓ Valores infinitos: "
    f"{n_inf_total:,}"
)

print(
    f"✓ Variables con cobertura completa: "
    f"{n_cobertura_completa}"
)

print(
    f"✓ Variables con cobertura parcial: "
    f"{n_cobertura_parcial}"
)

print(
    "✓ No se ha realizado ninguna imputación."
)

print(
    "✓ No se han eliminado registros."
)

print(
    "✓ No se ha realizado selección "
    "de variables para ML."
)

print(
    "✓ No se ha realizado "
    "tratamiento de outliers."
)

print(
    "\n✓ Dataset preparado para "
    "visualización y posterior modelado."
)

8.1 — AUDITORÍA GLOBAL DEL DATASET MAESTRO INTEGRADO

ESTRUCTURA GENERAL

✓ Filas: 43,642
✓ Columnas: 73
✓ Memoria aproximada: 43.00 MB

VARIABLES DEL DATASET
01. fecha
02. codi_estacio
03. nom_cabina
04. contaminant
05. valor
06. horas_validas
07. codi_estacio_str
08. estacion_fisica
09. estacion_geo
10. lat
11. lon
12. tipo
13. X_ETRS89
14. Y_ETRS89
15. codigo_meteo
16. estacion_meteo_asignada
17. distancia_meteo_m
18. distancia_meteo_km
19. TM
20. TN
21. TX
22. HRM
23. HRN
24. HRX
25. PM
26. PN
27. PX
28. PPT
29. RS24h
30. DVM10
31. DVVX10
32. VVM10
33. VVX10
34. n_aforos_total_500m
35. trafico_media_500m
36. trafico_suma_500m
37. n_aforos_obs_500m
38. dist_aforo_min_500m
39. trafico_idw_500m
40. n_aforos_total_750m
41. trafico_media_750m
42. trafico_suma_750m
43. n_aforos_obs_750m
44. dist_aforo_min_750m
45. trafico_idw_750m
46. n_aforos_total_1000m
47. trafico_media_1000m
48. trafico_suma_1000m
49. n_aforos_obs_1000m
50. dist_aforo_min_1000m
51. trafico_idw_1000m
52. anio
53. n_af

,dtype,n_variables
0,float64,58
1,object,9
2,int64,5
3,datetime64[ns],1



CONTROL DE FECHA

✓ Fechas inválidas: 0

COBERTURA TEMPORAL

✓ Fecha inicial: 2018-06-12
✓ Fecha final: 2024-12-31
✓ Fechas diferentes: 2,249
✓ Años presentes: [2018, 2019, 2020, 2021, 2022, 2023, 2024]

COBERTURA TEMPORAL POR AÑO


,anio,registros,dias
0,2018,2830,183
1,2019,4311,256
2,2020,6894,355
3,2021,7051,359
4,2022,7446,365
5,2023,7634,365
6,2024,7476,366



INTEGRIDAD DE LA CLAVE PRINCIPAL

✓ Duplicados fecha + estacion_fisica + contaminant: 0

NaN en la clave:


...


COLUMNAS COMPLETAMENTE VACÍAS

✓ Columnas 100 % NaN: 0

INVENTARIO GLOBAL DE VALORES AUSENTES

✓ Variables con algún NaN: 50
✓ Variables sin NaN: 23


,variable,n_nan,pct_nan
0,PM,15904,36.442
1,PN,15904,36.442
2,PX,15904,36.442
3,RS24h,15904,36.442
4,DVVX10,15904,36.442
5,PPT,10103,23.150
6,DVM10,10103,23.150
7,VVM10,10103,23.150
8,VVX10,10103,23.150
9,HRN,7568,17.341



CONTROL DE VALORES INFINITOS

✓ Valores infinitos totales: 0

VARIABLES DETECTADAS POR BLOQUE

Meteorología: 19 variables
  • codigo_meteo
  • estacion_meteo_asignada
  • distancia_meteo_m
  • distancia_meteo_km
  • TM
  • TN
  • TX
  • HRM
  • HRN
  • HRX
  • PM
  • PN
  • PX
  • PPT
  • RS24h
  • DVM10
  • DVVX10
  • VVM10
  • VVX10

Tráfico: 25 variables
  • n_aforos_total_500m
  • trafico_media_500m
  • trafico_suma_500m
  • n_aforos_obs_500m
  • dist_aforo_min_500m
  • trafico_idw_500m
  • n_aforos_total_750m
  • trafico_media_750m
  • trafico_suma_750m
  • n_aforos_obs_750m
  • dist_aforo_min_750m
  • trafico_idw_750m
  • n_aforos_total_1000m
  • trafico_media_1000m
  • trafico_suma_1000m
  • n_aforos_obs_1000m
  • dist_aforo_min_1000m
  • trafico_idw_1000m
  • n_aforos_disponibles_500m
  • n_aforos_disponibles_750m
  • n_aforos_disponibles_1000m
  • cobertura_aforos_500m
  • cobertura_aforos_750m
  • cobertura_aforos_1000m
  • dist_aforo_min_m

Ruido: 4 variables
  • LAeq_idw_1

,bloque,n_variables,cobertura_media_pct,cobertura_min_pct,cobertura_max_pct
0,Meteorología,19,83.30,63.56,99.02
1,Tráfico,25,99.32,98.99,100.00
2,Ruido,4,99.21,99.21,99.21
3,Vuelos,4,89.11,89.11,89.11
4,Puerto,6,99.98,99.95,100.00



CONTROL DE LA LIMITACIÓN TEMPORAL DE VUELOS

✓ Registros anteriores al 01/07/2019: 4,751
✓ NaN Vuelos_total antes del 01/07/2019: 4,751
✓ NaN Vuelos_total desde el 01/07/2019: 0

CONTROL FINAL DEL BLOQUE ACÚSTICO
✓ LAeq_idw_1500m: 344 NaN — 99.21 % cobertura
✓ n_sensores_obs_1500m: 344 NaN — 99.21 % cobertura
✓ dist_sensor_min_1500m: 344 NaN — 99.21 % cobertura
✓ cobertura_media_sensores_1500m: 344 NaN — 99.21 % cobertura

CONTROL FINAL DEL BLOQUE MARÍTIMO
✓ Puerto_movimientos: 0 NaN — 100.00 % cobertura
✓ Puerto_eslora_media: 20 NaN — 99.95 % cobertura
✓ Puerto_calado_medio: 20 NaN — 99.95 % cobertura
✓ Puerto_ferry_pasajeros: 0 NaN — 100.00 % cobertura
✓ Puerto_portacontenedores: 0 NaN — 100.00 % cobertura
✓ Puerto_tanque: 0 NaN — 100.00 % cobertura

RESUMEN DE COBERTURA GLOBAL

✓ Variables con cobertura completa: 23
✓ Variables con cobertura parcial: 50
✓ Variables completamente vacías: 0

RESULTADO DE LA AUDITORÍA GLOBAL

✓ AUDITORÍA ESTRUCTURAL SUPERADA.
✓ No se detectan errores 

### Resultados — 8.1. Auditoría estructural y de cobertura del dataset maestro integrado

La auditoría global del **Checkpoint 06** confirma la integridad estructural del dataset maestro tras completar la integración de las diferentes fuentes de información consideradas en el estudio.

El dataset final contiene **43.642 registros y 73 variables**. No se detectan columnas duplicadas, fechas inválidas, columnas completamente vacías ni valores infinitos.

El periodo cubierto por el maestro se extiende desde el **12 de junio de 2018 hasta el 31 de diciembre de 2024**, con **2.249 fechas diferentes** y presencia de los siete años incluidos en el periodo de estudio.

La clave principal `fecha + estacion_fisica + contaminant` presenta **0 duplicados y 0 valores ausentes**, confirmándose la unicidad de los registros y la conservación de la estructura definida durante la construcción del dataset maestro.

La estructura espacial incluye **10 identificadores de estación física**, correspondientes a **8 localizaciones geográficas**. Esta diferencia responde a cambios históricos en los códigos e identificadores de determinadas estaciones y no supone una duplicación espacial de los puntos de medida.

El dataset contiene los cuatro contaminantes seleccionados para el estudio:

- `NO2`;
- `O3`;
- `PM10`;
- `PM2.5`.

La **contaminación atmosférica constituye el bloque base del dataset maestro** y determina su granularidad mediante la combinación `fecha + estacion_fisica + contaminant`. Sobre esta estructura se han incorporado sucesivamente las variables explicativas procedentes de las restantes fuentes.

De este modo, el dataset maestro integra seis componentes principales:

- **contaminación atmosférica**, correspondiente a las observaciones diarias de `NO2`, `O3`, `PM10` y `PM2.5`;
- **meteorología**, con 19 variables detectadas;
- **tráfico rodado**, con 25 variables;
- **contaminación acústica**, con 4 variables;
- **actividad aérea**, con 4 variables;
- **actividad marítima**, con 6 variables.

La auditoría de cobertura confirma que **23 variables presentan cobertura completa y 50 cobertura parcial**, sin existir ninguna variable completamente vacía.

Para los bloques explicativos integrados, la cobertura obtenida es la siguiente:

| Bloque | Nº variables | Cobertura media | Cobertura mínima | Cobertura máxima |
|---|---:|---:|---:|---:|
| Meteorología | 19 | 83,30 % | 63,56 % | 99,02 % |
| Tráfico rodado | 25 | 99,32 % | 98,99 % | 100,00 % |
| Contaminación acústica | 4 | 99,21 % | 99,21 % | 99,21 % |
| Actividad aérea | 4 | 89,11 % | 89,11 % | 89,11 % |
| Actividad marítima | 6 | 99,98 % | 99,95 % | 100,00 % |

Las diferencias de cobertura observadas son coherentes con las características y limitaciones previamente documentadas para cada fuente y no indican errores derivados del proceso de integración.

En el bloque de **actividad aérea**, los **4.751 registros sin información** corresponden exclusivamente al periodo anterior al **1 de julio de 2019**. A partir de esa fecha no se detectan valores ausentes en `Vuelos_total`, por lo que la ausencia de información queda correctamente delimitada temporalmente.

El bloque de **contaminación acústica** mantiene **344 valores ausentes** en sus variables integradas, equivalentes a una cobertura del **99,21 %**. Estos valores se conservan sin imputación y reflejan las limitaciones espaciales y temporales previamente identificadas en la fuente acústica.

El bloque de **actividad marítima** presenta una cobertura prácticamente completa. `Puerto_movimientos`, `Puerto_ferry_pasajeros`, `Puerto_portacontenedores` y `Puerto_tanque` alcanzan una cobertura del **100 %**, mientras que `Puerto_eslora_media` y `Puerto_calado_medio` conservan únicamente **20 valores ausentes cada una**, correspondientes al día sin actividad marítima identificado durante la integración. Estos valores se mantienen como `NaN` al no existir buques sobre los que calcular las correspondientes magnitudes medias.

El bloque de **tráfico rodado** presenta una cobertura muy elevada, con una media del **99,32 %** y valores comprendidos entre el **98,99 % y el 100 %**, confirmando la correcta incorporación de las variables derivadas de los aforos y de sus relaciones espaciales con las estaciones de calidad del aire.

La mayor heterogeneidad de cobertura se concentra en el bloque de **meteorología**, cuya cobertura media es del **83,30 %**. Las diferencias entre variables responden a la disponibilidad diferencial de las magnitudes registradas por las estaciones meteorológicas utilizadas y se mantienen explícitamente como valores ausentes para su posterior tratamiento durante la preparación del modelado.

La auditoría técnica finaliza sin detectar errores críticos de integración:

- **43.642 registros conservados**;
- **73 variables**;
- **0 duplicados en la clave principal**;
- **0 valores ausentes en la clave principal**;
- **0 columnas duplicadas**;
- **0 columnas completamente vacías**;
- **0 valores infinitos**;
- conservación de las limitaciones de cobertura previamente identificadas;
- ausencia de pérdida o multiplicación de registros durante las integraciones.

No se ha realizado ninguna imputación, eliminación de registros, tratamiento de valores extremos ni selección de variables para aprendizaje automático. Estas operaciones se reservan para las posteriores fases de análisis y modelado.

> **Conclusión:** la auditoría estructural del Checkpoint 06 se considera superada. El dataset maestro presenta una estructura consistente, trazable y temporalmente coherente, manteniendo la contaminación atmosférica como núcleo del análisis e integrando sobre ella información meteorológica, de tráfico rodado, contaminación acústica, actividad aérea y actividad marítima. Las limitaciones de cobertura permanecen explícitamente documentadas y el dataset queda preparado para la fase de visualización y posterior modelado.